In [ ]:
# === CELL 0: Install Step 5 dependencies (Moved to top) ===

!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu118
!pip install -q shapely geopandas anndata h5py tqdm

import torch
print(f"PyTorch {torch.__version__}, CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    print(f"  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


In [ ]:
# ==============================================================================
# COSMX CELL 1 — LOAD CORRECTED STEP 4 EXPORTS + VIEW FILES + VERIFY RAW COUNTS
# Replaces Xenium Step 5 Cell 1 for CosMx.
# ==============================================================================

import numpy as np
import pandas as pd
from scipy import sparse
import anndata as ad
import json, os, gc, warnings
warnings.filterwarnings("ignore")

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

# ------------------------------------------------------------------------------
# Path setup
# ------------------------------------------------------------------------------

# CosMx Step 4 export folder.
# This folder should contain:
#   molecules.parquet
#   cell_data.npz
#   denoised_adata.h5ad
#   was_corrected.npy
#   step4_config.json
STEP4_EXPORT_DIR = "/content/drive/MyDrive/diffusion/step4_cosmx/step4_exports"

# CosMx Step 5 / imputation output folder.
# All Step 5 model checkpoints, reports, imputed records, and completed molecule
# tables should be saved here.
CHECKPOINT_DIR = "/content/drive/MyDrive/diffusion/step4_cosmx/imputation"
IMPUTATION_DIR = CHECKPOINT_DIR
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# Backward-compatible alias used by later Step 5 cells.
EXPORT_DIR = STEP4_EXPORT_DIR

if not os.path.exists(STEP4_EXPORT_DIR):
    raise FileNotFoundError(
        f"CosMx Step 4 export folder not found:\n{STEP4_EXPORT_DIR}\n"
        "Check that your final CosMx cell-level denoising export cell saved files there."
    )

print("=" * 80)
print("COSMX STEP 5: Loading corrected Step 4 exports")
print("=" * 80)
print(f"STEP4_EXPORT_DIR: {STEP4_EXPORT_DIR}")
print(f"CHECKPOINT_DIR  : {CHECKPOINT_DIR}")
print(f"IMPUTATION_DIR  : {IMPUTATION_DIR}")

# ------------------------------------------------------------------------------
# Required Step 4 files
# ------------------------------------------------------------------------------

required_files = [
    "molecules.parquet",
    "cell_data.npz",
    "denoised_adata.h5ad",
    "was_corrected.npy",
    "step4_config.json",
]

file_paths = {}
missing = []

print("\nRequired CosMx Step 4 export files:")
for fname in required_files:
    fpath = os.path.join(STEP4_EXPORT_DIR, fname)
    file_paths[fname] = fpath

    if os.path.exists(fpath):
        print(f"  ✓ {fname:35s} ({os.path.getsize(fpath) / 1e6:.1f} MB)")
    else:
        print(f"  ✗ MISSING: {fname}")
        missing.append(fname)

if missing:
    raise FileNotFoundError(f"Missing required CosMx Step 4 export files: {missing}")

# ------------------------------------------------------------------------------
# Helper
# ------------------------------------------------------------------------------

def ensure_dense(X):
    """Convert sparse matrix to dense NumPy array if needed."""
    if sparse.issparse(X):
        return X.toarray()
    return np.asarray(X)

# ==============================================================================
# FILE 1: Load and view step4_config.json
# ==============================================================================

print("\n" + "=" * 80)
print("FILE 1: step4_config.json")
print("=" * 80)

with open(file_paths["step4_config.json"], "r") as f:
    step4_config = json.load(f)

print("Top-level keys in step4_config:")
print(list(step4_config.keys()))

print("\nStep 4 config preview:")
for key, value in step4_config.items():
    if isinstance(value, list):
        print(f"  {key}: list with {len(value):,} items")
        print(f"    first 10: {value[:10]}")
    elif isinstance(value, dict):
        print(f"  {key}: dictionary")
        for subkey, subval in value.items():
            print(f"    {subkey}: {subval}")
    else:
        print(f"  {key}: {value}")

if "shared_genes" not in step4_config:
    raise KeyError("step4_config.json is missing required key: 'shared_genes'")

shared_genes = [str(g) for g in step4_config["shared_genes"]]

# CosMx may use 'cell_type', 'Final_CosMx_Cell_Type', or compatibility aliases.
ct_column = step4_config.get("cell_type_column", "cell_type")
unique_cell_types = sorted([str(x) for x in step4_config.get("cell_types", [])])

print(f"\nShared genes from config: {len(shared_genes):,}")
print(f"Cell-type column from config: {ct_column}")
print(f"Cell types from config: {len(unique_cell_types):,}")

# ==============================================================================
# FILE 2: Load and view denoised_adata.h5ad
# ==============================================================================

print("\n" + "=" * 80)
print("FILE 2: denoised_adata.h5ad")
print("=" * 80)

denoised_adata = ad.read_h5ad(file_paths["denoised_adata.h5ad"])

print(f"denoised_adata shape: {denoised_adata.shape}")
print(f"Number of cells: {denoised_adata.n_obs:,}")
print(f"Number of genes: {denoised_adata.n_vars:,}")

print("\nFirst 10 cell IDs:")
print(list(denoised_adata.obs_names[:10]))

print("\nFirst 10 genes:")
print(list(denoised_adata.var_names[:10]))

print("\nobs columns:")
print(list(denoised_adata.obs.columns))

print("\nlayers:")
print(list(denoised_adata.layers.keys()))

print("\nobs preview:")
display(denoised_adata.obs.head())

print("\nvar preview:")
display(denoised_adata.var.head())

# ------------------------------------------------------------------------------
# CosMx-safe cell-type column selection
# ------------------------------------------------------------------------------

if ct_column not in denoised_adata.obs.columns:
    if "cell_type" in denoised_adata.obs.columns:
        ct_column = "cell_type"
    elif "Final_CosMx_Cell_Type" in denoised_adata.obs.columns:
        ct_column = "Final_CosMx_Cell_Type"
    elif "Assigned_Xenium_Cell_Type" in denoised_adata.obs.columns:
        ct_column = "Assigned_Xenium_Cell_Type"
    elif "Reference_Cell_Type" in denoised_adata.obs.columns:
        ct_column = "Reference_Cell_Type"
    else:
        raise KeyError(
            "No valid cell-type column found in denoised_adata.obs. "
            "Expected one of: cell_type, Final_CosMx_Cell_Type, "
            "Assigned_Xenium_Cell_Type, Reference_Cell_Type."
        )

# Add Xenium-compatible alias for later Step 5 cells that still expect this name.
if "Assigned_Xenium_Cell_Type" not in denoised_adata.obs.columns:
    denoised_adata.obs["Assigned_Xenium_Cell_Type"] = denoised_adata.obs[ct_column].astype(str)

# Add generic cell_type if missing.
if "cell_type" not in denoised_adata.obs.columns:
    denoised_adata.obs["cell_type"] = denoised_adata.obs[ct_column].astype(str)

cell_types_series = denoised_adata.obs[ct_column].astype(str)

if not unique_cell_types:
    unique_cell_types = sorted(cell_types_series.unique().tolist())

print(f"\nFinal cell-type column used: {ct_column}")
print("\nCell-type counts:")
display(
    cell_types_series
    .value_counts()
    .rename_axis("cell_type")
    .reset_index(name="n_cells")
)

ct_to_idx = {ct: i for i, ct in enumerate(unique_cell_types)}
cell_type_indices = cell_types_series.map(ct_to_idx).values

if pd.isna(cell_type_indices).any():
    missing_ct = sorted(set(cell_types_series[pd.isna(cell_type_indices)].astype(str)))
    raise ValueError(
        "Some cell types in denoised_adata are missing from unique_cell_types:\n"
        f"{missing_ct[:20]}"
    )

cell_type_indices = cell_type_indices.astype(np.int64)

# ------------------------------------------------------------------------------
# Matrix loading
# ------------------------------------------------------------------------------

X_denoised = ensure_dense(denoised_adata.X).astype(np.float32)

if "raw" not in denoised_adata.layers:
    raise KeyError("denoised_adata.layers['raw'] is missing.")

X_raw = ensure_dense(denoised_adata.layers["raw"]).astype(np.float32)

if "uncertainty" in denoised_adata.layers:
    uncertainty = ensure_dense(denoised_adata.layers["uncertainty"]).astype(np.float32)
else:
    uncertainty = None
    print("WARNING: uncertainty layer not found.")

# CosMx cell IDs are strings like '1_25'. Never convert them to int.
cell_ids_step4 = np.array(denoised_adata.obs_names.astype(str), dtype=str)

print("\nMatrix summaries:")
print(f"  X_denoised shape: {X_denoised.shape}")
print(f"  X_raw shape     : {X_raw.shape}")
print(f"  X_raw sum       : {X_raw.sum(dtype=np.float64):,.0f}")
print(f"  X_denoised sum  : {X_denoised.sum(dtype=np.float64):,.2f}")

if uncertainty is not None:
    print(f"  uncertainty shape: {uncertainty.shape}")
    print(
        "  uncertainty min/mean/max: "
        f"{uncertainty.min():.4f} / {uncertainty.mean():.4f} / {uncertainty.max():.4f}"
    )

print("\nSmall raw count preview: first 5 cells × first 5 genes")
display(
    pd.DataFrame(
        X_raw[:5, :5],
        index=denoised_adata.obs_names[:5],
        columns=denoised_adata.var_names[:5],
    )
)

print("\nSmall denoised count preview: first 5 cells × first 5 genes")
display(
    pd.DataFrame(
        X_denoised[:5, :5],
        index=denoised_adata.obs_names[:5],
        columns=denoised_adata.var_names[:5],
    )
)

# ==============================================================================
# FILE 3: Load and view was_corrected.npy
# ==============================================================================

print("\n" + "=" * 80)
print("FILE 3: was_corrected.npy")
print("=" * 80)

was_corrected = np.load(file_paths["was_corrected.npy"])

if was_corrected.shape != X_denoised.shape:
    raise ValueError(
        f"was_corrected shape {was_corrected.shape} does not match "
        f"X_denoised shape {X_denoised.shape}"
    )

was_corrected = was_corrected.astype(bool)

print(f"was_corrected shape: {was_corrected.shape}")
print(f"Total cell-gene pairs: {was_corrected.size:,}")
print(f"Corrected pairs: {was_corrected.sum():,}")
print(f"Correction rate: {100 * was_corrected.sum() / was_corrected.size:.2f}%")

corrected_per_cell = was_corrected.sum(axis=1)
corrected_per_gene = was_corrected.sum(axis=0)

print("\nCorrected genes per cell summary:")
display(pd.Series(corrected_per_cell).describe().to_frame("corrected_genes_per_cell"))

print("\nTop 20 genes by number of corrected cells:")
display(
    pd.DataFrame({
        "gene_id": denoised_adata.var_names.astype(str),
        "n_corrected_cells": corrected_per_gene,
    })
    .sort_values("n_corrected_cells", ascending=False)
    .head(20)
)

# ==============================================================================
# FILE 4: Load and view molecules.parquet
# ==============================================================================

print("\n" + "=" * 80)
print("FILE 4: molecules.parquet")
print("=" * 80)

molecules = pd.read_parquet(file_paths["molecules.parquet"])

print(f"Molecules loaded: {len(molecules):,} rows")
print(f"Columns: {list(molecules.columns)}")

# CosMx-safe dtype handling.
if "cell_id" not in molecules.columns:
    raise KeyError("molecules.parquet is missing required column: cell_id")
if "gene_id" not in molecules.columns:
    raise KeyError("molecules.parquet is missing required column: gene_id")

molecules["cell_id"] = molecules["cell_id"].astype(str)
molecules["gene_id"] = molecules["gene_id"].astype(str)

# Add Xenium-compatible cell-type alias if needed.
if "Assigned_Xenium_Cell_Type" not in molecules.columns:
    if "Final_CosMx_Cell_Type" in molecules.columns:
        molecules["Assigned_Xenium_Cell_Type"] = molecules["Final_CosMx_Cell_Type"].astype(str)
    elif "cell_type" in molecules.columns:
        molecules["Assigned_Xenium_Cell_Type"] = molecules["cell_type"].astype(str)
    elif ct_column in molecules.columns:
        molecules["Assigned_Xenium_Cell_Type"] = molecules[ct_column].astype(str)
    else:
        raise KeyError(
            "No usable cell-type column found in molecules.parquet. "
            "Expected Assigned_Xenium_Cell_Type, Final_CosMx_Cell_Type, or cell_type."
        )

# Add generic cell_type if missing.
if "cell_type" not in molecules.columns:
    molecules["cell_type"] = molecules["Assigned_Xenium_Cell_Type"].astype(str)

# CosMx should already have compatibility columns from Step 4 export.
required_mol_cols = [
    "transcript_id",
    "cell_id",
    "gene_id",
    "x",
    "y",
    "z",
    "quality",
    "overlaps_nucleus",
    "Assigned_Xenium_Cell_Type",
]

missing_cols = [c for c in required_mol_cols if c not in molecules.columns]
if missing_cols:
    raise KeyError(f"molecules.parquet is missing required columns: {missing_cols}")

print("\nMolecule table preview:")
display(molecules.head())

print("\nMolecule table dtypes:")
display(molecules.dtypes.to_frame("dtype"))

print("\nClean molecule table checks:")
print(f"  total molecules: {len(molecules):,}")
print(f"  unique cells: {molecules['cell_id'].nunique():,}")
print(f"  unique genes: {molecules['gene_id'].nunique():,}")

if "quality" in molecules.columns:
    print(f"  quality < 20 compatibility check: {(molecules['quality'] < 20).sum():,}")
    print("  Note: for CosMx, quality is a compatibility placeholder, not real Xenium QV.")

print(f"  missing cell type: {molecules['Assigned_Xenium_Cell_Type'].isna().sum():,}")
print(f"  missing coordinates: {molecules[['x', 'y', 'z']].isna().any(axis=1).sum():,}")

print("\nMolecule quality summary:")
display(molecules["quality"].describe().to_frame("quality"))

print("\nTop 20 genes by molecule count:")
display(
    molecules["gene_id"]
    .value_counts()
    .head(20)
    .rename_axis("gene_id")
    .reset_index(name="n_molecules")
)

print("\nTop 20 cell types by molecule count:")
display(
    molecules["Assigned_Xenium_Cell_Type"]
    .value_counts()
    .head(20)
    .rename_axis("cell_type")
    .reset_index(name="n_molecules")
)

# ==============================================================================
# FILE 5: Load and view cell_data.npz
# CosMx-safe version: supports summary geometry instead of Xenium polygon geometry
# ==============================================================================

print("\n" + "=" * 80)
print("FILE 5: cell_data.npz")
print("=" * 80)

cell_geom = np.load(file_paths["cell_data.npz"], allow_pickle=False)

print("Keys in cell_data.npz:")
print(list(cell_geom.keys()))

# ------------------------------------------------------------------------------
# Required CosMx geometry keys
# ------------------------------------------------------------------------------

required_cosmx_geom_keys = [
    "cell_ids",
    "centroids",
    "center_x_global_px",
    "center_y_global_px",
    "cell_area_px",
    "cell_area_um2",
    "label_cell_area_px",
    "label_nuclear_area_px",
    "label_membrane_area_px",
    "label_cytoplasm_area_px",
    "label_extracellular_area_px",
    "label_nuclear_frac",
    "label_membrane_frac",
    "label_cytoplasm_frac",
    "label_extracellular_frac",
    "fov",
    "cell_ID_original",
    "cell_type",
    "Final_CosMx_Cell_Type",
]

missing_cosmx_geom_keys = [
    k for k in required_cosmx_geom_keys
    if k not in cell_geom.keys()
]

if missing_cosmx_geom_keys:
    raise KeyError(
        "cell_data.npz is missing required CosMx geometry-summary keys:\n"
        f"{missing_cosmx_geom_keys}\n\n"
        "This loader expects the CosMx cell_data.npz exported from the corrected "
        "CosMx cell-level denoising file."
    )

# ------------------------------------------------------------------------------
# Load CosMx geometry arrays
# ------------------------------------------------------------------------------

cell_ids_geom = cell_geom["cell_ids"].astype(str)
centroids = cell_geom["centroids"].astype(np.float32)

center_x_global_px = cell_geom["center_x_global_px"].astype(np.float32)
center_y_global_px = cell_geom["center_y_global_px"].astype(np.float32)

center_x_um = cell_geom["center_x_um"].astype(np.float32) if "center_x_um" in cell_geom.keys() else None
center_y_um = cell_geom["center_y_um"].astype(np.float32) if "center_y_um" in cell_geom.keys() else None

cell_area_px = cell_geom["cell_area_px"].astype(np.float32)
cell_area_um2 = cell_geom["cell_area_um2"].astype(np.float32)

label_cell_area_px = cell_geom["label_cell_area_px"].astype(np.float32)
label_nuclear_area_px = cell_geom["label_nuclear_area_px"].astype(np.float32)
label_membrane_area_px = cell_geom["label_membrane_area_px"].astype(np.float32)
label_cytoplasm_area_px = cell_geom["label_cytoplasm_area_px"].astype(np.float32)
label_extracellular_area_px = cell_geom["label_extracellular_area_px"].astype(np.float32)

label_nuclear_frac = cell_geom["label_nuclear_frac"].astype(np.float32)
label_membrane_frac = cell_geom["label_membrane_frac"].astype(np.float32)
label_cytoplasm_frac = cell_geom["label_cytoplasm_frac"].astype(np.float32)
label_extracellular_frac = cell_geom["label_extracellular_frac"].astype(np.float32)

geom_fov = cell_geom["fov"].astype(str)
geom_cell_ID_original = cell_geom["cell_ID_original"].astype(str)
geom_cell_type = cell_geom["cell_type"].astype(str)
geom_final_cell_type = cell_geom["Final_CosMx_Cell_Type"].astype(str)

print(f"Number of geometry cells: {len(cell_ids_geom):,}")
print(f"centroids shape        : {centroids.shape}")
print(f"cell_area_px shape     : {cell_area_px.shape}")
print(f"cell_area_um2 shape    : {cell_area_um2.shape}")
print(f"nuclear area shape     : {label_nuclear_area_px.shape}")
print(f"nuclear fraction shape : {label_nuclear_frac.shape}")

print("\nFirst 10 geometry cell IDs:")
print(cell_ids_geom[:10].tolist())

print("\nGeometry preview:")
display(
    pd.DataFrame({
        "cell_id": cell_ids_geom[:10],
        "center_x_global_px": center_x_global_px[:10],
        "center_y_global_px": center_y_global_px[:10],
        "cell_area_px": cell_area_px[:10],
        "label_cell_area_px": label_cell_area_px[:10],
        "label_nuclear_area_px": label_nuclear_area_px[:10],
        "label_nuclear_frac": label_nuclear_frac[:10],
        "cell_type": geom_cell_type[:10],
    })
)

print("\nCosMx geometry summary:")
display(
    pd.DataFrame({
        "cell_area_px": cell_area_px,
        "cell_area_um2": cell_area_um2,
        "label_cell_area_px": label_cell_area_px,
        "label_nuclear_area_px": label_nuclear_area_px,
        "label_nuclear_frac": label_nuclear_frac,
        "label_cytoplasm_frac": label_cytoplasm_frac,
        "label_membrane_frac": label_membrane_frac,
    }).describe()
)

# ------------------------------------------------------------------------------
# Backward-compatible placeholders
# ------------------------------------------------------------------------------
# Xenium Step 5 uses polygon arrays:
#   cell_offsets, cell_vertices, nuc_offsets, nuc_vertices, nuc_present
#
# CosMx Step 4 currently exports centroid/area/compartment-summary geometry,
# not polygon vertices. Therefore, define these as None so later CosMx-specific
# cells can detect summary geometry and avoid using Xenium polygon logic.

cell_offsets = None
cell_vertices = None
nuc_offsets = None
nuc_vertices = None

# For CosMx, nucleus presence can be approximated by whether nuclear area > 0.
nuc_present = label_nuclear_area_px > 0

print("\nCosMx geometry mode:")
print("  geometry_mode: summary_area_centroid")
print("  polygon vertices available: False")
print("  nucleus present cells:", int(nuc_present.sum()), "/", len(nuc_present))

# ------------------------------------------------------------------------------
# Geometry lookup tables for later CosMx Step 5 cells
# ------------------------------------------------------------------------------

geom_id_to_idx = {
    str(cid): i
    for i, cid in enumerate(cell_ids_geom)
}

centroid_lookup = {
    str(cid): centroids[i].astype(np.float32)
    for i, cid in enumerate(cell_ids_geom)
}

cell_area_lookup_px = {
    str(cid): float(cell_area_px[i])
    for i, cid in enumerate(cell_ids_geom)
}

cell_area_lookup_um2 = {
    str(cid): float(cell_area_um2[i])
    for i, cid in enumerate(cell_ids_geom)
}

nuc_area_lookup_px = {
    str(cid): float(label_nuclear_area_px[i])
    for i, cid in enumerate(cell_ids_geom)
}

nuc_frac_lookup = {
    str(cid): float(label_nuclear_frac[i])
    for i, cid in enumerate(cell_ids_geom)
}

cyto_frac_lookup = {
    str(cid): float(label_cytoplasm_frac[i])
    for i, cid in enumerate(cell_ids_geom)
}

membrane_frac_lookup = {
    str(cid): float(label_membrane_frac[i])
    for i, cid in enumerate(cell_ids_geom)
}

# Approximate cell and nuclear radii from area.
# These are useful later because CosMx does not currently provide polygon vertices.
cell_radius_lookup_px = {
    str(cid): float(np.sqrt(max(label_cell_area_px[i], 1.0) / np.pi))
    for i, cid in enumerate(cell_ids_geom)
}

nuc_radius_lookup_px = {
    str(cid): float(np.sqrt(max(label_nuclear_area_px[i], 0.0) / np.pi))
    for i, cid in enumerate(cell_ids_geom)
}

print("\nBuilt geometry lookups:")
print(f"  geom_id_to_idx         : {len(geom_id_to_idx):,}")
print(f"  centroid_lookup        : {len(centroid_lookup):,}")
print(f"  cell_radius_lookup_px  : {len(cell_radius_lookup_px):,}")
print(f"  nuc_radius_lookup_px   : {len(nuc_radius_lookup_px):,}")

# ------------------------------------------------------------------------------
# Geometry coverage
# ------------------------------------------------------------------------------

print("\nGeometry coverage:")
step4_cell_set = set(cell_ids_step4.astype(str))
geom_cell_set = set(cell_ids_geom.astype(str))
mol_cell_set = set(molecules["cell_id"].astype(str).unique())

print(f"  Step4 cells: {len(step4_cell_set):,}")
print(f"  Geometry cells: {len(geom_cell_set):,}")
print(f"  Molecule cells: {len(mol_cell_set):,}")
print(f"  Step4 ∩ geometry: {len(step4_cell_set & geom_cell_set):,}")
print(f"  Step4 ∩ molecules: {len(step4_cell_set & mol_cell_set):,}")
print(f"  Step4 ∩ geometry ∩ molecules: {len(step4_cell_set & geom_cell_set & mol_cell_set):,}")

if len(step4_cell_set - geom_cell_set) > 0:
    print("\nWARNING: Some Step4 cells are missing geometry.")
    print("First 10 missing geometry cell IDs:")
    print(list(step4_cell_set - geom_cell_set)[:10])

if len(mol_cell_set - geom_cell_set) > 0:
    print("\nWARNING: Some molecule cells are missing geometry.")
    print("First 10 molecule cell IDs missing geometry:")
    print(list(mol_cell_set - geom_cell_set)[:10])

# Store a flag for later Step 5 cells.
GEOMETRY_MODE = "cosmx_summary_area_centroid"

print("\nPASS: CosMx cell_data.npz loaded using summary area/centroid geometry.")

# ==============================================================================
# Lookup dictionaries
# ==============================================================================

print("\n" + "=" * 80)
print("Building Step 5 lookup dictionaries")
print("=" * 80)

# Gene order for Step 5 should match denoised_adata.var_names.
adata_genes = list(denoised_adata.var_names.astype(str))

print("Checking gene order:")
print(f"  len(shared_genes)        : {len(shared_genes):,}")
print(f"  denoised_adata.n_vars    : {denoised_adata.n_vars:,}")
print(f"  shared_genes == var_names: {shared_genes == adata_genes}")

if shared_genes != adata_genes:
    print("\nWARNING: step4_config['shared_genes'] does not exactly match denoised_adata.var_names.")
    print("Using denoised_adata.var_names as the authoritative Step 5 gene order.")
    shared_genes = adata_genes

gene_to_col = {str(g): j for j, g in enumerate(shared_genes)}
gene_idx_map = gene_to_col

# CosMx cell IDs are strings. Do not use int().
step4_cell_to_row = {str(c): i for i, c in enumerate(cell_ids_step4)}
cell_idx_map = step4_cell_to_row

print(f"gene_to_col size: {len(gene_to_col):,}")
print(f"cell_idx_map size: {len(cell_idx_map):,}")

# ==============================================================================
# CHECK: Is X_raw a true molecule-count matrix?
# ==============================================================================

print("\n" + "=" * 80)
print("CHECK: Is X_raw a true molecule-count matrix?")
print("=" * 80)

count_source = molecules[
    molecules["cell_id"].isin(step4_cell_to_row.keys())
    & molecules["gene_id"].isin(gene_to_col.keys())
].copy()

# Step 4 molecule table should normally contain only observed molecules.
# If status exists because a later completed table was accidentally reused,
# restrict to observed molecules for raw-count reconstruction.
if "status" in count_source.columns:
    print("status column found in molecule table; using status == 'observed' for raw count check.")
    count_source = count_source[count_source["status"] == "observed"].copy()

counts = (
    count_source
    .groupby(["cell_id", "gene_id"])
    .size()
    .reset_index(name="count")
)

X_raw_counts = np.zeros(X_raw.shape, dtype=np.int32)

rows = counts["cell_id"].map(step4_cell_to_row)
cols = counts["gene_id"].map(gene_to_col)

ok = rows.notna() & cols.notna()

X_raw_counts[
    rows[ok].astype(int).to_numpy(),
    cols[ok].astype(int).to_numpy()
] = counts.loc[ok, "count"].to_numpy(dtype=np.int32)

raw_frac = np.abs(X_raw - np.round(X_raw))
diff = X_raw.astype(np.float64) - X_raw_counts.astype(np.float64)
abs_diff = np.abs(diff)

print(f"X_raw shape: {X_raw.shape}")
print(f"X_raw sum: {X_raw.sum(dtype=np.float64):,.0f}")
print(f"X_raw_counts sum from molecule table: {X_raw_counts.sum(dtype=np.float64):,.0f}")
print(f"Number of molecule rows used for raw-count rebuild: {len(count_source):,}")
print(f"Max fractional part in X_raw: {raw_frac.max():.6f}")
print(f"Non-integer X_raw entries: {(raw_frac > 1e-6).sum():,} / {X_raw.size:,}")
print(f"Max abs diff X_raw vs molecule counts: {abs_diff.max():.6f}")
print(f"Pairs with abs diff > 1e-6: {(abs_diff > 1e-6).sum():,}")
print(f"Pairs with abs diff > 0.5: {(abs_diff > 0.5).sum():,}")

RAW_COUNTS_MATCH_X_RAW = bool(abs_diff.max() < 1e-6)

if RAW_COUNTS_MATCH_X_RAW:
    print("\nVERDICT: X_raw matches direct molecule-table counts.")
else:
    print("\nVERDICT: X_raw does NOT match direct molecule-table counts.")
    print("For coherence, Step 5 will use X_raw_counts rebuilt from the molecule table.")
    print("Also updating denoised_adata.layers['raw'] to X_raw_counts for this session.")
    X_raw = X_raw_counts.astype(np.float32)
    denoised_adata.layers["raw"] = X_raw.copy()
    denoised_adata.layers["raw_molecule_counts_clean"] = X_raw.copy()

# Backward-compatible aliases used by later Step 5 cells.
X_observed_counts = X_raw_counts
X_raw_for_step5 = X_raw_counts

# ==============================================================================
# Step 5 target diagnostics
# ==============================================================================

print("\n" + "=" * 80)
print("Step 5 count-direction diagnostics")
print("=" * 80)

X_den_round = np.rint(X_denoised).astype(np.int64)
X_raw_int = X_raw_counts.astype(np.int64)

delta_counts = X_den_round - X_raw_int

n_need_impute = int((delta_counts > 0).sum())
n_need_prune = int((delta_counts < 0).sum())

n_added_molecules = int(np.maximum(delta_counts, 0).sum())
n_prune_molecules = int((-np.minimum(delta_counts, 0)).sum())

n_need_impute_corrected = int(((delta_counts > 0) & was_corrected).sum())
n_need_prune_corrected = int(((delta_counts < 0) & was_corrected).sum())

print(f"  Pairs where round(X_denoised) > X_raw_counts: {n_need_impute:,}")
print(f"  Pairs where round(X_denoised) < X_raw_counts: {n_need_prune:,}")
print(f"  Corrected pairs needing imputation: {n_need_impute_corrected:,}")
print(f"  Corrected pairs needing pruning: {n_need_prune_corrected:,}")
print(f"  Total molecules to impute: {n_added_molecules:,}")
print(f"  Total molecules to prune : {n_prune_molecules:,}")

if n_need_prune > 0:
    print("\nWARNING: Some rounded denoised counts are below raw counts.")
    print("If your CosMx Step 4 denoising was selective dropout correction, this should usually be 0.")
    print("If this was a global smoothing ablation, negative deltas may be expected.")
else:
    print("\nPASS: No pruning needed based on rounded denoised counts.")

print("\n" + "=" * 80)
print("CosMx Step 4 inputs loaded and verified. Ready for Step 5 imputation.")
print("=" * 80)

# Keep memory manageable.
gc.collect()

In [ ]:
# ============================================================
# VIEW X_raw_counts MATRIX — CosMx-safe
# ============================================================

import pandas as pd
import numpy as np
from scipy import sparse

print("=" * 70)
print("VIEW X_raw_counts")
print("=" * 70)

# Safety checks
assert "X_raw_counts" in globals(), "X_raw_counts is missing. Run Cell 1 first."
assert "denoised_adata" in globals(), "denoised_adata is missing. Run Cell 1 first."

# Convert to dense array if needed
if sparse.issparse(X_raw_counts):
    X_raw_counts_view = X_raw_counts.toarray()
else:
    X_raw_counts_view = np.asarray(X_raw_counts)

if X_raw_counts_view.shape != denoised_adata.shape:
    raise ValueError(
        f"Shape mismatch: X_raw_counts {X_raw_counts_view.shape} "
        f"vs denoised_adata {denoised_adata.shape}"
    )

# Convert a small part of X_raw_counts into a readable dataframe
X_raw_counts_preview = pd.DataFrame(
    X_raw_counts_view[:10, :10],
    index=denoised_adata.obs_names[:10],
    columns=denoised_adata.var_names[:10]
)

print("First 10 cells × first 10 genes from X_raw_counts:")
display(X_raw_counts_preview)

print("\nMatrix summary:")
print(f"Shape: {X_raw_counts_view.shape}")
print(f"Total molecule counts: {X_raw_counts_view.sum(dtype=np.float64):,.0f}")
print(f"Minimum value: {X_raw_counts_view.min()}")
print(f"Maximum value: {X_raw_counts_view.max()}")
print(f"Nonzero entries: {np.count_nonzero(X_raw_counts_view):,}")
print(f"Sparsity: {100 * (1 - np.count_nonzero(X_raw_counts_view) / X_raw_counts_view.size):.2f}%")

# Per-cell total counts
cell_total_counts = X_raw_counts_view.sum(axis=1)

print("\nPer-cell total count summary:")
display(pd.Series(cell_total_counts).describe().to_frame("raw_counts_per_cell"))

# Per-gene total counts
gene_total_counts = X_raw_counts_view.sum(axis=0)

gene_count_df = pd.DataFrame({
    "gene_id": denoised_adata.var_names.astype(str),
    "total_raw_counts": gene_total_counts
}).sort_values("total_raw_counts", ascending=False)

print("\nTop 20 genes by total raw molecule count:")
display(gene_count_df.head(20))

In [ ]:
'''# ==============================================================================
# FIX AND SAVE denoised_adata.h5ad WITH CORRECT RAW GENE ORDER
# ==============================================================================

import os
import shutil
import numpy as np
import pandas as pd
import anndata as ad

print("=" * 70)
print("FIXING RAW LAYERS USING MOLECULE-DERIVED X_raw_counts")
print("=" * 70)

# ------------------------------------------------------------------
# 1. Sanity checks
# ------------------------------------------------------------------

assert "denoised_adata" in globals(), "denoised_adata is missing."
assert "X_raw_counts" in globals(), "X_raw_counts is missing."
assert "molecules" in globals(), "molecules dataframe is missing."
assert "STEP4_EXPORT_DIR" in globals(), "STEP4_EXPORT_DIR is missing."

if X_raw_counts.shape != denoised_adata.shape:
    raise ValueError(
        f"Shape mismatch: X_raw_counts {X_raw_counts.shape} "
        f"vs denoised_adata {denoised_adata.shape}"
    )

print(f"denoised_adata shape: {denoised_adata.shape}")
print(f"X_raw_counts shape  : {X_raw_counts.shape}")
print(f"X_raw_counts sum    : {X_raw_counts.sum(dtype=np.float64):,.0f}")

# ------------------------------------------------------------------
# 2. Rebuild one more molecule-count matrix for strict verification
# ------------------------------------------------------------------

gene_to_col_check = {g: j for j, g in enumerate(denoised_adata.var_names.astype(str))}
cell_to_row_check = {int(c): i for i, c in enumerate(denoised_adata.obs_names.astype(str))}

counts_check = (
    molecules[
        molecules["cell_id"].isin(cell_to_row_check.keys())
        & molecules["gene_id"].isin(gene_to_col_check.keys())
    ]
    .groupby(["cell_id", "gene_id"])
    .size()
    .reset_index(name="count")
)

X_check = np.zeros(denoised_adata.shape, dtype=np.int32)

rows = counts_check["cell_id"].map(cell_to_row_check)
cols = counts_check["gene_id"].map(gene_to_col_check)
ok = rows.notna() & cols.notna()

X_check[
    rows[ok].astype(int).to_numpy(),
    cols[ok].astype(int).to_numpy()
] = counts_check.loc[ok, "count"].to_numpy(dtype=np.int32)

max_diff_check = np.abs(X_check - X_raw_counts).max()

print(f"Verification max diff between X_check and X_raw_counts: {max_diff_check}")

if max_diff_check != 0:
    raise ValueError("X_raw_counts does not match molecule-table reconstruction. Do not save.")

# ------------------------------------------------------------------
# 3. Overwrite raw layers with corrected gene-order matrix
# ------------------------------------------------------------------

X_raw_fixed = X_raw_counts.astype(np.float32)

denoised_adata.layers["raw"] = X_raw_fixed.copy()
denoised_adata.layers["raw_molecule_counts_clean"] = X_raw_fixed.copy()

# Optional: keep old official raw layer as-is if present.
# Do not use raw_official_10x for Step 5 unless separately verified.
print("Updated:")
print("  denoised_adata.layers['raw']")
print("  denoised_adata.layers['raw_molecule_counts_clean']")

# ------------------------------------------------------------------
# 4. Verify fixed raw layer now matches X_raw_counts
# ------------------------------------------------------------------

fixed_raw = np.asarray(denoised_adata.layers["raw"])
abs_diff_fixed = np.abs(fixed_raw.astype(np.float64) - X_raw_counts.astype(np.float64))

print("\nPost-fix verification:")
print(f"  fixed raw sum: {fixed_raw.sum(dtype=np.float64):,.0f}")
print(f"  X_raw_counts sum: {X_raw_counts.sum(dtype=np.float64):,.0f}")
print(f"  max abs diff: {abs_diff_fixed.max():.6f}")
print(f"  mismatched pairs: {(abs_diff_fixed > 1e-6).sum():,}")

if abs_diff_fixed.max() > 1e-6:
    raise ValueError("Fix failed: raw layer still does not match X_raw_counts.")

# ------------------------------------------------------------------
# 5. Backup old h5ad and save corrected h5ad
# ------------------------------------------------------------------

old_h5ad_path = os.path.join(STEP4_EXPORT_DIR, "denoised_adata.h5ad")
backup_h5ad_path = os.path.join(STEP4_EXPORT_DIR, "denoised_adata_BEFORE_RAW_FIX.h5ad")
fixed_h5ad_path = os.path.join(STEP4_EXPORT_DIR, "denoised_adata.h5ad")

if os.path.exists(old_h5ad_path) and not os.path.exists(backup_h5ad_path):
    print("\nCreating backup of old denoised_adata.h5ad...")
    shutil.copy2(old_h5ad_path, backup_h5ad_path)
    print(f"Backup saved to: {backup_h5ad_path}")
elif os.path.exists(backup_h5ad_path):
    print("\nBackup already exists, not overwriting it:")
    print(backup_h5ad_path)

print("\nSaving corrected denoised_adata.h5ad...")
denoised_adata.write_h5ad(fixed_h5ad_path)

print("\nSaved fixed h5ad:")
print(fixed_h5ad_path)

# ------------------------------------------------------------------
# 6. Reload fixed file and verify from disk
# ------------------------------------------------------------------

print("\nReloading fixed h5ad from disk for final verification...")
adata_test = ad.read_h5ad(fixed_h5ad_path)

raw_test = np.asarray(adata_test.layers["raw"])
abs_diff_disk = np.abs(raw_test.astype(np.float64) - X_raw_counts.astype(np.float64))

print("\nDisk verification:")
print(f"  raw layer sum: {raw_test.sum(dtype=np.float64):,.0f}")
print(f"  X_raw_counts sum: {X_raw_counts.sum(dtype=np.float64):,.0f}")
print(f"  max abs diff: {abs_diff_disk.max():.6f}")
print(f"  mismatched pairs: {(abs_diff_disk > 1e-6).sum():,}")

if abs_diff_disk.max() < 1e-6:
    print("\nSUCCESS: Future Step 5 loads should no longer show raw-layer mismatch.")
else:
    raise ValueError("Disk verification failed: saved h5ad still mismatches.")'''

In [ ]:
# ==============================================================================
# COSMX CELL 4: Build summary-geometry lookup
# Replaces Xenium polygon-based geometry lookup.
# ==============================================================================

import numpy as np
import pandas as pd
import time

print("=" * 70)
print("SUBSTEP 5A-prep: Building CosMx summary-geometry lookup")
print("=" * 70)

t0 = time.time()

# ------------------------------------------------------------------------------
# Required objects from corrected Cell 1
# ------------------------------------------------------------------------------

required_objects = [
    "cell_ids_geom",
    "centroids",
    "cell_ids_step4",
    "molecules",
    "cell_radius_lookup_px",
    "nuc_radius_lookup_px",
    "cell_area_lookup_px",
    "nuc_area_lookup_px",
    "nuc_frac_lookup",
]

missing = [obj for obj in required_objects if obj not in globals()]
if missing:
    raise NameError(
        "Missing required objects from Cell 1:\n"
        + "\n".join(missing)
        + "\n\nRun corrected CosMx Cell 1 first."
    )

# ------------------------------------------------------------------------------
# CosMx cell IDs must stay as strings
# ------------------------------------------------------------------------------

cell_ids_geom = np.array(cell_ids_geom).astype(str)
cell_ids_step4 = np.array(cell_ids_step4).astype(str)

molecules["cell_id"] = molecules["cell_id"].astype(str)
molecules["gene_id"] = molecules["gene_id"].astype(str)

# Build mapping: cell_id -> index in geometry arrays
geom_id_to_idx = {
    str(cid): i
    for i, cid in enumerate(cell_ids_geom)
}

# ------------------------------------------------------------------------------
# CosMx does not have polygon vertices in current export
# ------------------------------------------------------------------------------

cell_polygons = {}
nuc_polygons = {}

print("CosMx geometry mode:")
print("  Using centroid + area-derived radius geometry.")
print("  Exact Shapely cell/nucleus polygons are not available.")
print("  cell_polygons and nuc_polygons are kept as empty dictionaries for compatibility.")

# ------------------------------------------------------------------------------
# Cell and nucleus area dictionaries
# ------------------------------------------------------------------------------

cell_areas = {
    str(cid): float(cell_area_lookup_px[str(cid)])
    for cid in cell_ids_geom
    if str(cid) in cell_area_lookup_px
}

nuc_areas = {
    str(cid): float(nuc_area_lookup_px[str(cid)])
    for cid in cell_ids_geom
    if str(cid) in nuc_area_lookup_px
}

# ------------------------------------------------------------------------------
# Centroid lookup
# ------------------------------------------------------------------------------

centroid_lookup = {
    str(cid): np.asarray(centroids[i], dtype=np.float32)
    for i, cid in enumerate(cell_ids_geom)
}

# ------------------------------------------------------------------------------
# Estimate nucleus centroid
# ------------------------------------------------------------------------------
# Xenium has nucleus polygons, so nucleus centroid can be computed directly.
# CosMx summary export does not have nucleus polygons.
#
# Better CosMx approximation:
#   If a cell has observed nuclear molecules, use the mean x/y of nuclear molecules.
#   Otherwise, fall back to the cell centroid.
# ------------------------------------------------------------------------------

print("\nEstimating nucleus centroids from observed nuclear molecules...")

mol_nuc = molecules[molecules["overlaps_nucleus"].astype(np.int8) == 1]

if len(mol_nuc) > 0:
    nuc_xy_df = (
        mol_nuc
        .groupby("cell_id")[["x", "y"]]
        .mean()
        .astype(np.float32)
    )
else:
    nuc_xy_df = pd.DataFrame(columns=["x", "y"])

nuc_centroids = {}

for cid in cell_ids_geom:
    cid = str(cid)

    if cid in nuc_xy_df.index:
        nuc_centroids[cid] = nuc_xy_df.loc[cid, ["x", "y"]].to_numpy(dtype=np.float32)
    else:
        nuc_centroids[cid] = centroid_lookup[cid]

# ------------------------------------------------------------------------------
# Valid-cell set
# ------------------------------------------------------------------------------

valid_cell_ids = (
    set(cell_ids_step4.astype(str))
    & set(cell_ids_geom.astype(str))
    & set(molecules["cell_id"].astype(str).unique())
)

print("\nGeometry lookup summary:")
print(f"  Geometry cells             : {len(cell_ids_geom):,}")
print(f"  Step4 cells                : {len(cell_ids_step4):,}")
print(f"  Molecule cells             : {molecules['cell_id'].nunique():,}")
print(f"  Valid cells for Step 5     : {len(valid_cell_ids):,}")
print(f"  Cell area lookup           : {len(cell_areas):,}")
print(f"  Nucleus area lookup        : {len(nuc_areas):,}")
print(f"  Cell centroid lookup       : {len(centroid_lookup):,}")
print(f"  Nucleus centroid lookup    : {len(nuc_centroids):,}")
print(f"  Cells with nuclear-molecule centroid: {len(nuc_xy_df):,}")

# ------------------------------------------------------------------------------
# Sanity checks
# ------------------------------------------------------------------------------

if len(valid_cell_ids) == 0:
    raise ValueError("No valid cells found across Step4, geometry, and molecules.")

missing_centroid = [cid for cid in list(valid_cell_ids)[:1000] if cid not in centroid_lookup]
if missing_centroid:
    raise ValueError(f"Some valid cells are missing centroids. Example: {missing_centroid[:10]}")

GEOMETRY_MODE = "cosmx_summary_area_centroid"

elapsed = time.time() - t0

print("\nPASS: Built CosMx summary-geometry lookup.")
print(f"Completed in {elapsed:.1f}s")
print("=" * 70)

In [ ]:
# ==============================================================================
# COSMX CELL 6: Cell-relative coordinate normalization — direct non-chunked version
# Replaces Xenium polygon-based coordinate normalization.
# ==============================================================================

import numpy as np
import pandas as pd
import time
import gc

print("=" * 70)
print("SUBSTEP 5A: CosMx cell-relative coordinate normalization")
print("Direct full-table vectorized version — no chunks")
print("=" * 70)

t0 = time.time()

# ------------------------------------------------------------------------------
# Required objects
# ------------------------------------------------------------------------------

required_objects = [
    "molecules",
    "valid_cell_ids",
    "cell_ids_step4",
    "centroid_lookup",
    "nuc_centroids",
    "cell_radius_lookup_px",
    "nuc_radius_lookup_px",
]

missing = [obj for obj in required_objects if obj not in globals()]
if missing:
    raise NameError(
        "Missing required objects from Cell 1 / Cell 4:\n"
        + "\n".join(missing)
        + "\n\nRun corrected CosMx Cell 1 and CosMx Cell 4 first."
    )

# ------------------------------------------------------------------------------
# Keep only molecules from valid cells
# ------------------------------------------------------------------------------

molecules["cell_id"] = molecules["cell_id"].astype(str)
molecules["gene_id"] = molecules["gene_id"].astype(str)

valid_cell_ids = set(str(c) for c in valid_cell_ids)

mol_mask = molecules["cell_id"].isin(valid_cell_ids)
mol = molecules[mol_mask].copy()

print(f"Molecules in valid cells: {len(mol):,} / {len(molecules):,}")
print(f"Unique valid molecule cells: {mol['cell_id'].nunique():,}")

if len(mol) == 0:
    raise ValueError("No molecules found in valid cells.")

n_mol = len(mol)

# ------------------------------------------------------------------------------
# Build scalar lookup dictionaries for direct mapping
# ------------------------------------------------------------------------------

nuc_x_lookup = {str(cid): float(v[0]) for cid, v in nuc_centroids.items()}
nuc_y_lookup = {str(cid): float(v[1]) for cid, v in nuc_centroids.items()}

cell_center_x_lookup = {str(cid): float(v[0]) for cid, v in centroid_lookup.items()}
cell_center_y_lookup = {str(cid): float(v[1]) for cid, v in centroid_lookup.items()}

cell_radius_lookup_safe = {
    str(cid): max(float(r), 1.0)
    for cid, r in cell_radius_lookup_px.items()
}

nuc_radius_lookup_safe = {
    str(cid): max(float(r), 0.0)
    for cid, r in nuc_radius_lookup_px.items()
}

# ------------------------------------------------------------------------------
# Pre-compute per-cell z range
# ------------------------------------------------------------------------------

print("\nComputing per-cell z ranges...")

cell_z_range = mol.groupby("cell_id")["z"].agg(["min", "max"])
z_min_lookup = cell_z_range["min"].astype(np.float32).to_dict()
z_max_lookup = cell_z_range["max"].astype(np.float32).to_dict()

# ------------------------------------------------------------------------------
# Direct full-table vectorized coordinate normalization
# ------------------------------------------------------------------------------

print("\nNormalizing molecule coordinates using CosMx summary geometry...")
print("Geometry approximation:")
print("  r_norm = distance from estimated nucleus/cell center divided by approximate cell radius")
print("  theta  = angle around estimated nucleus/cell center")
print("  z_rel  = z normalized within each cell")
print("  p_nuclear = CosMx observed nuclear compartment flag")

cids = mol["cell_id"].astype(str)

x_m = mol["x"].to_numpy(dtype=np.float32)
y_m = mol["y"].to_numpy(dtype=np.float32)
z_m = mol["z"].to_numpy(dtype=np.float32)

# Estimated nucleus centroid.
# If nuclear molecules exist for a cell, this came from their mean x/y.
# Otherwise it falls back to cell centroid.
nc_x = cids.map(nuc_x_lookup).to_numpy(dtype=np.float32)
nc_y = cids.map(nuc_y_lookup).to_numpy(dtype=np.float32)

# Fallback to cell centroid if nucleus centroid mapping is missing.
missing_nc = np.isnan(nc_x) | np.isnan(nc_y)

if missing_nc.any():
    print(f"Missing nucleus centroid mapping for {missing_nc.sum():,} molecules; using cell centroid fallback.")

    cc_x = cids.map(cell_center_x_lookup).to_numpy(dtype=np.float32)
    cc_y = cids.map(cell_center_y_lookup).to_numpy(dtype=np.float32)

    nc_x[missing_nc] = cc_x[missing_nc]
    nc_y[missing_nc] = cc_y[missing_nc]

cell_radius = cids.map(cell_radius_lookup_safe).to_numpy(dtype=np.float32)
cell_radius = np.maximum(cell_radius, 1.0)

# Radial distance and angle
dx = x_m - nc_x
dy = y_m - nc_y

d_center = np.sqrt(dx * dx + dy * dy).astype(np.float32)

# CosMx summary approximation:
# cell is approximated as a circle, so normalized radial position is:
# distance from estimated center / area-derived cell radius.
r_norm = d_center / (cell_radius + 1e-8)
r_norm = np.clip(r_norm, 0.0, 1.0).astype(np.float32)

theta = np.arctan2(dy, dx).astype(np.float32)

# z-relative per cell
z_min = cids.map(z_min_lookup).to_numpy(dtype=np.float32)
z_max = cids.map(z_max_lookup).to_numpy(dtype=np.float32)
z_range = z_max - z_min

z_rel = np.full(n_mol, 0.5, dtype=np.float32)

good_z = z_range > 1e-8
z_rel[good_z] = (z_m[good_z] - z_min[good_z]) / z_range[good_z]
z_rel = np.clip(z_rel, 0.0, 1.0).astype(np.float32)

# Nuclear probability
# For CosMx, the observed compartment label is the most reliable available target.
p_nuclear = mol["overlaps_nucleus"].to_numpy(dtype=np.float32)
p_nuclear = np.clip(p_nuclear, 0.0, 1.0).astype(np.float32)

# ------------------------------------------------------------------------------
# Assign normalized coordinates to molecule table
# ------------------------------------------------------------------------------

mol["r_norm"] = r_norm
mol["theta"] = theta
mol["z_rel"] = z_rel
mol["p_nuclear"] = p_nuclear

mol["status"] = "observed"
mol["weight"] = 1.0
mol["is_imputed"] = False

elapsed = time.time() - t0

print(f"\nCoordinate normalization complete in {elapsed / 60:.1f} min")

print("\nNormalized coordinate ranges:")
print(f"  r_norm range: [{mol['r_norm'].min():.3f}, {mol['r_norm'].max():.3f}]")
print(f"  theta range : [{mol['theta'].min():.3f}, {mol['theta'].max():.3f}]")
print(f"  z_rel range : [{mol['z_rel'].min():.3f}, {mol['z_rel'].max():.3f}]")
print(
    f"  p_nuclear > 0.5: {(mol['p_nuclear'] > 0.5).sum():,} / {n_mol:,} "
    f"({100 * (mol['p_nuclear'] > 0.5).sum() / n_mol:.1f}%)"
)

print("\nPreview of normalized molecule table:")
display(
    mol[
        [
            "transcript_id",
            "cell_id",
            "gene_id",
            "x",
            "y",
            "z",
            "r_norm",
            "theta",
            "z_rel",
            "p_nuclear",
            "status",
            "is_imputed",
        ]
    ].head()
)

print("\nSummary:")
display(
    mol[["r_norm", "theta", "z_rel", "p_nuclear"]].describe()
)

# ------------------------------------------------------------------------------
# Cleanup temporary arrays to reduce memory pressure after assignment
# ------------------------------------------------------------------------------

del (
    cids,
    x_m,
    y_m,
    z_m,
    nc_x,
    nc_y,
    cell_radius,
    dx,
    dy,
    d_center,
    r_norm,
    theta,
    z_min,
    z_max,
    z_range,
    z_rel,
    p_nuclear,
)

gc.collect()

print("\nPASS: CosMx coordinate normalization completed.")
print("=" * 70)

In [ ]:
# ==============================================================================
# COSMX CELL 7 — CHECKPOINT AFTER COORDINATE NORMALIZATION
# Saves mol and CosMx summary-geometry objects after coordinate normalization.
# ==============================================================================

import os
import pickle
import json
import gc
import numpy as np
import pandas as pd

print("=" * 80)
print("COSMX CHECKPOINT A: Save molecule table after coordinate normalization")
print("=" * 80)

# ------------------------------------------------------------------------------
# Use CosMx imputation folder
# ------------------------------------------------------------------------------

# Prefer CHECKPOINT_DIR from Cell 1.
# If it does not exist for some reason, define the correct CosMx path.
if "CHECKPOINT_DIR" not in globals():
    CHECKPOINT_DIR = "/content/drive/MyDrive/diffusion/step4_cosmx/imputation"

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

print(f"CHECKPOINT_DIR: {CHECKPOINT_DIR}")

# ------------------------------------------------------------------------------
# Required object checks
# ------------------------------------------------------------------------------

required_objects = [
    "mol",
    "GEOMETRY_MODE",
    "valid_cell_ids",
    "centroid_lookup",
    "nuc_centroids",
    "cell_radius_lookup_px",
    "nuc_radius_lookup_px",
    "cell_area_lookup_px",
    "nuc_area_lookup_px",
    "nuc_frac_lookup",
]

missing = [obj for obj in required_objects if obj not in globals()]

if missing:
    raise NameError(
        "Missing required objects before checkpointing:\n"
        + "\n".join(missing)
        + "\n\nRun corrected CosMx Cell 1, Cell 4, and Cell 6 first."
    )

required_mol_cols = [
    "transcript_id",
    "cell_id",
    "gene_id",
    "x",
    "y",
    "z",
    "r_norm",
    "theta",
    "z_rel",
    "p_nuclear",
    "status",
    "weight",
    "is_imputed",
]

missing_mol_cols = [c for c in required_mol_cols if c not in mol.columns]

if missing_mol_cols:
    raise KeyError(
        "mol is missing normalized-coordinate columns:\n"
        + "\n".join(missing_mol_cols)
        + "\n\nRun corrected CosMx Cell 6 first."
    )

# ------------------------------------------------------------------------------
# Save processed molecule table
# ------------------------------------------------------------------------------

mol_path = os.path.join(CHECKPOINT_DIR, "checkpoint_mol_after_5A.parquet")

mol.to_parquet(
    mol_path,
    index=False,
    compression="snappy"
)

print("\nSaved normalized molecule table:")
print(f"  path  : {mol_path}")
print(f"  rows  : {len(mol):,}")
print(f"  cells : {mol['cell_id'].nunique():,}")
print(f"  genes : {mol['gene_id'].nunique():,}")
print(f"  memory: {mol.memory_usage(deep=True).sum() / 1e9:.2f} GB")

# ------------------------------------------------------------------------------
# Save CosMx summary geometry objects
# ------------------------------------------------------------------------------

geometry_checkpoint = {
    "GEOMETRY_MODE": GEOMETRY_MODE,
    "valid_cell_ids": sorted([str(c) for c in valid_cell_ids]),

    # Main CosMx geometry lookups
    "centroid_lookup": centroid_lookup,
    "nuc_centroids": nuc_centroids,
    "cell_radius_lookup_px": cell_radius_lookup_px,
    "nuc_radius_lookup_px": nuc_radius_lookup_px,
    "cell_area_lookup_px": cell_area_lookup_px,
    "nuc_area_lookup_px": nuc_area_lookup_px,
    "nuc_frac_lookup": nuc_frac_lookup,

    # Optional lookups if available
    "cell_area_lookup_um2": globals().get("cell_area_lookup_um2", None),
    "cyto_frac_lookup": globals().get("cyto_frac_lookup", None),
    "membrane_frac_lookup": globals().get("membrane_frac_lookup", None),

    # Compatibility objects
    # These are empty for CosMx because exact polygon vertices are not available.
    "cell_polygons": globals().get("cell_polygons", {}),
    "nuc_polygons": globals().get("nuc_polygons", {}),
    "cell_areas": globals().get("cell_areas", {}),
    "nuc_areas": globals().get("nuc_areas", {}),
}

geom_path = os.path.join(CHECKPOINT_DIR, "checkpoint_cosmx_geometry_after_5A.pkl")

with open(geom_path, "wb") as f:
    pickle.dump(geometry_checkpoint, f, protocol=pickle.HIGHEST_PROTOCOL)

print("\nSaved CosMx geometry checkpoint:")
print(f"  path: {geom_path}")
print(f"  geometry mode: {GEOMETRY_MODE}")
print(f"  valid cells: {len(geometry_checkpoint['valid_cell_ids']):,}")
print(f"  centroid lookup: {len(centroid_lookup):,}")
print(f"  nucleus centroid lookup: {len(nuc_centroids):,}")
print(f"  cell radius lookup: {len(cell_radius_lookup_px):,}")
print(f"  nucleus radius lookup: {len(nuc_radius_lookup_px):,}")

# ------------------------------------------------------------------------------
# Save individual compatibility pickle files too
# ------------------------------------------------------------------------------
# Some later Xenium-style cells may expect these filenames.
# For CosMx, polygon dictionaries are empty, but saving them avoids file-not-found
# errors in recovery code. Later CosMx-specific cells should use the summary
# geometry checkpoint instead.

objects_to_save = {
    "cell_polygons.pkl": globals().get("cell_polygons", {}),
    "nuc_polygons.pkl": globals().get("nuc_polygons", {}),
    "cell_areas.pkl": globals().get("cell_areas", {}),
    "nuc_areas.pkl": globals().get("nuc_areas", {}),
    "nuc_centroids.pkl": nuc_centroids,
    "centroid_lookup.pkl": centroid_lookup,
    "cell_radius_lookup_px.pkl": cell_radius_lookup_px,
    "nuc_radius_lookup_px.pkl": nuc_radius_lookup_px,
}

print("\nSaving compatibility lookup files:")

for fname, obj in objects_to_save.items():
    fpath = os.path.join(CHECKPOINT_DIR, fname)
    with open(fpath, "wb") as f:
        pickle.dump(obj, f, protocol=pickle.HIGHEST_PROTOCOL)
    print(f"  Saved: {fpath}")

# ------------------------------------------------------------------------------
# Save a small JSON summary
# ------------------------------------------------------------------------------

summary = {
    "checkpoint_name": "checkpoint_after_5A_coordinate_normalization",
    "platform": "CosMx",
    "geometry_mode": GEOMETRY_MODE,
    "checkpoint_dir": CHECKPOINT_DIR,
    "mol_path": mol_path,
    "geometry_checkpoint_path": geom_path,
    "n_molecules": int(len(mol)),
    "n_cells": int(mol["cell_id"].nunique()),
    "n_genes": int(mol["gene_id"].nunique()),
    "r_norm_min": float(mol["r_norm"].min()),
    "r_norm_max": float(mol["r_norm"].max()),
    "r_norm_mean": float(mol["r_norm"].mean()),
    "theta_min": float(mol["theta"].min()),
    "theta_max": float(mol["theta"].max()),
    "z_rel_min": float(mol["z_rel"].min()),
    "z_rel_max": float(mol["z_rel"].max()),
    "z_rel_mean": float(mol["z_rel"].mean()),
    "p_nuclear_mean": float(mol["p_nuclear"].mean()),
    "p_nuclear_positive": int((mol["p_nuclear"] > 0.5).sum()),
}

summary_path = os.path.join(CHECKPOINT_DIR, "checkpoint_after_5A_summary.json")

with open(summary_path, "w") as f:
    json.dump(summary, f, indent=2)

print("\nSaved checkpoint summary:")
print(f"  path: {summary_path}")

# ------------------------------------------------------------------------------
# Final verification
# ------------------------------------------------------------------------------

print("\nCheckpoint verification:")
print(f"  molecule checkpoint exists: {os.path.exists(mol_path)}")
print(f"  geometry checkpoint exists: {os.path.exists(geom_path)}")
print(f"  summary exists: {os.path.exists(summary_path)}")

if not os.path.exists(mol_path):
    raise FileNotFoundError("Molecule checkpoint was not saved.")
if not os.path.exists(geom_path):
    raise FileNotFoundError("Geometry checkpoint was not saved.")
if not os.path.exists(summary_path):
    raise FileNotFoundError("Summary checkpoint was not saved.")

gc.collect()

print("\nCHECKPOINT A complete.")
print("=" * 80)

In [ ]:
'''# === CELL 8: FREE MEMORY — run this before S5-5 ===

import gc
import sys

def sizeof_fmt(num):
    for unit in ['B', 'KB', 'MB', 'GB']:
        if abs(num) < 1024:
            return f"{num:.1f} {unit}"
        num /= 1024
    return f"{num:.1f} TB"

# Show what's using memory before cleanup
print("=" * 60)
print("MEMORY CLEANUP")
print("=" * 60)

big_vars = []
for name, obj in list(globals().items()):
    if name.startswith('_'):
        continue
    try:
        size = sys.getsizeof(obj)
        big_vars.append((name, size, type(obj).__name__))
    except:
        pass

big_vars.sort(key=lambda x: -x[1])
print("Top memory consumers before cleanup:")
for name, size, typ in big_vars[:20]:
    print(f"  {name:40s} {sizeof_fmt(size):>10s}  ({typ})")

# --- Delete variables no longer needed ---

# The original unfiltered molecules DataFrame — mol already has everything we need
if 'molecules' in dir():
    del molecules
    print("\n  Deleted: molecules (original DataFrame)")

# merged_transcripts_df — was used to create molecules.parquet, no longer needed
if 'merged_transcripts_df' in dir():
    del merged_transcripts_df
    print("  Deleted: merged_transcripts_df")

# filtered_transcripts_df
if 'filtered_transcripts_df' in dir():
    del filtered_transcripts_df
    print("  Deleted: filtered_transcripts_df")

# molecules_csv_df
if 'molecules_csv_df' in dir():
    del molecules_csv_df
    print("  Deleted: molecules_csv_df")

# molecules_df
if 'molecules_df' in dir():
    del molecules_df
    print("  Deleted: molecules_df")

# transcripts_df — the raw CSV load
if 'transcripts_df' in dir():
    del transcripts_df
    print("  Deleted: transcripts_df")

# filtered_molecules_df
if 'filtered_molecules_df' in dir():
    del filtered_molecules_df
    print("  Deleted: filtered_molecules_df")

# The full scRNA-seq reference — we only need the denoised outputs now
if 'adata_ref' in dir():
    del adata_ref
    print("  Deleted: adata_ref")

if 'annotated_ref_adata' in dir():
    del annotated_ref_adata
    print("  Deleted: annotated_ref_adata")

# The raw spatial adata — we have denoised_adata which is what we need
if 'annotated_spatial_adata' in dir():
    del annotated_spatial_adata
    print("  Deleted: annotated_spatial_adata")

# cell_by_gene_adata — raw cell×gene matrix, already captured in X_raw
if 'cell_by_gene_adata' in dir():
    del cell_by_gene_adata
    print("  Deleted: cell_by_gene_adata")

# Various intermediate DataFrames from exploration cells
for varname in ['diff_exp_df', 'clusters_df', 'cells_df',
                'cell_boundaries_df', 'nucleus_boundaries_df',
                'filtered_molecules_csv_df', 'count_pivot', 'conf_pivot',
                'imp_pivot', 'xenium_annotations_df']:
    if varname in dir():
        exec(f'del {varname}')
        print(f"  Deleted: {varname}")

# The raw cell geometry arrays — we already built polygons from them
if 'cell_geom' in dir():
    del cell_geom
    print("  Deleted: cell_geom (raw npz)")

if 'cell_vertices' in dir():
    del cell_vertices
    print("  Deleted: cell_vertices")

if 'nuc_vertices' in dir():
    del nuc_vertices
    print("  Deleted: nuc_vertices")

# Force garbage collection (multiple passes)
gc.collect()
gc.collect()
gc.collect()

# Check memory after cleanup
import psutil
mem = psutil.virtual_memory()
print(f"\nSystem RAM after cleanup: {mem.used/1e9:.1f} / {mem.total/1e9:.1f} GB "
      f"({mem.percent}%)")
print(f"Available: {mem.available/1e9:.1f} GB")'''


In [ ]:
'''# ==============================================================================
# CELL 10 — CONSERVATIVE CONTAMINATION PRUNING
# Only prune where Step 4 actually corrected AND reduction >= 2 molecules.
# ==============================================================================

print("=" * 70)
print("SUBSTEP 5B: Contamination pruning — conservative corrected-count version")
print("=" * 70)

import numpy as np
import pandas as pd
import gc, time, os
from collections import defaultdict

t0 = time.time()

# ------------------------------------------------------------------
# Required variables
# ------------------------------------------------------------------
required_vars = [
    "mol",
    "X_raw_counts",
    "X_denoised",
    "was_corrected",
    "cell_ids_step4",
    "shared_genes",
    "valid_cell_ids",
]

for v in required_vars:
    if v not in globals():
        raise NameError(f"Required variable missing: {v}")

if was_corrected.shape != X_denoised.shape:
    raise ValueError(
        f"was_corrected shape {was_corrected.shape} does not match "
        f"X_denoised shape {X_denoised.shape}"
    )

if X_raw_counts.shape != X_denoised.shape:
    raise ValueError(
        f"X_raw_counts shape {X_raw_counts.shape} does not match "
        f"X_denoised shape {X_denoised.shape}"
    )

# Make sure status/weight columns exist
if "status" not in mol.columns:
    mol["status"] = "observed"

if "weight" not in mol.columns:
    mol["weight"] = 1.0

if "is_imputed" not in mol.columns:
    mol["is_imputed"] = False

# Convert status to string to avoid categorical assignment issues
mol["status"] = mol["status"].astype(str)

# Reset previous pruning if this cell is re-run
mol.loc[mol["status"] == "pruned", "status"] = "observed"
mol.loc[mol["weight"] == 0.0, "weight"] = 1.0

# ------------------------------------------------------------------
# Build conservative overcounted-pair list
# ------------------------------------------------------------------
step4_cell_to_row = {int(c): i for i, c in enumerate(cell_ids_step4)}
gene_to_col = {g: j for j, g in enumerate(shared_genes)}

raw_int = X_raw_counts.astype(np.int32)
den_round = np.rint(X_denoised).astype(np.int32)

reduction = raw_int - den_round

# Required condition:
#   Step 4 actually corrected this cell-gene pair
#   and denoising reduced the count by at least 2 molecules
candidate_mask = (was_corrected.astype(bool)) & (reduction >= 2)

# Optional extra conservativeness:
# avoid pruning from tiny raw counts.
# Set MIN_RAW_FOR_PRUNE = 0 if you want exactly only the two conditions above.
MIN_RAW_FOR_PRUNE = 4
candidate_mask &= raw_int >= MIN_RAW_FOR_PRUNE

# Only consider cells with geometry and molecules
valid_row_mask = np.array(
    [int(cid) in valid_cell_ids for cid in cell_ids_step4],
    dtype=bool
)
candidate_mask &= valid_row_mask[:, None]

candidate_positions = np.argwhere(candidate_mask)

overcounted_pairs = {
    (int(cell_ids_step4[i]), shared_genes[j]): int(reduction[i, j])
    for i, j in candidate_positions
}

print(f"Overcounted pairs eligible for pruning: {len(overcounted_pairs):,}")
print(f"Conditions:")
print(f"  was_corrected == True")
print(f"  X_raw_counts - round(X_denoised) >= 2")
print(f"  X_raw_counts >= {MIN_RAW_FOR_PRUNE}")
print(f"  cell has valid geometry")

if len(overcounted_pairs) == 0:
    print("No pruning needed.")
    n_pruned_total = 0

else:
    # ------------------------------------------------------------------
    # Find candidate molecules only from affected cell-gene pairs
    # ------------------------------------------------------------------
    oc_cells = set(cid for cid, gid in overcounted_pairs.keys())
    oc_genes = set(gid for cid, gid in overcounted_pairs.keys())

    mask_candidates = (
        mol["cell_id"].isin(oc_cells)
        & mol["gene_id"].isin(oc_genes)
        & (mol["status"] == "observed")
    )

    candidate_idx = mol.index[mask_candidates]

    print(f"Candidate molecule rows from affected cells/genes: {len(candidate_idx):,}")

    # Pull arrays once for speed
    cids_arr = mol.loc[candidate_idx, "cell_id"].values
    gids_arr = mol.loc[candidate_idx, "gene_id"].astype(str).values
    rnorm_arr = mol.loc[candidate_idx, "r_norm"].values.astype(np.float32)
    qual_arr = mol.loc[candidate_idx, "quality"].values.astype(np.float32)

    pair_to_molecules = defaultdict(list)

    for k in range(len(candidate_idx)):
        key = (int(cids_arr[k]), gids_arr[k])
        if key in overcounted_pairs:
            pair_to_molecules[key].append((k, candidate_idx[k]))

    # ------------------------------------------------------------------
    # Choose specific molecules to prune
    # Suspicion score:
    #   higher r_norm = closer to cell boundary
    #   lower QV = less confident detection
    # ------------------------------------------------------------------
    prune_indices = []

    for key, n_to_prune in overcounted_pairs.items():
        mol_list = pair_to_molecules.get(key, [])

        if not mol_list:
            continue

        local_k = np.array([m[0] for m in mol_list], dtype=int)
        local_idx = np.array([m[1] for m in mol_list])

        quality_scaled = np.clip(qual_arr[local_k] / 40.0, 0.0, 1.0)

        suspicion = (
            0.6 * rnorm_arr[local_k]
            + 0.4 * (1.0 - quality_scaled)
        )

        order = np.argsort(-suspicion)
        n_actual = min(int(n_to_prune), len(order))

        prune_indices.extend(local_idx[order[:n_actual]].tolist())

    if prune_indices:
        mol.loc[prune_indices, "status"] = "pruned"
        mol.loc[prune_indices, "weight"] = 0.0

    n_pruned_total = len(prune_indices)

    del candidate_idx, cids_arr, gids_arr, rnorm_arr, qual_arr, pair_to_molecules
    gc.collect()

pct_pruned = 100 * n_pruned_total / len(mol)

print(f"\nPruned: {n_pruned_total:,} molecules ({pct_pruned:.4f}%)")
print(f"Remaining observed: {(mol['status'] == 'observed').sum():,}")
print(f"Pruned status count: {(mol['status'] == 'pruned').sum():,}")
print(f"Completed in {time.time() - t0:.1f}s")

if pct_pruned > 10:
    print(f"WARNING: {pct_pruned:.2f}% pruned is high. Check Step 4 correction settings.")
elif pct_pruned < 5:
    print(f"Pruning rate {pct_pruned:.2f}% is conservative/healthy.")'''

In [ ]:
# ==============================================================================
# COSMX CELL 11 — SAVE PROCESSED MOLECULE TABLE FOR RECOVERY
# Pruning is skipped because no denoised counts are below raw counts.
# ==============================================================================

import os
import json
import gc

print("=" * 80)
print("COSMX CELL 11: Save processed molecule table for recovery")
print("=" * 80)

# ------------------------------------------------------------------------------
# Use CosMx imputation folder
# ------------------------------------------------------------------------------

if "CHECKPOINT_DIR" not in globals():
    CHECKPOINT_DIR = "/content/drive/MyDrive/diffusion/step4_cosmx/imputation"

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

print(f"CHECKPOINT_DIR: {CHECKPOINT_DIR}")

# ------------------------------------------------------------------------------
# Required object checks
# ------------------------------------------------------------------------------

required_objects = ["mol"]

missing = [obj for obj in required_objects if obj not in globals()]
if missing:
    raise NameError(
        "Missing required objects:\n"
        + "\n".join(missing)
        + "\n\nRun Cell 6 coordinate normalization first."
    )

required_cols = [
    "cell_id",
    "gene_id",
    "x",
    "y",
    "z",
    "r_norm",
    "theta",
    "z_rel",
    "p_nuclear",
]

missing_cols = [c for c in required_cols if c not in mol.columns]
if missing_cols:
    raise KeyError(
        "mol is missing required columns:\n"
        + "\n".join(missing_cols)
        + "\n\nRun corrected CosMx coordinate normalization first."
    )

# ------------------------------------------------------------------------------
# Ensure status columns exist
# ------------------------------------------------------------------------------

if "status" not in mol.columns:
    mol["status"] = "observed"

if "weight" not in mol.columns:
    mol["weight"] = 1.0

if "is_imputed" not in mol.columns:
    mol["is_imputed"] = False

# Since pruning is skipped, make sure there are no accidental pruned rows.
mol["status"] = mol["status"].astype(str)
mol.loc[mol["status"] == "pruned", "status"] = "observed"
mol.loc[mol["weight"] == 0.0, "weight"] = 1.0

# ------------------------------------------------------------------------------
# Save processed molecule table
# ------------------------------------------------------------------------------

post_prune_path = os.path.join(CHECKPOINT_DIR, "step5_mol_processed.parquet")

mol.to_parquet(
    post_prune_path,
    index=False,
    compression="snappy"
)

print("\nSaved processed molecule table:")
print(f"  path: {post_prune_path}")
print(f"  rows: {len(mol):,}")
print(f"  unique cells: {mol['cell_id'].nunique():,}")
print(f"  unique genes: {mol['gene_id'].nunique():,}")
print(f"  observed: {(mol['status'] == 'observed').sum():,}")
print(f"  pruned: {(mol['status'] == 'pruned').sum():,}")
print(f"  imputed: {(mol['status'] == 'imputed').sum() if 'imputed' in set(mol['status']) else 0:,}")

# ------------------------------------------------------------------------------
# Save small summary
# ------------------------------------------------------------------------------

summary = {
    "platform": "CosMx",
    "checkpoint_name": "step5_mol_processed_after_coordinate_normalization",
    "pruning_run": False,
    "reason_pruning_skipped": "round(X_denoised) < X_raw_counts was zero in Cell 1 diagnostics",
    "path": post_prune_path,
    "n_rows": int(len(mol)),
    "n_cells": int(mol["cell_id"].nunique()),
    "n_genes": int(mol["gene_id"].nunique()),
    "observed_rows": int((mol["status"] == "observed").sum()),
    "pruned_rows": int((mol["status"] == "pruned").sum()),
    "imputed_rows": int((mol["status"] == "imputed").sum() if "imputed" in set(mol["status"]) else 0),
}

summary_path = os.path.join(CHECKPOINT_DIR, "step5_mol_processed_summary.json")

with open(summary_path, "w") as f:
    json.dump(summary, f, indent=2)

print("\nSaved summary:")
print(f"  path: {summary_path}")

# ------------------------------------------------------------------------------
# Verification
# ------------------------------------------------------------------------------

if not os.path.exists(post_prune_path):
    raise FileNotFoundError("Processed molecule checkpoint was not saved.")

if not os.path.exists(summary_path):
    raise FileNotFoundError("Processed molecule summary was not saved.")

gc.collect()

print("\nPASS: Processed molecule table saved for recovery.")
print("=" * 80)

# session crash1


In [ ]:
# ==============================================================================
# COSMX RECOVERY CELL — LOAD STEP 5 STATE AFTER RUNTIME DISCONNECT
# Use this after you have already run:
#   - corrected CosMx Cell 1
#   - CosMx Cell 4 geometry lookup
#   - CosMx Cell 6 coordinate normalization
#   - CosMx Cell 7 checkpoint
#   - CosMx Cell 11 processed molecule save
# ==============================================================================

import os
import gc
import json
import pickle
import warnings
import numpy as np
import pandas as pd
from scipy import sparse
import anndata as ad

warnings.filterwarnings("ignore")

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

print("=" * 80)
print("COSMX STEP 5 RECOVERY: Loading processed molecule checkpoint and Step 4 state")
print("=" * 80)

# ------------------------------------------------------------------------------
# Paths
# ------------------------------------------------------------------------------

STEP4_EXPORT_DIR = "/content/drive/MyDrive/diffusion/step4_cosmx/step4_exports"
CHECKPOINT_DIR = "/content/drive/MyDrive/diffusion/step4_cosmx/imputation"
IMPUTATION_DIR = CHECKPOINT_DIR
EXPORT_DIR = STEP4_EXPORT_DIR

POST_PRUNE_MOL_PATH = os.path.join(CHECKPOINT_DIR, "step5_mol_processed.parquet")
MOL_AFTER_5A_PATH = os.path.join(CHECKPOINT_DIR, "checkpoint_mol_after_5A.parquet")
GEOM_CHECKPOINT_PATH = os.path.join(CHECKPOINT_DIR, "checkpoint_cosmx_geometry_after_5A.pkl")

print(f"STEP4_EXPORT_DIR: {STEP4_EXPORT_DIR}")
print(f"CHECKPOINT_DIR  : {CHECKPOINT_DIR}")
print(f"IMPUTATION_DIR  : {IMPUTATION_DIR}")

# ------------------------------------------------------------------------------
# Helper
# ------------------------------------------------------------------------------

def ensure_dense(X):
    if sparse.issparse(X):
        return X.toarray()
    return np.asarray(X)

# ------------------------------------------------------------------------------
# Check required files
# ------------------------------------------------------------------------------

required_files = {
    "Step 4 config": os.path.join(STEP4_EXPORT_DIR, "step4_config.json"),
    "Step 4 denoised AnnData": os.path.join(STEP4_EXPORT_DIR, "denoised_adata.h5ad"),
    "Step 4 was_corrected": os.path.join(STEP4_EXPORT_DIR, "was_corrected.npy"),
    "Step 4 molecules": os.path.join(STEP4_EXPORT_DIR, "molecules.parquet"),
    "Step 4 cell geometry": os.path.join(STEP4_EXPORT_DIR, "cell_data.npz"),
}

# Prefer Cell 11 checkpoint. If unavailable, fall back to Cell 7 checkpoint.
if os.path.exists(POST_PRUNE_MOL_PATH):
    mol_checkpoint_path = POST_PRUNE_MOL_PATH
    print("\nUsing Cell 11 processed molecule checkpoint:")
    print(f"  {mol_checkpoint_path}")
elif os.path.exists(MOL_AFTER_5A_PATH):
    mol_checkpoint_path = MOL_AFTER_5A_PATH
    print("\nWARNING: Cell 11 checkpoint not found.")
    print("Falling back to Cell 7 coordinate-normalization checkpoint:")
    print(f"  {mol_checkpoint_path}")
else:
    raise FileNotFoundError(
        "Neither step5_mol_processed.parquet nor checkpoint_mol_after_5A.parquet was found. "
        "Rerun CosMx Cells 1, 4, 6, 7, and 11."
    )

required_files["Step 5 molecule checkpoint"] = mol_checkpoint_path

missing = []

print("\nChecking required files:")
for name, path in required_files.items():
    if os.path.exists(path):
        print(f"  ✓ {name:32s}: {path}")
    else:
        print(f"  ✗ MISSING {name:32s}: {path}")
        missing.append((name, path))

if missing:
    raise FileNotFoundError(f"Missing required recovery files: {missing}")

# ==============================================================================
# 1. Load Step 4 config
# ==============================================================================

print("\n" + "=" * 80)
print("1. Loading Step 4 config")
print("=" * 80)

with open(os.path.join(STEP4_EXPORT_DIR, "step4_config.json"), "r") as f:
    step4_config = json.load(f)

shared_genes = [str(g) for g in step4_config["shared_genes"]]
ct_column = step4_config.get("cell_type_column", "cell_type")
unique_cell_types = sorted([str(x) for x in step4_config.get("cell_types", [])])

print(f"Shared genes: {len(shared_genes):,}")
print(f"Cell-type column from config: {ct_column}")
print(f"Cell types from config: {len(unique_cell_types):,}")

# ==============================================================================
# 2. Load denoised AnnData and Step 4 matrices
# ==============================================================================

print("\n" + "=" * 80)
print("2. Loading denoised_adata and Step 4 matrices")
print("=" * 80)

denoised_adata = ad.read_h5ad(os.path.join(STEP4_EXPORT_DIR, "denoised_adata.h5ad"))

print(f"denoised_adata shape: {denoised_adata.shape}")
print(f"obs columns: {list(denoised_adata.obs.columns)}")
print(f"layers: {list(denoised_adata.layers.keys())}")

# CosMx-safe cell-type column selection
if ct_column not in denoised_adata.obs.columns:
    if "cell_type" in denoised_adata.obs.columns:
        ct_column = "cell_type"
    elif "Final_CosMx_Cell_Type" in denoised_adata.obs.columns:
        ct_column = "Final_CosMx_Cell_Type"
    elif "Assigned_Xenium_Cell_Type" in denoised_adata.obs.columns:
        ct_column = "Assigned_Xenium_Cell_Type"
    elif "Reference_Cell_Type" in denoised_adata.obs.columns:
        ct_column = "Reference_Cell_Type"
    else:
        raise KeyError(
            "No valid cell-type column found. Expected cell_type, "
            "Final_CosMx_Cell_Type, Assigned_Xenium_Cell_Type, or Reference_Cell_Type."
        )

if "Assigned_Xenium_Cell_Type" not in denoised_adata.obs.columns:
    denoised_adata.obs["Assigned_Xenium_Cell_Type"] = denoised_adata.obs[ct_column].astype(str)

if "cell_type" not in denoised_adata.obs.columns:
    denoised_adata.obs["cell_type"] = denoised_adata.obs[ct_column].astype(str)

cell_types_series = denoised_adata.obs[ct_column].astype(str)

if not unique_cell_types:
    unique_cell_types = sorted(cell_types_series.unique().tolist())

ct_to_idx = {ct: i for i, ct in enumerate(unique_cell_types)}
mapped_ct = cell_types_series.map(ct_to_idx)

if mapped_ct.isna().any():
    missing_ct = sorted(set(cell_types_series[mapped_ct.isna()].astype(str)))
    raise ValueError(f"Some cell types could not be mapped: {missing_ct[:20]}")

cell_type_indices = mapped_ct.to_numpy(dtype=np.int64)

X_denoised = ensure_dense(denoised_adata.X).astype(np.float32)

if "uncertainty" in denoised_adata.layers:
    uncertainty = ensure_dense(denoised_adata.layers["uncertainty"]).astype(np.float32)
else:
    uncertainty = None
    print("WARNING: uncertainty layer not found.")

# CosMx cell IDs are strings. Never convert to int.
cell_ids_step4 = np.array(denoised_adata.obs_names.astype(str), dtype=str)

# Gene order check
adata_genes = list(denoised_adata.var_names.astype(str))

print("\nGene order check:")
print(f"  shared_genes length       : {len(shared_genes):,}")
print(f"  denoised_adata.n_vars     : {denoised_adata.n_vars:,}")
print(f"  shared_genes == var_names : {shared_genes == adata_genes}")

if shared_genes != adata_genes:
    print("WARNING: Config shared_genes differs from denoised_adata.var_names.")
    print("Using denoised_adata.var_names as authoritative Step 5 gene order.")
    shared_genes = adata_genes

print(f"\nFinal cell-type column used: {ct_column}")
print(f"X_denoised shape: {X_denoised.shape}")
print(f"X_denoised sum  : {X_denoised.sum(dtype=np.float64):,.2f}")

if uncertainty is not None:
    print(f"uncertainty shape: {uncertainty.shape}")

# ==============================================================================
# 3. Load was_corrected
# ==============================================================================

print("\n" + "=" * 80)
print("3. Loading was_corrected mask")
print("=" * 80)

was_corrected = np.load(os.path.join(STEP4_EXPORT_DIR, "was_corrected.npy")).astype(bool)

if was_corrected.shape != X_denoised.shape:
    raise ValueError(
        f"was_corrected shape {was_corrected.shape} does not match "
        f"X_denoised shape {X_denoised.shape}"
    )

print(f"was_corrected shape: {was_corrected.shape}")
print(f"Corrected pairs: {was_corrected.sum():,}")
print(f"Correction rate: {100 * was_corrected.sum() / was_corrected.size:.2f}%")

# ==============================================================================
# 4. Load recovered Step 5 molecule table
# ==============================================================================

print("\n" + "=" * 80)
print("4. Loading recovered Step 5 molecule table")
print("=" * 80)

mol = pd.read_parquet(mol_checkpoint_path)

print(f"mol loaded: {len(mol):,} rows")
print(f"mol columns: {list(mol.columns)}")

# CosMx-safe dtype handling
if "cell_id" not in mol.columns:
    raise KeyError("Recovered mol table is missing cell_id.")
if "gene_id" not in mol.columns:
    raise KeyError("Recovered mol table is missing gene_id.")

mol["cell_id"] = mol["cell_id"].astype(str)
mol["gene_id"] = mol["gene_id"].astype(str)

# Add Xenium-compatible alias if needed
if "Assigned_Xenium_Cell_Type" not in mol.columns:
    if "Final_CosMx_Cell_Type" in mol.columns:
        mol["Assigned_Xenium_Cell_Type"] = mol["Final_CosMx_Cell_Type"].astype(str)
    elif "cell_type" in mol.columns:
        mol["Assigned_Xenium_Cell_Type"] = mol["cell_type"].astype(str)
    else:
        raise KeyError(
            "Recovered mol table is missing a usable cell-type column. "
            "Expected Assigned_Xenium_Cell_Type, Final_CosMx_Cell_Type, or cell_type."
        )

if "cell_type" not in mol.columns:
    mol["cell_type"] = mol["Assigned_Xenium_Cell_Type"].astype(str)

required_mol_cols = [
    "transcript_id",
    "cell_id",
    "gene_id",
    "x",
    "y",
    "z",
    "quality",
    "overlaps_nucleus",
    "Assigned_Xenium_Cell_Type",
    "r_norm",
    "theta",
    "z_rel",
    "p_nuclear",
    "status",
    "weight",
    "is_imputed",
]

missing_cols = [c for c in required_mol_cols if c not in mol.columns]
if missing_cols:
    raise KeyError(f"Recovered mol table is missing required columns: {missing_cols}")

mol["status"] = mol["status"].astype(str)
mol["weight"] = mol["weight"].astype(np.float32)
mol["is_imputed"] = mol["is_imputed"].astype(bool)

print("\nRecovered molecule status counts:")
display(mol["status"].value_counts().rename_axis("status").reset_index(name="n_molecules"))

print("\nRecovered coordinate-feature checks:")
print(f"  r_norm range   : [{mol['r_norm'].min():.4f}, {mol['r_norm'].max():.4f}]")
print(f"  theta range    : [{mol['theta'].min():.4f}, {mol['theta'].max():.4f}]")
print(f"  z_rel range    : [{mol['z_rel'].min():.4f}, {mol['z_rel'].max():.4f}]")
print(f"  p_nuclear range: [{mol['p_nuclear'].min():.4f}, {mol['p_nuclear'].max():.4f}]")

# For compatibility with older later cells
molecules = mol

# ==============================================================================
# 5. Rebuild X_raw_counts from the original clean Step 4 molecule table
# ==============================================================================

print("\n" + "=" * 80)
print("5. Rebuilding X_raw_counts from clean Step 4 molecules")
print("=" * 80)

step4_molecules_path = os.path.join(STEP4_EXPORT_DIR, "molecules.parquet")
step4_molecules = pd.read_parquet(step4_molecules_path)

step4_molecules["cell_id"] = step4_molecules["cell_id"].astype(str)
step4_molecules["gene_id"] = step4_molecules["gene_id"].astype(str)

gene_to_col = {str(g): j for j, g in enumerate(shared_genes)}
gene_idx_map = gene_to_col

step4_cell_to_row = {str(c): i for i, c in enumerate(cell_ids_step4)}
cell_idx_map = step4_cell_to_row

count_source = step4_molecules[
    step4_molecules["cell_id"].isin(step4_cell_to_row.keys())
    & step4_molecules["gene_id"].isin(gene_to_col.keys())
].copy()

if "status" in count_source.columns:
    print("status column found in Step 4 molecule table; using status == 'observed' for raw reconstruction.")
    count_source = count_source[count_source["status"] == "observed"].copy()

counts = (
    count_source
    .groupby(["cell_id", "gene_id"])
    .size()
    .reset_index(name="count")
)

X_raw_counts = np.zeros(X_denoised.shape, dtype=np.int32)

rows = counts["cell_id"].map(step4_cell_to_row)
cols = counts["gene_id"].map(gene_to_col)
ok = rows.notna() & cols.notna()

X_raw_counts[
    rows[ok].astype(int).to_numpy(),
    cols[ok].astype(int).to_numpy()
] = counts.loc[ok, "count"].to_numpy(dtype=np.int32)

X_raw = X_raw_counts.astype(np.float32)
X_observed_counts = X_raw_counts
X_raw_for_step5 = X_raw_counts

denoised_adata.layers["raw"] = X_raw.copy()
denoised_adata.layers["raw_molecule_counts_clean"] = X_raw.copy()

print(f"X_raw_counts shape: {X_raw_counts.shape}")
print(f"X_raw_counts sum  : {X_raw_counts.sum(dtype=np.float64):,.0f}")
print(f"Step 4 molecule rows used: {len(count_source):,}")

# Check against raw layer if present
raw_layer = ensure_dense(denoised_adata.layers["raw"]).astype(np.float32)
raw_diff = np.abs(raw_layer.astype(np.float64) - X_raw_counts.astype(np.float64))

print("\nRaw layer check:")
print(f"  raw layer sum     : {raw_layer.sum(dtype=np.float64):,.0f}")
print(f"  X_raw_counts sum  : {X_raw_counts.sum(dtype=np.float64):,.0f}")
print(f"  max abs diff      : {raw_diff.max():.6f}")
print(f"  mismatched entries: {(raw_diff > 1e-6).sum():,}")

# Status-aware current observed count matrix
observed_mol = mol[mol["status"] == "observed"].copy()

counts_current = (
    observed_mol
    .groupby(["cell_id", "gene_id"])
    .size()
    .reset_index(name="count")
)

X_current_observed_counts = np.zeros(X_denoised.shape, dtype=np.int32)

rows2 = counts_current["cell_id"].map(step4_cell_to_row)
cols2 = counts_current["gene_id"].map(gene_to_col)
ok2 = rows2.notna() & cols2.notna()

X_current_observed_counts[
    rows2[ok2].astype(int).to_numpy(),
    cols2[ok2].astype(int).to_numpy()
] = counts_current.loc[ok2, "count"].to_numpy(dtype=np.int32)

current_diff = np.abs(
    X_current_observed_counts.astype(np.float64)
    - X_raw_counts.astype(np.float64)
)

print("\nStatus-aware observed-count check:")
print(f"  current observed count sum: {X_current_observed_counts.sum(dtype=np.float64):,.0f}")
print(f"  original raw count sum    : {X_raw_counts.sum(dtype=np.float64):,.0f}")
print(f"  max abs diff              : {current_diff.max():.6f}")
print(f"  mismatched pairs          : {(current_diff > 1e-6).sum():,}")

if current_diff.max() > 1e-6:
    print("\nWARNING: Current observed counts differ from original raw counts.")
    print("This is expected only if pruning was performed.")
else:
    print("\nPASS: Current observed molecule table matches original raw counts.")

del step4_molecules, count_source, counts, observed_mol, counts_current
gc.collect()

# ==============================================================================
# 6. Load CosMx geometry checkpoint / rebuild summary geometry
# ==============================================================================

print("\n" + "=" * 80)
print("6. Loading CosMx summary geometry")
print("=" * 80)

GEOMETRY_MODE = "cosmx_summary_area_centroid"

# First, try the combined checkpoint from corrected Cell 7.
if os.path.exists(GEOM_CHECKPOINT_PATH):
    print("Loading combined CosMx geometry checkpoint:")
    print(f"  {GEOM_CHECKPOINT_PATH}")

    with open(GEOM_CHECKPOINT_PATH, "rb") as f:
        geometry_checkpoint = pickle.load(f)

    GEOMETRY_MODE = geometry_checkpoint.get("GEOMETRY_MODE", GEOMETRY_MODE)
    valid_cell_ids = set(geometry_checkpoint.get("valid_cell_ids", []))

    centroid_lookup = geometry_checkpoint.get("centroid_lookup", {})
    nuc_centroids = geometry_checkpoint.get("nuc_centroids", {})
    cell_radius_lookup_px = geometry_checkpoint.get("cell_radius_lookup_px", {})
    nuc_radius_lookup_px = geometry_checkpoint.get("nuc_radius_lookup_px", {})
    cell_area_lookup_px = geometry_checkpoint.get("cell_area_lookup_px", {})
    nuc_area_lookup_px = geometry_checkpoint.get("nuc_area_lookup_px", {})
    nuc_frac_lookup = geometry_checkpoint.get("nuc_frac_lookup", {})

    cell_area_lookup_um2 = geometry_checkpoint.get("cell_area_lookup_um2", None)
    cyto_frac_lookup = geometry_checkpoint.get("cyto_frac_lookup", None)
    membrane_frac_lookup = geometry_checkpoint.get("membrane_frac_lookup", None)

    cell_polygons = geometry_checkpoint.get("cell_polygons", {})
    nuc_polygons = geometry_checkpoint.get("nuc_polygons", {})
    cell_areas = geometry_checkpoint.get("cell_areas", {})
    nuc_areas = geometry_checkpoint.get("nuc_areas", {})

else:
    print("Combined geometry checkpoint not found.")
    print("Rebuilding CosMx summary geometry from cell_data.npz.")

    cell_geom = np.load(os.path.join(STEP4_EXPORT_DIR, "cell_data.npz"), allow_pickle=False)

    cell_ids_geom = cell_geom["cell_ids"].astype(str)
    centroids = cell_geom["centroids"].astype(np.float32)

    label_cell_area_px = cell_geom["label_cell_area_px"].astype(np.float32)
    label_nuclear_area_px = cell_geom["label_nuclear_area_px"].astype(np.float32)
    label_nuclear_frac = cell_geom["label_nuclear_frac"].astype(np.float32)

    if "cell_area_px" in cell_geom.keys():
        cell_area_px = cell_geom["cell_area_px"].astype(np.float32)
    else:
        cell_area_px = label_cell_area_px

    if "cell_area_um2" in cell_geom.keys():
        cell_area_um2 = cell_geom["cell_area_um2"].astype(np.float32)
    else:
        cell_area_um2 = None

    centroid_lookup = {
        str(cid): centroids[i].astype(np.float32)
        for i, cid in enumerate(cell_ids_geom)
    }

    cell_area_lookup_px = {
        str(cid): float(cell_area_px[i])
        for i, cid in enumerate(cell_ids_geom)
    }

    if cell_area_um2 is not None:
        cell_area_lookup_um2 = {
            str(cid): float(cell_area_um2[i])
            for i, cid in enumerate(cell_ids_geom)
        }
    else:
        cell_area_lookup_um2 = None

    nuc_area_lookup_px = {
        str(cid): float(label_nuclear_area_px[i])
        for i, cid in enumerate(cell_ids_geom)
    }

    nuc_frac_lookup = {
        str(cid): float(label_nuclear_frac[i])
        for i, cid in enumerate(cell_ids_geom)
    }

    cell_radius_lookup_px = {
        str(cid): float(np.sqrt(max(label_cell_area_px[i], 1.0) / np.pi))
        for i, cid in enumerate(cell_ids_geom)
    }

    nuc_radius_lookup_px = {
        str(cid): float(np.sqrt(max(label_nuclear_area_px[i], 0.0) / np.pi))
        for i, cid in enumerate(cell_ids_geom)
    }

    # Re-estimate nucleus centroid from nuclear molecules, otherwise use cell centroid.
    mol_nuc = mol[mol["overlaps_nucleus"].astype(np.int8) == 1]

    if len(mol_nuc) > 0:
        nuc_xy_df = (
            mol_nuc
            .groupby("cell_id")[["x", "y"]]
            .mean()
            .astype(np.float32)
        )
    else:
        nuc_xy_df = pd.DataFrame(columns=["x", "y"])

    nuc_centroids = {}

    for cid in cell_ids_geom:
        cid = str(cid)
        if cid in nuc_xy_df.index:
            nuc_centroids[cid] = nuc_xy_df.loc[cid, ["x", "y"]].to_numpy(dtype=np.float32)
        else:
            nuc_centroids[cid] = centroid_lookup[cid]

    cell_polygons = {}
    nuc_polygons = {}
    cell_areas = cell_area_lookup_px
    nuc_areas = nuc_area_lookup_px

    valid_cell_ids = (
        set(cell_ids_step4.astype(str))
        & set(cell_ids_geom.astype(str))
        & set(mol["cell_id"].astype(str).unique())
    )

    del cell_geom
    gc.collect()

# Ensure valid_cell_ids exists and is string-based
if "valid_cell_ids" not in globals() or len(valid_cell_ids) == 0:
    valid_cell_ids = (
        set(cell_ids_step4.astype(str))
        & set(mol["cell_id"].astype(str).unique())
    )

valid_cell_ids = set(str(c) for c in valid_cell_ids)

print(f"GEOMETRY_MODE: {GEOMETRY_MODE}")
print(f"valid_cell_ids: {len(valid_cell_ids):,}")
print(f"centroid_lookup: {len(centroid_lookup):,}")
print(f"nuc_centroids: {len(nuc_centroids):,}")
print(f"cell_radius_lookup_px: {len(cell_radius_lookup_px):,}")
print(f"nuc_radius_lookup_px: {len(nuc_radius_lookup_px):,}")
print(f"cell_polygons: {len(cell_polygons):,}  # expected 0 for CosMx summary geometry")
print(f"nuc_polygons : {len(nuc_polygons):,}  # expected 0 for CosMx summary geometry")

# ==============================================================================
# 7. Load compact cell_data.npz arrays in CosMx summary mode
# ==============================================================================

print("\n" + "=" * 80)
print("7. Loading compact CosMx cell_data.npz arrays")
print("=" * 80)

cell_geom = np.load(os.path.join(STEP4_EXPORT_DIR, "cell_data.npz"), allow_pickle=False)

print("cell_data.npz keys:")
print(list(cell_geom.keys()))

cell_ids_geom = cell_geom["cell_ids"].astype(str)
centroids = cell_geom["centroids"].astype(np.float32)

center_x_global_px = cell_geom["center_x_global_px"].astype(np.float32)
center_y_global_px = cell_geom["center_y_global_px"].astype(np.float32)

cell_area_px = cell_geom["cell_area_px"].astype(np.float32)
cell_area_um2 = cell_geom["cell_area_um2"].astype(np.float32)

label_cell_area_px = cell_geom["label_cell_area_px"].astype(np.float32)
label_nuclear_area_px = cell_geom["label_nuclear_area_px"].astype(np.float32)
label_membrane_area_px = cell_geom["label_membrane_area_px"].astype(np.float32)
label_cytoplasm_area_px = cell_geom["label_cytoplasm_area_px"].astype(np.float32)
label_extracellular_area_px = cell_geom["label_extracellular_area_px"].astype(np.float32)

label_nuclear_frac = cell_geom["label_nuclear_frac"].astype(np.float32)
label_membrane_frac = cell_geom["label_membrane_frac"].astype(np.float32)
label_cytoplasm_frac = cell_geom["label_cytoplasm_frac"].astype(np.float32)
label_extracellular_frac = cell_geom["label_extracellular_frac"].astype(np.float32)

# Compatibility placeholders: CosMx export does not contain polygon vertices.
cell_offsets = None
cell_vertices = None
nuc_offsets = None
nuc_vertices = None
nuc_present = label_nuclear_area_px > 0

geom_id_to_idx = {
    str(cid): i
    for i, cid in enumerate(cell_ids_geom)
}

print(f"cell_ids_geom shape: {cell_ids_geom.shape}")
print(f"centroids shape    : {centroids.shape}")
print(f"cell_area_px shape : {cell_area_px.shape}")
print(f"nuc_present cells  : {int(nuc_present.sum()):,} / {len(nuc_present):,}")
print("Polygon arrays are set to None because CosMx summary export has no polygon vertices.")

del cell_geom
gc.collect()

# ==============================================================================
# 8. Step 5 target diagnostics after recovery
# ==============================================================================

print("\n" + "=" * 80)
print("8. Step 5 target diagnostics after recovery")
print("=" * 80)

den_round = np.rint(X_denoised).astype(np.int32)

need_impute_mask = den_round > X_raw_counts
need_prune_mask = den_round < X_raw_counts

n_need_impute = int(need_impute_mask.sum())
n_need_prune = int(need_prune_mask.sum())

n_need_impute_corrected = int((need_impute_mask & was_corrected).sum())
n_need_prune_corrected = int((need_prune_mask & was_corrected).sum())

n_molecules_to_impute = int(np.maximum(den_round - X_raw_counts, 0).sum())
n_molecules_to_prune = int((-np.minimum(den_round - X_raw_counts, 0)).sum())

print(f"Pairs where round(X_denoised) > X_raw_counts: {n_need_impute:,}")
print(f"Pairs where round(X_denoised) < X_raw_counts: {n_need_prune:,}")
print(f"Corrected pairs needing imputation: {n_need_impute_corrected:,}")
print(f"Corrected pairs needing pruning: {n_need_prune_corrected:,}")
print(f"Total molecules to impute: {n_molecules_to_impute:,}")
print(f"Total molecules to prune : {n_molecules_to_prune:,}")

if n_need_prune == 0:
    print("PASS: No pruning needed after recovery.")
else:
    print("WARNING: Pruning would be needed. Check whether this was a global smoothing run.")

# ==============================================================================
# Final sanity summary
# ==============================================================================

print("\n" + "=" * 80)
print("COSMX RECOVERY COMPLETE")
print("=" * 80)

print("Recovered variables ready for later Step 5 cells:")
print("  mol")
print("  molecules")
print("  denoised_adata")
print("  X_denoised")
print("  X_raw_counts / X_observed_counts / X_raw_for_step5")
print("  X_current_observed_counts")
print("  was_corrected")
print("  uncertainty")
print("  shared_genes")
print("  cell_ids_step4")
print("  gene_to_col / gene_idx_map")
print("  step4_cell_to_row / cell_idx_map")
print("  cell_type_indices")
print("  valid_cell_ids")
print("  GEOMETRY_MODE")
print("  centroid_lookup")
print("  nuc_centroids")
print("  cell_radius_lookup_px / nuc_radius_lookup_px")
print("  cell_ids_geom, centroids")
print("  cell_offsets/cell_vertices/nuc_offsets/nuc_vertices = None for CosMx summary geometry")
print("  cell_polygons/nuc_polygons are empty compatibility dictionaries")

print("\nYou can now continue with later Step 5 cells for building/training data.")

In [ ]:
# ==============================================================================
# COSMX CELL S5-6-DIST — Build training data + empirical baseline pools
# Replaces old Xenium / PC2PC-based training-data cell.
# ==============================================================================

import os
import gc
import time
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from collections import defaultdict

print("=" * 80)
print("COSMX SUBSTEP 5C: Building training data + empirical baselines")
print("=" * 80)

t0 = time.time()

# ------------------------------------------------------------------------------
# Required variables from recovery cell
# ------------------------------------------------------------------------------

required_vars = [
    "mol",
    "X_raw_counts",
    "X_denoised",
    "was_corrected",
    "shared_genes",
    "cell_ids_step4",
    "gene_to_col",
    "step4_cell_to_row",
    "cell_type_indices",
    "unique_cell_types",
    "denoised_adata",
    "centroid_lookup",
    "cell_radius_lookup_px",
    "nuc_radius_lookup_px",
    "cell_area_lookup_px",
    "nuc_area_lookup_px",
]

missing = [v for v in required_vars if v not in globals()]
if missing:
    raise NameError(
        "Required variable(s) missing:\n"
        + "\n".join(missing)
        + "\n\nRun the CosMx recovery cell first."
    )

# ------------------------------------------------------------------------------
# Basic shape checks
# ------------------------------------------------------------------------------

if was_corrected.shape != X_denoised.shape:
    raise ValueError(
        f"was_corrected shape {was_corrected.shape} does not match "
        f"X_denoised shape {X_denoised.shape}"
    )

if X_raw_counts.shape != X_denoised.shape:
    raise ValueError(
        f"X_raw_counts shape {X_raw_counts.shape} does not match "
        f"X_denoised shape {X_denoised.shape}"
    )

if len(shared_genes) != X_denoised.shape[1]:
    raise ValueError(
        f"shared_genes length {len(shared_genes)} does not match "
        f"X_denoised n_genes {X_denoised.shape[1]}"
    )

# CosMx cell IDs must stay as strings.
cell_ids_step4 = np.array(cell_ids_step4).astype(str)

# Rebuild string-safe maps, just in case an older cell created int maps.
step4_cell_to_row = {
    str(cid): i
    for i, cid in enumerate(cell_ids_step4)
}

cell_idx_map = step4_cell_to_row

gene_to_col = {
    str(g): j
    for j, g in enumerate(shared_genes)
}

gene_idx_map = gene_to_col

# ------------------------------------------------------------------------------
# Use only active observed molecules
# ------------------------------------------------------------------------------

mol["cell_id"] = mol["cell_id"].astype(str)
mol["gene_id"] = mol["gene_id"].astype(str)
mol["status"] = mol["status"].astype(str)

mol_clean = mol[mol["status"] == "observed"].copy()

print(f"Observed molecules for training/baselines: {len(mol_clean):,}")

# Add compatibility alias if needed.
if "Assigned_Xenium_Cell_Type" not in mol_clean.columns:
    if "Final_CosMx_Cell_Type" in mol_clean.columns:
        mol_clean["Assigned_Xenium_Cell_Type"] = mol_clean["Final_CosMx_Cell_Type"].astype(str)
    elif "cell_type" in mol_clean.columns:
        mol_clean["Assigned_Xenium_Cell_Type"] = mol_clean["cell_type"].astype(str)
    else:
        raise KeyError(
            "mol is missing a usable cell-type column. Expected one of: "
            "Assigned_Xenium_Cell_Type, Final_CosMx_Cell_Type, cell_type."
        )

required_cols = [
    "cell_id",
    "gene_id",
    "r_norm",
    "theta",
    "z_rel",
    "p_nuclear",
    "Assigned_Xenium_Cell_Type",
]

missing_cols = [c for c in required_cols if c not in mol_clean.columns]
if missing_cols:
    raise KeyError(f"mol is missing required columns: {missing_cols}")

# ------------------------------------------------------------------------------
# Build cell coordinate lookup for spatial-kNN empirical baseline
# ------------------------------------------------------------------------------

cell_xy_lookup = {}

if {"x_centroid", "y_centroid"}.issubset(set(denoised_adata.obs.columns)):
    print("Using denoised_adata.obs x_centroid/y_centroid for cell coordinates.")

    for cid, x, y in zip(
        denoised_adata.obs_names.astype(str),
        denoised_adata.obs["x_centroid"].values,
        denoised_adata.obs["y_centroid"].values,
    ):
        cell_xy_lookup[str(cid)] = np.array([float(x), float(y)], dtype=np.float32)

elif "centroid_lookup" in globals() and len(centroid_lookup) > 0:
    print("Using centroid_lookup from CosMx summary geometry.")

    for cid, xy in centroid_lookup.items():
        cell_xy_lookup[str(cid)] = np.asarray(xy, dtype=np.float32)

elif "cell_ids_geom" in globals() and "centroids" in globals():
    print("Using cell_ids_geom/centroids for cell coordinates.")

    for cid, xy in zip(cell_ids_geom.astype(str), centroids):
        cell_xy_lookup[str(cid)] = np.asarray(xy, dtype=np.float32)

else:
    raise NameError(
        "No cell coordinate source found. Need denoised_adata.obs x/y centroids, "
        "centroid_lookup, or cell_ids_geom + centroids."
    )

print(f"Cell coordinate lookup entries: {len(cell_xy_lookup):,}")

# ------------------------------------------------------------------------------
# Area lookup for CosMx summary geometry
# ------------------------------------------------------------------------------

# For CosMx, use summary area lookup from cell_data.npz.
# If missing for any cell, use fallback values.
cell_area_lookup_safe = {
    str(cid): max(float(area), 1.0)
    for cid, area in cell_area_lookup_px.items()
}

nuc_area_lookup_safe = {
    str(cid): max(float(area), 0.0)
    for cid, area in nuc_area_lookup_px.items()
}

print(f"Cell area lookup entries: {len(cell_area_lookup_safe):,}")
print(f"Nucleus area lookup entries: {len(nuc_area_lookup_safe):,}")

# ------------------------------------------------------------------------------
# Build one training example per eligible (cell, gene) pair
# ------------------------------------------------------------------------------

MIN_MOL_FOR_TRAINING = 6

print("\nBuilding examples from observed molecules...")

examples_by_pair = []

# Grouping 30M rows can take some time.
for (cid, gid), group in mol_clean.groupby(["cell_id", "gene_id"], sort=False):
    cid = str(cid)
    gid = str(gid)
    n = len(group)

    if n < MIN_MOL_FOR_TRAINING:
        continue

    if cid not in step4_cell_to_row:
        continue

    if gid not in gene_to_col:
        continue

    if cid not in cell_xy_lookup:
        continue

    row = step4_cell_to_row[cid]
    gene_idx = gene_to_col[gid]
    ct_idx = int(cell_type_indices[row])

    c_area = float(cell_area_lookup_safe.get(cid, 100.0))
    n_area = float(nuc_area_lookup_safe.get(cid, 30.0))

    values = group[["r_norm", "theta", "z_rel", "p_nuclear"]].values.astype(np.float32)

    examples_by_pair.append({
        "cell_id": cid,                 # original CosMx string ID
        "gene_id": gid,
        "gene_idx": int(gene_idx),
        "ct_idx": int(ct_idx),
        "cell_xy": cell_xy_lookup[cid],
        "cell_area": c_area,
        "nuc_area": n_area,
        "values": values,
    })

print(f"Eligible observed (cell, gene) pairs: {len(examples_by_pair):,}")

if len(examples_by_pair) == 0:
    raise RuntimeError(
        "No eligible training examples found. "
        "Check mol/status/features and MIN_MOL_FOR_TRAINING."
    )

# ------------------------------------------------------------------------------
# Encode CosMx string cell IDs to integer codes for PyTorch batches
# ------------------------------------------------------------------------------

all_example_cell_ids = sorted({ex["cell_id"] for ex in examples_by_pair})
cell_id_to_code = {
    cid: i
    for i, cid in enumerate(all_example_cell_ids)
}

cell_code_to_id = {
    i: cid
    for cid, i in cell_id_to_code.items()
}

for ex in examples_by_pair:
    ex["cell_code"] = int(cell_id_to_code[ex["cell_id"]])

print(f"Encoded cell IDs for PyTorch: {len(cell_id_to_code):,}")

# ------------------------------------------------------------------------------
# Cell-level train/validation split
# Important: baselines are built only from training cells to avoid validation leakage.
# ------------------------------------------------------------------------------

P_DROP_MIN = 0.15
P_DROP_MAX = 0.40

all_cells = sorted({ex["cell_id"] for ex in examples_by_pair})

rng_split = np.random.default_rng(42)
rng_split.shuffle(all_cells)

n_val = max(1, int(0.10 * len(all_cells)))
val_cells = set(all_cells[:n_val])
train_cells = set(all_cells[n_val:])

training_examples = [
    ex for ex in examples_by_pair
    if ex["cell_id"] in train_cells
]

validation_examples = [
    ex for ex in examples_by_pair
    if ex["cell_id"] in val_cells
]

print(f"Train cells: {len(train_cells):,}")
print(f"Validation cells: {len(val_cells):,}")
print(f"Training examples: {len(training_examples):,}")
print(f"Validation examples: {len(validation_examples):,}")
print(f"Dynamic molecule dropout: {P_DROP_MIN*100:.0f}%–{P_DROP_MAX*100:.0f}%")

if len(training_examples) == 0 or len(validation_examples) == 0:
    raise RuntimeError("Training or validation examples are empty. Check split/settings.")

# ------------------------------------------------------------------------------
# Build empirical baseline pools from training examples only
# ------------------------------------------------------------------------------

print("\nBuilding empirical baseline pools from TRAINING examples only...")

gene_pool_lists = defaultdict(list)
ct_gene_pool_lists = defaultdict(list)

# For spatial-kNN:
# key = (ct_idx, gene_idx)
# value = list of entries with original string cell_id, encoded cell_code, xy, values
spatial_entry_lists = defaultdict(list)

for ex in training_examples:
    gene_idx = int(ex["gene_idx"])
    ct_idx = int(ex["ct_idx"])
    values = ex["values"].astype(np.float32)

    gene_pool_lists[gene_idx].append(values)
    ct_gene_pool_lists[(ct_idx, gene_idx)].append(values)

    spatial_entry_lists[(ct_idx, gene_idx)].append({
        "cell_id": ex["cell_id"],
        "cell_code": int(ex["cell_code"]),
        "xy": ex["cell_xy"].astype(np.float32),
        "values": values,
    })

baseline_gene_pools = {}
for gene_idx, arrs in gene_pool_lists.items():
    baseline_gene_pools[int(gene_idx)] = np.concatenate(arrs, axis=0).astype(np.float32)

baseline_ct_gene_pools = {}
for key, arrs in ct_gene_pool_lists.items():
    baseline_ct_gene_pools[key] = np.concatenate(arrs, axis=0).astype(np.float32)

baseline_spatial_index = {}
for key, entries in spatial_entry_lists.items():
    # Keep both string IDs and integer codes.
    # String IDs are needed for CosMx compatibility.
    # Integer codes can be useful for tensor-based logic.
    cell_ids_arr = np.array([e["cell_id"] for e in entries], dtype=object)
    cell_codes_arr = np.array([e["cell_code"] for e in entries], dtype=np.int64)
    xy_arr = np.stack([e["xy"] for e in entries], axis=0).astype(np.float32)
    values_list = [e["values"].astype(np.float32) for e in entries]

    baseline_spatial_index[key] = {
        "cell_ids": cell_ids_arr,
        "cell_codes": cell_codes_arr,
        "xy": xy_arr,
        "values": values_list,
    }

print(f"Gene-level empirical pools: {len(baseline_gene_pools):,}")
print(f"Cell-type gene empirical pools: {len(baseline_ct_gene_pools):,}")
print(f"Spatial-kNN empirical index groups: {len(baseline_spatial_index):,}")

# A few diagnostics
pool_size_summary = pd.Series(
    {
        shared_genes[int(g)]: len(v)
        for g, v in baseline_gene_pools.items()
    }
).sort_values(ascending=False)

print("\nTop 10 gene-level baseline pool sizes:")
display(
    pool_size_summary
    .head(10)
    .rename("n_molecules")
    .reset_index()
    .rename(columns={"index": "gene"})
)

# ------------------------------------------------------------------------------
# Dataset
# ------------------------------------------------------------------------------

class LocalizationDistributionDataset(Dataset):
    """
    Dynamically hides observed molecules and asks the model to recover their
    within-cell localization distribution.

    Baselines are NOT generated here. They are generated during validation from
    training-only empirical pools.
    """

    def __init__(
        self,
        examples,
        max_context=60,
        max_target=20,
        p_drop_min=0.15,
        p_drop_max=0.40,
        training=True,
        seed=42,
    ):
        self.examples = examples
        self.max_context = max_context
        self.max_target = max_target
        self.p_drop_min = p_drop_min
        self.p_drop_max = p_drop_max
        self.training = training
        self.seed = seed

    def __len__(self):
        return len(self.examples)

    def _rng(self, idx):
        if self.training:
            return np.random.default_rng()
        return np.random.default_rng(self.seed + idx)

    @staticmethod
    def _features(vals):
        r = vals[:, 0]
        theta = vals[:, 1]
        z = vals[:, 2]
        p_nuc = vals[:, 3]

        # Use sin/cos so theta wraps correctly around -pi/pi.
        return np.stack(
            [r, np.sin(theta), np.cos(theta), z, p_nuc],
            axis=-1
        ).astype(np.float32)

    def __getitem__(self, idx):
        ex = self.examples[idx]
        vals = ex["values"]
        n = len(vals)
        rng = self._rng(idx)

        p_drop = rng.uniform(self.p_drop_min, self.p_drop_max)
        k = int(rng.binomial(n, p_drop))
        k = max(1, min(n - 3, k, self.max_target))

        perm = rng.permutation(n)
        target_vals = vals[perm[:k]]
        context_vals = vals[perm[k:]]

        if len(context_vals) > self.max_context:
            ctx_idx = rng.choice(len(context_vals), size=self.max_context, replace=False)
            context_used = context_vals[ctx_idx]
        else:
            context_used = context_vals

        n_ctx = min(len(context_used), self.max_context)
        n_tgt = min(len(target_vals), self.max_target)

        ctx_padded = np.zeros((self.max_context, 5), dtype=np.float32)
        ctx_mask = np.zeros(self.max_context, dtype=np.float32)

        if n_ctx > 0:
            ctx_padded[:n_ctx] = self._features(context_used[:n_ctx])
            ctx_mask[:n_ctx] = 1.0

        target_coords = np.zeros((self.max_target, 3), dtype=np.float32)
        target_pnuc = np.zeros(self.max_target, dtype=np.float32)
        target_mask = np.zeros(self.max_target, dtype=np.float32)

        target_coords[:n_tgt] = target_vals[:n_tgt, :3]
        target_pnuc[:n_tgt] = target_vals[:n_tgt, 3]
        target_mask[:n_tgt] = 1.0

        c_area = float(ex["cell_area"])
        n_area = float(ex["nuc_area"])

        # Context summary features from visible/context molecules.
        # Feature order in ctx_features:
        #   0 = r_norm
        #   1 = sin(theta)
        #   2 = cos(theta)
        #   3 = z_rel
        #   4 = p_nuclear
        if n_ctx > 0:
            ctx_features = ctx_padded[:n_ctx]

            ctx_r_mean = float(ctx_features[:, 0].mean())
            ctx_r_std = float(ctx_features[:, 0].std())

            ctx_z_mean = float(ctx_features[:, 3].mean())
            ctx_z_std = float(ctx_features[:, 3].std())

            ctx_pnuc_mean = float(ctx_features[:, 4].mean())
        else:
            ctx_r_mean = 0.5
            ctx_r_std = 0.0
            ctx_z_mean = 0.5
            ctx_z_std = 0.0
            ctx_pnuc_mean = 0.0

        geom = np.array([
            c_area / 500.0,
            n_area / 200.0,
            n_area / c_area if c_area > 0 else 0.3,
            n_ctx / 50.0,
            ctx_r_mean,
            ctx_r_std,
            ctx_z_mean,
            ctx_z_std,
            ctx_pnuc_mean,
        ], dtype=np.float32)

        return {
            "context": torch.tensor(ctx_padded, dtype=torch.float32),
            "context_mask": torch.tensor(ctx_mask, dtype=torch.float32),
            "target_coords": torch.tensor(target_coords, dtype=torch.float32),
            "target_pnuc": torch.tensor(target_pnuc, dtype=torch.float32),
            "target_mask": torch.tensor(target_mask, dtype=torch.float32),
            "gene_idx": torch.tensor(ex["gene_idx"], dtype=torch.long),
            "ct_idx": torch.tensor(ex["ct_idx"], dtype=torch.long),

            # Keep the old key name "cell_id", but it now stores an integer code,
            # not the original CosMx string ID. This avoids PyTorch tensor errors.
            "cell_id": torch.tensor(ex["cell_code"], dtype=torch.long),

            "cell_xy": torch.tensor(ex["cell_xy"], dtype=torch.float32),
            "geom": torch.tensor(geom, dtype=torch.float32),
        }

# ------------------------------------------------------------------------------
# DataLoaders
# ------------------------------------------------------------------------------

train_dataset = LocalizationDistributionDataset(
    training_examples,
    max_context=96,
    max_target=24,
    p_drop_min=P_DROP_MIN,
    p_drop_max=P_DROP_MAX,
    training=True,
)

val_dataset = LocalizationDistributionDataset(
    validation_examples,
    max_context=96,
    max_target=24,
    p_drop_min=P_DROP_MIN,
    p_drop_max=P_DROP_MAX,
    training=False,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=256,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    drop_last=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=256,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
)

print(f"\nTrain batches: {len(train_loader):,}")
print(f"Val batches: {len(val_loader):,}")

# ------------------------------------------------------------------------------
# Quick batch sanity check
# ------------------------------------------------------------------------------

batch = next(iter(train_loader))

print("\nBatch sanity check:")
for k, v in batch.items():
    print(f"  {k:15s}: shape={tuple(v.shape)}, dtype={v.dtype}")

print(f"\nCell S5-6-DIST completed in {(time.time() - t0) / 60:.1f} min")

# ------------------------------------------------------------------------------
# Clean temporary large objects
# ------------------------------------------------------------------------------

del examples_by_pair
del gene_pool_lists, ct_gene_pool_lists, spatial_entry_lists

if "mol_clean" in globals():
    del mol_clean

gc.collect()

print("\nPASS: CosMx training data and empirical baseline pools are ready.")

In [ ]:
# ==============================================================================
# COSMX CELL S5-7-DIST: Conditional localization distribution model
# ==============================================================================

import torch
import torch.nn as nn
import torch.nn.functional as F

print("=" * 80)
print("COSMX SUBSTEP 5D: Conditional Localization Distribution Model")
print("=" * 80)

# ------------------------------------------------------------------------------
# Required object checks
# ------------------------------------------------------------------------------

required_objects = [
    "shared_genes",
    "unique_cell_types",
    "train_loader",
    "val_loader",
    "training_examples",
    "validation_examples",
]

missing = [obj for obj in required_objects if obj not in globals()]

if missing:
    raise NameError(
        "Missing required objects before model creation:\n"
        + "\n".join(missing)
        + "\n\nRun COSMX CELL S5-6-DIST first."
    )

n_genes = len(shared_genes)
n_cell_types = len(unique_cell_types)

if n_genes <= 0:
    raise ValueError("n_genes is zero. Check shared_genes.")

if n_cell_types <= 0:
    raise ValueError("n_cell_types is zero. Check unique_cell_types.")

# Check gene/cell-type index ranges from the training examples.
max_gene_idx = max(int(ex["gene_idx"]) for ex in training_examples + validation_examples)
max_ct_idx = max(int(ex["ct_idx"]) for ex in training_examples + validation_examples)

print("Index checks:")
print(f"  n_genes: {n_genes:,}")
print(f"  max gene_idx in examples: {max_gene_idx:,}")
print(f"  n_cell_types: {n_cell_types:,}")
print(f"  max ct_idx in examples: {max_ct_idx:,}")

if max_gene_idx >= n_genes:
    raise ValueError(
        f"gene_idx out of range: max_gene_idx={max_gene_idx}, n_genes={n_genes}"
    )

if max_ct_idx >= n_cell_types:
    raise ValueError(
        f"ct_idx out of range: max_ct_idx={max_ct_idx}, n_cell_types={n_cell_types}"
    )

# ------------------------------------------------------------------------------
# Context encoder
# ------------------------------------------------------------------------------

class PointNetContextEncoder(nn.Module):
    def __init__(self, in_dim=5, hidden=192, out_dim=192):
        super().__init__()

        self.mlp = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.SiLU(),
            nn.Linear(hidden, hidden),
            nn.SiLU(),
            nn.Linear(hidden, out_dim),
        )

        self.out_proj = nn.Sequential(
            nn.Linear(out_dim * 2, out_dim),
            nn.SiLU(),
            nn.Linear(out_dim, out_dim),
        )

    def forward(self, points, mask):
        """
        points: batch × max_context × 5
        mask  : batch × max_context
        """

        h = self.mlp(points)

        mask_exp = mask.unsqueeze(-1)

        # Max pooling over valid context molecules.
        h_masked = h.masked_fill(mask_exp == 0, -1e4)
        max_pool = h_masked.max(dim=1).values
        max_pool = torch.where(
            torch.isfinite(max_pool),
            max_pool,
            torch.zeros_like(max_pool),
        )

        # Mean pooling over valid context molecules.
        denom = mask_exp.sum(dim=1).clamp_min(1.0)
        mean_pool = (h * mask_exp).sum(dim=1) / denom

        return self.out_proj(torch.cat([max_pool, mean_pool], dim=-1))


# ------------------------------------------------------------------------------
# Conditional localization model
# ------------------------------------------------------------------------------

class LocalizationDistributionModel(nn.Module):
    def __init__(
        self,
        n_genes,
        n_cell_types,
        ctx_dim=192,
        gene_dim=96,
        ct_dim=32,
        geom_dim=9,
        geom_hidden=64,
        hidden=384,
    ):
        super().__init__()

        self.n_genes = n_genes
        self.n_cell_types = n_cell_types
        self.geom_dim = geom_dim

        self.context_encoder = PointNetContextEncoder(
            in_dim=5,
            hidden=192,
            out_dim=ctx_dim,
        )

        self.gene_embed = nn.Embedding(n_genes, gene_dim)
        self.ct_embed = nn.Embedding(n_cell_types, ct_dim)

        self.geom_mlp = nn.Sequential(
            nn.Linear(geom_dim, geom_hidden),
            nn.SiLU(),
            nn.Linear(geom_hidden, geom_hidden),
            nn.SiLU(),
        )

        fused = ctx_dim + gene_dim + ct_dim + geom_hidden

        self.backbone = nn.Sequential(
            nn.Linear(fused, hidden),
            nn.SiLU(),
            nn.Dropout(0.08),
            nn.Linear(hidden, hidden),
            nn.SiLU(),
            nn.Dropout(0.08),
            nn.Linear(hidden, hidden),
            nn.SiLU(),
        )

        # r_norm and z_rel are in [0, 1], so Beta distributions are appropriate.
        self.r_head = nn.Linear(hidden, 2)      # alpha, beta for r_norm
        self.z_head = nn.Linear(hidden, 2)      # alpha, beta for z_rel

        # p_nuclear is binary/probability-like.
        self.pnuc_head = nn.Linear(hidden, 1)   # logit p_nuclear

        # theta is circular, represented using a 2D direction proxy.
        self.theta_head = nn.Linear(hidden, 2)

    def forward(self, context, context_mask, gene_idx, ct_idx, geom):
        h_ctx = self.context_encoder(context, context_mask)
        h_gene = self.gene_embed(gene_idx)
        h_ct = self.ct_embed(ct_idx)
        h_geom = self.geom_mlp(geom)

        h = torch.cat([h_ctx, h_gene, h_ct, h_geom], dim=-1)
        h = self.backbone(h)

        # Add >1 offset to keep Beta parameters stable.
        r_ab = F.softplus(self.r_head(h)) + 1.05
        z_ab = F.softplus(self.z_head(h)) + 1.05

        pnuc_logit = self.pnuc_head(h).squeeze(-1)

        theta_vec = self.theta_head(h)

        return {
            "r_alpha": r_ab[:, 0],
            "r_beta": r_ab[:, 1],
            "z_alpha": z_ab[:, 0],
            "z_beta": z_ab[:, 1],
            "pnuc_logit": pnuc_logit,
            "theta_vec": theta_vec,
        }


# ------------------------------------------------------------------------------
# Build model
# ------------------------------------------------------------------------------

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = LocalizationDistributionModel(
    n_genes=n_genes,
    n_cell_types=n_cell_types,
    geom_dim=9,
).to(device)

n_params = sum(p.numel() for p in model.parameters())

print("\nModel summary:")
print(f"  Model parameters: {n_params:,} ({n_params * 4 / 1e6:.1f} MB)")
print(f"  Device: {device}")

if torch.cuda.is_available():
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    print(
        f"  GPU memory: "
        f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB"
    )

# ------------------------------------------------------------------------------
# Forward-pass sanity check
# ------------------------------------------------------------------------------

print("\nRunning one-batch forward-pass sanity check...")

batch = next(iter(train_loader))

batch_gpu = {}
for k, v in batch.items():
    if torch.is_tensor(v):
        batch_gpu[k] = v.to(device)
    else:
        batch_gpu[k] = v

with torch.no_grad():
    out = model(
        context=batch_gpu["context"],
        context_mask=batch_gpu["context_mask"],
        gene_idx=batch_gpu["gene_idx"],
        ct_idx=batch_gpu["ct_idx"],
        geom=batch_gpu["geom"],
    )

print("Model output sanity check:")
for k, v in out.items():
    print(
        f"  {k:12s}: shape={tuple(v.shape)}, "
        f"dtype={v.dtype}, "
        f"finite={torch.isfinite(v).all().item()}"
    )

# Check expected output shapes.
batch_size = batch_gpu["gene_idx"].shape[0]

expected_keys = [
    "r_alpha",
    "r_beta",
    "z_alpha",
    "z_beta",
    "pnuc_logit",
    "theta_vec",
]

for key in expected_keys:
    if key not in out:
        raise KeyError(f"Model output missing key: {key}")

if out["r_alpha"].shape != (batch_size,):
    raise ValueError("Unexpected r_alpha shape.")

if out["z_alpha"].shape != (batch_size,):
    raise ValueError("Unexpected z_alpha shape.")

if out["pnuc_logit"].shape != (batch_size,):
    raise ValueError("Unexpected pnuc_logit shape.")

if out["theta_vec"].shape != (batch_size, 2):
    raise ValueError("Unexpected theta_vec shape.")

print("\nPASS: CosMx localization distribution model is ready.")
print("=" * 80)

In [ ]:
# ==============================================================================
# COSMX CELL S5-8-DIST — Train localization distribution model
# Baselines:
#   1. Gene-level empirical distribution
#   2. Cell-type gene empirical distribution
#   3. Spatial-kNN empirical distribution
# ==============================================================================

import os
import gc
import time
import json
import math
import numpy as np
import pandas as pd

import torch
import torch.nn.functional as F
from torch.distributions import Beta

from sklearn.neighbors import BallTree

print("=" * 80)
print("COSMX SUBSTEP 5E: Training localization distribution model")
print("Baselines: gene empirical / cell-type gene empirical / spatial-kNN empirical")
print("=" * 80)

# ------------------------------------------------------------------------------
# 0. Required variable checks
# ------------------------------------------------------------------------------

required_vars = [
    "model",
    "device",
    "train_loader",
    "val_loader",
    "train_dataset",
    "val_dataset",
    "training_examples",
    "validation_examples",
    "shared_genes",
    "unique_cell_types",
    "cell_ids_step4",
    "cell_type_indices",
    "gene_to_col",
    "step4_cell_to_row",
    "centroids",
    "cell_ids_geom",
]

missing = [v for v in required_vars if v not in globals()]
if missing:
    raise NameError(
        "Missing required variable(s):\n"
        + "\n".join(missing)
        + "\n\nRun COSMX CELL S5-6-DIST and S5-7-DIST first."
    )

if "P_DROP_MIN" not in globals():
    P_DROP_MIN = 0.15
if "P_DROP_MAX" not in globals():
    P_DROP_MAX = 0.40

# CosMx cell IDs must stay strings.
cell_ids_step4 = np.array(cell_ids_step4).astype(str)
cell_ids_geom = np.array(cell_ids_geom).astype(str)

# Rebuild string-safe maps, just in case an older cell created int maps.
step4_cell_to_row = {
    str(cid): i
    for i, cid in enumerate(cell_ids_step4)
}
cell_idx_map = step4_cell_to_row

gene_to_col = {
    str(g): j
    for j, g in enumerate(shared_genes)
}
gene_idx_map = gene_to_col

print(f"Training examples: {len(training_examples):,}")
print(f"Validation examples: {len(validation_examples):,}")
print(f"Genes: {len(shared_genes):,}")
print(f"Cell types: {len(unique_cell_types):,}")
print(f"Device: {device}")

# ------------------------------------------------------------------------------
# 1. Training hyperparameters
# ------------------------------------------------------------------------------

RUN_NAME = "cosmx_9E_distribution_empirical_baselines"

# Keep 60 for final thesis run. You can reduce to 3–5 for smoke test.
EPOCHS = globals().get("EPOCHS", 60)
LR = globals().get("LR", 2e-4)
ETA_MIN = globals().get("ETA_MIN", 1e-5)

RECOVERY_EVAL_EVERY = globals().get("RECOVERY_EVAL_EVERY", 5)
RECOVERY_EVAL_BATCH_SIZE = globals().get("RECOVERY_EVAL_BATCH_SIZE", 256)
RECOVERY_EVAL_MAX_ITEMS = globals().get("RECOVERY_EVAL_MAX_ITEMS", 4096)

GENE_POOL_CAP = globals().get("GENE_POOL_CAP", 50_000)
CT_GENE_POOL_CAP = globals().get("CT_GENE_POOL_CAP", 20_000)
SPATIAL_KNN_K = globals().get("SPATIAL_KNN_K", 80)
SPATIAL_KNN_MIN_POOL = globals().get("SPATIAL_KNN_MIN_POOL", 5)

EPS = 1e-4

if "CHECKPOINT_DIR" not in globals():
    CHECKPOINT_DIR = "/content/drive/MyDrive/diffusion/step4_cosmx/imputation"

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

LOCAL_BEST_MODEL_PATH = os.path.join(CHECKPOINT_DIR, f"{RUN_NAME}_best_model.pt")
BEST_COORD_MODEL_PATH = os.path.join(CHECKPOINT_DIR, f"{RUN_NAME}_best_coordNN_model.pt")
RECOVERY_HISTORY_PATH = os.path.join(CHECKPOINT_DIR, f"{RUN_NAME}_recovery_history.json")
TRAINING_HISTORY_PATH = os.path.join(CHECKPOINT_DIR, f"{RUN_NAME}_training_history.json")

print("\nTraining config:")
print(f"  RUN_NAME: {RUN_NAME}")
print(f"  CHECKPOINT_DIR: {CHECKPOINT_DIR}")
print(f"  EPOCHS: {EPOCHS}")
print(f"  LR: {LR}")
print(f"  ETA_MIN: {ETA_MIN}")
print(f"  RECOVERY_EVAL_EVERY: {RECOVERY_EVAL_EVERY}")
print(f"  RECOVERY_EVAL_MAX_ITEMS: {RECOVERY_EVAL_MAX_ITEMS}")
print(f"  GENE_POOL_CAP: {GENE_POOL_CAP}")
print(f"  CT_GENE_POOL_CAP: {CT_GENE_POOL_CAP}")
print(f"  SPATIAL_KNN_K: {SPATIAL_KNN_K}")

# ------------------------------------------------------------------------------
# 2. Build empirical baseline pools from TRAINING examples only
# ------------------------------------------------------------------------------

print("\n" + "=" * 80)
print("Building empirical baseline pools from training examples only")
print("=" * 80)

t_pool = time.time()
rng_pool = np.random.default_rng(123)

_gene_lists = {}
_ct_gene_lists = {}

# String cell_id -> {gene_idx -> values}
train_cell_gene_values = {}

# String cell_id -> centroid
geom_cell_to_centroid = {
    str(cid): centroids[i].astype(np.float32)
    for i, cid in enumerate(cell_ids_geom)
}

for ex in training_examples:
    gid = int(ex["gene_idx"])
    ct = int(ex["ct_idx"])
    cid = str(ex["cell_id"])
    vals = ex["values"].astype(np.float32)

    _gene_lists.setdefault(gid, []).append(vals)
    _ct_gene_lists.setdefault((ct, gid), []).append(vals)

    if cid not in train_cell_gene_values:
        train_cell_gene_values[cid] = {}
    train_cell_gene_values[cid][gid] = vals

def _concat_and_cap(list_of_arrays, cap, rng):
    """
    Concatenate empirical molecule-coordinate arrays and cap them to avoid
    huge memory growth. Each row is [r_norm, theta, z_rel, p_nuclear].
    """
    if len(list_of_arrays) == 0:
        return None

    arr = np.concatenate(list_of_arrays, axis=0).astype(np.float32)

    if len(arr) > cap:
        idx = rng.choice(len(arr), size=cap, replace=False)
        arr = arr[idx]

    return arr

gene_empirical_pools = {
    int(gid): _concat_and_cap(arr_list, GENE_POOL_CAP, rng_pool)
    for gid, arr_list in _gene_lists.items()
}

ct_gene_empirical_pools = {
    key: _concat_and_cap(arr_list, CT_GENE_POOL_CAP, rng_pool)
    for key, arr_list in _ct_gene_lists.items()
}

del _gene_lists, _ct_gene_lists
gc.collect()

print(f"Gene empirical pools: {len(gene_empirical_pools):,}")
print(f"Cell-type gene empirical pools: {len(ct_gene_empirical_pools):,}")
print(f"Training cells with molecule pools: {len(train_cell_gene_values):,}")

# Build BallTree over training-cell centroids for spatial-kNN baseline.
train_spatial_cells = []
train_spatial_coords = []

for cid in train_cell_gene_values.keys():
    cid = str(cid)
    if cid in geom_cell_to_centroid:
        train_spatial_cells.append(cid)
        train_spatial_coords.append(geom_cell_to_centroid[cid])

train_spatial_cells = np.array(train_spatial_cells, dtype=object)
train_spatial_coords = np.asarray(train_spatial_coords, dtype=np.float32)

if len(train_spatial_cells) == 0:
    spatial_tree = None
    print("WARNING: No training-cell centroids found. Spatial-kNN baseline will fall back.")
else:
    spatial_tree = BallTree(train_spatial_coords)
    print(f"Spatial BallTree cells: {len(train_spatial_cells):,}")

print(f"Baseline pools built in {(time.time() - t_pool) / 60:.2f} min")

# ------------------------------------------------------------------------------
# 3. Distribution loss
# ------------------------------------------------------------------------------

MEAN_LOSS_WEIGHT = globals().get("MEAN_LOSS_WEIGHT", 0.05)
THETA_LOSS_WEIGHT = globals().get("THETA_LOSS_WEIGHT", 0.05)

def beta_nll(x, alpha, beta):
    x = x.clamp(EPS, 1 - EPS)
    dist = Beta(alpha.unsqueeze(1), beta.unsqueeze(1))
    return -dist.log_prob(x)

def mean_location_loss(outputs, target_coords, target_mask):
    r_target = target_coords[:, :, 0]
    z_target = target_coords[:, :, 2]

    denom_per_example = target_mask.sum(dim=1).clamp_min(1.0)

    r_true_mean = (r_target * target_mask).sum(dim=1) / denom_per_example
    z_true_mean = (z_target * target_mask).sum(dim=1) / denom_per_example

    r_pred_mean = outputs["r_alpha"] / (
        outputs["r_alpha"] + outputs["r_beta"] + 1e-8
    )
    z_pred_mean = outputs["z_alpha"] / (
        outputs["z_alpha"] + outputs["z_beta"] + 1e-8
    )

    r_mean_loss = F.smooth_l1_loss(r_pred_mean, r_true_mean)
    z_mean_loss = F.smooth_l1_loss(z_pred_mean, z_true_mean)

    return r_mean_loss + z_mean_loss

def theta_direction_loss(theta_vec, target_coords, target_mask):
    target_theta = target_coords[:, :, 1]

    target_vec = torch.stack(
        [torch.sin(target_theta), torch.cos(target_theta)],
        dim=-1
    )

    pred_vec = F.normalize(theta_vec, dim=-1).unsqueeze(1)
    target_vec = F.normalize(target_vec, dim=-1)

    cosine_sim = (pred_vec * target_vec).sum(dim=-1)
    loss = 1.0 - cosine_sim

    denom = target_mask.sum() + 1e-8
    return (loss * target_mask).sum() / denom

def distribution_loss(outputs, target_coords, target_pnuc, target_mask):
    r = target_coords[:, :, 0]
    z = target_coords[:, :, 2]

    r_nll = beta_nll(r, outputs["r_alpha"], outputs["r_beta"])
    z_nll = beta_nll(z, outputs["z_alpha"], outputs["z_beta"])

    pnuc_logits = outputs["pnuc_logit"].unsqueeze(1).expand_as(target_pnuc)

    pnuc_bce = F.binary_cross_entropy_with_logits(
        pnuc_logits,
        target_pnuc,
        reduction="none"
    )

    denom = target_mask.sum() + 1e-8

    r_loss = (r_nll * target_mask).sum() / denom
    z_loss = (z_nll * target_mask).sum() / denom
    pnuc_loss = (pnuc_bce * target_mask).sum() / denom

    mean_loss = mean_location_loss(outputs, target_coords, target_mask)
    theta_loss = theta_direction_loss(
        outputs["theta_vec"],
        target_coords,
        target_mask,
    )

    total = (
        r_loss
        + z_loss
        + 0.5 * pnuc_loss
        + MEAN_LOSS_WEIGHT * mean_loss
        + THETA_LOSS_WEIGHT * theta_loss
    )

    return total, r_loss, z_loss, pnuc_loss

# ------------------------------------------------------------------------------
# 4. Learned model sampler
# ------------------------------------------------------------------------------

@torch.no_grad()
def sample_from_learned_distribution(outputs, target_mask, context=None, context_mask=None):
    dev = target_mask.device
    B, K = target_mask.shape

    r_dist = torch.distributions.Beta(
        outputs["r_alpha"].clamp_min(1e-4),
        outputs["r_beta"].clamp_min(1e-4),
    )
    r = r_dist.sample((K,)).transpose(0, 1).to(dev).clamp(0.0, 1.0)

    z_dist = torch.distributions.Beta(
        outputs["z_alpha"].clamp_min(1e-4),
        outputs["z_beta"].clamp_min(1e-4),
    )
    z = z_dist.sample((K,)).transpose(0, 1).to(dev).clamp(0.0, 1.0)

    theta_vec = F.normalize(outputs["theta_vec"], dim=-1)
    theta_mu = torch.atan2(theta_vec[:, 0], theta_vec[:, 1])

    theta = theta_mu.unsqueeze(1).expand(B, K).clone()

    theta_jitter = 0.20
    theta = theta + torch.randn((B, K), device=dev) * theta_jitter
    theta = ((theta + torch.pi) % (2 * torch.pi)) - torch.pi

    return torch.stack([r, theta, z], dim=-1)

# ------------------------------------------------------------------------------
# 5. Empirical baseline samplers
# ------------------------------------------------------------------------------

def _sample_empirical_pool(pool, n, rng):
    if pool is None or len(pool) == 0:
        r = rng.uniform(0, 1, size=n)
        theta = rng.uniform(-np.pi, np.pi, size=n)
        z = rng.uniform(0, 1, size=n)
        return np.stack([r, theta, z], axis=1).astype(np.float32)

    idx = rng.integers(0, len(pool), size=n)
    vals = pool[idx, :3].astype(np.float32)

    vals[:, 0] = np.clip(vals[:, 0], 0, 1)
    vals[:, 1] = np.arctan2(np.sin(vals[:, 1]), np.cos(vals[:, 1]))
    vals[:, 2] = np.clip(vals[:, 2], 0, 1)

    return vals

def sample_gene_empirical(ex, n, rng):
    gid = int(ex["gene_idx"])
    pool = gene_empirical_pools.get(gid, None)
    return _sample_empirical_pool(pool, n, rng)

def sample_ct_gene_empirical(ex, n, rng):
    gid = int(ex["gene_idx"])
    ct = int(ex["ct_idx"])

    pool = ct_gene_empirical_pools.get((ct, gid), None)

    if pool is None or len(pool) < 3:
        pool = gene_empirical_pools.get(gid, None)

    return _sample_empirical_pool(pool, n, rng)

def sample_spatial_knn_empirical(ex, n, rng):
    gid = int(ex["gene_idx"])
    cid = str(ex["cell_id"])

    if spatial_tree is None or cid not in geom_cell_to_centroid:
        return sample_ct_gene_empirical(ex, n, rng)

    query_coord = geom_cell_to_centroid[cid].reshape(1, -1)

    k = min(SPATIAL_KNN_K, len(train_spatial_cells))
    _, nn_idx = spatial_tree.query(query_coord, k=k)

    candidate_arrays = []

    for idx in nn_idx[0]:
        nb_cid = str(train_spatial_cells[idx])
        gene_dict = train_cell_gene_values.get(nb_cid, {})
        vals = gene_dict.get(gid, None)

        if vals is not None and len(vals) > 0:
            candidate_arrays.append(vals)

    if len(candidate_arrays) > 0:
        pool = np.concatenate(candidate_arrays, axis=0).astype(np.float32)

        if len(pool) >= SPATIAL_KNN_MIN_POOL:
            return _sample_empirical_pool(pool, n, rng)

    return sample_ct_gene_empirical(ex, n, rng)

# ------------------------------------------------------------------------------
# 6. Held-out recovery metrics
# ------------------------------------------------------------------------------

def _wasserstein_1d(a, b):
    if len(a) == 0 or len(b) == 0:
        return np.nan

    a = np.sort(np.asarray(a, dtype=np.float64))
    b = np.sort(np.asarray(b, dtype=np.float64))

    q = np.linspace(0, 1, max(len(a), len(b)))
    aq = np.interp(q, np.linspace(0, 1, len(a)), a)
    bq = np.interp(q, np.linspace(0, 1, len(b)), b)

    return float(np.mean(np.abs(aq - bq)))

def _theta_set_distance(pred_theta, true_theta):
    if len(pred_theta) == 0 or len(true_theta) == 0:
        return np.nan

    pred_theta = np.asarray(pred_theta, dtype=np.float64)
    true_theta = np.asarray(true_theta, dtype=np.float64)

    d = np.abs(
        np.arctan2(
            np.sin(pred_theta[:, None] - true_theta[None, :]),
            np.cos(pred_theta[:, None] - true_theta[None, :])
        )
    )

    return float(np.mean(np.min(d, axis=1)) / np.pi)

def compute_recovery_metrics_np(pred, target):
    matched_dists = []
    r_wass = []
    z_wass = []
    theta_dist = []
    hist_l1 = []

    for p, t in zip(pred, target):
        if len(p) == 0 or len(t) == 0:
            continue

        p = np.asarray(p, dtype=np.float32)
        t = np.asarray(t, dtype=np.float32)

        dr = p[:, None, 0] - t[None, :, 0]

        dtheta = np.arctan2(
            np.sin(p[:, None, 1] - t[None, :, 1]),
            np.cos(p[:, None, 1] - t[None, :, 1])
        ) / np.pi

        dz = p[:, None, 2] - t[None, :, 2]

        dist = np.sqrt(dr * dr + dtheta * dtheta + dz * dz)

        matched_dists.append(float(np.mean(np.min(dist, axis=1))))
        r_wass.append(_wasserstein_1d(p[:, 0], t[:, 0]))
        z_wass.append(_wasserstein_1d(p[:, 2], t[:, 2]))
        theta_dist.append(_theta_set_distance(p[:, 1], t[:, 1]))

        p_hist, _ = np.histogram(p[:, 0], bins=np.linspace(0, 1, 11), density=False)
        t_hist, _ = np.histogram(t[:, 0], bins=np.linspace(0, 1, 11), density=False)

        p_hist = p_hist / max(p_hist.sum(), 1)
        t_hist = t_hist / max(t_hist.sum(), 1)

        hist_l1.append(float(np.abs(p_hist - t_hist).sum()))

    return {
        "coord_nn": float(np.nanmean(matched_dists)) if matched_dists else np.nan,
        "r_wasserstein": float(np.nanmean(r_wass)) if r_wass else np.nan,
        "z_wasserstein": float(np.nanmean(z_wass)) if z_wass else np.nan,
        "theta_distance": float(np.nanmean(theta_dist)) if theta_dist else np.nan,
        "radial_hist_l1": float(np.nanmean(hist_l1)) if hist_l1 else np.nan,
    }

# ------------------------------------------------------------------------------
# 7. Deterministic held-out construction for empirical recovery evaluation
# ------------------------------------------------------------------------------

def _features_np(vals):
    r = vals[:, 0]
    th = vals[:, 1]
    z = vals[:, 2]
    pn = vals[:, 3]

    return np.stack(
        [r, np.sin(th), np.cos(th), z, pn],
        axis=-1
    ).astype(np.float32)

def make_eval_item_from_example(
    ex,
    idx,
    max_context=96,
    max_target=24,
    p_drop_min=0.15,
    p_drop_max=0.40,
    seed=12345,
):
    vals = ex["values"]
    n = len(vals)

    rng = np.random.default_rng(seed + idx)
    p_drop = rng.uniform(p_drop_min, p_drop_max)

    k = int(rng.binomial(n, p_drop))
    k = max(1, min(n - 3, k, max_target))

    perm = rng.permutation(n)

    target_vals = vals[perm[:k]]
    context_vals = vals[perm[k:]]

    if len(context_vals) > max_context:
        ctx_idx = rng.choice(len(context_vals), size=max_context, replace=False)
        context_used = context_vals[ctx_idx]
    else:
        context_used = context_vals

    n_ctx = min(len(context_used), max_context)
    n_tgt = min(len(target_vals), max_target)

    ctx_padded = np.zeros((max_context, 5), dtype=np.float32)

    if n_ctx > 0:
        ctx_padded[:n_ctx] = _features_np(context_used[:n_ctx])

    ctx_mask = np.zeros(max_context, dtype=np.float32)
    ctx_mask[:n_ctx] = 1.0

    target_coords = np.zeros((max_target, 3), dtype=np.float32)
    target_pnuc = np.zeros(max_target, dtype=np.float32)
    target_mask = np.zeros(max_target, dtype=np.float32)

    target_coords[:n_tgt] = target_vals[:n_tgt, :3]
    target_pnuc[:n_tgt] = target_vals[:n_tgt, 3]
    target_mask[:n_tgt] = 1.0

    c_area = float(ex.get("cell_area", 100.0))
    n_area = float(ex.get("nuc_area", 30.0))

    if n_ctx > 0:
        ctx_features = ctx_padded[:n_ctx]

        ctx_r_mean = float(ctx_features[:, 0].mean())
        ctx_r_std = float(ctx_features[:, 0].std())

        ctx_z_mean = float(ctx_features[:, 3].mean())
        ctx_z_std = float(ctx_features[:, 3].std())

        ctx_pnuc_mean = float(ctx_features[:, 4].mean())
    else:
        ctx_r_mean = 0.5
        ctx_r_std = 0.0
        ctx_z_mean = 0.5
        ctx_z_std = 0.0
        ctx_pnuc_mean = 0.0

    geom = np.array([
        c_area / 500.0,
        n_area / 200.0,
        n_area / c_area if c_area > 0 else 0.3,
        n_ctx / 50.0,
        ctx_r_mean,
        ctx_r_std,
        ctx_z_mean,
        ctx_z_std,
        ctx_pnuc_mean,
    ], dtype=np.float32)

    return {
        "context": ctx_padded,
        "context_mask": ctx_mask,
        "target_coords": target_coords,
        "target_pnuc": target_pnuc,
        "target_mask": target_mask,
        "gene_idx": int(ex["gene_idx"]),
        "ct_idx": int(ex["ct_idx"]),
        "geom": geom,
        "ex": ex,
        "n_tgt": n_tgt,
    }

@torch.no_grad()
def evaluate_heldout_recovery_empirical(
    model,
    validation_examples,
    device,
    max_items=2048,
    batch_size=256,
    seed=12345,
):
    model.eval()

    n_eval = min(max_items, len(validation_examples))

    learned_preds = []
    gene_preds = []
    ct_gene_preds = []
    spatial_knn_preds = []
    targets = []

    rng_eval = np.random.default_rng(seed)

    for start in range(0, n_eval, batch_size):
        end = min(start + batch_size, n_eval)

        items = [
            make_eval_item_from_example(
                validation_examples[i],
                idx=i,
                p_drop_min=P_DROP_MIN,
                p_drop_max=P_DROP_MAX,
                seed=seed,
            )
            for i in range(start, end)
        ]

        ctx = torch.tensor(np.stack([it["context"] for it in items]), dtype=torch.float32, device=device)
        ctx_mask = torch.tensor(np.stack([it["context_mask"] for it in items]), dtype=torch.float32, device=device)
        target_coords_t = torch.tensor(np.stack([it["target_coords"] for it in items]), dtype=torch.float32, device=device)
        target_mask_t = torch.tensor(np.stack([it["target_mask"] for it in items]), dtype=torch.float32, device=device)

        gene_idx_t = torch.tensor([it["gene_idx"] for it in items], dtype=torch.long, device=device)
        ct_idx_t = torch.tensor([it["ct_idx"] for it in items], dtype=torch.long, device=device)
        geom_t = torch.tensor(np.stack([it["geom"] for it in items]), dtype=torch.float32, device=device)

        outputs = model(ctx, ctx_mask, gene_idx_t, ct_idx_t, geom_t)
        learned_sample_t = sample_from_learned_distribution(outputs, target_mask_t, ctx, ctx_mask)

        learned_sample = learned_sample_t.detach().cpu().numpy()
        target_coords_np = target_coords_t.detach().cpu().numpy()
        target_mask_np = target_mask_t.detach().cpu().numpy()

        for local_i, it in enumerate(items):
            m = target_mask_np[local_i] > 0
            n_tgt = int(m.sum())

            if n_tgt <= 0:
                continue

            true_coords = target_coords_np[local_i, m, :]
            learned_coords = learned_sample[local_i, m, :]

            ex = it["ex"]

            gene_coords = sample_gene_empirical(ex, n_tgt, rng_eval)
            ct_gene_coords = sample_ct_gene_empirical(ex, n_tgt, rng_eval)
            spatial_coords = sample_spatial_knn_empirical(ex, n_tgt, rng_eval)

            targets.append(true_coords)
            learned_preds.append(learned_coords)
            gene_preds.append(gene_coords)
            ct_gene_preds.append(ct_gene_coords)
            spatial_knn_preds.append(spatial_coords)

    metrics = {}

    all_methods = {
        "learned": learned_preds,
        "gene_emp": gene_preds,
        "ct_gene_emp": ct_gene_preds,
        "spatial_knn_emp": spatial_knn_preds,
    }

    for name, preds in all_methods.items():
        m = compute_recovery_metrics_np(preds, targets)
        for k, v in m.items():
            metrics[f"{name}_{k}"] = v

    learned_coord = metrics.get("learned_coord_nn", np.nan)

    for base in ["gene_emp", "ct_gene_emp", "spatial_knn_emp"]:
        base_coord = metrics.get(f"{base}_coord_nn", np.nan)
        metrics[f"improvement_vs_{base}_coord_nn"] = float(
            (base_coord - learned_coord) / (abs(base_coord) + 1e-8)
        )

    metrics["n_eval_examples"] = len(targets)
    metrics["n_eval_target_molecules"] = int(sum(len(t) for t in targets))

    return metrics

# ------------------------------------------------------------------------------
# 8. Optimizer / scheduler / history containers
# ------------------------------------------------------------------------------

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)

scheduler_lr = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=EPOCHS,
    eta_min=ETA_MIN,
)

train_losses, val_losses = [], []
train_r_losses, train_z_losses, train_pnuc_losses = [], [], []
val_r_losses, val_z_losses, val_pnuc_losses = [], [], []

recovery_history = []
best_val_loss = float("inf")
best_coord_nn = float("inf")

t0_train = time.time()

# ------------------------------------------------------------------------------
# 9. Main training loop
# ------------------------------------------------------------------------------

print("\n" + "=" * 80)
print("TRAINING: learned localization distribution model")
print("=" * 80)

for epoch in range(EPOCHS):
    epoch_start = time.time()

    model.train()

    train_total = 0.0
    train_r = 0.0
    train_z = 0.0
    train_p = 0.0
    n_train_batches = 0

    for batch in train_loader:
        ctx = batch["context"].to(device)
        ctx_mask = batch["context_mask"].to(device)
        target = batch["target_coords"].to(device)
        target_pnuc = batch["target_pnuc"].to(device)
        mask = batch["target_mask"].to(device)

        gene_idx = batch["gene_idx"].to(device)
        ct_idx = batch["ct_idx"].to(device)
        geom = batch["geom"].to(device)

        optimizer.zero_grad(set_to_none=True)

        outputs = model(ctx, ctx_mask, gene_idx, ct_idx, geom)

        loss, r_loss, z_loss, pnuc_loss = distribution_loss(
            outputs,
            target,
            target_pnuc,
            mask,
        )

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()

        train_total += float(loss.item())
        train_r += float(r_loss.item())
        train_z += float(z_loss.item())
        train_p += float(pnuc_loss.item())
        n_train_batches += 1

    avg_train = train_total / max(n_train_batches, 1)
    avg_train_r = train_r / max(n_train_batches, 1)
    avg_train_z = train_z / max(n_train_batches, 1)
    avg_train_p = train_p / max(n_train_batches, 1)

    train_losses.append(avg_train)
    train_r_losses.append(avg_train_r)
    train_z_losses.append(avg_train_z)
    train_pnuc_losses.append(avg_train_p)

    # --------------------------------------------------------------------------
    # Validation
    # --------------------------------------------------------------------------
    model.eval()

    val_total = 0.0
    val_r = 0.0
    val_z = 0.0
    val_p = 0.0
    n_val_batches = 0

    with torch.no_grad():
        for batch in val_loader:
            ctx = batch["context"].to(device)
            ctx_mask = batch["context_mask"].to(device)
            target = batch["target_coords"].to(device)
            target_pnuc = batch["target_pnuc"].to(device)
            mask = batch["target_mask"].to(device)

            gene_idx = batch["gene_idx"].to(device)
            ct_idx = batch["ct_idx"].to(device)
            geom = batch["geom"].to(device)

            outputs = model(ctx, ctx_mask, gene_idx, ct_idx, geom)

            loss, r_loss, z_loss, pnuc_loss = distribution_loss(
                outputs,
                target,
                target_pnuc,
                mask,
            )

            val_total += float(loss.item())
            val_r += float(r_loss.item())
            val_z += float(z_loss.item())
            val_p += float(pnuc_loss.item())
            n_val_batches += 1

    avg_val = val_total / max(n_val_batches, 1)
    avg_val_r = val_r / max(n_val_batches, 1)
    avg_val_z = val_z / max(n_val_batches, 1)
    avg_val_p = val_p / max(n_val_batches, 1)

    val_losses.append(avg_val)
    val_r_losses.append(avg_val_r)
    val_z_losses.append(avg_val_z)
    val_pnuc_losses.append(avg_val_p)

    scheduler_lr.step()

    # --------------------------------------------------------------------------
    # Held-out recovery evaluation
    # --------------------------------------------------------------------------
    recovery_metrics = None

    should_eval_recovery = (
        epoch == 0
        or (epoch + 1) % RECOVERY_EVAL_EVERY == 0
        or (epoch + 1) == EPOCHS
    )

    if should_eval_recovery:
        recovery_metrics = evaluate_heldout_recovery_empirical(
            model=model,
            validation_examples=validation_examples,
            device=device,
            max_items=RECOVERY_EVAL_MAX_ITEMS,
            batch_size=RECOVERY_EVAL_BATCH_SIZE,
            seed=12345,
        )

        recovery_metrics["epoch"] = epoch + 1
        recovery_history.append(recovery_metrics)

    # --------------------------------------------------------------------------
    # Save best models
    # --------------------------------------------------------------------------
    marker = ""

    if recovery_metrics is not None:
        current_coord = recovery_metrics.get("learned_coord_nn", np.nan)

        if np.isfinite(current_coord) and current_coord < best_coord_nn:
            best_coord_nn = current_coord
            torch.save(model.state_dict(), BEST_COORD_MODEL_PATH)
            print(f"  ★ saved best coordNN model: {best_coord_nn:.4f}")

    if avg_val < best_val_loss:
        best_val_loss = avg_val
        torch.save(model.state_dict(), LOCAL_BEST_MODEL_PATH)
        marker = " ★ saved"

    # Save lightweight history each epoch so progress is not lost.
    history_payload = {
        "run_name": RUN_NAME,
        "epoch_completed": epoch + 1,
        "EPOCHS": EPOCHS,
        "train_losses": train_losses,
        "val_losses": val_losses,
        "train_r_losses": train_r_losses,
        "train_z_losses": train_z_losses,
        "train_pnuc_losses": train_pnuc_losses,
        "val_r_losses": val_r_losses,
        "val_z_losses": val_z_losses,
        "val_pnuc_losses": val_pnuc_losses,
        "recovery_history": recovery_history,
        "best_val_loss": best_val_loss,
        "best_coord_nn": best_coord_nn,
        "best_model_path": LOCAL_BEST_MODEL_PATH,
        "best_coord_model_path": BEST_COORD_MODEL_PATH,
    }

    with open(TRAINING_HISTORY_PATH, "w") as f:
        json.dump(history_payload, f, indent=2)

    if recovery_history:
        with open(RECOVERY_HISTORY_PATH, "w") as f:
            json.dump(recovery_history, f, indent=2)

    # --------------------------------------------------------------------------
    # Console logging
    # --------------------------------------------------------------------------
    if (epoch + 1) % 5 == 0 or epoch == 0 or recovery_metrics is not None:
        rec_msg = ""

        if recovery_metrics is not None:
            rec_msg = (
                f" | learned coordNN={recovery_metrics.get('learned_coord_nn', np.nan):.4f}"
                f" | gene={recovery_metrics.get('gene_emp_coord_nn', np.nan):.4f}"
                f" | ct_gene={recovery_metrics.get('ct_gene_emp_coord_nn', np.nan):.4f}"
                f" | spatial_kNN={recovery_metrics.get('spatial_knn_emp_coord_nn', np.nan):.4f}"
                f" | imp_vs_gene={100 * recovery_metrics.get('improvement_vs_gene_emp_coord_nn', np.nan):.2f}%"
                f" | imp_vs_ct_gene={100 * recovery_metrics.get('improvement_vs_ct_gene_emp_coord_nn', np.nan):.2f}%"
                f" | imp_vs_spatial={100 * recovery_metrics.get('improvement_vs_spatial_knn_emp_coord_nn', np.nan):.2f}%"
            )

        print(
            f"Epoch {epoch + 1:3d}/{EPOCHS} | "
            f"Train total/r/z/pnuc: "
            f"{avg_train:.4f}/{avg_train_r:.4f}/{avg_train_z:.4f}/{avg_train_p:.4f} | "
            f"Val total/r/z/pnuc: "
            f"{avg_val:.4f}/{avg_val_r:.4f}/{avg_val_z:.4f}/{avg_val_p:.4f} | "
            f"LR: {optimizer.param_groups[0]['lr']:.2e} | "
            f"{time.time() - epoch_start:.1f}s{marker}{rec_msg}"
        )

# ------------------------------------------------------------------------------
# 10. Finish training
# ------------------------------------------------------------------------------

print("\n" + "=" * 80)
print("TRAINING COMPLETE")
print("=" * 80)
print(f"Best validation loss: {best_val_loss:.6f}")
print(f"Best coordNN: {best_coord_nn:.6f}")
print(f"Best validation-loss model path: {LOCAL_BEST_MODEL_PATH}")
print(f"Best coordNN model path: {BEST_COORD_MODEL_PATH}")
print(f"Training history path: {TRAINING_HISTORY_PATH}")
print(f"Recovery history path: {RECOVERY_HISTORY_PATH}")
print(f"Total training time: {(time.time() - t0_train) / 60:.1f} min")

# Load best validation-loss model by default.
model.load_state_dict(torch.load(LOCAL_BEST_MODEL_PATH, map_location=device))
model.to(device)
model.eval()

print("Loaded best validation-loss model weights.")

In [ ]:
# ==============================================================================
# COSMX CHECKPOINT DIST
# Save model/report with empirical-baseline metrics
# ==============================================================================

import os
import json
import torch
import numpy as np
import pandas as pd
import gc

print("=" * 80)
print("COSMX CHECKPOINT DIST: Saving learned distribution model and baseline report")
print("=" * 80)

# ------------------------------------------------------------------------------
# CosMx paths and run name
# ------------------------------------------------------------------------------

if "CHECKPOINT_DIR" not in globals():
    CHECKPOINT_DIR = "/content/drive/MyDrive/diffusion/step4_cosmx/imputation"

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

RUN_NAME = globals().get(
    "RUN_NAME",
    "cosmx_9E_distribution_empirical_baselines"
)

# These should already exist from the training cell.
BEST_VAL_STATE_DICT_PATH = globals().get(
    "LOCAL_BEST_MODEL_PATH",
    os.path.join(CHECKPOINT_DIR, f"{RUN_NAME}_best_model.pt")
)

BEST_COORD_STATE_DICT_PATH = globals().get(
    "BEST_COORD_MODEL_PATH",
    os.path.join(CHECKPOINT_DIR, f"{RUN_NAME}_best_coordNN_model.pt")
)

# Save full checkpoints separately so we do NOT overwrite the plain state_dict files.
BEST_VAL_FULL_CKPT_PATH = os.path.join(
    CHECKPOINT_DIR,
    f"{RUN_NAME}_best_val_full_checkpoint.pt"
)

BEST_COORD_FULL_CKPT_PATH = os.path.join(
    CHECKPOINT_DIR,
    f"{RUN_NAME}_best_coordNN_full_checkpoint.pt"
)

REPORT_PATH = os.path.join(
    CHECKPOINT_DIR,
    f"{RUN_NAME}_training_report.txt"
)

RECOVERY_CSV_PATH = os.path.join(
    CHECKPOINT_DIR,
    f"{RUN_NAME}_recovery_history.csv"
)

CURVES_NPZ_PATH = os.path.join(
    CHECKPOINT_DIR,
    f"{RUN_NAME}_training_curves.npz"
)

SUMMARY_JSON_PATH = os.path.join(
    CHECKPOINT_DIR,
    f"{RUN_NAME}_checkpoint_summary.json"
)

print(f"RUN_NAME: {RUN_NAME}")
print(f"CHECKPOINT_DIR: {CHECKPOINT_DIR}")
print(f"Best validation-loss state_dict path: {BEST_VAL_STATE_DICT_PATH}")
print(f"Best coordNN state_dict path        : {BEST_COORD_STATE_DICT_PATH}")

# ------------------------------------------------------------------------------
# Required variables
# ------------------------------------------------------------------------------

required_vars = [
    "model",
    "device",
    "shared_genes",
    "unique_cell_types",
    "train_loader",
    "training_examples",
    "validation_examples",
    "train_losses",
    "val_losses",
    "train_r_losses",
    "train_z_losses",
    "train_pnuc_losses",
    "val_r_losses",
    "val_z_losses",
    "val_pnuc_losses",
    "recovery_history",
    "best_val_loss",
]

missing = [v for v in required_vars if v not in globals()]

if missing:
    raise NameError(
        "Missing required variables before checkpoint save:\n"
        + "\n".join(missing)
        + "\n\nRun the CosMx training cell first."
    )

# Optional variables with safe defaults
EPOCHS = globals().get("EPOCHS", len(train_losses))
LR = globals().get("LR", None)
ETA_MIN = globals().get("ETA_MIN", None)
P_DROP_MIN = globals().get("P_DROP_MIN", None)
P_DROP_MAX = globals().get("P_DROP_MAX", None)
RECOVERY_EVAL_EVERY = globals().get("RECOVERY_EVAL_EVERY", None)
RECOVERY_EVAL_MAX_ITEMS = globals().get("RECOVERY_EVAL_MAX_ITEMS", None)
RECOVERY_EVAL_BATCH_SIZE = globals().get("RECOVERY_EVAL_BATCH_SIZE", None)
best_coord_nn = globals().get("best_coord_nn", np.nan)

# ------------------------------------------------------------------------------
# Build shared checkpoint metadata
# ------------------------------------------------------------------------------

training_config = {
    "epochs": int(EPOCHS) if EPOCHS is not None else None,
    "batch_size": getattr(train_loader, "batch_size", None),
    "lr": LR,
    "eta_min": ETA_MIN,
    "coord_model": "beta_r_beta_z_bernoulli_pnuc_theta_from_context",
    "p_drop_min": P_DROP_MIN,
    "p_drop_max": P_DROP_MAX,
    "baselines": [
        "gene_empirical_distribution",
        "cell_type_gene_empirical_distribution",
        "spatial_knn_empirical",
    ],
    "n_train_examples": int(len(training_examples)),
    "n_val_examples": int(len(validation_examples)),
    "recovery_eval_every": RECOVERY_EVAL_EVERY,
    "recovery_eval_max_items": RECOVERY_EVAL_MAX_ITEMS,
    "recovery_eval_batch_size": RECOVERY_EVAL_BATCH_SIZE,
}

base_payload = {
    "platform": "CosMx",
    "architecture": "Conditional_Beta_Localization_Distribution_v1",
    "run_name": RUN_NAME,
    "n_genes": int(len(shared_genes)),
    "n_cell_types": int(len(unique_cell_types)),
    "gene_names": list(map(str, shared_genes)),
    "cell_type_names": list(map(str, unique_cell_types)),
    "training_config": training_config,
    "train_losses": [float(x) for x in train_losses],
    "val_losses": [float(x) for x in val_losses],
    "train_r_losses": [float(x) for x in train_r_losses],
    "train_z_losses": [float(x) for x in train_z_losses],
    "train_pnuc_losses": [float(x) for x in train_pnuc_losses],
    "val_r_losses": [float(x) for x in val_r_losses],
    "val_z_losses": [float(x) for x in val_z_losses],
    "val_pnuc_losses": [float(x) for x in val_pnuc_losses],
    "recovery_history": recovery_history,
    "best_val_loss": float(best_val_loss),
    "best_coord_nn": float(best_coord_nn) if np.isfinite(best_coord_nn) else None,
}

# ------------------------------------------------------------------------------
# Save full checkpoint for best validation-loss model
# ------------------------------------------------------------------------------

if os.path.exists(BEST_VAL_STATE_DICT_PATH):
    model.load_state_dict(torch.load(BEST_VAL_STATE_DICT_PATH, map_location=device))
    model.to(device)
    model.eval()
    print(f"\nLoaded best validation-loss state_dict from:\n  {BEST_VAL_STATE_DICT_PATH}")
else:
    print("\nWARNING: Best validation-loss state_dict not found.")
    print("Saving current model state as best-validation full checkpoint.")

best_val_payload = dict(base_payload)
best_val_payload["selection_metric"] = "best_validation_loss"
best_val_payload["selected_model_state_dict_path"] = BEST_VAL_STATE_DICT_PATH
best_val_payload["model_state_dict"] = model.state_dict()

torch.save(best_val_payload, BEST_VAL_FULL_CKPT_PATH)

print("\nSaved full checkpoint for best validation-loss model:")
print(f"  {BEST_VAL_FULL_CKPT_PATH}")

# ------------------------------------------------------------------------------
# Save full checkpoint for best coordNN model
# ------------------------------------------------------------------------------

if os.path.exists(BEST_COORD_STATE_DICT_PATH):
    model.load_state_dict(torch.load(BEST_COORD_STATE_DICT_PATH, map_location=device))
    model.to(device)
    model.eval()

    best_coord_payload = dict(base_payload)
    best_coord_payload["selection_metric"] = "best_learned_coord_nn"
    best_coord_payload["selected_model_state_dict_path"] = BEST_COORD_STATE_DICT_PATH
    best_coord_payload["model_state_dict"] = model.state_dict()

    torch.save(best_coord_payload, BEST_COORD_FULL_CKPT_PATH)

    print("\nSaved full checkpoint for best coordNN model:")
    print(f"  {BEST_COORD_FULL_CKPT_PATH}")
    if np.isfinite(best_coord_nn):
        print(f"  best_coord_nn: {best_coord_nn:.6f}")
else:
    print("\nWARNING: Best coordNN state_dict not found. Skipping coordNN full checkpoint.")

# ------------------------------------------------------------------------------
# Save recovery history as CSV
# ------------------------------------------------------------------------------

if recovery_history:
    recovery_df = pd.DataFrame(recovery_history)
    recovery_df.to_csv(RECOVERY_CSV_PATH, index=False)

    print("\nSaved recovery history CSV:")
    print(f"  {RECOVERY_CSV_PATH}")
else:
    recovery_df = pd.DataFrame()
    print("\nWARNING: recovery_history is empty. No recovery CSV saved.")

# ------------------------------------------------------------------------------
# Save training curves as NPZ
# ------------------------------------------------------------------------------

np.savez_compressed(
    CURVES_NPZ_PATH,
    train_losses=np.array(train_losses, dtype=np.float32),
    val_losses=np.array(val_losses, dtype=np.float32),
    train_r_losses=np.array(train_r_losses, dtype=np.float32),
    train_z_losses=np.array(train_z_losses, dtype=np.float32),
    train_pnuc_losses=np.array(train_pnuc_losses, dtype=np.float32),
    val_r_losses=np.array(val_r_losses, dtype=np.float32),
    val_z_losses=np.array(val_z_losses, dtype=np.float32),
    val_pnuc_losses=np.array(val_pnuc_losses, dtype=np.float32),
)

print("\nSaved training curves NPZ:")
print(f"  {CURVES_NPZ_PATH}")

# ------------------------------------------------------------------------------
# Save text report
# ------------------------------------------------------------------------------

with open(REPORT_PATH, "w") as f:
    f.write("=" * 80 + "\n")
    f.write("COSMX STEP 5 LOCALIZATION DISTRIBUTION MODEL TRAINING REPORT\n")
    f.write("=" * 80 + "\n\n")

    f.write(f"Run: {RUN_NAME}\n")
    f.write(f"Checkpoint directory: {CHECKPOINT_DIR}\n")
    f.write(f"Best validation total loss: {float(best_val_loss):.6f}\n")

    if np.isfinite(best_coord_nn):
        f.write(f"Best learned coordNN: {float(best_coord_nn):.6f}\n")

    f.write("\nSaved files:\n")
    f.write(f"  Best validation state_dict: {BEST_VAL_STATE_DICT_PATH}\n")
    f.write(f"  Best coordNN state_dict: {BEST_COORD_STATE_DICT_PATH}\n")
    f.write(f"  Best validation full checkpoint: {BEST_VAL_FULL_CKPT_PATH}\n")
    f.write(f"  Best coordNN full checkpoint: {BEST_COORD_FULL_CKPT_PATH}\n")
    f.write(f"  Recovery CSV: {RECOVERY_CSV_PATH}\n")
    f.write(f"  Curves NPZ: {CURVES_NPZ_PATH}\n\n")

    f.write("Training config:\n")
    for k, v in training_config.items():
        f.write(f"  {k}: {v}\n")

    f.write("\nEpoch losses:\n")
    f.write(
        "epoch\t"
        "train_total\ttrain_r\ttrain_z\ttrain_pnuc\t"
        "val_total\tval_r\tval_z\tval_pnuc\n"
    )

    for idx in range(len(train_losses)):
        f.write(
            f"{idx + 1}\t"
            f"{train_losses[idx]:.6f}\t"
            f"{train_r_losses[idx]:.6f}\t"
            f"{train_z_losses[idx]:.6f}\t"
            f"{train_pnuc_losses[idx]:.6f}\t"
            f"{val_losses[idx]:.6f}\t"
            f"{val_r_losses[idx]:.6f}\t"
            f"{val_z_losses[idx]:.6f}\t"
            f"{val_pnuc_losses[idx]:.6f}\n"
        )

    f.write("\nHeld-out recovery metrics:\n")
    f.write("Compared methods:\n")
    f.write("  learned\n")
    f.write("  gene_emp\n")
    f.write("  ct_gene_emp\n")
    f.write("  spatial_knn_emp\n\n")

    if recovery_history:
        keys = sorted([k for k in recovery_history[-1].keys() if k != "epoch"])
        f.write("epoch\t" + "\t".join(keys) + "\n")

        for rec in recovery_history:
            f.write(str(rec.get("epoch", "")))

            for k in keys:
                val = rec.get(k, np.nan)

                if isinstance(val, (int, np.integer)):
                    f.write(f"\t{val}")
                elif isinstance(val, (float, np.floating)):
                    f.write(f"\t{val:.6f}")
                else:
                    f.write(f"\t{val}")

            f.write("\n")
    else:
        f.write("No recovery metrics were recorded.\n")

print("\nSaved training report:")
print(f"  {REPORT_PATH}")

# ------------------------------------------------------------------------------
# Save compact JSON summary
# ------------------------------------------------------------------------------

summary = {
    "platform": "CosMx",
    "run_name": RUN_NAME,
    "checkpoint_dir": CHECKPOINT_DIR,
    "best_val_loss": float(best_val_loss),
    "best_coord_nn": float(best_coord_nn) if np.isfinite(best_coord_nn) else None,
    "n_epochs_completed": int(len(train_losses)),
    "n_train_examples": int(len(training_examples)),
    "n_val_examples": int(len(validation_examples)),
    "best_validation_state_dict_path": BEST_VAL_STATE_DICT_PATH,
    "best_coordNN_state_dict_path": BEST_COORD_STATE_DICT_PATH,
    "best_validation_full_checkpoint_path": BEST_VAL_FULL_CKPT_PATH,
    "best_coordNN_full_checkpoint_path": BEST_COORD_FULL_CKPT_PATH,
    "report_path": REPORT_PATH,
    "recovery_csv_path": RECOVERY_CSV_PATH,
    "curves_npz_path": CURVES_NPZ_PATH,
}

with open(SUMMARY_JSON_PATH, "w") as f:
    json.dump(summary, f, indent=2)

print("\nSaved checkpoint summary JSON:")
print(f"  {SUMMARY_JSON_PATH}")

# ------------------------------------------------------------------------------
# Final verification
# ------------------------------------------------------------------------------

expected_outputs = [
    BEST_VAL_FULL_CKPT_PATH,
    REPORT_PATH,
    CURVES_NPZ_PATH,
    SUMMARY_JSON_PATH,
]

if recovery_history:
    expected_outputs.append(RECOVERY_CSV_PATH)

if os.path.exists(BEST_COORD_STATE_DICT_PATH):
    expected_outputs.append(BEST_COORD_FULL_CKPT_PATH)

missing_outputs = [p for p in expected_outputs if not os.path.exists(p)]

if missing_outputs:
    raise FileNotFoundError(
        "Some checkpoint/report outputs were not saved:\n"
        + "\n".join(missing_outputs)
    )

gc.collect()

print("\nCheckpoint save complete.")
print("=" * 80)

In [ ]:
# ==============================================================================
# COSMX S5-DIST TRAINING CURVES
# Plot loss curves + empirical-baseline recovery curves
# ==============================================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("=" * 80)
print("COSMX S5-DIST: Plotting training curves")
print("=" * 80)

# ------------------------------------------------------------------------------
# CosMx paths
# ------------------------------------------------------------------------------

if "CHECKPOINT_DIR" not in globals():
    CHECKPOINT_DIR = "/content/drive/MyDrive/diffusion/step4_cosmx/imputation"

RUN_NAME = globals().get(
    "RUN_NAME",
    "cosmx_9E_distribution_empirical_baselines"
)

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

CURVE_PATH = os.path.join(
    CHECKPOINT_DIR,
    f"{RUN_NAME}_training_curves.png"
)

RECOVERY_CURVE_PATH = os.path.join(
    CHECKPOINT_DIR,
    f"{RUN_NAME}_recovery_baseline_curves.png"
)

print(f"CHECKPOINT_DIR: {CHECKPOINT_DIR}")
print(f"RUN_NAME      : {RUN_NAME}")

# ------------------------------------------------------------------------------
# Required objects
# ------------------------------------------------------------------------------

required_vars = [
    "train_losses",
    "val_losses",
    "train_r_losses",
    "train_z_losses",
    "train_pnuc_losses",
    "val_r_losses",
    "val_z_losses",
    "val_pnuc_losses",
]

missing = [v for v in required_vars if v not in globals()]

if missing:
    raise NameError(
        "Missing required training-curve variables:\n"
        + "\n".join(missing)
        + "\n\nRun the CosMx training cell first."
    )

if len(train_losses) == 0 or len(val_losses) == 0:
    raise ValueError("Training/validation loss lists are empty.")

epochs = np.arange(1, len(train_losses) + 1)

# ------------------------------------------------------------------------------
# 1. Training / validation loss curves
# ------------------------------------------------------------------------------

fig, axes = plt.subplots(1, 4, figsize=(24, 4))

axes[0].plot(epochs, train_losses, label="Train total")
axes[0].plot(epochs, val_losses, label="Val total")
axes[0].set_title("Total distribution loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs, train_r_losses, label="Train r")
axes[1].plot(epochs, val_r_losses, label="Val r")
axes[1].set_title("r_norm Beta NLL")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

axes[2].plot(epochs, train_z_losses, label="Train z")
axes[2].plot(epochs, val_z_losses, label="Val z")
axes[2].set_title("z_rel Beta NLL")
axes[2].set_xlabel("Epoch")
axes[2].set_ylabel("Loss")
axes[2].legend()
axes[2].grid(True, alpha=0.3)

axes[3].plot(epochs, train_pnuc_losses, label="Train p_nuclear")
axes[3].plot(epochs, val_pnuc_losses, label="Val p_nuclear")
axes[3].set_title("p_nuclear BCE")
axes[3].set_xlabel("Epoch")
axes[3].set_ylabel("Loss")
axes[3].legend()
axes[3].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(CURVE_PATH, dpi=150)
plt.show()

print("\nSaved training loss curves:")
print(f"  {CURVE_PATH}")

# ------------------------------------------------------------------------------
# 2. Held-out recovery curves against empirical baselines
# ------------------------------------------------------------------------------

if "recovery_history" in globals() and len(recovery_history) > 0:
    rec_df = pd.DataFrame(recovery_history).sort_values("epoch")

    RECOVERY_CSV_PATH = os.path.join(
        CHECKPOINT_DIR,
        f"{RUN_NAME}_recovery_history.csv"
    )
    rec_df.to_csv(RECOVERY_CSV_PATH, index=False)

    fig, axes = plt.subplots(1, 5, figsize=(28, 4))

    metric_specs = [
        ("coord_nn", "coordNN ↓"),
        ("r_wasserstein", "r Wasserstein ↓"),
        ("z_wasserstein", "z Wasserstein ↓"),
        ("theta_distance", "Theta distance ↓"),
        ("radial_hist_l1", "Radial hist L1 ↓"),
    ]

    methods = [
        ("learned", "Learned"),
        ("gene_emp", "Gene empirical"),
        ("ct_gene_emp", "Cell-type gene empirical"),
        ("spatial_knn_emp", "Spatial-kNN empirical"),
    ]

    for ax, (metric_key, title) in zip(axes, metric_specs):
        for method_key, method_label in methods:
            col = f"{method_key}_{metric_key}"

            if col in rec_df.columns:
                ax.plot(
                    rec_df["epoch"],
                    rec_df[col],
                    marker="o",
                    label=method_label,
                )

        ax.set_title(title)
        ax.set_xlabel("Epoch")
        ax.set_ylabel("Lower is better")
        ax.grid(True, alpha=0.3)

    axes[0].legend(loc="best")

    plt.tight_layout()
    plt.savefig(RECOVERY_CURVE_PATH, dpi=150)
    plt.show()

    print("\nSaved recovery baseline curves:")
    print(f"  {RECOVERY_CURVE_PATH}")

    print("\nSaved recovery history CSV:")
    print(f"  {RECOVERY_CSV_PATH}")

    print("\nLatest held-out recovery metrics:")
    latest = rec_df.iloc[-1].to_dict()

    summary_rows = []

    for method_key, method_label in methods:
        row = {"method": method_label}

        for metric_key, _ in metric_specs:
            row[metric_key] = latest.get(f"{method_key}_{metric_key}", np.nan)

        summary_rows.append(row)

    recovery_summary_df = pd.DataFrame(summary_rows)
    display(recovery_summary_df)

    print("\nLatest learned improvement over baselines using coordNN:")

    for base in ["gene_emp", "ct_gene_emp", "spatial_knn_emp"]:
        k = f"improvement_vs_{base}_coord_nn"

        if k in latest:
            print(f"  learned vs {base}: {100 * latest[k]:.2f}%")

else:
    print("\nNo recovery_history found. Skipping empirical-baseline recovery curves.")

# ------------------------------------------------------------------------------
# 3. Optional quick overfitting summary
# ------------------------------------------------------------------------------

train_final = float(train_losses[-1])
val_final = float(val_losses[-1])
gap_final = val_final - train_final

best_val = float(np.min(val_losses))
best_val_epoch = int(np.argmin(val_losses) + 1)

print("\nTraining curve summary:")
print(f"  Final train loss: {train_final:.6f}")
print(f"  Final val loss  : {val_final:.6f}")
print(f"  Final val-train gap: {gap_final:.6f}")
print(f"  Best val loss   : {best_val:.6f} at epoch {best_val_epoch}")

if gap_final < 0.01:
    print("  Interpretation: no serious overfitting based on train/val loss gap.")
else:
    print("  Interpretation: possible overfitting; inspect recovery curves too.")

print("\nPASS: CosMx training/recovery curves generated.")
print("=" * 80)

In [ ]:
# ==============================================================================
# COSMX CHECK: Is it safe to disconnect?
# Checks that important Step 5 training/recovery files exist in Drive.
# ==============================================================================

import os

print("=" * 80)
print("COSMX CHECKING WHETHER IT IS SAFE TO DISCONNECT")
print("=" * 80)

if "CHECKPOINT_DIR" not in globals():
    CHECKPOINT_DIR = "/content/drive/MyDrive/diffusion/step4_cosmx/imputation"

RUN_NAME = globals().get(
    "RUN_NAME",
    "cosmx_9E_distribution_empirical_baselines"
)

print(f"CHECKPOINT_DIR: {CHECKPOINT_DIR}")
print(f"RUN_NAME      : {RUN_NAME}")

# ------------------------------------------------------------------------------
# Important files that should exist after your current Step 5 stage
# ------------------------------------------------------------------------------

files_to_check = [
    # Main molecule recovery files
    "checkpoint_mol_after_5A.parquet",
    "step5_mol_processed.parquet",
    "step5_mol_processed_summary.json",

    # CosMx geometry checkpoint files
    "checkpoint_cosmx_geometry_after_5A.pkl",
    "checkpoint_after_5A_summary.json",
    "cell_polygons.pkl",
    "nuc_polygons.pkl",
    "cell_areas.pkl",
    "nuc_areas.pkl",
    "nuc_centroids.pkl",
    "centroid_lookup.pkl",
    "cell_radius_lookup_px.pkl",
    "nuc_radius_lookup_px.pkl",

    # Model state_dict checkpoints from training cell
    f"{RUN_NAME}_best_model.pt",
    f"{RUN_NAME}_best_coordNN_model.pt",

    # Full checkpoints from checkpoint-report cell
    f"{RUN_NAME}_best_val_full_checkpoint.pt",
    f"{RUN_NAME}_best_coordNN_full_checkpoint.pt",

    # Training reports / histories / curves
    f"{RUN_NAME}_training_history.json",
    f"{RUN_NAME}_recovery_history.json",
    f"{RUN_NAME}_recovery_history.csv",
    f"{RUN_NAME}_training_report.txt",
    f"{RUN_NAME}_training_curves.npz",
    f"{RUN_NAME}_checkpoint_summary.json",

    # Figures
    f"{RUN_NAME}_training_curves.png",
    f"{RUN_NAME}_recovery_baseline_curves.png",
]

all_ok = True

print("\nChecking important files:")

for fname in files_to_check:
    path = os.path.join(CHECKPOINT_DIR, fname)

    if os.path.exists(path):
        print(f"✓ {fname:75s} {os.path.getsize(path) / 1e6:10.2f} MB")
    else:
        print(f"✗ MISSING: {fname}")
        all_ok = False

print("\n" + "-" * 80)

if all_ok:
    print("SAFE: Important CosMx Step 5 training/recovery files are saved to Drive.")
    print("You can disconnect the runtime if you do not need to continue immediately.")
else:
    print("NOT SAFE YET: Some expected files are missing.")
    print("If you have not run the checkpoint/report or plotting cells yet, run them first.")
    print("If a missing file is intentionally skipped, then this warning may be acceptable.")

print("=" * 80)

===========RAN TILL THIS\=================

In [ ]:
'''# ==============================================================================
# RECOVERY CELL FOR 9E FINAL LARGE EVALUATION
# Run this in a fresh runtime before running final 10k/20k recovery evaluation.
# ==============================================================================

import os
import gc
import json
import pickle
import warnings
from collections import defaultdict

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.distributions import Beta

from scipy import sparse
import anndata as ad
from sklearn.neighbors import BallTree

warnings.filterwarnings("ignore")

# ------------------------------------------------------------------------------
# 0. Mount Google Drive
# ------------------------------------------------------------------------------

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

print("=" * 90)
print("9E FINAL-EVAL RECOVERY: loading files from Drive")
print("=" * 90)

# ------------------------------------------------------------------------------
# 1. Paths
# ------------------------------------------------------------------------------

STEP4_EXPORT_DIR = "/content/drive/MyDrive/diffusion/step4_exports"
CHECKPOINT_DIR = "/content/drive/MyDrive/diffusion/latest_run"

RUN_NAME = "attempt_9E_distribution_empirical_baselines"

POST_PRUNE_MOL_PATH = os.path.join(CHECKPOINT_DIR, "step5_mol_processed.parquet")

DENOISED_ADATA_PATH = os.path.join(STEP4_EXPORT_DIR, "denoised_adata.h5ad")
STEP4_CONFIG_PATH = os.path.join(STEP4_EXPORT_DIR, "step4_config.json")
WAS_CORRECTED_PATH = os.path.join(STEP4_EXPORT_DIR, "was_corrected.npy")
CELL_DATA_PATH = os.path.join(STEP4_EXPORT_DIR, "cell_data.npz")

CELL_POLYGONS_PATH = os.path.join(CHECKPOINT_DIR, "cell_polygons.pkl")
NUC_POLYGONS_PATH = os.path.join(CHECKPOINT_DIR, "nuc_polygons.pkl")
CELL_AREAS_PATH = os.path.join(CHECKPOINT_DIR, "cell_areas.pkl")
NUC_AREAS_PATH = os.path.join(CHECKPOINT_DIR, "nuc_areas.pkl")
NUC_CENTROIDS_PATH = os.path.join(CHECKPOINT_DIR, "nuc_centroids.pkl")

BEST_VAL_MODEL_PATH = os.path.join(CHECKPOINT_DIR, f"{RUN_NAME}_best_model.pt")
BEST_COORD_MODEL_PATH = os.path.join(CHECKPOINT_DIR, f"{RUN_NAME}_best_coordNN_model.pt")
RECOVERY_HISTORY_PATH = os.path.join(CHECKPOINT_DIR, f"{RUN_NAME}_recovery_history.csv")
TRAINING_CURVES_PATH = os.path.join(CHECKPOINT_DIR, f"{RUN_NAME}_training_curves.npz")
TRAINING_REPORT_PATH = os.path.join(CHECKPOINT_DIR, f"{RUN_NAME}_training_report.txt")

required_files = {
    "processed Step 5 molecule table": POST_PRUNE_MOL_PATH,
    "Step 4 denoised AnnData": DENOISED_ADATA_PATH,
    "Step 4 config": STEP4_CONFIG_PATH,
    "Step 4 was_corrected": WAS_CORRECTED_PATH,
    "Step 4 compact cell_data": CELL_DATA_PATH,
    "cell polygons": CELL_POLYGONS_PATH,
    "nucleus polygons": NUC_POLYGONS_PATH,
    "cell areas": CELL_AREAS_PATH,
    "nucleus areas": NUC_AREAS_PATH,
    "nucleus centroids": NUC_CENTROIDS_PATH,
    "9E best validation-loss checkpoint": BEST_VAL_MODEL_PATH,
    "9E best coordNN checkpoint": BEST_COORD_MODEL_PATH,
    "9E recovery history CSV": RECOVERY_HISTORY_PATH,
    "9E training curves NPZ": TRAINING_CURVES_PATH,
    "9E training report": TRAINING_REPORT_PATH,
}

print("\nFiles expected from Drive:")
all_ok = True
for label, path in required_files.items():
    exists = os.path.exists(path)
    if exists:
        print(f"  ✓ {label:38s}: {path}  ({os.path.getsize(path) / 1e6:.2f} MB)")
    else:
        print(f"  ✗ MISSING {label:30s}: {path}")
        all_ok = False

if not all_ok:
    raise FileNotFoundError(
        "Some required recovery files are missing. Check the paths above before continuing."
    )

# ------------------------------------------------------------------------------
# 2. Helper
# ------------------------------------------------------------------------------

def ensure_dense(X):
    if sparse.issparse(X):
        return X.toarray()
    return np.asarray(X)

# ------------------------------------------------------------------------------
# 3. Load Step 5 molecule table
# ------------------------------------------------------------------------------

print("\n" + "=" * 90)
print("1. Loading processed Step 5 molecule table")
print("=" * 90)

print(f"Loading: {POST_PRUNE_MOL_PATH}")
mol = pd.read_parquet(POST_PRUNE_MOL_PATH)

# Compatibility alias, because some older cells may use molecules.
molecules = mol

print(f"mol shape: {mol.shape}")
print(f"mol columns: {list(mol.columns)}")
print("\nMolecule status counts:")
display(mol["status"].value_counts(dropna=False).reset_index().rename(
    columns={"index": "status", "status": "n_molecules"}
))

# ------------------------------------------------------------------------------
# 4. Load Step 4 AnnData and config/state
# ------------------------------------------------------------------------------

print("\n" + "=" * 90)
print("2. Loading Step 4 AnnData/config/state")
print("=" * 90)

print(f"Loading: {DENOISED_ADATA_PATH}")
denoised_adata = ad.read_h5ad(DENOISED_ADATA_PATH)

print(f"Loading: {STEP4_CONFIG_PATH}")
with open(STEP4_CONFIG_PATH, "r") as f:
    step4_config = json.load(f)

print(f"Loading: {WAS_CORRECTED_PATH}")
was_corrected = np.load(WAS_CORRECTED_PATH)

print(f"denoised_adata shape: {denoised_adata.shape}")
print(f"denoised_adata layers: {list(denoised_adata.layers.keys())}")
print(f"was_corrected shape: {was_corrected.shape}")

shared_genes = list(denoised_adata.var_names.astype(str))
cell_ids_step4 = np.array([int(x) for x in denoised_adata.obs_names.astype(str)])

gene_to_col = {g: i for i, g in enumerate(shared_genes)}
step4_cell_to_row = {int(cid): i for i, cid in enumerate(cell_ids_step4)}

# These are not strictly needed for final recovery evaluation, but useful for consistency.
X_denoised = ensure_dense(denoised_adata.X).astype(np.float32)

if "raw_molecule_counts_clean" in denoised_adata.layers:
    X_raw_counts = ensure_dense(denoised_adata.layers["raw_molecule_counts_clean"]).astype(np.float32)
elif "raw" in denoised_adata.layers:
    X_raw_counts = ensure_dense(denoised_adata.layers["raw"]).astype(np.float32)
else:
    print("WARNING: no raw count layer found in denoised_adata. Rebuilding from mol.")
    X_raw_counts = None

# Cell type indices.
if "cell_type" not in denoised_adata.obs.columns:
    raise KeyError("denoised_adata.obs must contain 'cell_type'.")

cell_type_labels = denoised_adata.obs["cell_type"].astype(str).values
unique_cell_types = sorted(pd.unique(cell_type_labels))
ct_to_idx = {ct: i for i, ct in enumerate(unique_cell_types)}
cell_type_indices = np.array([ct_to_idx[ct] for ct in cell_type_labels], dtype=np.int64)

print(f"shared_genes: {len(shared_genes)}")
print(f"cell_ids_step4: {len(cell_ids_step4):,}")
print(f"unique_cell_types: {len(unique_cell_types)}")
print(f"X_denoised shape: {X_denoised.shape}")
if X_raw_counts is not None:
    print(f"X_raw_counts shape: {X_raw_counts.shape}")
    print(f"X_raw_counts sum: {X_raw_counts.sum():,.0f}")

# ------------------------------------------------------------------------------
# 5. Load geometry support objects
# ------------------------------------------------------------------------------

print("\n" + "=" * 90)
print("3. Loading geometry objects")
print("=" * 90)

print(f"Loading: {CELL_POLYGONS_PATH}")
with open(CELL_POLYGONS_PATH, "rb") as f:
    cell_polygons = pickle.load(f)

print(f"Loading: {NUC_POLYGONS_PATH}")
with open(NUC_POLYGONS_PATH, "rb") as f:
    nuc_polygons = pickle.load(f)

print(f"Loading: {CELL_AREAS_PATH}")
with open(CELL_AREAS_PATH, "rb") as f:
    cell_areas = pickle.load(f)

print(f"Loading: {NUC_AREAS_PATH}")
with open(NUC_AREAS_PATH, "rb") as f:
    nuc_areas = pickle.load(f)

print(f"Loading: {NUC_CENTROIDS_PATH}")
with open(NUC_CENTROIDS_PATH, "rb") as f:
    nuc_centroids = pickle.load(f)

print(f"cell_polygons: {len(cell_polygons):,}")
print(f"nuc_polygons : {len(nuc_polygons):,}")
print(f"cell_areas   : {len(cell_areas):,}")
print(f"nuc_areas    : {len(nuc_areas):,}")
print(f"nuc_centroids: {len(nuc_centroids):,}")

print(f"Loading: {CELL_DATA_PATH}")
cell_data = np.load(CELL_DATA_PATH, allow_pickle=True)
cell_ids_geom = cell_data["cell_ids"]
centroids = cell_data["centroids"]

print(f"cell_ids_geom shape: {cell_ids_geom.shape}")
print(f"centroids shape    : {centroids.shape}")

# ------------------------------------------------------------------------------
# 6. Rebuild observed examples and empirical baseline pools
# ------------------------------------------------------------------------------

print("\n" + "=" * 90)
print("4. Rebuilding validation examples and empirical baseline pools")
print("=" * 90)

P_DROP_MIN = 0.15
P_DROP_MAX = 0.40
MIN_MOL_FOR_TRAINING = 6

mol["status"] = mol["status"].astype(str)
mol_clean = mol[mol["status"] == "observed"].copy()

required_cols = [
    "cell_id",
    "gene_id",
    "r_norm",
    "theta",
    "z_rel",
    "p_nuclear",
    "Assigned_Xenium_Cell_Type",
]
missing_cols = [c for c in required_cols if c not in mol_clean.columns]
if missing_cols:
    raise KeyError(f"mol is missing required columns: {missing_cols}")

print(f"Observed molecules used for examples/baselines: {len(mol_clean):,}")

# Cell coordinate lookup.
cell_xy_lookup = {}

if {"x_centroid", "y_centroid"}.issubset(set(denoised_adata.obs.columns)):
    print("Using denoised_adata.obs x_centroid/y_centroid for cell coordinates.")
    for cid, x, y in zip(
        denoised_adata.obs_names.astype(str),
        denoised_adata.obs["x_centroid"].values,
        denoised_adata.obs["y_centroid"].values,
    ):
        cell_xy_lookup[int(cid)] = np.array([float(x), float(y)], dtype=np.float32)
else:
    print("Using cell_ids_geom/centroids for cell coordinates.")
    for cid, xy in zip(cell_ids_geom, centroids):
        cell_xy_lookup[int(cid)] = np.asarray(xy, dtype=np.float32)

examples_by_pair = []

print("Building examples from observed molecules...")
for (cid, gid), group in mol_clean.groupby(["cell_id", "gene_id"], sort=False):
    n = len(group)
    if n < MIN_MOL_FOR_TRAINING:
        continue

    cid_int = int(cid)

    if cid_int not in step4_cell_to_row:
        continue
    if gid not in gene_to_col:
        continue
    if cid_int not in cell_xy_lookup:
        continue

    row = step4_cell_to_row[cid_int]
    gene_idx = gene_to_col[gid]
    ct_idx = int(cell_type_indices[row])

    c_area = float(cell_areas.get(cid_int, 100.0))
    n_area = float(nuc_areas.get(cid_int, 30.0))

    values = group[["r_norm", "theta", "z_rel", "p_nuclear"]].values.astype(np.float32)

    examples_by_pair.append({
        "cell_id": cid_int,
        "gene_id": str(gid),
        "gene_idx": gene_idx,
        "ct_idx": ct_idx,
        "cell_xy": cell_xy_lookup[cid_int],
        "cell_area": c_area,
        "nuc_area": n_area,
        "values": values,
    })

print(f"Eligible observed (cell, gene) pairs: {len(examples_by_pair):,}")

# Same deterministic split as 9E training.
all_cells = sorted({ex["cell_id"] for ex in examples_by_pair})
rng_split = np.random.default_rng(42)
rng_split.shuffle(all_cells)

n_val = max(1, int(0.10 * len(all_cells)))
val_cells = set(all_cells[:n_val])
train_cells = set(all_cells[n_val:])

training_examples = [ex for ex in examples_by_pair if ex["cell_id"] in train_cells]
validation_examples = [ex for ex in examples_by_pair if ex["cell_id"] in val_cells]

print(f"Train cells: {len(train_cells):,}")
print(f"Validation cells: {len(val_cells):,}")
print(f"Training examples: {len(training_examples):,}")
print(f"Validation examples: {len(validation_examples):,}")

# Baseline pools from training examples only.
print("Building empirical baseline pools from training examples only...")

gene_pool_lists = defaultdict(list)
ct_gene_pool_lists = defaultdict(list)
spatial_entry_lists = defaultdict(list)

for ex in training_examples:
    gene_idx = int(ex["gene_idx"])
    ct_idx = int(ex["ct_idx"])
    values = ex["values"].astype(np.float32)

    gene_pool_lists[gene_idx].append(values)
    ct_gene_pool_lists[(ct_idx, gene_idx)].append(values)

    spatial_entry_lists[(ct_idx, gene_idx)].append({
        "cell_id": int(ex["cell_id"]),
        "xy": ex["cell_xy"].astype(np.float32),
        "values": values,
    })

baseline_gene_pools = {
    gene_idx: np.concatenate(arrs, axis=0).astype(np.float32)
    for gene_idx, arrs in gene_pool_lists.items()
}

baseline_ct_gene_pools = {
    key: np.concatenate(arrs, axis=0).astype(np.float32)
    for key, arrs in ct_gene_pool_lists.items()
}

baseline_spatial_index = {}
for key, entries in spatial_entry_lists.items():
    cell_ids_arr = np.array([e["cell_id"] for e in entries], dtype=np.int64)
    xy_arr = np.stack([e["xy"] for e in entries], axis=0).astype(np.float32)
    values_list = [e["values"].astype(np.float32) for e in entries]

    baseline_spatial_index[key] = {
        "cell_ids": cell_ids_arr,
        "xy": xy_arr,
        "values": values_list,
    }

print(f"Gene empirical pools: {len(baseline_gene_pools):,}")
print(f"Cell-type gene empirical pools: {len(baseline_ct_gene_pools):,}")
print(f"Spatial-kNN empirical index groups: {len(baseline_spatial_index):,}")

# Free the huge filtered copy.
del mol_clean
gc.collect()

# ------------------------------------------------------------------------------
# 7. Define 9E model architecture
# ------------------------------------------------------------------------------

print("\n" + "=" * 90)
print("5. Defining 9E model architecture")
print("=" * 90)

class PointNetContextEncoder(nn.Module):
    def __init__(self, in_dim=5, hidden=192, out_dim=192):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.SiLU(),
            nn.Linear(hidden, hidden), nn.SiLU(),
            nn.Linear(hidden, out_dim),
        )
        self.out_proj = nn.Sequential(
            nn.Linear(out_dim * 2, out_dim),
            nn.SiLU(),
            nn.Linear(out_dim, out_dim),
        )

    def forward(self, points, mask):
        h = self.mlp(points)
        mask_exp = mask.unsqueeze(-1)

        h_masked = h.masked_fill(mask_exp == 0, -1e4)
        max_pool = h_masked.max(dim=1).values
        max_pool = torch.where(torch.isfinite(max_pool), max_pool, torch.zeros_like(max_pool))

        denom = mask_exp.sum(dim=1).clamp_min(1.0)
        mean_pool = (h * mask_exp).sum(dim=1) / denom

        return self.out_proj(torch.cat([max_pool, mean_pool], dim=-1))


class LocalizationDistributionModel(nn.Module):
    def __init__(self, n_genes, n_cell_types, ctx_dim=192, gene_dim=96,
                 ct_dim=32, geom_dim=9, geom_hidden=64, hidden=384):
        super().__init__()

        self.context_encoder = PointNetContextEncoder(in_dim=5, hidden=192, out_dim=ctx_dim)
        self.gene_embed = nn.Embedding(n_genes, gene_dim)
        self.ct_embed = nn.Embedding(n_cell_types, ct_dim)

        self.geom_mlp = nn.Sequential(
            nn.Linear(geom_dim, geom_hidden), nn.SiLU(),
            nn.Linear(geom_hidden, geom_hidden), nn.SiLU(),
        )

        fused = ctx_dim + gene_dim + ct_dim + geom_hidden

        self.backbone = nn.Sequential(
            nn.Linear(fused, hidden), nn.SiLU(), nn.Dropout(0.08),
            nn.Linear(hidden, hidden), nn.SiLU(), nn.Dropout(0.08),
            nn.Linear(hidden, hidden), nn.SiLU(),
        )

        self.r_head = nn.Linear(hidden, 2)
        self.z_head = nn.Linear(hidden, 2)
        self.pnuc_head = nn.Linear(hidden, 1)
        self.theta_head = nn.Linear(hidden, 2)

    def forward(self, context, context_mask, gene_idx, ct_idx, geom):
        h_ctx = self.context_encoder(context, context_mask)
        h_gene = self.gene_embed(gene_idx)
        h_ct = self.ct_embed(ct_idx)
        h_geom = self.geom_mlp(geom)

        h = self.backbone(torch.cat([h_ctx, h_gene, h_ct, h_geom], dim=-1))

        r_ab = F.softplus(self.r_head(h)) + 1.05
        z_ab = F.softplus(self.z_head(h)) + 1.05
        pnuc_logit = self.pnuc_head(h).squeeze(-1)
        theta_vec = self.theta_head(h)

        return {
            "r_alpha": r_ab[:, 0],
            "r_beta": r_ab[:, 1],
            "z_alpha": z_ab[:, 0],
            "z_beta": z_ab[:, 1],
            "pnuc_logit": pnuc_logit,
            "theta_vec": theta_vec,
        }

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = LocalizationDistributionModel(
    n_genes=len(shared_genes),
    n_cell_types=len(unique_cell_types),
).to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {n_params:,} ({n_params * 4 / 1e6:.1f} MB)")
print(f"Device: {device}")

# ------------------------------------------------------------------------------
# 8. Load best 9E coordNN checkpoint
# ------------------------------------------------------------------------------

print("\n" + "=" * 90)
print("6. Loading best 9E coordNN checkpoint")
print("=" * 90)

print(f"Loading checkpoint: {BEST_COORD_MODEL_PATH}")
ckpt = torch.load(BEST_COORD_MODEL_PATH, map_location=device)

if isinstance(ckpt, dict) and "model_state_dict" in ckpt:
    model.load_state_dict(ckpt["model_state_dict"])
    print("Loaded model_state_dict from full checkpoint payload.")
    if "best_coord_nn" in ckpt:
        print(f"Checkpoint best_coord_nn: {ckpt['best_coord_nn']}")
    if "selection_metric" in ckpt:
        print(f"Checkpoint selection_metric: {ckpt['selection_metric']}")
else:
    model.load_state_dict(ckpt)
    print("Loaded raw model state_dict.")

model.to(device)
model.eval()

print("\n" + "=" * 90)
print("9E RECOVERY COMPLETE")
print("=" * 90)
print("Ready for final large evaluation.")
print(f"Use validation_examples: {len(validation_examples):,}")
print(f"Use baseline_gene_pools: {len(baseline_gene_pools):,}")
print(f"Use baseline_ct_gene_pools: {len(baseline_ct_gene_pools):,}")
print(f"Use baseline_spatial_index: {len(baseline_spatial_index):,}")
print(f"Device for final eval: {device}")'''

In [ ]:
# ==============================================================================
# COSMX FINAL LARGE RECOVERY EVALUATION FOR 9E
# Run this after CosMx model training + checkpoint saving.
# ==============================================================================

import os
import time
import json
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.distributions import Beta
from sklearn.neighbors import BallTree

print("=" * 90)
print("COSMX FINAL LARGE RECOVERY EVALUATION — 9E")
print("=" * 90)

# ------------------------------------------------------------------------------
# 0. Required variables
# ------------------------------------------------------------------------------

required_vars = [
    "model",
    "device",
    "validation_examples",
    "shared_genes",
    "unique_cell_types",
]

missing = [v for v in required_vars if v not in globals()]
if missing:
    raise NameError(
        "Missing required variable(s):\n"
        + "\n".join(missing)
        + "\n\nRun the CosMx recovery/model/training cells first."
    )

# Baseline pools may come from either S5-6-DIST or S5-8-DIST.
# Prefer the names created by the final training cell if available.
if "gene_empirical_pools" in globals():
    FINAL_GENE_POOLS = gene_empirical_pools
elif "baseline_gene_pools" in globals():
    FINAL_GENE_POOLS = baseline_gene_pools
else:
    raise NameError("Missing gene empirical pools: gene_empirical_pools or baseline_gene_pools")

if "ct_gene_empirical_pools" in globals():
    FINAL_CT_GENE_POOLS = ct_gene_empirical_pools
elif "baseline_ct_gene_pools" in globals():
    FINAL_CT_GENE_POOLS = baseline_ct_gene_pools
else:
    raise NameError("Missing cell-type gene empirical pools.")

# Spatial baseline source.
# Prefer training-cell dictionary if it exists; otherwise use baseline_spatial_index from S5-6.
HAS_TRAIN_SPATIAL_DICT = (
    "train_cell_gene_values" in globals()
    and "geom_cell_to_centroid" in globals()
    and "train_spatial_cells" in globals()
    and "train_spatial_coords" in globals()
)

HAS_BASELINE_SPATIAL_INDEX = "baseline_spatial_index" in globals()

if not HAS_TRAIN_SPATIAL_DICT and not HAS_BASELINE_SPATIAL_INDEX:
    raise NameError(
        "Missing spatial baseline structures. Need either training-cell spatial dictionaries "
        "or baseline_spatial_index."
    )

# ------------------------------------------------------------------------------
# 1. Paths and configuration
# ------------------------------------------------------------------------------

if "CHECKPOINT_DIR" not in globals():
    CHECKPOINT_DIR = "/content/drive/MyDrive/diffusion/step4_cosmx/imputation"

RUN_NAME = globals().get(
    "RUN_NAME",
    "cosmx_9E_distribution_empirical_baselines"
)

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

BEST_COORD_MODEL_PATH = globals().get(
    "BEST_COORD_MODEL_PATH",
    os.path.join(CHECKPOINT_DIR, f"{RUN_NAME}_best_coordNN_model.pt")
)

BEST_VAL_MODEL_PATH = globals().get(
    "LOCAL_BEST_MODEL_PATH",
    os.path.join(CHECKPOINT_DIR, f"{RUN_NAME}_best_model.pt")
)

FINAL_EVAL_CSV_PATH = os.path.join(
    CHECKPOINT_DIR,
    f"{RUN_NAME}_final_large_recovery_metrics.csv"
)

FINAL_EVAL_JSON_PATH = os.path.join(
    CHECKPOINT_DIR,
    f"{RUN_NAME}_final_large_recovery_metrics.json"
)

FINAL_EVAL_REPORT_PATH = os.path.join(
    CHECKPOINT_DIR,
    f"{RUN_NAME}_final_large_recovery_report.txt"
)

# Recommended:
#   10,000 = faster
#   20,000 = stronger final estimate
FINAL_RECOVERY_EVAL_MAX_ITEMS = 45_000
FINAL_RECOVERY_EVAL_BATCH_SIZE = globals().get("FINAL_RECOVERY_EVAL_BATCH_SIZE", 256)
FINAL_EVAL_SEED = globals().get("FINAL_EVAL_SEED", 22345)

P_DROP_MIN = globals().get("P_DROP_MIN", 0.15)
P_DROP_MAX = globals().get("P_DROP_MAX", 0.40)

SPATIAL_KNN_K = globals().get("SPATIAL_KNN_K", 80)
SPATIAL_KNN_MIN_POOL = globals().get("SPATIAL_KNN_MIN_POOL", 5)

print(f"CHECKPOINT_DIR             : {CHECKPOINT_DIR}")
print(f"RUN_NAME                   : {RUN_NAME}")
print(f"Validation examples        : {len(validation_examples):,}")
print(f"Final eval max items       : {FINAL_RECOVERY_EVAL_MAX_ITEMS:,}")
print(f"Final eval batch size      : {FINAL_RECOVERY_EVAL_BATCH_SIZE}")
print(f"Device                     : {device}")
print(f"P_DROP range               : {P_DROP_MIN:.2f}–{P_DROP_MAX:.2f}")

# ------------------------------------------------------------------------------
# 2. Load best coordNN checkpoint for final localization evaluation
# ------------------------------------------------------------------------------

print("\n" + "=" * 90)
print("1. Loading best coordNN checkpoint")
print("=" * 90)

loaded_checkpoint_path = None

if os.path.exists(BEST_COORD_MODEL_PATH):
    loaded_checkpoint_path = BEST_COORD_MODEL_PATH
    print(f"Loading best coordNN state_dict:")
    print(f"  {BEST_COORD_MODEL_PATH}")

    ckpt = torch.load(BEST_COORD_MODEL_PATH, map_location=device)

elif os.path.exists(BEST_VAL_MODEL_PATH):
    loaded_checkpoint_path = BEST_VAL_MODEL_PATH
    print("WARNING: best coordNN model not found.")
    print("Falling back to best validation-loss state_dict:")
    print(f"  {BEST_VAL_MODEL_PATH}")

    ckpt = torch.load(BEST_VAL_MODEL_PATH, map_location=device)

else:
    ckpt = None
    print("WARNING: no saved model checkpoint found.")
    print("Using currently loaded model.")

if ckpt is not None:
    if isinstance(ckpt, dict) and "model_state_dict" in ckpt:
        model.load_state_dict(ckpt["model_state_dict"])
        print("Loaded model_state_dict from full checkpoint payload.")
        if "best_coord_nn" in ckpt:
            print(f"Checkpoint best_coord_nn: {ckpt['best_coord_nn']}")
        if "selection_metric" in ckpt:
            print(f"Selection metric: {ckpt['selection_metric']}")
    else:
        model.load_state_dict(ckpt)
        print("Loaded raw state_dict checkpoint.")

model.to(device)
model.eval()

# ------------------------------------------------------------------------------
# 3. Helper functions
# ------------------------------------------------------------------------------

def _features_np(vals):
    """
    Convert [r_norm, theta, z_rel, p_nuclear] into:
      [r_norm, sin(theta), cos(theta), z_rel, p_nuclear]
    """
    r = vals[:, 0]
    theta = vals[:, 1]
    z = vals[:, 2]
    p_nuc = vals[:, 3]

    return np.stack(
        [r, np.sin(theta), np.cos(theta), z, p_nuc],
        axis=-1
    ).astype(np.float32)


def make_eval_item_from_example(
    ex,
    idx,
    max_context=96,
    max_target=24,
    p_drop_min=0.15,
    p_drop_max=0.40,
    seed=12345,
):
    """
    Recreates the same validation hide/recover format used during training.
    geom has 9 features because the 9E model uses geom_dim=9.
    """
    vals = ex["values"]
    n = len(vals)

    rng = np.random.default_rng(seed + idx)
    p_drop = rng.uniform(p_drop_min, p_drop_max)

    k = int(rng.binomial(n, p_drop))
    k = max(1, min(n - 3, k, max_target))

    perm = rng.permutation(n)

    target_vals = vals[perm[:k]]
    context_vals = vals[perm[k:]]

    if len(context_vals) > max_context:
        ctx_idx = rng.choice(len(context_vals), size=max_context, replace=False)
        context_used = context_vals[ctx_idx]
    else:
        context_used = context_vals

    n_ctx = min(len(context_used), max_context)
    n_tgt = min(len(target_vals), max_target)

    ctx_padded = np.zeros((max_context, 5), dtype=np.float32)
    if n_ctx > 0:
        ctx_padded[:n_ctx] = _features_np(context_used[:n_ctx])

    ctx_mask = np.zeros(max_context, dtype=np.float32)
    ctx_mask[:n_ctx] = 1.0

    target_coords = np.zeros((max_target, 3), dtype=np.float32)
    target_pnuc = np.zeros(max_target, dtype=np.float32)
    target_mask = np.zeros(max_target, dtype=np.float32)

    target_coords[:n_tgt] = target_vals[:n_tgt, :3]
    target_pnuc[:n_tgt] = target_vals[:n_tgt, 3]
    target_mask[:n_tgt] = 1.0

    c_area = float(ex.get("cell_area", 100.0))
    n_area = float(ex.get("nuc_area", 30.0))

    if n_ctx > 0:
        ctx_features = ctx_padded[:n_ctx]

        ctx_r_mean = float(ctx_features[:, 0].mean())
        ctx_r_std = float(ctx_features[:, 0].std())

        ctx_z_mean = float(ctx_features[:, 3].mean())
        ctx_z_std = float(ctx_features[:, 3].std())

        ctx_pnuc_mean = float(ctx_features[:, 4].mean())
    else:
        ctx_r_mean = 0.5
        ctx_r_std = 0.0
        ctx_z_mean = 0.5
        ctx_z_std = 0.0
        ctx_pnuc_mean = 0.0

    geom = np.array([
        c_area / 500.0,
        n_area / 200.0,
        n_area / c_area if c_area > 0 else 0.3,
        n_ctx / 50.0,
        ctx_r_mean,
        ctx_r_std,
        ctx_z_mean,
        ctx_z_std,
        ctx_pnuc_mean,
    ], dtype=np.float32)

    return {
        "context": ctx_padded,
        "context_mask": ctx_mask,
        "target_coords": target_coords,
        "target_pnuc": target_pnuc,
        "target_mask": target_mask,
        "gene_idx": int(ex["gene_idx"]),
        "ct_idx": int(ex["ct_idx"]),
        "geom": geom,
        "ex": ex,
        "n_tgt": n_tgt,
    }


@torch.no_grad()
def sample_from_learned_distribution(outputs, target_mask, context=None, context_mask=None):
    """
    9E learned sampler:
      r_norm sampled from predicted Beta distribution
      z_rel sampled from predicted Beta distribution
      theta sampled around learned theta direction
    """
    dev = target_mask.device
    B, K = target_mask.shape

    r_dist = Beta(
        outputs["r_alpha"].clamp_min(1e-4),
        outputs["r_beta"].clamp_min(1e-4),
    )
    r = r_dist.sample((K,)).transpose(0, 1).to(dev).clamp(0.0, 1.0)

    z_dist = Beta(
        outputs["z_alpha"].clamp_min(1e-4),
        outputs["z_beta"].clamp_min(1e-4),
    )
    z = z_dist.sample((K,)).transpose(0, 1).to(dev).clamp(0.0, 1.0)

    theta_vec = F.normalize(outputs["theta_vec"], dim=-1)
    theta_mu = torch.atan2(theta_vec[:, 0], theta_vec[:, 1])

    theta = theta_mu.unsqueeze(1).expand(B, K).clone()

    theta_jitter = 0.20
    theta = theta + torch.randn((B, K), device=dev) * theta_jitter
    theta = ((theta + torch.pi) % (2 * torch.pi)) - torch.pi

    return torch.stack([r, theta, z], dim=-1)


def _sample_empirical_pool(pool, n_tgt, rng):
    if pool is None or len(pool) == 0:
        r = rng.uniform(0, 1, size=n_tgt)
        theta = rng.uniform(-np.pi, np.pi, size=n_tgt)
        z = rng.uniform(0, 1, size=n_tgt)
        return np.stack([r, theta, z], axis=1).astype(np.float32)

    idx = rng.choice(len(pool), size=n_tgt, replace=True)
    vals = pool[idx, :3].astype(np.float32)

    vals[:, 0] = np.clip(vals[:, 0], 0, 1)
    vals[:, 1] = np.arctan2(np.sin(vals[:, 1]), np.cos(vals[:, 1]))
    vals[:, 2] = np.clip(vals[:, 2], 0, 1)

    return vals


def sample_gene_empirical(ex, n_tgt, rng):
    gene_idx = int(ex["gene_idx"])
    pool = FINAL_GENE_POOLS.get(gene_idx, None)
    return _sample_empirical_pool(pool, n_tgt, rng)


def sample_ct_gene_empirical(ex, n_tgt, rng):
    key = (int(ex["ct_idx"]), int(ex["gene_idx"]))
    pool = FINAL_CT_GENE_POOLS.get(key, None)

    if pool is None or len(pool) == 0:
        return sample_gene_empirical(ex, n_tgt, rng)

    return _sample_empirical_pool(pool, n_tgt, rng)


# Cache BallTrees by (ct_idx, gene_idx) so we do not rebuild the same tree thousands of times.
_spatial_tree_cache = {}

def sample_spatial_knn_empirical(ex, n_tgt, rng, k_neighbors=80, min_pool=5):
    """
    CosMx-safe spatial-kNN empirical baseline.

    Uses either:
      1. train_cell_gene_values + geom_cell_to_centroid, or
      2. baseline_spatial_index from S5-6-DIST.
    """
    gid = int(ex["gene_idx"])
    ct = int(ex["ct_idx"])

    # --------------------------------------------------------------------------
    # Preferred version: training-cell dictionary from S5-8-DIST
    # --------------------------------------------------------------------------
    if HAS_TRAIN_SPATIAL_DICT:
        cid = str(ex["cell_id"])

        if "spatial_tree" in globals() and spatial_tree is not None and cid in geom_cell_to_centroid:
            query_coord = geom_cell_to_centroid[cid].reshape(1, -1)

            k = min(k_neighbors, len(train_spatial_cells))
            _, nn_idx = spatial_tree.query(query_coord, k=k)

            candidate_arrays = []

            for idx in nn_idx[0]:
                nb_cid = str(train_spatial_cells[idx])
                gene_dict = train_cell_gene_values.get(nb_cid, {})
                vals = gene_dict.get(gid, None)

                if vals is not None and len(vals) > 0:
                    candidate_arrays.append(vals)

            if len(candidate_arrays) > 0:
                pool = np.concatenate(candidate_arrays, axis=0).astype(np.float32)

                if len(pool) >= min_pool:
                    return _sample_empirical_pool(pool, n_tgt, rng)

        return sample_ct_gene_empirical(ex, n_tgt, rng)

    # --------------------------------------------------------------------------
    # Fallback version: baseline_spatial_index from S5-6-DIST
    # --------------------------------------------------------------------------
    key = (ct, gid)

    info = baseline_spatial_index.get(key, None)

    if info is None or len(info["cell_ids"]) == 0:
        return sample_ct_gene_empirical(ex, n_tgt, rng)

    xy = info["xy"]
    values_list = info["values"]

    query_xy = np.asarray(ex["cell_xy"], dtype=np.float32).reshape(1, -1)

    k = min(k_neighbors, len(xy))

    if k < 1:
        return sample_ct_gene_empirical(ex, n_tgt, rng)

    if key not in _spatial_tree_cache:
        _spatial_tree_cache[key] = BallTree(xy)

    tree = _spatial_tree_cache[key]
    _, ind = tree.query(query_xy, k=k)

    candidate_vals = []

    for idx in ind[0]:
        vals = values_list[int(idx)]
        if vals is not None and len(vals) > 0:
            candidate_vals.append(vals)

    if len(candidate_vals) < min_pool:
        return sample_ct_gene_empirical(ex, n_tgt, rng)

    pool = np.concatenate(candidate_vals, axis=0)

    if len(pool) == 0:
        return sample_ct_gene_empirical(ex, n_tgt, rng)

    return _sample_empirical_pool(pool, n_tgt, rng)


# ------------------------------------------------------------------------------
# 4. Metrics — use the SAME coordinate metric style as the training recovery cell
# ------------------------------------------------------------------------------

def _wasserstein_1d(a, b):
    if len(a) == 0 or len(b) == 0:
        return np.nan

    a = np.sort(np.asarray(a, dtype=np.float64))
    b = np.sort(np.asarray(b, dtype=np.float64))

    q = np.linspace(0, 1, max(len(a), len(b)))
    aq = np.interp(q, np.linspace(0, 1, len(a)), a)
    bq = np.interp(q, np.linspace(0, 1, len(b)), b)

    return float(np.mean(np.abs(aq - bq)))


def _theta_set_distance(pred_theta, true_theta):
    if len(pred_theta) == 0 or len(true_theta) == 0:
        return np.nan

    pred_theta = np.asarray(pred_theta, dtype=np.float64)
    true_theta = np.asarray(true_theta, dtype=np.float64)

    d = np.abs(
        np.arctan2(
            np.sin(pred_theta[:, None] - true_theta[None, :]),
            np.cos(pred_theta[:, None] - true_theta[None, :])
        )
    )

    return float(np.mean(np.min(d, axis=1)) / np.pi)


def compute_recovery_metrics_np(pred, target):
    """
    Same metric style used during the training recovery evaluation.

    Each coordinate array has shape [n_molecules, 3]:
      [r_norm, theta, z_rel]
    """
    matched_dists = []
    r_wass = []
    z_wass = []
    theta_dist = []
    hist_l1 = []

    for p, t in zip(pred, target):
        if len(p) == 0 or len(t) == 0:
            continue

        p = np.asarray(p, dtype=np.float32)
        t = np.asarray(t, dtype=np.float32)

        dr = p[:, None, 0] - t[None, :, 0]

        dtheta = np.arctan2(
            np.sin(p[:, None, 1] - t[None, :, 1]),
            np.cos(p[:, None, 1] - t[None, :, 1])
        ) / np.pi

        dz = p[:, None, 2] - t[None, :, 2]

        dist = np.sqrt(dr * dr + dtheta * dtheta + dz * dz)

        matched_dists.append(float(np.mean(np.min(dist, axis=1))))
        r_wass.append(_wasserstein_1d(p[:, 0], t[:, 0]))
        z_wass.append(_wasserstein_1d(p[:, 2], t[:, 2]))
        theta_dist.append(_theta_set_distance(p[:, 1], t[:, 1]))

        p_hist, _ = np.histogram(p[:, 0], bins=np.linspace(0, 1, 11), density=False)
        t_hist, _ = np.histogram(t[:, 0], bins=np.linspace(0, 1, 11), density=False)

        p_hist = p_hist / max(p_hist.sum(), 1)
        t_hist = t_hist / max(t_hist.sum(), 1)

        hist_l1.append(float(np.abs(p_hist - t_hist).sum()))

    return {
        "coord_nn": float(np.nanmean(matched_dists)) if matched_dists else np.nan,
        "r_wasserstein": float(np.nanmean(r_wass)) if r_wass else np.nan,
        "z_wasserstein": float(np.nanmean(z_wass)) if z_wass else np.nan,
        "theta_distance": float(np.nanmean(theta_dist)) if theta_dist else np.nan,
        "radial_hist_l1": float(np.nanmean(hist_l1)) if hist_l1 else np.nan,
    }


@torch.no_grad()
def evaluate_heldout_recovery_empirical_large(
    model,
    validation_examples,
    device,
    max_items=10_000,
    batch_size=256,
    seed=12345,
):
    """
    Large final evaluation of learned model against 3 empirical baselines.
    """
    model.eval()

    n_eval = min(max_items, len(validation_examples))

    learned_preds = []
    gene_preds = []
    ct_gene_preds = []
    spatial_knn_preds = []
    targets = []

    rng_eval = np.random.default_rng(seed)

    t0_eval = time.time()

    for start in range(0, n_eval, batch_size):
        end = min(start + batch_size, n_eval)

        items = [
            make_eval_item_from_example(
                validation_examples[i],
                idx=i,
                max_context=96,
                max_target=24,
                p_drop_min=P_DROP_MIN,
                p_drop_max=P_DROP_MAX,
                seed=seed,
            )
            for i in range(start, end)
        ]

        ctx = torch.tensor(
            np.stack([it["context"] for it in items]),
            dtype=torch.float32,
            device=device,
        )

        ctx_mask = torch.tensor(
            np.stack([it["context_mask"] for it in items]),
            dtype=torch.float32,
            device=device,
        )

        target_coords_t = torch.tensor(
            np.stack([it["target_coords"] for it in items]),
            dtype=torch.float32,
            device=device,
        )

        target_mask_t = torch.tensor(
            np.stack([it["target_mask"] for it in items]),
            dtype=torch.float32,
            device=device,
        )

        gene_idx_t = torch.tensor(
            [it["gene_idx"] for it in items],
            dtype=torch.long,
            device=device,
        )

        ct_idx_t = torch.tensor(
            [it["ct_idx"] for it in items],
            dtype=torch.long,
            device=device,
        )

        geom_t = torch.tensor(
            np.stack([it["geom"] for it in items]),
            dtype=torch.float32,
            device=device,
        )

        outputs = model(ctx, ctx_mask, gene_idx_t, ct_idx_t, geom_t)

        learned_sample_t = sample_from_learned_distribution(
            outputs,
            target_mask_t,
            ctx,
            ctx_mask,
        )

        learned_sample = learned_sample_t.detach().cpu().numpy()
        target_coords_np = target_coords_t.detach().cpu().numpy()
        target_mask_np = target_mask_t.detach().cpu().numpy()

        for local_i, it in enumerate(items):
            m = target_mask_np[local_i] > 0
            n_tgt = int(m.sum())

            if n_tgt <= 0:
                continue

            true_coords = target_coords_np[local_i, m, :]
            learned_coords = learned_sample[local_i, m, :]

            ex = it["ex"]

            gene_coords = sample_gene_empirical(ex, n_tgt, rng_eval)
            ct_gene_coords = sample_ct_gene_empirical(ex, n_tgt, rng_eval)
            spatial_coords = sample_spatial_knn_empirical(
                ex,
                n_tgt,
                rng_eval,
                k_neighbors=SPATIAL_KNN_K,
                min_pool=SPATIAL_KNN_MIN_POOL,
            )

            targets.append(true_coords)
            learned_preds.append(learned_coords)
            gene_preds.append(gene_coords)
            ct_gene_preds.append(ct_gene_coords)
            spatial_knn_preds.append(spatial_coords)

        if (end % 2048 == 0) or (end == n_eval):
            elapsed = (time.time() - t0_eval) / 60
            print(f"  evaluated {end:,}/{n_eval:,} examples ({elapsed:.1f} min elapsed)")

    metrics = {}

    all_methods = {
        "learned": learned_preds,
        "gene_emp": gene_preds,
        "ct_gene_emp": ct_gene_preds,
        "spatial_knn_emp": spatial_knn_preds,
    }

    for name, preds in all_methods.items():
        m = compute_recovery_metrics_np(preds, targets)

        for k, v in m.items():
            metrics[f"{name}_{k}"] = v

    learned_coord = metrics.get("learned_coord_nn", np.nan)

    for base in ["gene_emp", "ct_gene_emp", "spatial_knn_emp"]:
        base_coord = metrics.get(f"{base}_coord_nn", np.nan)
        metrics[f"improvement_vs_{base}_coord_nn"] = float(
            (base_coord - learned_coord) / (abs(base_coord) + 1e-8)
        )

    metrics["n_eval_examples"] = int(n_eval)
    metrics["n_eval_target_molecules"] = int(sum(len(t) for t in targets))
    metrics["seed"] = int(seed)
    metrics["checkpoint_loaded"] = str(loaded_checkpoint_path)

    return metrics


# ------------------------------------------------------------------------------
# 5. Run final large evaluation
# ------------------------------------------------------------------------------

print("\n" + "=" * 90)
print("2. Running final large evaluation")
print("=" * 90)

t0 = time.time()

final_metrics = evaluate_heldout_recovery_empirical_large(
    model=model,
    validation_examples=validation_examples,
    device=device,
    max_items=FINAL_RECOVERY_EVAL_MAX_ITEMS,
    batch_size=FINAL_RECOVERY_EVAL_BATCH_SIZE,
    seed=FINAL_EVAL_SEED,
)

elapsed = time.time() - t0

print("\n" + "=" * 90)
print("FINAL LARGE EVALUATION COMPLETE")
print("=" * 90)
print(f"Elapsed: {elapsed / 60:.1f} min")

# ------------------------------------------------------------------------------
# 6. Display final metrics
# ------------------------------------------------------------------------------

rows = []

method_map = {
    "learned": "Learned 9E",
    "gene_emp": "Gene empirical",
    "ct_gene_emp": "Cell-type gene empirical",
    "spatial_knn_emp": "Spatial-kNN empirical",
}

for key, label in method_map.items():
    rows.append({
        "method": label,
        "coord_nn": final_metrics.get(f"{key}_coord_nn", np.nan),
        "r_wasserstein": final_metrics.get(f"{key}_r_wasserstein", np.nan),
        "z_wasserstein": final_metrics.get(f"{key}_z_wasserstein", np.nan),
        "theta_distance": final_metrics.get(f"{key}_theta_distance", np.nan),
        "radial_hist_l1": final_metrics.get(f"{key}_radial_hist_l1", np.nan),
    })

final_df = pd.DataFrame(rows)

print("\nFinal large recovery metrics:")
display(final_df)

print("\nFinal learned improvement over baselines using coordNN:")
for base in ["gene_emp", "ct_gene_emp", "spatial_knn_emp"]:
    imp = final_metrics.get(f"improvement_vs_{base}_coord_nn", np.nan)
    print(f"  learned vs {base}: {100 * imp:.2f}%")

# ------------------------------------------------------------------------------
# 7. Save final results
# ------------------------------------------------------------------------------

final_df.to_csv(FINAL_EVAL_CSV_PATH, index=False)

# Save all metrics as JSON too.
with open(FINAL_EVAL_JSON_PATH, "w") as f:
    json.dump(final_metrics, f, indent=2)

with open(FINAL_EVAL_REPORT_PATH, "w") as f:
    f.write("COSMX FINAL LARGE RECOVERY EVALUATION — 9E\n")
    f.write("=" * 80 + "\n\n")
    f.write(f"RUN_NAME: {RUN_NAME}\n")
    f.write(f"CHECKPOINT_DIR: {CHECKPOINT_DIR}\n")
    f.write(f"checkpoint_loaded: {loaded_checkpoint_path}\n")
    f.write(f"n_eval_examples: {final_metrics['n_eval_examples']}\n")
    f.write(f"n_eval_target_molecules: {final_metrics['n_eval_target_molecules']}\n")
    f.write(f"seed: {final_metrics['seed']}\n")
    f.write(f"elapsed_min: {elapsed / 60:.2f}\n\n")

    f.write("Metrics:\n")
    f.write(final_df.to_string(index=False))
    f.write("\n\n")

    f.write("CoordNN improvements:\n")
    for base in ["gene_emp", "ct_gene_emp", "spatial_knn_emp"]:
        imp = final_metrics.get(f"improvement_vs_{base}_coord_nn", np.nan)
        f.write(f"  learned vs {base}: {100 * imp:.2f}%\n")

print("\nSaved final large evaluation files:")
print(f"  CSV   : {FINAL_EVAL_CSV_PATH}")
print(f"  JSON  : {FINAL_EVAL_JSON_PATH}")
print(f"  Report: {FINAL_EVAL_REPORT_PATH}")

print("\nPASS: CosMx final large recovery evaluation completed.")
print("=" * 90)

======EVALUATION ENDS======

In [ ]:
# ==============================================================================
# COSMX CELL S5-9E-INFER-1 — Prepare 9E imputation targets + summary-geometry cache
# Run this after final large evaluation.
# ==============================================================================

import os
import gc
import time
import pickle
import numpy as np
import pandas as pd
from tqdm import tqdm

print("=" * 90)
print("COSMX SUBSTEP 5F-9E: Prepare 9E imputation targets and summary geometry")
print("=" * 90)

t0 = time.time()

# ------------------------------------------------------------------------------
# 0. Required variables
# ------------------------------------------------------------------------------

required_vars = [
    "mol",
    "model",
    "device",
    "X_denoised",
    "X_raw_counts",
    "was_corrected",
    "shared_genes",
    "cell_ids_step4",
    "gene_to_col",
    "step4_cell_to_row",
    "cell_type_indices",
    "unique_cell_types",
    "denoised_adata",
    "nuc_centroids",
    "cell_radius_lookup_px",
    "nuc_radius_lookup_px",
]

missing = [v for v in required_vars if v not in globals()]

if missing:
    raise NameError(
        "Missing required variable(s):\n"
        + "\n".join(missing)
        + "\n\nRun the CosMx recovery, training, and final evaluation cells first."
    )

# ------------------------------------------------------------------------------
# 1. CosMx paths and run config
# ------------------------------------------------------------------------------

if "CHECKPOINT_DIR" not in globals():
    CHECKPOINT_DIR = "/content/drive/MyDrive/diffusion/step4_cosmx/imputation"

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

RUN_NAME = globals().get(
    "RUN_NAME",
    "cosmx_9E_distribution_empirical_baselines"
)

MAX_CONTEXT_9E = globals().get("MAX_CONTEXT_9E", 96)
INFER_BATCH_SIZE = globals().get("INFER_BATCH_SIZE", 256)

print(f"CHECKPOINT_DIR : {CHECKPOINT_DIR}")
print(f"RUN_NAME       : {RUN_NAME}")
print(f"MAX_CONTEXT_9E : {MAX_CONTEXT_9E}")
print(f"INFER_BATCH_SIZE: {INFER_BATCH_SIZE}")
print(f"Device         : {device}")

# ------------------------------------------------------------------------------
# 2. Make all IDs string-safe
# ------------------------------------------------------------------------------

cell_ids_step4 = np.array(cell_ids_step4).astype(str)

step4_cell_to_row = {
    str(cid): i
    for i, cid in enumerate(cell_ids_step4)
}

cell_idx_map = step4_cell_to_row

gene_to_col = {
    str(g): j
    for j, g in enumerate(shared_genes)
}

gene_idx_map = gene_to_col

mol["cell_id"] = mol["cell_id"].astype(str)
mol["gene_id"] = mol["gene_id"].astype(str)

# ------------------------------------------------------------------------------
# 3. Cell type labels
# ------------------------------------------------------------------------------

if "cell_type" in denoised_adata.obs.columns:
    cell_type_labels = denoised_adata.obs["cell_type"].astype(str).values
elif "Final_CosMx_Cell_Type" in denoised_adata.obs.columns:
    cell_type_labels = denoised_adata.obs["Final_CosMx_Cell_Type"].astype(str).values
elif "Assigned_Xenium_Cell_Type" in denoised_adata.obs.columns:
    cell_type_labels = denoised_adata.obs["Assigned_Xenium_Cell_Type"].astype(str).values
else:
    raise KeyError(
        "Could not find cell type column in denoised_adata.obs. "
        "Expected cell_type, Final_CosMx_Cell_Type, or Assigned_Xenium_Cell_Type."
    )

print(f"Cell type labels loaded: {len(cell_type_labels):,}")

if len(cell_type_labels) != len(cell_ids_step4):
    raise ValueError(
        f"cell_type_labels length {len(cell_type_labels)} does not match "
        f"cell_ids_step4 length {len(cell_ids_step4)}"
    )

# ------------------------------------------------------------------------------
# 4. Identify imputation targets from corrected Step 4 counts
# ------------------------------------------------------------------------------

print("\nIdentifying imputation targets from Step 4 denoised counts...")

if X_denoised.shape != X_raw_counts.shape:
    raise ValueError(
        f"Shape mismatch: X_denoised {X_denoised.shape} "
        f"vs X_raw_counts {X_raw_counts.shape}"
    )

if was_corrected is not None and was_corrected.shape != X_denoised.shape:
    raise ValueError(
        f"was_corrected shape {was_corrected.shape} does not match "
        f"X_denoised shape {X_denoised.shape}"
    )

X_den_round = np.rint(X_denoised).astype(np.int32)
X_raw_int = np.rint(X_raw_counts).astype(np.int32)

delta_counts = X_den_round - X_raw_int

# For selective denoising, negative deltas should already be zero.
# Clip only as a safety guard.
negative_pairs = int((delta_counts < 0).sum())
negative_molecules = int((-np.minimum(delta_counts, 0)).sum())

print(f"Negative delta pairs before clipping: {negative_pairs:,}")
print(f"Negative molecules before clipping  : {negative_molecules:,}")

delta_counts[delta_counts < 0] = 0

# Only impute pairs Step 4 actually corrected.
if was_corrected is not None:
    delta_counts = delta_counts * was_corrected.astype(bool)

rows, cols = np.where(delta_counts > 0)

imputation_targets = []
total_to_generate = 0

missing_geometry = 0
missing_nucleus_centroid = 0
missing_cell_radius = 0

for row, col in zip(rows, cols):
    n_imp = int(delta_counts[row, col])

    if n_imp <= 0:
        continue

    cid = str(cell_ids_step4[row])
    gid = str(shared_genes[col])

    # CosMx uses summary geometry, not polygon geometry.
    # Required for inverse conversion:
    #   nucleus/cell center + approximate cell radius
    if cid not in nuc_centroids:
        missing_nucleus_centroid += 1
        continue

    if cid not in cell_radius_lookup_px:
        missing_cell_radius += 1
        continue

    radius = float(cell_radius_lookup_px[cid])

    if not np.isfinite(radius) or radius <= 0:
        missing_geometry += 1
        continue

    target = {
        "row": int(row),
        "col": int(col),
        "cell_id": cid,
        "gene_id": gid,
        "gene_idx": int(col),
        "ct_idx": int(cell_type_indices[row]),
        "ct_label": str(cell_type_labels[row]),
        "n_impute": n_imp,
    }

    imputation_targets.append(target)
    total_to_generate += n_imp

print(f"Candidate corrected cell-gene pairs: {len(rows):,}")
print(f"Pairs needing imputation: {len(imputation_targets):,}")
print(f"Total molecules to generate: {total_to_generate:,}")

print("\nSkipped targets due to missing CosMx summary geometry:")
print(f"  missing nucleus/cell centroid: {missing_nucleus_centroid:,}")
print(f"  missing cell radius          : {missing_cell_radius:,}")
print(f"  invalid radius/geometry      : {missing_geometry:,}")

if len(imputation_targets) == 0:
    raise RuntimeError(
        "No imputation targets found. "
        "Check X_denoised, X_raw_counts, was_corrected, and CosMx geometry lookups."
    )

imputation_targets_df = pd.DataFrame(imputation_targets)

print("\nImputation target preview:")
display(imputation_targets_df.head())

print("\nImputation target summary:")
print(f"  unique cells: {imputation_targets_df['cell_id'].nunique():,}")
print(f"  unique genes: {imputation_targets_df['gene_id'].nunique():,}")
print(f"  total imputed molecules: {imputation_targets_df['n_impute'].sum():,}")

print("\nTop 20 genes by imputed molecules:")
display(
    imputation_targets_df
    .groupby("gene_id")["n_impute"]
    .sum()
    .sort_values(ascending=False)
    .head(20)
    .rename_axis("gene_id")
    .reset_index(name="n_impute")
)

# Save target summary for traceability.
targets_path = os.path.join(
    CHECKPOINT_DIR,
    f"{RUN_NAME}_imputation_targets.csv"
)

imputation_targets_df.to_csv(targets_path, index=False)

print(f"\nSaved target list:")
print(f"  {targets_path}")

# ------------------------------------------------------------------------------
# 5. Build observed molecule context lookup for target pairs only
# ------------------------------------------------------------------------------

print("\nBuilding observed molecule context lookup for target pairs only...")

target_pairs = imputation_targets_df[["cell_id", "gene_id"]].drop_duplicates().copy()
target_pairs["cell_id"] = target_pairs["cell_id"].astype(str)
target_pairs["gene_id"] = target_pairs["gene_id"].astype(str)

mol_obs_cols = [
    "cell_id",
    "gene_id",
    "r_norm",
    "theta",
    "z_rel",
    "p_nuclear",
    "status",
]

missing_mol_cols = [c for c in mol_obs_cols if c not in mol.columns]

if missing_mol_cols:
    raise KeyError(f"mol is missing required columns: {missing_mol_cols}")

mol_obs = mol[mol["status"].astype(str) == "observed"][mol_obs_cols].copy()

mol_obs["cell_id"] = mol_obs["cell_id"].astype(str)
mol_obs["gene_id"] = mol_obs["gene_id"].astype(str)

# Inner merge keeps only observed molecules for target cell-gene pairs.
mol_target_obs = mol_obs.merge(
    target_pairs,
    on=["cell_id", "gene_id"],
    how="inner",
)

print(f"Observed molecules in target pairs: {len(mol_target_obs):,}")

mol_context_lookup = {}

for (cid, gid), group in tqdm(
    mol_target_obs.groupby(["cell_id", "gene_id"], sort=False),
    desc="Building context lookup",
    mininterval=5,
):
    cid = str(cid)
    gid = str(gid)

    vals = group[["r_norm", "theta", "z_rel", "p_nuclear"]].values.astype(np.float32)

    mol_context_lookup[(cid, gid)] = vals

print(f"Context lookup entries: {len(mol_context_lookup):,}")

# Some targets may have no observed molecule of the same gene in the same cell.
# Later inference should fall back to empty/default context for those cases.
n_targets_with_context = int(
    imputation_targets_df.apply(
        lambda r: (str(r["cell_id"]), str(r["gene_id"])) in mol_context_lookup,
        axis=1,
    ).sum()
)

print(f"Targets with same-cell same-gene observed context: {n_targets_with_context:,}")
print(f"Targets without direct context: {len(imputation_targets_df) - n_targets_with_context:,}")

del mol_obs, mol_target_obs, target_pairs
gc.collect()

# ------------------------------------------------------------------------------
# 6. Build cell z-range lookup
# ------------------------------------------------------------------------------

print("\nBuilding cell z-range lookup...")

mol_active = mol[mol["status"].astype(str) != "pruned"][["cell_id", "z"]].copy()
mol_active["cell_id"] = mol_active["cell_id"].astype(str)

cell_z_stats = (
    mol_active
    .groupby("cell_id")["z"]
    .agg(["min", "max"])
    .to_dict("index")
)

print(f"Cell z-range entries: {len(cell_z_stats):,}")

if len(cell_z_stats) == 0:
    raise RuntimeError("cell_z_stats is empty. Cannot convert z_rel to absolute z.")

del mol_active
gc.collect()

# ------------------------------------------------------------------------------
# 7. Geometry helper functions for CosMx summary geometry
# ------------------------------------------------------------------------------

def _centroid_to_xy(c):
    """
    Convert stored centroid to a 2D numpy array.
    Handles numpy arrays, lists, tuples, and point-like objects.
    """
    if hasattr(c, "x") and hasattr(c, "y"):
        return np.array([float(c.x), float(c.y)], dtype=np.float32)

    arr = np.asarray(c, dtype=np.float32).reshape(-1)

    if len(arr) < 2:
        raise ValueError(f"Invalid centroid object: {c}")

    return arr[:2].astype(np.float32)


def _safe_cell_radius(cid):
    """
    Get approximate CosMx cell radius in pixels.
    """
    cid = str(cid)

    r = float(cell_radius_lookup_px.get(cid, 1.0))

    if not np.isfinite(r) or r <= 0:
        r = 1.0

    return r


def _safe_nuc_radius(cid):
    """
    Get approximate CosMx nucleus radius in pixels.
    """
    cid = str(cid)

    r = float(nuc_radius_lookup_px.get(cid, 0.0))

    if not np.isfinite(r) or r < 0:
        r = 0.0

    return r


def _safe_nuc_centroid(cid):
    """
    Get estimated nucleus/cell centroid for CosMx summary geometry.
    """
    cid = str(cid)

    if cid in nuc_centroids:
        return _centroid_to_xy(nuc_centroids[cid])

    if "centroid_lookup" in globals() and cid in centroid_lookup:
        return _centroid_to_xy(centroid_lookup[cid])

    raise KeyError(f"No centroid found for cell_id={cid}")


# ------------------------------------------------------------------------------
# 8. Precompute approximate circular edge lookup
# ------------------------------------------------------------------------------

print("\nPrecomputing approximate circular geometry for cells needing imputation...")

GEOM_CACHE_PATH = os.path.join(
    CHECKPOINT_DIR,
    f"{RUN_NAME}_cosmx_summary_geometry_cache.pkl"
)

N_ANGLES = globals().get("N_ANGLES", 72)

precomputed_angles = np.linspace(
    -np.pi,
    np.pi,
    N_ANGLES,
    endpoint=False,
).astype(np.float32)

cells_needing_imputation = sorted(
    set(str(t["cell_id"]) for t in imputation_targets)
)

# If a previous failed run created a broken/partial cache, remove it.
if os.path.exists(GEOM_CACHE_PATH):
    try:
        with open(GEOM_CACHE_PATH, "rb") as f:
            test_cache = pickle.load(f)

        valid_cache = (
            isinstance(test_cache, dict)
            and "cell_edge_lookup" in test_cache
            and "N_ANGLES" in test_cache
            and "precomputed_angles" in test_cache
            and test_cache.get("geometry_mode", "") == "cosmx_summary_area_centroid"
        )

        if not valid_cache:
            print("Existing geometry cache is incomplete or incompatible. Removing it.")
            os.remove(GEOM_CACHE_PATH)

        del test_cache

    except Exception as e:
        print(f"Existing geometry cache could not be loaded: {e}")
        print("Removing broken geometry cache.")
        os.remove(GEOM_CACHE_PATH)

if os.path.exists(GEOM_CACHE_PATH):
    print(f"Loading geometry cache:")
    print(f"  {GEOM_CACHE_PATH}")

    with open(GEOM_CACHE_PATH, "rb") as f:
        geom_cache = pickle.load(f)

    cell_edge_lookup = geom_cache["cell_edge_lookup"]
    N_ANGLES = int(geom_cache["N_ANGLES"])
    precomputed_angles = geom_cache["precomputed_angles"].astype(np.float32)

    print(f"Loaded cell_edge_lookup: {len(cell_edge_lookup):,}")
    print(f"N_ANGLES: {N_ANGLES}")

else:
    cell_edge_lookup = {}

    for cid in tqdm(
        cells_needing_imputation,
        desc="Precomputing CosMx circular cell geometry",
        mininterval=5,
    ):
        cid = str(cid)

        try:
            nc = _safe_nuc_centroid(cid)
            radius = _safe_cell_radius(cid)

            edge_points = np.zeros((N_ANGLES, 2), dtype=np.float32)

            edge_points[:, 0] = nc[0] + radius * np.cos(precomputed_angles)
            edge_points[:, 1] = nc[1] + radius * np.sin(precomputed_angles)

            cell_edge_lookup[cid] = edge_points

        except Exception:
            continue

    geom_cache = {
        "geometry_mode": "cosmx_summary_area_centroid",
        "cell_edge_lookup": cell_edge_lookup,
        "N_ANGLES": int(N_ANGLES),
        "precomputed_angles": precomputed_angles,
    }

    with open(GEOM_CACHE_PATH, "wb") as f:
        pickle.dump(geom_cache, f, protocol=pickle.HIGHEST_PROTOCOL)

    print(f"Saved geometry cache:")
    print(f"  {GEOM_CACHE_PATH}")
    print(f"cell_edge_lookup: {len(cell_edge_lookup):,}")

if len(cell_edge_lookup) == 0:
    raise RuntimeError("cell_edge_lookup is empty. Cannot convert normalized coordinates to absolute coordinates.")

# ------------------------------------------------------------------------------
# 9. Compatibility placeholder for nucleus preparation
# ------------------------------------------------------------------------------

# Xenium used prepared Shapely nucleus polygons.
# CosMx summary geometry has no nucleus polygons, so this stays empty.
cell_nuc_prep = {}

print("\nNucleus geometry mode:")
print("  CosMx summary geometry has no exact nucleus polygons.")
print("  p_nuclear geometry will be estimated using distance <= approximate nucleus radius.")
print(f"  In-memory prepared nucleus polygons: {len(cell_nuc_prep):,}")

# ------------------------------------------------------------------------------
# 10. Context feature and absolute coordinate conversion
# ------------------------------------------------------------------------------

def context_features(vals):
    """
    Convert observed molecule values [r_norm, theta, z_rel, p_nuclear]
    into model context features [r_norm, sin(theta), cos(theta), z_rel, p_nuclear].
    """
    vals = np.asarray(vals, dtype=np.float32)

    if vals.ndim != 2 or vals.shape[1] < 4:
        raise ValueError(f"Expected vals shape [n, 4], got {vals.shape}")

    r = vals[:, 0]
    th = vals[:, 1]
    z = vals[:, 2]
    pn = vals[:, 3]

    return np.stack(
        [r, np.sin(th), np.cos(th), z, pn],
        axis=-1,
    ).astype(np.float32)


def convert_to_absolute_fast(r_norms, thetas, z_rels, cid):
    """
    Convert normalized 9E molecule coordinates [r_norm, theta, z_rel]
    back to absolute CosMx-like x/y/z coordinates.

    CosMx geometry approximation:
      x/y = estimated nucleus/cell centroid + r_norm * cell_radius * direction(theta)
      z   = z_min + z_rel * observed_cell_z_range

    Returns:
      abs_x, abs_y, abs_z, p_nuc_geom
    """
    cid = str(cid)

    nc = _safe_nuc_centroid(cid)

    if cid not in cell_edge_lookup:
        # Fallback: construct one-off circular edge lookup if not cached.
        radius = _safe_cell_radius(cid)
        edge_pts = np.zeros((N_ANGLES, 2), dtype=np.float32)
        edge_pts[:, 0] = nc[0] + radius * np.cos(precomputed_angles)
        edge_pts[:, 1] = nc[1] + radius * np.sin(precomputed_angles)
    else:
        edge_pts = cell_edge_lookup[cid]

    zr = cell_z_stats.get(cid, {"min": 0.0, "max": 15.0})

    z_min_c = float(zr["min"])
    z_max_c = float(zr["max"])
    z_range_c = max(z_max_c - z_min_c, 0.1)

    r_norms = np.asarray(r_norms, dtype=np.float32)
    thetas = np.asarray(thetas, dtype=np.float32)
    z_rels = np.asarray(z_rels, dtype=np.float32)

    r_norms = np.clip(r_norms, 0.0, 1.0)
    z_rels = np.clip(z_rels, 0.0, 1.0)

    n = len(r_norms)

    abs_x = np.zeros(n, dtype=np.float32)
    abs_y = np.zeros(n, dtype=np.float32)
    abs_z = np.zeros(n, dtype=np.float32)
    p_nuc_geom = np.zeros(n, dtype=np.float32)

    angle_step = 2.0 * np.pi / N_ANGLES
    first_angle = float(precomputed_angles[0])

    for j in range(n):
        theta_w = ((float(thetas[j]) + np.pi) % (2.0 * np.pi)) - np.pi

        frac_pos = (theta_w - first_angle) / angle_step
        frac_pos = frac_pos % N_ANGLES

        idx_lo = int(np.floor(frac_pos)) % N_ANGLES
        idx_hi = (idx_lo + 1) % N_ANGLES
        frac = frac_pos - np.floor(frac_pos)

        edge_x = edge_pts[idx_lo, 0] * (1.0 - frac) + edge_pts[idx_hi, 0] * frac
        edge_y = edge_pts[idx_lo, 1] * (1.0 - frac) + edge_pts[idx_hi, 1] * frac

        abs_x[j] = nc[0] + r_norms[j] * (edge_x - nc[0])
        abs_y[j] = nc[1] + r_norms[j] * (edge_y - nc[1])
        abs_z[j] = z_min_c + z_rels[j] * z_range_c

        # Approximate nuclear assignment from radius.
        nuc_radius = _safe_nuc_radius(cid)
        if nuc_radius > 0:
            d_nuc = np.sqrt(
                (abs_x[j] - nc[0]) ** 2
                + (abs_y[j] - nc[1]) ** 2
            )
            p_nuc_geom[j] = 1.0 if d_nuc <= nuc_radius else 0.0
        else:
            p_nuc_geom[j] = 0.0

    return abs_x, abs_y, abs_z, p_nuc_geom


# ------------------------------------------------------------------------------
# 11. Final summary
# ------------------------------------------------------------------------------

elapsed = time.time() - t0

print("\n" + "=" * 90)
print("COSMX 9E INFERENCE SETUP COMPLETE")
print("=" * 90)
print(f"Pairs needing imputation: {len(imputation_targets):,}")
print(f"Total molecules to generate: {total_to_generate:,}")
print(f"Context lookup entries: {len(mol_context_lookup):,}")
print(f"Cells needing imputation: {len(cells_needing_imputation):,}")
print(f"Cells with geometry cache: {len(cell_edge_lookup):,}")
print(f"Geometry mode: cosmx_summary_area_centroid")
print(f"Setup time: {elapsed / 60:.1f} min")

print("\nImportant outputs ready for the next inference cell:")
print("  imputation_targets")
print("  imputation_targets_df")
print("  mol_context_lookup")
print("  cell_z_stats")
print("  cell_edge_lookup")
print("  context_features(vals)")
print("  convert_to_absolute_fast(r_norms, thetas, z_rels, cid)")

print("=" * 90)

In [ ]:
# Clear previous bad Infer-2 output before rerunning fixed Infer-2
if "imputed_records" in globals():
    del imputed_records

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("Cleared old imputed_records and GPU cache.")

In [ ]:
# ==============================================================================
# COSMX CELL S5-9E-INFER-2 — Generate 9E imputed molecule records
# CosMx-safe version.
# Run this after COSMX S5-9E-INFER-1.
# ==============================================================================

import os
import gc
import time
import json
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.distributions import Beta
from tqdm import tqdm

print("=" * 90)
print("COSMX SUBSTEP 5G-9E: Generate imputed molecule records")
print("=" * 90)

t0 = time.time()

# ------------------------------------------------------------------------------
# 0. Required variable checks
# ------------------------------------------------------------------------------

required_vars = [
    "model",
    "device",
    "imputation_targets",
    "imputation_targets_df",
    "total_to_generate",
    "mol_context_lookup",
    "convert_to_absolute_fast",
    "cell_type_labels",
    "cell_edge_lookup",
    "nuc_centroids",
    "cell_z_stats",
    "cell_radius_lookup_px",
    "nuc_radius_lookup_px",
]

missing = [v for v in required_vars if v not in globals()]

if missing:
    raise NameError(
        "Missing required variable(s) from Infer-1:\n"
        + "\n".join(missing)
        + "\n\nRun COSMX S5-9E-INFER-1 first."
    )

# ------------------------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------------------------

if "CHECKPOINT_DIR" not in globals():
    CHECKPOINT_DIR = "/content/drive/MyDrive/diffusion/step4_cosmx/imputation"

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

RUN_NAME = globals().get(
    "RUN_NAME",
    "cosmx_9E_distribution_empirical_baselines"
)

MAX_CONTEXT_9E = globals().get("MAX_CONTEXT_9E", 96)
INFER_BATCH_SIZE = globals().get("INFER_BATCH_SIZE", 256)
THETA_JITTER = globals().get("THETA_JITTER", 0.20)

# Reproducible stochastic sampling.
SEED_INFER = globals().get("SEED_INFER", 12345)

np.random.seed(SEED_INFER)
torch.manual_seed(SEED_INFER)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED_INFER)

model.to(device)
model.eval()

n_targets = len(imputation_targets)

print(f"CHECKPOINT_DIR                  : {CHECKPOINT_DIR}")
print(f"RUN_NAME                        : {RUN_NAME}")
print(f"Targets / cell-gene pairs       : {n_targets:,}")
print(f"Expected imputed molecules      : {total_to_generate:,}")
print(f"Context lookup entries          : {len(mol_context_lookup):,}")
print(f"Cells with geometry cache       : {len(cell_edge_lookup):,}")
print(f"Device                          : {device}")
print(f"MAX_CONTEXT_9E                  : {MAX_CONTEXT_9E}")
print(f"INFER_BATCH_SIZE                : {INFER_BATCH_SIZE}")
print(f"THETA_JITTER                    : {THETA_JITTER}")
print(f"SEED_INFER                      : {SEED_INFER}")

if len(imputation_targets) == 0:
    raise RuntimeError("imputation_targets is empty. Run/fix Infer-1 first.")

if int(total_to_generate) <= 0:
    raise RuntimeError("total_to_generate <= 0. Nothing to impute.")

# ------------------------------------------------------------------------------
# 2. CosMx-safe geometry helpers
# ------------------------------------------------------------------------------

def _safe_cell_area(cid):
    """
    Return approximate CosMx cell area in pixels.
    Used for geom feature construction.
    """
    cid = str(cid)

    if "cell_area_lookup_px" in globals() and cid in cell_area_lookup_px:
        val = float(cell_area_lookup_px[cid])
    elif "cell_areas" in globals() and cid in cell_areas:
        val = float(cell_areas[cid])
    else:
        # fallback from radius
        r = float(cell_radius_lookup_px.get(cid, 1.0))
        val = float(np.pi * max(r, 1.0) ** 2)

    if not np.isfinite(val) or val <= 0:
        val = 100.0

    return val


def _safe_nuc_area(cid):
    """
    Return approximate CosMx nuclear area in pixels.
    Used for geom feature construction.
    """
    cid = str(cid)

    if "nuc_area_lookup_px" in globals() and cid in nuc_area_lookup_px:
        val = float(nuc_area_lookup_px[cid])
    elif "nuc_areas" in globals() and cid in nuc_areas:
        val = float(nuc_areas[cid])
    else:
        r = float(nuc_radius_lookup_px.get(cid, 0.0))
        val = float(np.pi * max(r, 0.0) ** 2)

    if not np.isfinite(val) or val < 0:
        val = 0.0

    return val


def make_context_features_9e(vals):
    """
    Convert observed molecule values [r_norm, theta, z_rel, p_nuclear]
    into 9E model context features:
      [r_norm, sin(theta), cos(theta), z_rel, p_nuclear]
    """
    vals = np.asarray(vals, dtype=np.float32)

    if vals.ndim != 2 or vals.shape[1] < 4:
        raise ValueError(f"Expected vals shape [N, 4], got {vals.shape}")

    r = vals[:, 0]
    th = vals[:, 1]
    z = vals[:, 2]
    pn = vals[:, 3]

    return np.stack(
        [r, np.sin(th), np.cos(th), z, pn],
        axis=-1
    ).astype(np.float32)

# ------------------------------------------------------------------------------
# 2b. Fallback context helpers for no-direct-context targets
# ------------------------------------------------------------------------------

rng_context = np.random.default_rng(SEED_INFER)

# Prefer empirical pools from training cell.
if "ct_gene_empirical_pools" in globals():
    FALLBACK_CT_GENE_POOLS = ct_gene_empirical_pools
elif "baseline_ct_gene_pools" in globals():
    FALLBACK_CT_GENE_POOLS = baseline_ct_gene_pools
else:
    FALLBACK_CT_GENE_POOLS = {}

if "gene_empirical_pools" in globals():
    FALLBACK_GENE_POOLS = gene_empirical_pools
elif "baseline_gene_pools" in globals():
    FALLBACK_GENE_POOLS = baseline_gene_pools
else:
    FALLBACK_GENE_POOLS = {}

print("\nFallback context pools:")
print(f"  cell-type-gene pools: {len(FALLBACK_CT_GENE_POOLS):,}")
print(f"  gene pools          : {len(FALLBACK_GENE_POOLS):,}")

def get_fallback_context_values(gene_idx, ct_idx, max_context=96):
    """
    Return pseudo-context values [r_norm, theta, z_rel, p_nuclear]
    for targets that have no same-cell same-gene observed context.
    """

    gene_idx = int(gene_idx)
    ct_idx = int(ct_idx)

    pool = FALLBACK_CT_GENE_POOLS.get((ct_idx, gene_idx), None)

    if pool is None or len(pool) == 0:
        pool = FALLBACK_GENE_POOLS.get(gene_idx, None)

    if pool is not None and len(pool) > 0:
        n_ctx = min(max_context, len(pool))
        idx = rng_context.choice(len(pool), size=n_ctx, replace=len(pool) < n_ctx)
        vals = pool[idx, :4].astype(np.float32)

        # Safety clipping
        vals[:, 0] = np.clip(vals[:, 0], 0.0, 1.0)  # r_norm
        vals[:, 1] = np.arctan2(np.sin(vals[:, 1]), np.cos(vals[:, 1]))  # theta
        vals[:, 2] = np.clip(vals[:, 2], 0.0, 1.0)  # z_rel
        vals[:, 3] = np.clip(vals[:, 3], 0.0, 1.0)  # p_nuclear

        return vals

    # Final fallback: one neutral pseudo molecule.
    # Important: this is not empty context. mask should be 1 for this row.
    return np.array(
        [[0.5, 0.0, 0.5, 0.5]],
        dtype=np.float32
    )

# ------------------------------------------------------------------------------
# 3. Main inference loop
# ------------------------------------------------------------------------------

imputed_records = []
n_generated_so_far = 0

n_batches = (n_targets + INFER_BATCH_SIZE - 1) // INFER_BATCH_SIZE

print("\nStarting CosMx 9E imputation generation...")
print("Note: this creates ~5 million imputed records, so high-RAM runtime is preferred.")

for batch_start in tqdm(
    range(0, n_targets, INFER_BATCH_SIZE),
    desc="Generating CosMx 9E imputed molecules",
    total=n_batches,
    mininterval=10,
):
    batch_targets = imputation_targets[batch_start:batch_start + INFER_BATCH_SIZE]
    B = len(batch_targets)

    # Fixed-size model inputs.
    ctx_batch = np.zeros((B, MAX_CONTEXT_9E, 5), dtype=np.float32)
    mask_batch = np.zeros((B, MAX_CONTEXT_9E), dtype=np.float32)

    gene_batch = np.zeros(B, dtype=np.int64)
    ct_batch = np.zeros(B, dtype=np.int64)

    # 9E uses geom_dim = 9.
    geom_batch = np.zeros((B, 9), dtype=np.float32)

    n_gen_list = []

    # --------------------------------------------------------------------------
    # Build batch tensors from target cell-gene pairs
    # --------------------------------------------------------------------------
    for b, target in enumerate(batch_targets):
        cid = str(target["cell_id"])
        gid = str(target["gene_id"])
        n_imp = int(target["n_impute"])

        if n_imp <= 0:
            n_gen_list.append(0)
            continue

        vals = mol_context_lookup.get((cid, gid), None)

        if vals is not None and len(vals) > 0:
            n_ctx = min(len(vals), MAX_CONTEXT_9E)

            # Deterministic context selection for reproducibility.
            vals_used = vals[:n_ctx]

            ctx_batch[b, :n_ctx] = make_context_features_9e(vals_used)
            mask_batch[b, :n_ctx] = 1.0
        else:
            # No same-gene observed context in this cell.
            # Use empirical pseudo-context instead of empty context to avoid coordinate collapse.
            fallback_vals = get_fallback_context_values(
                gene_idx=int(target["gene_idx"]),
                ct_idx=int(target["ct_idx"]),
                max_context=MAX_CONTEXT_9E,
            )
            n_ctx = min(len(fallback_vals), MAX_CONTEXT_9E)

            ctx_batch[b, :n_ctx] = make_context_features_9e(fallback_vals[:n_ctx])
            mask_batch[b, :n_ctx] = 1.0

        gene_batch[b] = int(target["gene_idx"])
        ct_batch[b] = int(target["ct_idx"])

        c_area = _safe_cell_area(cid)
        n_area = _safe_nuc_area(cid)

        # ----------------------------------------------------------------------
        # 9E context-summary geometry features.
        # Must match training/evaluation geom_dim=9:
        #   0: cell_area / 500
        #   1: nuc_area / 200
        #   2: nuc_area / cell_area
        #   3: n_context / 50
        #   4: context_r_mean
        #   5: context_r_std
        #   6: context_z_mean
        #   7: context_z_std
        #   8: context_pnuc_mean
        # ----------------------------------------------------------------------
        if n_ctx > 0:
            ctx_summary_features = ctx_batch[b, :n_ctx]

            ctx_r_mean = float(ctx_summary_features[:, 0].mean())
            ctx_r_std = float(ctx_summary_features[:, 0].std())

            ctx_z_mean = float(ctx_summary_features[:, 3].mean())
            ctx_z_std = float(ctx_summary_features[:, 3].std())

            ctx_pnuc_mean = float(ctx_summary_features[:, 4].mean())
        else:
            ctx_r_mean = 0.5
            ctx_r_std = 0.0
            ctx_z_mean = 0.5
            ctx_z_std = 0.0
            ctx_pnuc_mean = 0.0

        geom_batch[b] = np.array([
            c_area / 500.0,
            n_area / 200.0,
            n_area / c_area if c_area > 0 else 0.3,
            n_ctx / 50.0,
            ctx_r_mean,
            ctx_r_std,
            ctx_z_mean,
            ctx_z_std,
            ctx_pnuc_mean,
        ], dtype=np.float32)

        n_gen_list.append(n_imp)

    K = int(max(n_gen_list))

    if K <= 0:
        continue

    # --------------------------------------------------------------------------
    # Move batch to GPU/CPU
    # --------------------------------------------------------------------------
    ctx_t = torch.tensor(ctx_batch, dtype=torch.float32, device=device)
    mask_t = torch.tensor(mask_batch, dtype=torch.float32, device=device)
    gene_t = torch.tensor(gene_batch, dtype=torch.long, device=device)
    ct_t = torch.tensor(ct_batch, dtype=torch.long, device=device)
    geom_t = torch.tensor(geom_batch, dtype=torch.float32, device=device)

    # --------------------------------------------------------------------------
    # Model inference + sampling
    # --------------------------------------------------------------------------
    with torch.no_grad():
        outputs = model(ctx_t, mask_t, gene_t, ct_t, geom_t)

        # Sample r_norm from predicted Beta distribution.
        r_alpha = outputs["r_alpha"].clamp_min(1e-4).unsqueeze(1).expand(B, K)
        r_beta = outputs["r_beta"].clamp_min(1e-4).unsqueeze(1).expand(B, K)

        r_dist = Beta(r_alpha, r_beta)
        r_samples = r_dist.sample().clamp(0.0, 1.0)

        # Sample z_rel from predicted Beta distribution.
        z_alpha = outputs["z_alpha"].clamp_min(1e-4).unsqueeze(1).expand(B, K)
        z_beta = outputs["z_beta"].clamp_min(1e-4).unsqueeze(1).expand(B, K)

        z_dist = Beta(z_alpha, z_beta)
        z_samples = z_dist.sample().clamp(0.0, 1.0)

        # 9E learned theta direction.
        # theta_vec is trained as [sin(theta), cos(theta)].
        theta_vec = F.normalize(outputs["theta_vec"], dim=-1)
        theta_mu = torch.atan2(theta_vec[:, 0], theta_vec[:, 1])

        theta_samples = theta_mu.unsqueeze(1).expand(B, K).clone()

        # Add jitter to prevent all generated molecules from collapsing to one angle.
        theta_samples = theta_samples + torch.randn((B, K), device=device) * THETA_JITTER

        # Wrap theta back to [-pi, pi].
        theta_samples = ((theta_samples + torch.pi) % (2 * torch.pi)) - torch.pi

        # Model-predicted nuclear probability.
        p_nuc_pred = torch.sigmoid(outputs["pnuc_logit"])

        # Move to CPU numpy.
        r_samples_np = r_samples.detach().cpu().numpy()
        z_samples_np = z_samples.detach().cpu().numpy()
        theta_samples_np = theta_samples.detach().cpu().numpy()
        p_nuc_pred_np = p_nuc_pred.detach().cpu().numpy()

    # Free GPU tensors from this batch.
    del ctx_t, mask_t, gene_t, ct_t, geom_t
    del outputs, r_samples, z_samples, theta_samples, p_nuc_pred

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # --------------------------------------------------------------------------
    # Convert normalized coordinates to absolute x/y/z and build records
    # --------------------------------------------------------------------------
    for b, target in enumerate(batch_targets):
        cid = str(target["cell_id"])
        gid = str(target["gene_id"])
        row = int(target["row"])
        n_imp = int(target["n_impute"])

        if n_imp <= 0:
            continue

        if cid not in cell_edge_lookup:
            raise KeyError(f"Missing cell_edge_lookup for cell_id={cid}")

        if cid not in nuc_centroids:
            raise KeyError(f"Missing nucleus/cell centroid for cell_id={cid}")

        r_norms = r_samples_np[b, :n_imp]
        z_rels = z_samples_np[b, :n_imp]
        thetas = theta_samples_np[b, :n_imp]

        abs_x, abs_y, abs_z, p_nuc_geom = convert_to_absolute_fast(
            r_norms,
            thetas,
            z_rels,
            cid,
        )

        # Prefer the label saved in the target itself.
        if "ct_label" in target:
            ct_label = str(target["ct_label"])
        else:
            ct_label = str(cell_type_labels[row])

        # One confidence value per cell-gene pair from the model's nuclear-probability head.
        # Keep bounded away from zero so downstream weighted analyses do not discard them.
        confidence = float(np.clip(p_nuc_pred_np[b], 0.1, 1.0))

        for k in range(n_imp):
            global_imp_idx = n_generated_so_far + k

            imputed_records.append({
                # Keep transcript_id numeric-compatible with old molecule table.
                # Add unique imputed_molecule_id for reliable downstream tracking.
                "transcript_id": -1,
                "imputed_molecule_id": f"imputed_9E_{global_imp_idx}",

                # CosMx cell IDs must remain strings, e.g. "1_1".
                "cell_id": cid,

                "overlaps_nucleus": int(p_nuc_geom[k] > 0.5),
                "gene_id": gid,

                # Absolute CosMx-like coordinates.
                "x": float(abs_x[k]),
                "y": float(abs_y[k]),
                "z": float(abs_z[k]),

                # CosMx Step 4 uses quality as compatibility placeholder.
                # For imputed molecules, use NaN to show no raw measured quality.
                "quality": np.nan,

                # Compatibility with older Xenium-style downstream code.
                "Assigned_Xenium_Cell_Type": ct_label,

                # Native/generic CosMx label columns.
                "cell_type": ct_label,
                "Final_CosMx_Cell_Type": ct_label,

                # Normalized learned coordinates.
                "r_norm": float(r_norms[k]),
                "theta": float(thetas[k]),
                "z_rel": float(z_rels[k]),

                # Geometry-derived nuclear assignment for generated absolute coordinate.
                "p_nuclear": float(p_nuc_geom[k]),

                # Model output probability for this cell-gene pair.
                "p_nuclear_model_prob": float(p_nuc_pred_np[b]),

                # Status flags.
                "status": "imputed",
                "weight": confidence,
                "is_imputed": True,
                "imputation_confidence": confidence,
                "imputation_model": "9E_best_coordNN",

                # Traceability.
                "source_run": RUN_NAME,
                "n_impute_for_pair": n_imp,
            })

        n_generated_so_far += n_imp

    # Optional progress every ~200k generated molecules.
    if n_generated_so_far > 0 and n_generated_so_far % 200_000 < max(n_gen_list):
        elapsed_min = (time.time() - t0) / 60
        print(
            f"  Generated {n_generated_so_far:,}/{total_to_generate:,} molecules "
            f"({elapsed_min:.1f} min elapsed)"
        )

    # Free CPU arrays.
    del ctx_batch, mask_batch, gene_batch, ct_batch, geom_batch
    del r_samples_np, z_samples_np, theta_samples_np, p_nuc_pred_np
    gc.collect()

# ------------------------------------------------------------------------------
# 4. Final checks
# ------------------------------------------------------------------------------

elapsed = time.time() - t0

print("\n" + "=" * 90)
print("COSMX 9E IMPUTATION GENERATION COMPLETE")
print("=" * 90)
print(f"Generated imputed molecules: {len(imputed_records):,}")
print(f"Expected imputed molecules : {int(total_to_generate):,}")
print(f"Difference                 : {len(imputed_records) - int(total_to_generate):,}")
print(f"Elapsed                    : {elapsed / 60:.1f} min")

if len(imputed_records) != int(total_to_generate):
    raise RuntimeError(
        f"Generated molecule count does not match expected total: "
        f"{len(imputed_records):,} vs {int(total_to_generate):,}"
    )

if len(imputed_records) > 0:
    confs = np.array(
        [r["imputation_confidence"] for r in imputed_records],
        dtype=np.float32,
    )

    print("\nConfidence summary:")
    print(f"  Mean confidence: {np.mean(confs):.4f}")
    print(f"  Min confidence : {np.min(confs):.4f}")
    print(f"  Max confidence : {np.max(confs):.4f}")

    preview_df = pd.DataFrame(imputed_records[:5])

    print("\nPreview of first 5 imputed records:")
    display(preview_df)

# Save a small summary immediately, even before Infer-3.
infer2_summary = {
    "platform": "CosMx",
    "run_name": RUN_NAME,
    "n_imputed_records": int(len(imputed_records)),
    "expected_imputed_records": int(total_to_generate),
    "difference": int(len(imputed_records) - int(total_to_generate)),
    "infer_batch_size": int(INFER_BATCH_SIZE),
    "max_context": int(MAX_CONTEXT_9E),
    "theta_jitter": float(THETA_JITTER),
    "seed_infer": int(SEED_INFER),
    "elapsed_min": float(elapsed / 60),
}

infer2_summary_path = os.path.join(
    CHECKPOINT_DIR,
    f"{RUN_NAME}_infer2_generation_summary.json"
)

with open(infer2_summary_path, "w") as f:
    json.dump(infer2_summary, f, indent=2)

print("\nSaved Infer-2 generation summary:")
print(f"  {infer2_summary_path}")

print("\nInfer-2 complete. Next run the corrected Infer-3 save/check cell.")
print("=" * 90)

In [ ]:
# ==============================================================================
# QUICK DIAGNOSTIC AFTER COSMX INFER-2
# Check whether imputed coordinates are spatially diverse or collapsed.
# ==============================================================================

import numpy as np
import pandas as pd

print("=" * 80)
print("POST-INFER-2 DIAGNOSTIC: Imputed coordinate diversity")
print("=" * 80)

assert "imputed_records" in globals(), "imputed_records not found. Run Infer-2 first."

n_total = len(imputed_records)
print(f"Total imputed records: {n_total:,}")

# Sample for fast diagnostics
N_CHECK = min(200_000, n_total)
rng = np.random.default_rng(42)
idx = rng.choice(n_total, size=N_CHECK, replace=False)

sample = [imputed_records[i] for i in idx]

x = np.array([r["x"] for r in sample], dtype=np.float32)
y = np.array([r["y"] for r in sample], dtype=np.float32)
z = np.array([r["z"] for r in sample], dtype=np.float32)
r_norm = np.array([r["r_norm"] for r in sample], dtype=np.float32)
theta = np.array([r["theta"] for r in sample], dtype=np.float32)
z_rel = np.array([r["z_rel"] for r in sample], dtype=np.float32)
p_nuc = np.array([r["p_nuclear"] for r in sample], dtype=np.float32)
p_model = np.array([r["p_nuclear_model_prob"] for r in sample], dtype=np.float32)
cell_ids = np.array([str(r["cell_id"]) for r in sample], dtype=object)

print("\nGlobal sampled coordinate summaries:")
summary_df = pd.DataFrame({
    "x": x,
    "y": y,
    "z": z,
    "r_norm": r_norm,
    "theta": theta,
    "z_rel": z_rel,
    "p_nuclear": p_nuc,
    "p_nuclear_model_prob": p_model,
}).describe()

display(summary_df)

# Approximate coordinate uniqueness after rounding
xy_round_001 = np.stack([np.round(x, 3), np.round(y, 3)], axis=1)
xy_round_01 = np.stack([np.round(x, 1), np.round(y, 1)], axis=1)

unique_xy_001 = len(np.unique(xy_round_001, axis=0))
unique_xy_01 = len(np.unique(xy_round_01, axis=0))

print("\nSample coordinate uniqueness:")
print(f"  Sample size: {N_CHECK:,}")
print(f"  Unique x/y rounded to 0.001: {unique_xy_001:,} ({100 * unique_xy_001 / N_CHECK:.2f}%)")
print(f"  Unique x/y rounded to 0.1  : {unique_xy_01:,} ({100 * unique_xy_01 / N_CHECK:.2f}%)")

# Check how many r_norm values are near zero
print("\nr_norm concentration:")
print(f"  r_norm < 0.001: {(r_norm < 0.001).sum():,} ({100 * (r_norm < 0.001).mean():.2f}%)")
print(f"  r_norm < 0.010: {(r_norm < 0.010).sum():,} ({100 * (r_norm < 0.010).mean():.2f}%)")
print(f"  r_norm < 0.050: {(r_norm < 0.050).sum():,} ({100 * (r_norm < 0.050).mean():.2f}%)")
print(f"  r_norm > 0.950: {(r_norm > 0.950).sum():,} ({100 * (r_norm > 0.950).mean():.2f}%)")

# Check nuclear probability saturation
print("\np_nuclear model saturation:")
print(f"  p_model > 0.99: {(p_model > 0.99).sum():,} ({100 * (p_model > 0.99).mean():.2f}%)")
print(f"  p_model < 0.01: {(p_model < 0.01).sum():,} ({100 * (p_model < 0.01).mean():.2f}%)")

# Per-cell diversity for a few highly sampled cells
sample_df = pd.DataFrame({
    "cell_id": cell_ids,
    "x": x,
    "y": y,
    "z": z,
    "r_norm": r_norm,
    "theta": theta,
})

cell_counts = sample_df["cell_id"].value_counts().head(10)

print("\nTop sampled cells and within-cell coordinate spread:")
rows = []

for cid, n in cell_counts.items():
    sub = sample_df[sample_df["cell_id"] == cid]
    rows.append({
        "cell_id": cid,
        "n_sampled": len(sub),
        "x_std": sub["x"].std(),
        "y_std": sub["y"].std(),
        "z_std": sub["z"].std(),
        "r_norm_mean": sub["r_norm"].mean(),
        "r_norm_std": sub["r_norm"].std(),
        "unique_xy_0.001": len(np.unique(np.stack([np.round(sub["x"], 3), np.round(sub["y"], 3)], axis=1), axis=0)),
    })

display(pd.DataFrame(rows))

print("\nInterpretation guide:")
print("  Good: many unique x/y values, r_norm not almost all near 0, reasonable within-cell x/y spread.")
print("  Warning: if most r_norm values are <0.01 or unique x/y is very low, coordinates are collapsed.")
print("=" * 80)

In [ ]:
# ==============================================================================
# POST-INFER-2 DIAGNOSTIC: Collapse by direct-context availability
# ==============================================================================

import numpy as np
import pandas as pd

print("=" * 80)
print("Collapse check: direct-context vs no-direct-context targets")
print("=" * 80)

assert "imputed_records" in globals()
assert "mol_context_lookup" in globals()

N_CHECK = min(300_000, len(imputed_records))
rng = np.random.default_rng(123)
idx = rng.choice(len(imputed_records), size=N_CHECK, replace=False)

rows = []
for i in idx:
    r = imputed_records[i]
    key = (str(r["cell_id"]), str(r["gene_id"]))
    rows.append({
        "has_direct_context": key in mol_context_lookup,
        "r_norm": float(r["r_norm"]),
        "p_model": float(r["p_nuclear_model_prob"]),
        "x": float(r["x"]),
        "y": float(r["y"]),
        "cell_id": str(r["cell_id"]),
    })

df = pd.DataFrame(rows)

summary = (
    df
    .groupby("has_direct_context")
    .agg(
        n=("r_norm", "size"),
        r_mean=("r_norm", "mean"),
        r_median=("r_norm", "median"),
        frac_r_lt_001=("r_norm", lambda x: float((x < 0.001).mean())),
        frac_r_lt_01=("r_norm", lambda x: float((x < 0.01).mean())),
        p_model_mean=("p_model", "mean"),
        frac_p_gt_099=("p_model", lambda x: float((x > 0.99).mean())),
    )
    .reset_index()
)

display(summary)

print("\nIf collapse is much worse for has_direct_context=False, update Infer-2 fallback context.")

SESSION CRASH

In [ ]:
import os

CHECKPOINT_DIR = "/content/drive/MyDrive/diffusion/step4_cosmx/imputation"
RUN_NAME = "cosmx_9E_distribution_empirical_baselines"

paths = {
    "imputed_records": os.path.join(CHECKPOINT_DIR, f"{RUN_NAME}_imputed_records.parquet"),
    "completed_molecule_table": os.path.join(CHECKPOINT_DIR, f"{RUN_NAME}_completed_molecule_table.parquet"),
    "count_reconciliation": os.path.join(CHECKPOINT_DIR, f"{RUN_NAME}_count_reconciliation.csv"),
    "imputation_summary": os.path.join(CHECKPOINT_DIR, f"{RUN_NAME}_imputation_summary.csv"),
    "imputation_targets": os.path.join(CHECKPOINT_DIR, f"{RUN_NAME}_imputation_targets.csv"),
}

for name, path in paths.items():
    print(f"{name:25s}: {os.path.exists(path)}  {path}")
    if os.path.exists(path):
        print(f"  size: {os.path.getsize(path)/1e9:.3f} GB")

In [ ]:
# ==============================================================================
# COSMX RECOVERY BEFORE INFER-3
# Use this when Infer-2 imputed_records.parquet exists but Infer-3 did not finish.
# This reloads the minimum required objects for corrected Infer-3.
# ==============================================================================

import os
import gc
import json
import numpy as np
import pandas as pd
from scipy import sparse
import anndata as ad

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

print("=" * 90)
print("COSMX RECOVERY BEFORE INFER-3")
print("=" * 90)

# ------------------------------------------------------------------------------
# 0. Paths
# ------------------------------------------------------------------------------

STEP4_EXPORT_DIR = "/content/drive/MyDrive/diffusion/step4_cosmx/step4_exports"
CHECKPOINT_DIR = "/content/drive/MyDrive/diffusion/step4_cosmx/imputation"
RUN_NAME = "cosmx_9E_distribution_empirical_baselines"

DENOISED_ADATA_PATH = os.path.join(STEP4_EXPORT_DIR, "denoised_adata.h5ad")
STEP4_MOLECULES_PATH = os.path.join(STEP4_EXPORT_DIR, "molecules.parquet")
STEP4_CONFIG_PATH = os.path.join(STEP4_EXPORT_DIR, "step4_config.json")

PROCESSED_MOL_PATH = os.path.join(CHECKPOINT_DIR, "step5_mol_processed.parquet")
IMPUTED_RECORDS_PATH = os.path.join(CHECKPOINT_DIR, f"{RUN_NAME}_imputed_records.parquet")
IMPUTATION_TARGETS_PATH = os.path.join(CHECKPOINT_DIR, f"{RUN_NAME}_imputation_targets.csv")
COUNT_CHECK_PATH = os.path.join(CHECKPOINT_DIR, f"{RUN_NAME}_count_reconciliation.csv")

COMPLETED_MOL_PATH = os.path.join(CHECKPOINT_DIR, f"{RUN_NAME}_completed_molecule_table.parquet")
IMPUTATION_SUMMARY_PATH = os.path.join(CHECKPOINT_DIR, f"{RUN_NAME}_imputation_summary.csv")

print(f"STEP4_EXPORT_DIR: {STEP4_EXPORT_DIR}")
print(f"CHECKPOINT_DIR  : {CHECKPOINT_DIR}")
print(f"RUN_NAME        : {RUN_NAME}")

# ------------------------------------------------------------------------------
# 1. Check required files
# ------------------------------------------------------------------------------

required_files = {
    "denoised_adata.h5ad": DENOISED_ADATA_PATH,
    "step4_config.json": STEP4_CONFIG_PATH,
    "processed observed molecule table": PROCESSED_MOL_PATH,
    "imputed records": IMPUTED_RECORDS_PATH,
    "imputation targets": IMPUTATION_TARGETS_PATH,
}

missing = []

print("\nChecking required files:")
for name, path in required_files.items():
    exists = os.path.exists(path)
    print(f"  {'✓' if exists else '✗'} {name:40s}: {path}")
    if exists:
        print(f"      size: {os.path.getsize(path) / 1e9:.3f} GB")
    else:
        missing.append(path)

if missing:
    raise FileNotFoundError(
        "Missing required files:\n" + "\n".join(missing)
    )

# ------------------------------------------------------------------------------
# 2. Load Step 4 config and denoised_adata
# ------------------------------------------------------------------------------

print("\n" + "=" * 90)
print("1. Loading Step 4 config and denoised_adata")
print("=" * 90)

with open(STEP4_CONFIG_PATH, "r") as f:
    step4_config = json.load(f)

shared_genes = [str(g) for g in step4_config["shared_genes"]]

denoised_adata = ad.read_h5ad(DENOISED_ADATA_PATH)

print(f"denoised_adata shape: {denoised_adata.shape}")
print(f"layers: {list(denoised_adata.layers.keys())}")

# Make gene order authoritative from denoised_adata.
shared_genes = list(denoised_adata.var_names.astype(str))
cell_ids_step4 = np.array(denoised_adata.obs_names.astype(str), dtype=str)

gene_to_col = {str(g): j for j, g in enumerate(shared_genes)}
gene_idx_map = gene_to_col

step4_cell_to_row = {str(c): i for i, c in enumerate(cell_ids_step4)}
cell_idx_map = step4_cell_to_row

print(f"Cells: {len(cell_ids_step4):,}")
print(f"Genes: {len(shared_genes):,}")

# ------------------------------------------------------------------------------
# 3. Load observed molecule table used by Step 5
# ------------------------------------------------------------------------------

print("\n" + "=" * 90)
print("2. Loading processed observed molecule table")
print("=" * 90)

mol = pd.read_parquet(PROCESSED_MOL_PATH)

mol["cell_id"] = mol["cell_id"].astype(str)
mol["gene_id"] = mol["gene_id"].astype(str)

if "status" not in mol.columns:
    mol["status"] = "observed"

mol["status"] = mol["status"].astype(str)

if "is_imputed" not in mol.columns:
    mol["is_imputed"] = False

if "weight" not in mol.columns:
    mol["weight"] = 1.0

print(f"mol shape: {mol.shape}")
print(f"unique cells: {mol['cell_id'].nunique():,}")
print(f"unique genes: {mol['gene_id'].nunique():,}")

print("\nObserved molecule status counts:")
display(
    mol["status"]
    .value_counts(dropna=False)
    .rename_axis("status")
    .reset_index(name="n_molecules")
)

# ------------------------------------------------------------------------------
# 4. Load imputation targets
# ------------------------------------------------------------------------------

print("\n" + "=" * 90)
print("3. Loading imputation targets")
print("=" * 90)

imputation_targets_df = pd.read_csv(IMPUTATION_TARGETS_PATH)

imputation_targets_df["cell_id"] = imputation_targets_df["cell_id"].astype(str)
imputation_targets_df["gene_id"] = imputation_targets_df["gene_id"].astype(str)
imputation_targets_df["n_impute"] = imputation_targets_df["n_impute"].astype(int)

total_to_generate = int(imputation_targets_df["n_impute"].sum())

print(f"imputation_targets_df shape: {imputation_targets_df.shape}")
print(f"target pairs: {len(imputation_targets_df):,}")
print(f"total_to_generate: {total_to_generate:,}")
print(f"unique target cells: {imputation_targets_df['cell_id'].nunique():,}")
print(f"unique target genes: {imputation_targets_df['gene_id'].nunique():,}")

# Optional list form, in case later cells need it.
imputation_targets = imputation_targets_df.to_dict("records")

# ------------------------------------------------------------------------------
# 5. Load saved imputed records
# ------------------------------------------------------------------------------

print("\n" + "=" * 90)
print("4. Loading saved imputed records")
print("=" * 90)

imputed_df = pd.read_parquet(IMPUTED_RECORDS_PATH)

imputed_df["cell_id"] = imputed_df["cell_id"].astype(str)
imputed_df["gene_id"] = imputed_df["gene_id"].astype(str)

if "status" in imputed_df.columns:
    imputed_df["status"] = imputed_df["status"].astype(str)
else:
    imputed_df["status"] = "imputed"

if "is_imputed" not in imputed_df.columns:
    imputed_df["is_imputed"] = True

print(f"imputed_df shape: {imputed_df.shape}")
print(f"imputed records: {len(imputed_df):,}")
print(f"unique imputed cells: {imputed_df['cell_id'].nunique():,}")
print(f"unique imputed genes: {imputed_df['gene_id'].nunique():,}")

if len(imputed_df) != total_to_generate:
    raise RuntimeError(
        f"Loaded imputed records count does not match expected total: "
        f"{len(imputed_df):,} vs {total_to_generate:,}"
    )

print("PASS: imputed_df row count matches total_to_generate.")

# ------------------------------------------------------------------------------
# 6. Re-run count reconciliation quickly
# ------------------------------------------------------------------------------

print("\n" + "=" * 90)
print("5. Rechecking imputed counts against targets")
print("=" * 90)

imputed_counts = (
    imputed_df
    .groupby(["cell_id", "gene_id"])
    .size()
    .reset_index(name="n_imputed_actual")
)

target_counts = imputation_targets_df[["cell_id", "gene_id", "n_impute"]].copy()
target_counts = target_counts.rename(columns={"n_impute": "n_imputed_expected"})

check_df = target_counts.merge(
    imputed_counts,
    on=["cell_id", "gene_id"],
    how="left",
)

check_df["n_imputed_actual"] = check_df["n_imputed_actual"].fillna(0).astype(int)
check_df["n_imputed_expected"] = check_df["n_imputed_expected"].astype(int)
check_df["diff"] = check_df["n_imputed_actual"] - check_df["n_imputed_expected"]

n_bad = int((check_df["diff"] != 0).sum())
max_abs_diff = int(check_df["diff"].abs().max()) if len(check_df) else 0

expected_sum = int(check_df["n_imputed_expected"].sum())
actual_sum = int(check_df["n_imputed_actual"].sum())

print(f"Target cell-gene pairs       : {len(check_df):,}")
print(f"Expected imputed molecules   : {expected_sum:,}")
print(f"Actual imputed molecules     : {actual_sum:,}")
print(f"Mismatched target pairs      : {n_bad:,}")
print(f"Max abs diff expected/actual : {max_abs_diff}")

# Save/overwrite count reconciliation check.
check_df.to_csv(COUNT_CHECK_PATH, index=False)
print(f"Saved count reconciliation: {COUNT_CHECK_PATH}")

if n_bad > 0:
    display(check_df[check_df["diff"] != 0].head(20))
    raise RuntimeError("Count reconciliation failed. Do not run Infer-3 until fixed.")

if actual_sum != total_to_generate:
    raise RuntimeError(
        f"Actual imputed molecule total {actual_sum:,} does not match "
        f"total_to_generate {total_to_generate:,}"
    )

print("PASS: Count reconciliation passed.")

# ------------------------------------------------------------------------------
# 7. Final recovery summary
# ------------------------------------------------------------------------------

print("\n" + "=" * 90)
print("RECOVERY BEFORE INFER-3 COMPLETE")
print("=" * 90)

print("Ready variables for corrected Infer-3:")
print("  mol")
print("  imputed_df")
print("  imputation_targets_df")
print("  imputation_targets")
print("  total_to_generate")
print("  check_df")
print("  n_bad")
print("  max_abs_diff")
print("  expected_sum")
print("  actual_sum")
print("  CHECKPOINT_DIR")
print("  RUN_NAME")

print("\nExpected final Infer-3 counts:")
print(f"  observed molecules : {(mol['status'].astype(str) == 'observed').sum():,}")
print(f"  imputed molecules  : {len(imputed_df):,}")
print(f"  completed molecules: {(mol['status'].astype(str) == 'observed').sum() + len(imputed_df):,}")

if os.path.exists(COMPLETED_MOL_PATH):
    print("\nNOTE: completed_molecule_table already exists.")
    print("Corrected Infer-3 will overwrite it safely using temp-file replacement.")
else:
    print("\ncompleted_molecule_table does not exist yet. Corrected Infer-3 will create it.")

gc.collect()

print("=" * 90)

In [ ]:
# ==============================================================================
# COSMX CELL S5-9E-INFER-3 — Save 9E imputed records + count reconciliation
# CosMx-safe version with string cell IDs.
# ==============================================================================

import os
import gc
import json
import numpy as np
import pandas as pd

print("=" * 90)
print("COSMX SUBSTEP 5H-9E: Save imputed records and verify counts")
print("=" * 90)

# ------------------------------------------------------------------------------
# 0. Paths / configuration
# ------------------------------------------------------------------------------

if "CHECKPOINT_DIR" not in globals():
    CHECKPOINT_DIR = "/content/drive/MyDrive/diffusion/step4_cosmx/imputation"

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

RUN_NAME = globals().get(
    "RUN_NAME",
    "cosmx_9E_distribution_empirical_baselines"
)

IMPUTED_RECORDS_PATH = os.path.join(
    CHECKPOINT_DIR,
    f"{RUN_NAME}_imputed_records.parquet"
)

COMPLETED_MOL_PATH = os.path.join(
    CHECKPOINT_DIR,
    f"{RUN_NAME}_completed_molecule_table.parquet"
)

IMPUTATION_SUMMARY_PATH = os.path.join(
    CHECKPOINT_DIR,
    f"{RUN_NAME}_imputation_summary.csv"
)

COUNT_CHECK_PATH = os.path.join(
    CHECKPOINT_DIR,
    f"{RUN_NAME}_count_reconciliation.csv"
)

IMPUTATION_SUMMARY_JSON_PATH = os.path.join(
    CHECKPOINT_DIR,
    f"{RUN_NAME}_imputation_summary.json"
)

# Temporary files for safer overwrite.
IMPUTED_RECORDS_TMP_PATH = IMPUTED_RECORDS_PATH + ".tmp"
COMPLETED_MOL_TMP_PATH = COMPLETED_MOL_PATH + ".tmp"
IMPUTATION_SUMMARY_TMP_PATH = IMPUTATION_SUMMARY_PATH + ".tmp"
COUNT_CHECK_TMP_PATH = COUNT_CHECK_PATH + ".tmp"
IMPUTATION_SUMMARY_JSON_TMP_PATH = IMPUTATION_SUMMARY_JSON_PATH + ".tmp"

print(f"CHECKPOINT_DIR: {CHECKPOINT_DIR}")
print(f"RUN_NAME: {RUN_NAME}")
print(f"IMPUTED_RECORDS_PATH: {IMPUTED_RECORDS_PATH}")
print(f"COMPLETED_MOL_PATH: {COMPLETED_MOL_PATH}")
print(f"IMPUTATION_SUMMARY_PATH: {IMPUTATION_SUMMARY_PATH}")
print(f"COUNT_CHECK_PATH: {COUNT_CHECK_PATH}")

# ------------------------------------------------------------------------------
# 1. Required variable checks
# ------------------------------------------------------------------------------

required_vars = [
    "mol",
    "imputation_targets_df",
    "total_to_generate",
]

missing = [v for v in required_vars if v not in globals()]
if missing:
    raise NameError(
        "Missing required variable(s):\n"
        + "\n".join(missing)
        + "\n\nRun corrected CosMx Infer-1 and Infer-2 first."
    )

# imputed_records may be in memory after Infer-2.
# If not, reload imputed_df from the already-saved parquet.
if "imputed_records" in globals() and len(imputed_records) > 0:
    print("\nBuilding imputed_df from imputed_records in memory...")
    imputed_df = pd.DataFrame(imputed_records)

elif "imputed_df" in globals() and len(imputed_df) > 0:
    print("\nUsing existing imputed_df from memory...")

elif os.path.exists(IMPUTED_RECORDS_PATH):
    print("\nimputed_records not found in memory. Loading saved imputed records:")
    print(f"  {IMPUTED_RECORDS_PATH}")
    imputed_df = pd.read_parquet(IMPUTED_RECORDS_PATH)

else:
    raise RuntimeError(
        "Could not find imputed_records, imputed_df, or saved imputed records parquet. "
        "Run corrected CosMx Infer-2 first."
    )

print(f"imputed_df shape: {imputed_df.shape}")

if len(imputed_df) == 0:
    raise RuntimeError("imputed_df is empty. Do not continue.")

# ------------------------------------------------------------------------------
# 2. Basic imputed_df schema checks
# ------------------------------------------------------------------------------

expected_imputed_cols = [
    "cell_id",
    "gene_id",
    "x",
    "y",
    "z",
    "r_norm",
    "theta",
    "z_rel",
    "p_nuclear",
    "status",
    "is_imputed",
]

missing_imputed_cols = [
    c for c in expected_imputed_cols
    if c not in imputed_df.columns
]

if missing_imputed_cols:
    raise KeyError(f"imputed_df is missing expected columns: {missing_imputed_cols}")

# CosMx cell IDs must remain strings.
imputed_df["cell_id"] = imputed_df["cell_id"].astype(str)
imputed_df["gene_id"] = imputed_df["gene_id"].astype(str)

imputation_targets_df = imputation_targets_df.copy()
imputation_targets_df["cell_id"] = imputation_targets_df["cell_id"].astype(str)
imputation_targets_df["gene_id"] = imputation_targets_df["gene_id"].astype(str)

# Ensure unique imputed molecule IDs exist.
if "imputed_molecule_id" not in imputed_df.columns:
    print("imputed_molecule_id missing; creating it.")
    imputed_df["imputed_molecule_id"] = [
        f"imputed_9E_{i}" for i in range(len(imputed_df))
    ]

# Add transcript_id if missing.
if "transcript_id" not in imputed_df.columns:
    print("transcript_id missing in imputed_df; creating from imputed_molecule_id.")
    imputed_df["transcript_id"] = imputed_df["imputed_molecule_id"].astype(str)

# Add inference_seed to imputed records if available.
if "SEED_INFER" in globals() and "inference_seed" not in imputed_df.columns:
    imputed_df["inference_seed"] = int(SEED_INFER)

# Ensure CosMx-native type columns exist.
if "cell_type" not in imputed_df.columns:
    if "Final_CosMx_Cell_Type" in imputed_df.columns:
        imputed_df["cell_type"] = imputed_df["Final_CosMx_Cell_Type"].astype(str)
    elif "Assigned_Xenium_Cell_Type" in imputed_df.columns:
        imputed_df["cell_type"] = imputed_df["Assigned_Xenium_Cell_Type"].astype(str)

if "Final_CosMx_Cell_Type" not in imputed_df.columns:
    if "cell_type" in imputed_df.columns:
        imputed_df["Final_CosMx_Cell_Type"] = imputed_df["cell_type"].astype(str)
    elif "Assigned_Xenium_Cell_Type" in imputed_df.columns:
        imputed_df["Final_CosMx_Cell_Type"] = imputed_df["Assigned_Xenium_Cell_Type"].astype(str)

if "Assigned_Xenium_Cell_Type" not in imputed_df.columns:
    if "cell_type" in imputed_df.columns:
        imputed_df["Assigned_Xenium_Cell_Type"] = imputed_df["cell_type"].astype(str)
    elif "Final_CosMx_Cell_Type" in imputed_df.columns:
        imputed_df["Assigned_Xenium_Cell_Type"] = imputed_df["Final_CosMx_Cell_Type"].astype(str)

# ------------------------------------------------------------------------------
# 3. Count reconciliation BEFORE saving huge completed table
# ------------------------------------------------------------------------------

print("\nRunning count reconciliation check...")

imputed_counts = (
    imputed_df
    .groupby(["cell_id", "gene_id"])
    .size()
    .reset_index(name="n_imputed_actual")
)

target_counts = imputation_targets_df[["cell_id", "gene_id", "n_impute"]].copy()
target_counts = target_counts.rename(columns={"n_impute": "n_imputed_expected"})

check_df = target_counts.merge(
    imputed_counts,
    on=["cell_id", "gene_id"],
    how="left",
)

check_df["n_imputed_actual"] = check_df["n_imputed_actual"].fillna(0).astype(int)
check_df["n_imputed_expected"] = check_df["n_imputed_expected"].astype(int)
check_df["diff"] = check_df["n_imputed_actual"] - check_df["n_imputed_expected"]

n_bad = int((check_df["diff"] != 0).sum())
max_abs_diff = int(check_df["diff"].abs().max()) if len(check_df) else 0

expected_sum = int(check_df["n_imputed_expected"].sum())
actual_sum = int(check_df["n_imputed_actual"].sum())

print(f"Target cell-gene pairs       : {len(check_df):,}")
print(f"Expected imputed molecules   : {expected_sum:,}")
print(f"Actual imputed molecules     : {actual_sum:,}")
print(f"Mismatched target pairs      : {n_bad:,}")
print(f"Max abs diff expected/actual : {max_abs_diff}")

# Save count reconciliation using temp file, then replace old file.
check_df.to_csv(COUNT_CHECK_TMP_PATH, index=False)
os.replace(COUNT_CHECK_TMP_PATH, COUNT_CHECK_PATH)

print(f"Saved count reconciliation:")
print(f"  {COUNT_CHECK_PATH}")

if n_bad > 0:
    print("\nWARNING: Some imputation target counts do not match generated records.")
    display(check_df[check_df["diff"] != 0].head(20))
    raise RuntimeError(
        "Count reconciliation failed. Do not save completed molecule table until this is fixed."
    )

if actual_sum != int(total_to_generate):
    raise RuntimeError(
        f"Total generated molecules mismatch: actual {actual_sum:,}, "
        f"expected total_to_generate {int(total_to_generate):,}"
    )

print("PASS: Count reconciliation passed for every target pair.")

# ------------------------------------------------------------------------------
# 4. Save imputed records
# ------------------------------------------------------------------------------

print("\nSaving imputed records to:")
print(f"  {IMPUTED_RECORDS_PATH}")

if os.path.exists(IMPUTED_RECORDS_TMP_PATH):
    os.remove(IMPUTED_RECORDS_TMP_PATH)

imputed_df.to_parquet(
    IMPUTED_RECORDS_TMP_PATH,
    index=False,
    compression="snappy",
)

os.replace(IMPUTED_RECORDS_TMP_PATH, IMPUTED_RECORDS_PATH)

print("Saved imputed records.")

# ------------------------------------------------------------------------------
# 5. Build observed molecule table
# ------------------------------------------------------------------------------

print("\nBuilding observed molecule table...")

mol["cell_id"] = mol["cell_id"].astype(str)
mol["gene_id"] = mol["gene_id"].astype(str)

observed_mol = mol[mol["status"].astype(str) == "observed"].copy()

print(f"observed_mol shape: {observed_mol.shape}")

if len(observed_mol) == 0:
    raise RuntimeError("observed_mol is empty. Check mol['status'].")

# Ensure observed CosMx cell labels exist.
if "cell_type" not in observed_mol.columns:
    if "Final_CosMx_Cell_Type" in observed_mol.columns:
        observed_mol["cell_type"] = observed_mol["Final_CosMx_Cell_Type"].astype(str)
    elif "Assigned_Xenium_Cell_Type" in observed_mol.columns:
        observed_mol["cell_type"] = observed_mol["Assigned_Xenium_Cell_Type"].astype(str)

if "Final_CosMx_Cell_Type" not in observed_mol.columns:
    if "cell_type" in observed_mol.columns:
        observed_mol["Final_CosMx_Cell_Type"] = observed_mol["cell_type"].astype(str)
    elif "Assigned_Xenium_Cell_Type" in observed_mol.columns:
        observed_mol["Final_CosMx_Cell_Type"] = observed_mol["Assigned_Xenium_Cell_Type"].astype(str)

if "Assigned_Xenium_Cell_Type" not in observed_mol.columns:
    if "cell_type" in observed_mol.columns:
        observed_mol["Assigned_Xenium_Cell_Type"] = observed_mol["cell_type"].astype(str)
    elif "Final_CosMx_Cell_Type" in observed_mol.columns:
        observed_mol["Assigned_Xenium_Cell_Type"] = observed_mol["Final_CosMx_Cell_Type"].astype(str)

# ------------------------------------------------------------------------------
# 6. Fix transcript_id dtype before concatenation
# ------------------------------------------------------------------------------

print("\nFixing transcript_id dtype for Parquet compatibility...")

if "transcript_id" not in observed_mol.columns:
    print("WARNING: observed_mol has no transcript_id column. Creating transcript_id from row index.")
    observed_mol["transcript_id"] = observed_mol.index.astype(str)

if "transcript_id" not in imputed_df.columns:
    imputed_df["transcript_id"] = imputed_df["imputed_molecule_id"].astype(str)

# For CosMx, safest is to store transcript_id as string in the completed parquet.
# This avoids Arrow mixed-type errors between observed numeric IDs and imputed IDs.
observed_mol["transcript_id"] = observed_mol["transcript_id"].astype(str)
imputed_df["transcript_id"] = imputed_df["transcript_id"].astype(str)

print("Converted observed and imputed transcript_id to string.")

# ------------------------------------------------------------------------------
# 7. Normalize common columns to avoid Arrow mixed-type errors
# ------------------------------------------------------------------------------

print("\nNormalizing common column dtypes...")

# Text-like columns.
text_cols = [
    "transcript_id",
    "cell_id",
    "gene_id",
    "Assigned_Xenium_Cell_Type",
    "cell_type",
    "Final_CosMx_Cell_Type",
    "status",
    "imputed_molecule_id",
    "imputation_model",
    "source_run",
]

for col in text_cols:
    if col in observed_mol.columns:
        observed_mol[col] = observed_mol[col].where(observed_mol[col].notna(), None)
        observed_mol[col] = observed_mol[col].astype("string")

    if col in imputed_df.columns:
        imputed_df[col] = imputed_df[col].where(imputed_df[col].notna(), None)
        imputed_df[col] = imputed_df[col].astype("string")

# Numeric columns.
# IMPORTANT: Do NOT include cell_id here. CosMx cell IDs are strings.
numeric_cols = [
    "overlaps_nucleus",
    "x",
    "y",
    "z",
    "quality",
    "r_norm",
    "theta",
    "z_rel",
    "p_nuclear",
    "p_nuclear_model_prob",
    "weight",
    "imputation_confidence",
    "n_impute_for_pair",
    "inference_seed",
]

for col in numeric_cols:
    if col in observed_mol.columns:
        observed_mol[col] = pd.to_numeric(observed_mol[col], errors="coerce")
    if col in imputed_df.columns:
        imputed_df[col] = pd.to_numeric(imputed_df[col], errors="coerce")

# Boolean column.
if "is_imputed" in observed_mol.columns:
    observed_mol["is_imputed"] = observed_mol["is_imputed"].fillna(False).astype(bool)
else:
    observed_mol["is_imputed"] = False

if "is_imputed" in imputed_df.columns:
    imputed_df["is_imputed"] = imputed_df["is_imputed"].fillna(True).astype(bool)
else:
    imputed_df["is_imputed"] = True

# ------------------------------------------------------------------------------
# 8. Align columns before concatenation
# ------------------------------------------------------------------------------

print("\nAligning observed and imputed table columns...")

all_cols = list(dict.fromkeys(list(observed_mol.columns) + list(imputed_df.columns)))

for col in all_cols:
    if col not in observed_mol.columns:
        observed_mol[col] = np.nan
    if col not in imputed_df.columns:
        imputed_df[col] = np.nan

observed_mol = observed_mol[all_cols]
imputed_df = imputed_df[all_cols]

print(f"Observed columns: {len(observed_mol.columns)}")
print(f"Imputed columns : {len(imputed_df.columns)}")

# ------------------------------------------------------------------------------
# 9. Build completed molecule table
# ------------------------------------------------------------------------------

print("\nBuilding completed molecule table...")

completed_mol = pd.concat([observed_mol, imputed_df], ignore_index=True)

print(f"observed molecules : {len(observed_mol):,}")
print(f"imputed molecules  : {len(imputed_df):,}")
print(f"completed molecules: {len(completed_mol):,}")

expected_completed = len(observed_mol) + len(imputed_df)

if len(completed_mol) != expected_completed:
    raise RuntimeError(
        f"Completed molecule table size mismatch: "
        f"{len(completed_mol):,} vs expected {expected_completed:,}"
    )

if int(len(imputed_df)) != int(total_to_generate):
    raise RuntimeError(
        f"Imputed count mismatch: {len(imputed_df):,} vs expected {int(total_to_generate):,}"
    )

# ------------------------------------------------------------------------------
# 10. Save completed molecule table
# ------------------------------------------------------------------------------

print("\nSaving completed molecule table to:")
print(f"  {COMPLETED_MOL_PATH}")

if os.path.exists(COMPLETED_MOL_TMP_PATH):
    os.remove(COMPLETED_MOL_TMP_PATH)

completed_mol.to_parquet(
    COMPLETED_MOL_TMP_PATH,
    index=False,
    compression="snappy",
)

os.replace(COMPLETED_MOL_TMP_PATH, COMPLETED_MOL_PATH)

print("Saved completed molecule table.")

# ------------------------------------------------------------------------------
# 11. Save summary
# ------------------------------------------------------------------------------

summary = {
    "platform": "CosMx",
    "run_name": RUN_NAME,
    "n_observed_molecules": int(len(observed_mol)),
    "n_imputed_molecules": int(len(imputed_df)),
    "n_completed_molecules": int(len(completed_mol)),
    "n_target_pairs": int(len(imputation_targets_df)),
    "expected_imputed_molecules": int(total_to_generate),
    "actual_imputed_molecules": int(len(imputed_df)),
    "count_mismatch_pairs": int(n_bad),
    "max_abs_count_diff": int(max_abs_diff),
    "imputed_records_path": IMPUTED_RECORDS_PATH,
    "completed_molecule_table_path": COMPLETED_MOL_PATH,
    "count_reconciliation_path": COUNT_CHECK_PATH,
    "transcript_id_fix": "converted observed and imputed transcript_id to string before parquet save",
    "cell_id_handling": "CosMx cell_id preserved as string",
}

if "SEED_INFER" in globals():
    summary["inference_seed"] = int(SEED_INFER)

summary_df = pd.DataFrame([summary])

summary_df.to_csv(IMPUTATION_SUMMARY_TMP_PATH, index=False)
os.replace(IMPUTATION_SUMMARY_TMP_PATH, IMPUTATION_SUMMARY_PATH)

with open(IMPUTATION_SUMMARY_JSON_TMP_PATH, "w") as f:
    json.dump(summary, f, indent=2)

os.replace(IMPUTATION_SUMMARY_JSON_TMP_PATH, IMPUTATION_SUMMARY_JSON_PATH)

print(f"\nSaved imputation summary:")
print(f"  {IMPUTATION_SUMMARY_PATH}")
print(f"  {IMPUTATION_SUMMARY_JSON_PATH}")

display(summary_df)

# ------------------------------------------------------------------------------
# 12. Optional preview / final checks
# ------------------------------------------------------------------------------

print("\nCompleted molecule table status counts:")
status_counts = (
    completed_mol["status"]
    .value_counts(dropna=False)
    .rename_axis("status")
    .reset_index(name="n_molecules")
)

display(status_counts)

print("\nCompleted molecule table cell_id dtype:")
print(completed_mol["cell_id"].dtype)

print("\nImputed molecule table preview:")
display(imputed_df.head())

# Check final status counts explicitly.
n_completed_observed = int((completed_mol["status"].astype(str) == "observed").sum())
n_completed_imputed = int((completed_mol["status"].astype(str) == "imputed").sum())

print("\nFinal count check:")
print(f"  observed rows in completed table: {n_completed_observed:,}")
print(f"  imputed rows in completed table : {n_completed_imputed:,}")
print(f"  total completed rows            : {len(completed_mol):,}")

if n_completed_observed != len(observed_mol):
    raise RuntimeError("Observed row count changed after concatenation.")

if n_completed_imputed != len(imputed_df):
    raise RuntimeError("Imputed row count changed after concatenation.")

# ------------------------------------------------------------------------------
# 13. Cleanup
# ------------------------------------------------------------------------------

gc.collect()

print("\n" + "=" * 90)
print("COSMX 9E IMPUTATION SAVE/CHECK COMPLETE")
print("=" * 90)
print("Saved files:")
print(f"  Imputed records          : {IMPUTED_RECORDS_PATH}")
print(f"  Completed molecule table : {COMPLETED_MOL_PATH}")
print(f"  Count reconciliation     : {COUNT_CHECK_PATH}")
print(f"  Summary CSV              : {IMPUTATION_SUMMARY_PATH}")
print(f"  Summary JSON             : {IMPUTATION_SUMMARY_JSON_PATH}")

In [ ]:
# ==============================================================================
# COSMX CHECK: Verify Step 5 / 9E imputation output files
# ==============================================================================

import os
import pandas as pd

CHECKPOINT_DIR = "/content/drive/MyDrive/diffusion/step4_cosmx/imputation"
RUN_NAME = "cosmx_9E_distribution_empirical_baselines"

paths = [
    os.path.join(CHECKPOINT_DIR, f"{RUN_NAME}_imputed_records.parquet"),
    os.path.join(CHECKPOINT_DIR, f"{RUN_NAME}_completed_molecule_table.parquet"),
    os.path.join(CHECKPOINT_DIR, f"{RUN_NAME}_count_reconciliation.csv"),
    os.path.join(CHECKPOINT_DIR, f"{RUN_NAME}_imputation_summary.csv"),
    os.path.join(CHECKPOINT_DIR, f"{RUN_NAME}_imputation_summary.json"),
]

print("=" * 90)
print("COSMX STEP 5 OUTPUT FILE CHECK")
print("=" * 90)

all_ok = True

for p in paths:
    exists = os.path.exists(p)
    status = "✓" if exists else "✗"
    size = f"{os.path.getsize(p) / 1e9:.3f} GB" if exists else ""
    print(f"{status} {p} {size}")

    if not exists:
        all_ok = False

print("\n" + "-" * 90)

if all_ok:
    print("PASS: All main CosMx Step 5 imputation output files exist.")
else:
    print("WARNING: Some expected CosMx Step 5 output files are missing.")

# Optional: read small summary/count files to verify contents.
summary_path = os.path.join(CHECKPOINT_DIR, f"{RUN_NAME}_imputation_summary.csv")
count_check_path = os.path.join(CHECKPOINT_DIR, f"{RUN_NAME}_count_reconciliation.csv")

if os.path.exists(summary_path):
    print("\nImputation summary:")
    summary_df = pd.read_csv(summary_path)
    display(summary_df)

if os.path.exists(count_check_path):
    print("\nCount reconciliation quick check:")
    count_df = pd.read_csv(count_check_path)

    n_bad = int((count_df["diff"] != 0).sum())
    expected_sum = int(count_df["n_imputed_expected"].sum())
    actual_sum = int(count_df["n_imputed_actual"].sum())
    max_abs_diff = int(count_df["diff"].abs().max()) if len(count_df) else 0

    print(f"Target cell-gene pairs       : {len(count_df):,}")
    print(f"Expected imputed molecules   : {expected_sum:,}")
    print(f"Actual imputed molecules     : {actual_sum:,}")
    print(f"Mismatched target pairs      : {n_bad:,}")
    print(f"Max abs diff expected/actual : {max_abs_diff}")

    if n_bad == 0 and expected_sum == actual_sum:
        print("PASS: Count reconciliation is clean.")
    else:
        print("WARNING: Count reconciliation has mismatches.")

print("=" * 90)

==========IMPUTATION===========

======SAFE TO DISCONNECT RUNTIME======

In [ ]:
# ==============================================================================
# COSMX RECOVERY BEFORE EMPIRICAL BASELINE GENERATION
#
# Run this in a new runtime before:
#   1. corrected CosMx S5-6-DIST
#   2. corrected COSMX CELL S5-9E-BASELINE-1
#
# Recovers:
#   mol
#   X_raw_counts
#   X_denoised
#   was_corrected
#   shared_genes
#   cell_ids_step4
#   gene_to_col
#   step4_cell_to_row
#   cell_type_indices
#   unique_cell_types
#   denoised_adata
#   imputation_targets_df
#   imputation_targets
#   total_to_generate
#   cell_edge_lookup
#   nuc_centroids
#   cell_type_labels
#   convert_to_absolute_fast
# ==============================================================================

import os
import gc
import json
import pickle
import warnings
import numpy as np
import pandas as pd
from scipy import sparse
import anndata as ad

warnings.filterwarnings("ignore")

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

print("=" * 90)
print("COSMX RECOVERY BEFORE EMPIRICAL BASELINE GENERATION")
print("=" * 90)

# ------------------------------------------------------------------------------
# 0. Paths
# ------------------------------------------------------------------------------

STEP4_EXPORT_DIR = "/content/drive/MyDrive/diffusion/step4_cosmx/step4_exports"
CHECKPOINT_DIR = "/content/drive/MyDrive/diffusion/step4_cosmx/imputation"
IMPUTATION_DIR = CHECKPOINT_DIR

RUN_NAME = "cosmx_9E_distribution_empirical_baselines"

STEP4_CONFIG_PATH = os.path.join(STEP4_EXPORT_DIR, "step4_config.json")
DENOISED_ADATA_PATH = os.path.join(STEP4_EXPORT_DIR, "denoised_adata.h5ad")
WAS_CORRECTED_PATH = os.path.join(STEP4_EXPORT_DIR, "was_corrected.npy")
CELL_DATA_PATH = os.path.join(STEP4_EXPORT_DIR, "cell_data.npz")

PROCESSED_MOL_PATH = os.path.join(CHECKPOINT_DIR, "step5_mol_processed.parquet")

IMPUTATION_TARGETS_PATH = os.path.join(
    CHECKPOINT_DIR,
    f"{RUN_NAME}_imputation_targets.csv"
)

GEOM_CHECKPOINT_PATH = os.path.join(
    CHECKPOINT_DIR,
    "checkpoint_cosmx_geometry_after_5A.pkl"
)

NUC_CENTROIDS_PATH = os.path.join(
    CHECKPOINT_DIR,
    "nuc_centroids.pkl"
)

GEOM_CACHE_PATH = os.path.join(
    CHECKPOINT_DIR,
    f"{RUN_NAME}_cosmx_summary_geometry_cache.pkl"
)

print(f"STEP4_EXPORT_DIR: {STEP4_EXPORT_DIR}")
print(f"CHECKPOINT_DIR  : {CHECKPOINT_DIR}")
print(f"RUN_NAME        : {RUN_NAME}")

# ------------------------------------------------------------------------------
# 1. Check required files
# ------------------------------------------------------------------------------

required_files = {
    "Step 4 config": STEP4_CONFIG_PATH,
    "Step 4 denoised AnnData": DENOISED_ADATA_PATH,
    "Step 4 was_corrected": WAS_CORRECTED_PATH,
    "Step 4 cell_data": CELL_DATA_PATH,
    "Step 5 processed molecule table": PROCESSED_MOL_PATH,
    "Imputation targets": IMPUTATION_TARGETS_PATH,
    "Infer-1 geometry cache": GEOM_CACHE_PATH,
}

missing = []

print("\nChecking required files:")
for name, path in required_files.items():
    exists = os.path.exists(path)
    print(f"  {'✓' if exists else '✗'} {name:40s}: {path}")
    if exists:
        print(f"      size: {os.path.getsize(path) / 1e6:.2f} MB")
    else:
        missing.append(path)

if os.path.exists(GEOM_CHECKPOINT_PATH):
    print(f"  ✓ Combined geometry checkpoint             : {GEOM_CHECKPOINT_PATH}")
elif os.path.exists(NUC_CENTROIDS_PATH):
    print(f"  ✓ Separate nuc_centroids pickle            : {NUC_CENTROIDS_PATH}")
else:
    print(f"  ✗ No nuc_centroids source found")
    missing.append(GEOM_CHECKPOINT_PATH)

if missing:
    raise FileNotFoundError(
        "Missing required recovery files:\n" + "\n".join(missing)
    )

# ------------------------------------------------------------------------------
# 2. Helper
# ------------------------------------------------------------------------------

def ensure_dense(X):
    if sparse.issparse(X):
        return X.toarray()
    return np.asarray(X)

# ------------------------------------------------------------------------------
# 3. Load Step 4 config
# ------------------------------------------------------------------------------

print("\n" + "=" * 90)
print("1. Loading Step 4 config")
print("=" * 90)

with open(STEP4_CONFIG_PATH, "r") as f:
    step4_config = json.load(f)

print(f"Config platform: {step4_config.get('platform')}")
print(f"Config sample  : {step4_config.get('sample_name')}")
print(f"Config genes   : {len(step4_config.get('shared_genes', [])):,}")
print(f"Config cells   : {len(step4_config.get('cell_ids', [])):,}")

# ------------------------------------------------------------------------------
# 4. Load denoised_adata, X_raw_counts, X_denoised, was_corrected
# ------------------------------------------------------------------------------

print("\n" + "=" * 90)
print("2. Loading denoised_adata and Step 4 matrices")
print("=" * 90)

denoised_adata = ad.read_h5ad(DENOISED_ADATA_PATH)

print(f"denoised_adata shape: {denoised_adata.shape}")
print(f"Available layers: {list(denoised_adata.layers.keys())}")

shared_genes = list(denoised_adata.var_names.astype(str))
cell_ids_step4 = np.array(denoised_adata.obs_names.astype(str), dtype=str)

gene_to_col = {str(g): j for j, g in enumerate(shared_genes)}
gene_idx_map = gene_to_col

step4_cell_to_row = {str(c): i for i, c in enumerate(cell_ids_step4)}
cell_idx_map = step4_cell_to_row

X_denoised = ensure_dense(denoised_adata.X).astype(np.float32)

# Prefer corrected raw layer saved from Step 4.
if "raw" in denoised_adata.layers:
    X_raw_counts = ensure_dense(denoised_adata.layers["raw"]).astype(np.float32)
elif "raw_molecule_counts_clean" in denoised_adata.layers:
    X_raw_counts = ensure_dense(denoised_adata.layers["raw_molecule_counts_clean"]).astype(np.float32)
else:
    raise KeyError(
        "Could not find raw count layer in denoised_adata. "
        "Expected layer 'raw' or 'raw_molecule_counts_clean'."
    )

was_corrected = np.load(WAS_CORRECTED_PATH)

if was_corrected.shape != X_denoised.shape:
    raise ValueError(
        f"was_corrected shape {was_corrected.shape} does not match "
        f"X_denoised shape {X_denoised.shape}"
    )

if X_raw_counts.shape != X_denoised.shape:
    raise ValueError(
        f"X_raw_counts shape {X_raw_counts.shape} does not match "
        f"X_denoised shape {X_denoised.shape}"
    )

print(f"shared_genes: {len(shared_genes):,}")
print(f"cell_ids_step4: {len(cell_ids_step4):,}")
print(f"X_raw_counts shape: {X_raw_counts.shape}")
print(f"X_denoised shape  : {X_denoised.shape}")
print(f"was_corrected shape: {was_corrected.shape}")
print(f"Raw sum      : {X_raw_counts.sum(dtype=np.float64):,.0f}")
print(f"Denoised sum : {X_denoised.sum(dtype=np.float64):,.2f}")
print(f"Added sum    : {(X_denoised - X_raw_counts).sum(dtype=np.float64):,.2f}")
print(f"Corrected pairs: {int(was_corrected.sum()):,}")

# ------------------------------------------------------------------------------
# 5. Recover cell-type labels and indices
# ------------------------------------------------------------------------------

print("\n" + "=" * 90)
print("3. Recovering cell-type labels")
print("=" * 90)

if "cell_type" in denoised_adata.obs.columns:
    CELL_TYPE_COLUMN_USED = "cell_type"
elif "Final_CosMx_Cell_Type" in denoised_adata.obs.columns:
    CELL_TYPE_COLUMN_USED = "Final_CosMx_Cell_Type"
elif "Assigned_Xenium_Cell_Type" in denoised_adata.obs.columns:
    CELL_TYPE_COLUMN_USED = "Assigned_Xenium_Cell_Type"
elif "Reference_Cell_Type" in denoised_adata.obs.columns:
    CELL_TYPE_COLUMN_USED = "Reference_Cell_Type"
else:
    raise KeyError(
        "Could not find a usable cell-type column in denoised_adata.obs. "
        "Expected one of: cell_type, Final_CosMx_Cell_Type, "
        "Assigned_Xenium_Cell_Type, Reference_Cell_Type."
    )

cell_type_labels = denoised_adata.obs[CELL_TYPE_COLUMN_USED].astype(str).values
unique_cell_types = np.array(sorted(pd.unique(cell_type_labels).astype(str)), dtype=object)
cell_type_to_idx = {ct: i for i, ct in enumerate(unique_cell_types)}
cell_type_indices = np.array([cell_type_to_idx[str(ct)] for ct in cell_type_labels], dtype=np.int64)

print(f"Using cell-type column: {CELL_TYPE_COLUMN_USED}")
print(f"cell_type_labels length: {len(cell_type_labels):,}")
print(f"unique_cell_types: {len(unique_cell_types):,}")
print("Cell types:")
print(list(unique_cell_types))

# ------------------------------------------------------------------------------
# 6. Load processed molecule table
# ------------------------------------------------------------------------------

print("\n" + "=" * 90)
print("4. Loading processed molecule table")
print("=" * 90)

mol = pd.read_parquet(PROCESSED_MOL_PATH)

mol["cell_id"] = mol["cell_id"].astype(str)
mol["gene_id"] = mol["gene_id"].astype(str)

if "status" not in mol.columns:
    mol["status"] = "observed"

mol["status"] = mol["status"].astype(str)

if "weight" not in mol.columns:
    mol["weight"] = 1.0

if "is_imputed" not in mol.columns:
    mol["is_imputed"] = False

# Make sure CosMx cell-type columns exist for later cells.
if "Assigned_Xenium_Cell_Type" not in mol.columns:
    if "cell_type" in mol.columns:
        mol["Assigned_Xenium_Cell_Type"] = mol["cell_type"].astype(str)
    elif "Final_CosMx_Cell_Type" in mol.columns:
        mol["Assigned_Xenium_Cell_Type"] = mol["Final_CosMx_Cell_Type"].astype(str)
    else:
        # Map from denoised_adata if missing.
        cell_to_type = dict(zip(cell_ids_step4, cell_type_labels))
        mol["Assigned_Xenium_Cell_Type"] = mol["cell_id"].map(cell_to_type).fillna("Unknown").astype(str)

if "cell_type" not in mol.columns:
    mol["cell_type"] = mol["Assigned_Xenium_Cell_Type"].astype(str)

if "Final_CosMx_Cell_Type" not in mol.columns:
    mol["Final_CosMx_Cell_Type"] = mol["Assigned_Xenium_Cell_Type"].astype(str)

print(f"mol shape: {mol.shape}")
print(f"unique molecule cells: {mol['cell_id'].nunique():,}")
print(f"unique molecule genes: {mol['gene_id'].nunique():,}")

print("\nMolecule status counts:")
display(
    mol["status"]
    .value_counts(dropna=False)
    .rename_axis("status")
    .reset_index(name="n_molecules")
)

required_mol_cols = ["cell_id", "gene_id", "r_norm", "theta", "z_rel", "p_nuclear", "status"]
missing_mol_cols = [c for c in required_mol_cols if c not in mol.columns]
if missing_mol_cols:
    raise KeyError(
        f"Processed molecule table is missing required normalized-coordinate columns: {missing_mol_cols}. "
        "You would need to rerun coordinate normalization if these are missing."
    )

# ------------------------------------------------------------------------------
# 7. Load imputation targets
# ------------------------------------------------------------------------------

print("\n" + "=" * 90)
print("5. Loading imputation targets")
print("=" * 90)

imputation_targets_df = pd.read_csv(IMPUTATION_TARGETS_PATH)

imputation_targets_df["cell_id"] = imputation_targets_df["cell_id"].astype(str)
imputation_targets_df["gene_id"] = imputation_targets_df["gene_id"].astype(str)
imputation_targets_df["n_impute"] = imputation_targets_df["n_impute"].astype(int)

total_to_generate = int(imputation_targets_df["n_impute"].sum())
imputation_targets = imputation_targets_df.to_dict("records")

print(f"imputation_targets_df shape: {imputation_targets_df.shape}")
print(f"target pairs: {len(imputation_targets_df):,}")
print(f"total_to_generate: {total_to_generate:,}")
print(f"unique target cells: {imputation_targets_df['cell_id'].nunique():,}")
print(f"unique target genes: {imputation_targets_df['gene_id'].nunique():,}")

# ------------------------------------------------------------------------------
# 8. Load geometry checkpoint
# ------------------------------------------------------------------------------

print("\n" + "=" * 90)
print("6. Loading CosMx summary geometry")
print("=" * 90)

GEOMETRY_MODE = "cosmx_summary_area_centroid"

centroid_lookup = {}
nuc_centroids = {}
cell_radius_lookup_px = {}
nuc_radius_lookup_px = {}
cell_area_lookup_px = {}
nuc_area_lookup_px = {}

if os.path.exists(GEOM_CHECKPOINT_PATH):
    print("Loading combined geometry checkpoint:")
    print(f"  {GEOM_CHECKPOINT_PATH}")

    with open(GEOM_CHECKPOINT_PATH, "rb") as f:
        geometry_checkpoint = pickle.load(f)

    GEOMETRY_MODE = geometry_checkpoint.get("GEOMETRY_MODE", GEOMETRY_MODE)

    centroid_lookup = geometry_checkpoint.get("centroid_lookup", {})
    nuc_centroids = geometry_checkpoint.get("nuc_centroids", {})
    cell_radius_lookup_px = geometry_checkpoint.get("cell_radius_lookup_px", {})
    nuc_radius_lookup_px = geometry_checkpoint.get("nuc_radius_lookup_px", {})
    cell_area_lookup_px = geometry_checkpoint.get("cell_area_lookup_px", {})
    nuc_area_lookup_px = geometry_checkpoint.get("nuc_area_lookup_px", {})

elif os.path.exists(NUC_CENTROIDS_PATH):
    print("Combined geometry checkpoint not found. Loading separate nuc_centroids.pkl:")
    print(f"  {NUC_CENTROIDS_PATH}")

    with open(NUC_CENTROIDS_PATH, "rb") as f:
        nuc_centroids = pickle.load(f)

else:
    raise FileNotFoundError("No geometry checkpoint or nuc_centroids.pkl found.")

# String-safe lookup keys.
centroid_lookup = {str(k): np.asarray(v, dtype=np.float32) for k, v in centroid_lookup.items()}
nuc_centroids = {str(k): np.asarray(v, dtype=np.float32) for k, v in nuc_centroids.items()}
cell_radius_lookup_px = {str(k): float(v) for k, v in cell_radius_lookup_px.items()}
nuc_radius_lookup_px = {str(k): float(v) for k, v in nuc_radius_lookup_px.items()}
cell_area_lookup_px = {str(k): float(v) for k, v in cell_area_lookup_px.items()}
nuc_area_lookup_px = {str(k): float(v) for k, v in nuc_area_lookup_px.items()}

# Compatibility aliases for cells that expect cell_areas/nuc_areas.
cell_areas = cell_area_lookup_px
nuc_areas = nuc_area_lookup_px

print(f"GEOMETRY_MODE        : {GEOMETRY_MODE}")
print(f"centroid_lookup      : {len(centroid_lookup):,}")
print(f"nuc_centroids        : {len(nuc_centroids):,}")
print(f"cell_radius_lookup_px: {len(cell_radius_lookup_px):,}")
print(f"nuc_radius_lookup_px : {len(nuc_radius_lookup_px):,}")
print(f"cell_area_lookup_px  : {len(cell_area_lookup_px):,}")
print(f"nuc_area_lookup_px   : {len(nuc_area_lookup_px):,}")

# ------------------------------------------------------------------------------
# 9. Load cell_edge_lookup from geometry cache
# ------------------------------------------------------------------------------

print("\n" + "=" * 90)
print("7. Loading cell_edge_lookup")
print("=" * 90)

with open(GEOM_CACHE_PATH, "rb") as f:
    geom_cache = pickle.load(f)

if "cell_edge_lookup" not in geom_cache:
    raise KeyError("Geometry cache does not contain cell_edge_lookup.")

cell_edge_lookup = geom_cache["cell_edge_lookup"]
N_ANGLES = int(geom_cache.get("N_ANGLES", 72))
precomputed_angles = geom_cache.get(
    "precomputed_angles",
    np.linspace(-np.pi, np.pi, N_ANGLES, endpoint=False).astype(np.float32)
).astype(np.float32)

cell_edge_lookup = {
    str(k): np.asarray(v, dtype=np.float32)
    for k, v in cell_edge_lookup.items()
}

print(f"cell_edge_lookup entries: {len(cell_edge_lookup):,}")
print(f"N_ANGLES: {N_ANGLES}")
print(f"precomputed_angles shape: {precomputed_angles.shape}")

if len(cell_edge_lookup) == 0:
    raise RuntimeError("cell_edge_lookup is empty.")

# ------------------------------------------------------------------------------
# 10. Recover cell_z_stats
# ------------------------------------------------------------------------------

print("\n" + "=" * 90)
print("8. Recovering cell_z_stats")
print("=" * 90)

if "z" not in mol.columns:
    raise KeyError("mol does not contain z column; cannot recover cell_z_stats.")

mol_active = mol[mol["status"].astype(str) != "pruned"][["cell_id", "z"]].copy()

cell_z_stats = (
    mol_active
    .groupby("cell_id")["z"]
    .agg(["min", "max"])
    .to_dict("index")
)

del mol_active
gc.collect()

print(f"cell_z_stats entries: {len(cell_z_stats):,}")

if len(cell_z_stats) == 0:
    raise RuntimeError("cell_z_stats is empty.")

# ------------------------------------------------------------------------------
# 11. Define convert_to_absolute_fast
# ------------------------------------------------------------------------------

print("\n" + "=" * 90)
print("9. Defining convert_to_absolute_fast")
print("=" * 90)

def _centroid_to_xy(c):
    if hasattr(c, "x") and hasattr(c, "y"):
        return np.array([float(c.x), float(c.y)], dtype=np.float32)

    arr = np.asarray(c, dtype=np.float32).reshape(-1)

    if len(arr) < 2:
        raise ValueError(f"Invalid centroid object: {c}")

    return arr[:2].astype(np.float32)


def _safe_cell_radius(cid):
    cid = str(cid)
    r = float(cell_radius_lookup_px.get(cid, 1.0))

    if not np.isfinite(r) or r <= 0:
        r = 1.0

    return r


def _safe_nuc_radius(cid):
    cid = str(cid)
    r = float(nuc_radius_lookup_px.get(cid, 0.0))

    if not np.isfinite(r) or r < 0:
        r = 0.0

    return r


def _safe_nuc_centroid(cid):
    cid = str(cid)

    if cid in nuc_centroids:
        return _centroid_to_xy(nuc_centroids[cid])

    if cid in centroid_lookup:
        return _centroid_to_xy(centroid_lookup[cid])

    raise KeyError(f"No centroid found for cell_id={cid}")


def convert_to_absolute_fast(r_norms, thetas, z_rels, cid):
    """
    Convert normalized coordinates [r_norm, theta, z_rel] back to
    absolute CosMx-like x/y/z coordinates.
    """
    cid = str(cid)

    nc = _safe_nuc_centroid(cid)

    if cid not in cell_edge_lookup:
        radius = _safe_cell_radius(cid)
        edge_pts = np.zeros((N_ANGLES, 2), dtype=np.float32)
        edge_pts[:, 0] = nc[0] + radius * np.cos(precomputed_angles)
        edge_pts[:, 1] = nc[1] + radius * np.sin(precomputed_angles)
    else:
        edge_pts = cell_edge_lookup[cid]

    zr = cell_z_stats.get(cid, {"min": 0.0, "max": 15.0})

    z_min_c = float(zr["min"])
    z_max_c = float(zr["max"])
    z_range_c = max(z_max_c - z_min_c, 0.1)

    r_norms = np.asarray(r_norms, dtype=np.float32)
    thetas = np.asarray(thetas, dtype=np.float32)
    z_rels = np.asarray(z_rels, dtype=np.float32)

    r_norms = np.clip(r_norms, 0.0, 1.0)
    z_rels = np.clip(z_rels, 0.0, 1.0)

    n = len(r_norms)

    abs_x = np.zeros(n, dtype=np.float32)
    abs_y = np.zeros(n, dtype=np.float32)
    abs_z = np.zeros(n, dtype=np.float32)
    p_nuc_geom = np.zeros(n, dtype=np.float32)

    angle_step = 2.0 * np.pi / N_ANGLES
    first_angle = float(precomputed_angles[0])

    for j in range(n):
        theta_w = ((float(thetas[j]) + np.pi) % (2.0 * np.pi)) - np.pi

        frac_pos = (theta_w - first_angle) / angle_step
        frac_pos = frac_pos % N_ANGLES

        idx_lo = int(np.floor(frac_pos)) % N_ANGLES
        idx_hi = (idx_lo + 1) % N_ANGLES
        frac = frac_pos - np.floor(frac_pos)

        edge_x = edge_pts[idx_lo, 0] * (1.0 - frac) + edge_pts[idx_hi, 0] * frac
        edge_y = edge_pts[idx_lo, 1] * (1.0 - frac) + edge_pts[idx_hi, 1] * frac

        abs_x[j] = nc[0] + r_norms[j] * (edge_x - nc[0])
        abs_y[j] = nc[1] + r_norms[j] * (edge_y - nc[1])
        abs_z[j] = z_min_c + z_rels[j] * z_range_c

        nuc_radius = _safe_nuc_radius(cid)

        if nuc_radius > 0:
            d_nuc = np.sqrt(
                (abs_x[j] - nc[0]) ** 2
                + (abs_y[j] - nc[1]) ** 2
            )
            p_nuc_geom[j] = 1.0 if d_nuc <= nuc_radius else 0.0
        else:
            p_nuc_geom[j] = 0.0

    return abs_x, abs_y, abs_z, p_nuc_geom

# ------------------------------------------------------------------------------
# 12. Quick geometry conversion test
# ------------------------------------------------------------------------------

print("\nTesting convert_to_absolute_fast...")

test_cid = next(iter(cell_edge_lookup.keys()))

test_r = np.array([0.1, 0.5, 0.9], dtype=np.float32)
test_theta = np.array([-np.pi / 2, 0.0, np.pi / 2], dtype=np.float32)
test_z = np.array([0.2, 0.5, 0.8], dtype=np.float32)

tx, ty, tz, tp = convert_to_absolute_fast(test_r, test_theta, test_z, test_cid)

print(f"test cell_id: {test_cid}")
print("x:", tx)
print("y:", ty)
print("z:", tz)
print("p_nuc_geom:", tp)

if not np.all(np.isfinite(tx)) or not np.all(np.isfinite(ty)) or not np.all(np.isfinite(tz)):
    raise RuntimeError("convert_to_absolute_fast produced non-finite coordinates.")

# ------------------------------------------------------------------------------
# 13. Final summary
# ------------------------------------------------------------------------------

print("\n" + "=" * 90)
print("COSMX RECOVERY BEFORE BASELINE GENERATION COMPLETE")
print("=" * 90)

print("Recovered variables:")
print(f"  mol                       : {mol.shape}")
print(f"  X_raw_counts              : {X_raw_counts.shape}")
print(f"  X_denoised                : {X_denoised.shape}")
print(f"  was_corrected             : {was_corrected.shape}")
print(f"  shared_genes              : {len(shared_genes):,}")
print(f"  cell_ids_step4            : {len(cell_ids_step4):,}")
print(f"  unique_cell_types          : {len(unique_cell_types):,}")
print(f"  cell_type_labels           : {len(cell_type_labels):,}")
print(f"  imputation_targets_df      : {imputation_targets_df.shape}")
print(f"  total_to_generate          : {total_to_generate:,}")
print(f"  cell_edge_lookup           : {len(cell_edge_lookup):,}")
print(f"  nuc_centroids              : {len(nuc_centroids):,}")
print(f"  cell_z_stats               : {len(cell_z_stats):,}")
print("  convert_to_absolute_fast   : defined")

print("\nNext cells to run:")
print("  1. corrected CosMx S5-6-DIST")
print("  2. corrected COSMX CELL S5-9E-BASELINE-1")

print("=" * 90)

gc.collect()

In [ ]:
# ==============================================================================
# COSMX CELL S5-9E-BASELINE-1 — Generate 3 empirical baseline imputed records
# Baselines:
#   1. Gene-level empirical distribution
#   2. Cell-type gene empirical distribution
#   3. Spatial-kNN empirical distribution
#
# CPU/RAM-heavy. GPU not needed.
# Run after:
#   - S5-6-DIST, so baseline_gene_pools / baseline_ct_gene_pools / baseline_spatial_index exist
#   - S5-9E-INFER-1, so convert_to_absolute_fast / cell_edge_lookup / nuc_centroids exist
# ==============================================================================

import os
import gc
import time
import json
import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.neighbors import BallTree

print("=" * 90)
print("COSMX SUBSTEP 5I-9E: Generate empirical baseline imputed records")
print("=" * 90)

t0_all = time.time()

# ------------------------------------------------------------------------------
# 0. Paths / configuration
# ------------------------------------------------------------------------------

if "CHECKPOINT_DIR" not in globals():
    CHECKPOINT_DIR = "/content/drive/MyDrive/diffusion/step4_cosmx/imputation"

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

RUN_NAME = globals().get(
    "RUN_NAME",
    "cosmx_9E_distribution_empirical_baselines"
)

BASELINE_SEED = globals().get("BASELINE_SEED", 12345)
SPATIAL_KNN_K = globals().get("SPATIAL_KNN_K", 80)
SPATIAL_KNN_MIN_POOL = globals().get("SPATIAL_KNN_MIN_POOL", 5)

baseline_paths = {
    "gene_emp": os.path.join(
        CHECKPOINT_DIR,
        f"{RUN_NAME}_gene_emp_imputed_records.parquet"
    ),
    "ct_gene_emp": os.path.join(
        CHECKPOINT_DIR,
        f"{RUN_NAME}_ct_gene_emp_imputed_records.parquet"
    ),
    "spatial_knn_emp": os.path.join(
        CHECKPOINT_DIR,
        f"{RUN_NAME}_spatial_knn_emp_imputed_records.parquet"
    ),
}

baseline_summary_path = os.path.join(
    CHECKPOINT_DIR,
    f"{RUN_NAME}_baseline_imputation_summary.csv"
)

baseline_summary_json_path = os.path.join(
    CHECKPOINT_DIR,
    f"{RUN_NAME}_baseline_imputation_summary.json"
)

print(f"CHECKPOINT_DIR: {CHECKPOINT_DIR}")
print(f"RUN_NAME      : {RUN_NAME}")
print(f"BASELINE_SEED : {BASELINE_SEED}")
print(f"SPATIAL_KNN_K : {SPATIAL_KNN_K}")

for k, p in baseline_paths.items():
    print(f"{k:16s}: {p}")

# ------------------------------------------------------------------------------
# 1. Required variable checks
# ------------------------------------------------------------------------------

required_vars = [
    "imputation_targets",
    "imputation_targets_df",
    "total_to_generate",
    "baseline_gene_pools",
    "baseline_ct_gene_pools",
    "baseline_spatial_index",
    "convert_to_absolute_fast",
    "cell_type_labels",
    "cell_edge_lookup",
    "nuc_centroids",
]

missing = [v for v in required_vars if v not in globals()]

if missing:
    raise NameError(
        "Missing required variable(s):\n"
        + "\n".join(missing)
        + "\n\nYou likely need to rerun S5-6-DIST and/or S5-9E-INFER-1 before this baseline cell."
    )

# CosMx cell IDs must remain strings.
imputation_targets_df = imputation_targets_df.copy()
imputation_targets_df["cell_id"] = imputation_targets_df["cell_id"].astype(str)
imputation_targets_df["gene_id"] = imputation_targets_df["gene_id"].astype(str)

# Convert target list to string-safe version.
imputation_targets = imputation_targets_df.to_dict("records")

print(f"\nTarget cell-gene pairs: {len(imputation_targets):,}")
print(f"Expected molecules per baseline: {int(total_to_generate):,}")
print(f"Gene empirical pools: {len(baseline_gene_pools):,}")
print(f"Cell-type gene empirical pools: {len(baseline_ct_gene_pools):,}")
print(f"Spatial-kNN empirical groups: {len(baseline_spatial_index):,}")

# ------------------------------------------------------------------------------
# 2. Build / recover cell coordinate lookup for spatial baseline
# ------------------------------------------------------------------------------

print("\nBuilding CosMx-safe cell coordinate lookup...")

cell_xy_lookup = {}

if "denoised_adata" in globals() and {"x_centroid", "y_centroid"}.issubset(set(denoised_adata.obs.columns)):
    for cid, x, y in zip(
        denoised_adata.obs_names.astype(str),
        denoised_adata.obs["x_centroid"].values,
        denoised_adata.obs["y_centroid"].values,
    ):
        cell_xy_lookup[str(cid)] = np.array([float(x), float(y)], dtype=np.float32)

elif "centroid_lookup" in globals() and len(centroid_lookup) > 0:
    for cid, xy in centroid_lookup.items():
        cell_xy_lookup[str(cid)] = np.asarray(xy, dtype=np.float32)

elif "cell_ids_geom" in globals() and "centroids" in globals():
    for cid, xy in zip(cell_ids_geom.astype(str), centroids):
        cell_xy_lookup[str(cid)] = np.asarray(xy, dtype=np.float32)

else:
    print("WARNING: No global cell coordinate lookup found.")
    print("Spatial-kNN baseline will fall back more often to cell-type-gene empirical baseline.")

print(f"cell_xy_lookup entries: {len(cell_xy_lookup):,}")

# ------------------------------------------------------------------------------
# 3. Build spatial BallTrees once for speed
# ------------------------------------------------------------------------------

print("\nBuilding spatial BallTrees for baseline_spatial_index...")

spatial_tree_index = {}

for key, info in tqdm(
    baseline_spatial_index.items(),
    desc="Building spatial BallTrees",
    mininterval=5,
):
    xy = np.asarray(info["xy"], dtype=np.float32)

    if len(xy) == 0:
        continue

    # Keep cell IDs as strings/object.
    if "cell_ids" in info:
        cell_ids_arr = np.array([str(c) for c in info["cell_ids"]], dtype=object)
    else:
        cell_ids_arr = np.array([], dtype=object)

    spatial_tree_index[key] = {
        "tree": BallTree(xy),
        "xy": xy,
        "cell_ids": cell_ids_arr,
        "values": info["values"],
    }

print(f"Spatial BallTrees built: {len(spatial_tree_index):,}")

# ------------------------------------------------------------------------------
# 4. Sampling helpers
# ------------------------------------------------------------------------------

def fallback_sample_uniform(n_imp, rng):
    """
    Used only if no empirical pool is available.
    Returns [r_norm, theta, z_rel, p_nuclear].
    """
    r = rng.uniform(0.0, 1.0, size=n_imp).astype(np.float32)
    theta = rng.uniform(-np.pi, np.pi, size=n_imp).astype(np.float32)
    z = rng.uniform(0.0, 1.0, size=n_imp).astype(np.float32)
    p_nuc = rng.uniform(0.0, 1.0, size=n_imp).astype(np.float32)

    return np.stack([r, theta, z, p_nuc], axis=-1).astype(np.float32)


def _sanitize_sampled_vals(vals, n_imp):
    vals = np.asarray(vals, dtype=np.float32)

    if vals.ndim != 2:
        raise ValueError(f"Expected sampled vals to be 2D, got shape {vals.shape}")

    if vals.shape[0] != n_imp:
        raise ValueError(f"Expected {n_imp} sampled rows, got {vals.shape[0]}")

    if vals.shape[1] < 4:
        pad = np.zeros((n_imp, 4 - vals.shape[1]), dtype=np.float32)
        vals = np.concatenate([vals, pad], axis=1)

    vals = vals[:, :4].astype(np.float32)

    vals[:, 0] = np.clip(vals[:, 0], 0.0, 1.0)
    vals[:, 1] = np.arctan2(np.sin(vals[:, 1]), np.cos(vals[:, 1]))
    vals[:, 2] = np.clip(vals[:, 2], 0.0, 1.0)
    vals[:, 3] = np.clip(vals[:, 3], 0.0, 1.0)

    return vals


def sample_gene_empirical_for_target(target, n_imp, rng):
    gene_idx = int(target["gene_idx"])

    pool = baseline_gene_pools.get(gene_idx, None)

    if pool is None or len(pool) == 0:
        return fallback_sample_uniform(n_imp, rng)

    idx = rng.choice(len(pool), size=n_imp, replace=True)
    return _sanitize_sampled_vals(pool[idx], n_imp)


def sample_ct_gene_empirical_for_target(target, n_imp, rng):
    key = (int(target["ct_idx"]), int(target["gene_idx"]))

    pool = baseline_ct_gene_pools.get(key, None)

    if pool is None or len(pool) == 0:
        return sample_gene_empirical_for_target(target, n_imp, rng)

    idx = rng.choice(len(pool), size=n_imp, replace=True)
    return _sanitize_sampled_vals(pool[idx], n_imp)


def sample_spatial_knn_empirical_for_target(target, n_imp, rng, k_neighbors=80, min_pool=5):
    key = (int(target["ct_idx"]), int(target["gene_idx"]))

    info = spatial_tree_index.get(key, None)

    if info is None:
        return sample_ct_gene_empirical_for_target(target, n_imp, rng)

    xy = info["xy"]
    values_list = info["values"]

    if len(xy) == 0:
        return sample_ct_gene_empirical_for_target(target, n_imp, rng)

    cid = str(target["cell_id"])

    # Target cell coordinate.
    if "cell_xy" in target and target["cell_xy"] is not None:
        query_xy = np.asarray(target["cell_xy"], dtype=np.float32).reshape(1, -1)
    elif cid in cell_xy_lookup:
        query_xy = np.asarray(cell_xy_lookup[cid], dtype=np.float32).reshape(1, -1)
    else:
        return sample_ct_gene_empirical_for_target(target, n_imp, rng)

    k = min(k_neighbors, len(xy))

    if k < 1:
        return sample_ct_gene_empirical_for_target(target, n_imp, rng)

    _, ind = info["tree"].query(query_xy, k=k)

    candidate_vals = []

    for idx in ind[0]:
        vals = values_list[int(idx)]
        if vals is not None and len(vals) > 0:
            candidate_vals.append(vals)

    if len(candidate_vals) < min_pool:
        return sample_ct_gene_empirical_for_target(target, n_imp, rng)

    pool = np.concatenate(candidate_vals, axis=0)

    if len(pool) == 0:
        return sample_ct_gene_empirical_for_target(target, n_imp, rng)

    idx = rng.choice(len(pool), size=n_imp, replace=True)
    return _sanitize_sampled_vals(pool[idx], n_imp)


def sample_for_method(method_key, target, n_imp, rng):
    if method_key == "gene_emp":
        return sample_gene_empirical_for_target(target, n_imp, rng)

    if method_key == "ct_gene_emp":
        return sample_ct_gene_empirical_for_target(target, n_imp, rng)

    if method_key == "spatial_knn_emp":
        return sample_spatial_knn_empirical_for_target(
            target,
            n_imp,
            rng,
            k_neighbors=SPATIAL_KNN_K,
            min_pool=SPATIAL_KNN_MIN_POOL,
        )

    raise ValueError(f"Unknown baseline method: {method_key}")


# ------------------------------------------------------------------------------
# 5. Generate one baseline at a time
# ------------------------------------------------------------------------------

def generate_baseline_imputed_records(method_key, method_label, seed_offset=0):
    """
    Generate imputed molecule records for one empirical baseline method.
    Saves one parquet file per baseline.
    """

    print("\n" + "=" * 90)
    print(f"Generating baseline: {method_key} — {method_label}")
    print("=" * 90)

    t0 = time.time()

    rng = np.random.default_rng(BASELINE_SEED + seed_offset)

    records = []
    generated = 0

    for target in tqdm(
        imputation_targets,
        desc=f"{method_key} imputation",
        mininterval=10,
    ):
        cid = str(target["cell_id"])
        gid = str(target["gene_id"])
        row = int(target["row"])
        n_imp = int(target["n_impute"])

        if n_imp <= 0:
            continue

        if cid not in cell_edge_lookup:
            raise KeyError(f"Missing cell_edge_lookup for cell_id={cid}")

        if cid not in nuc_centroids:
            raise KeyError(f"Missing nuc_centroids for cell_id={cid}")

        sampled_vals = sample_for_method(method_key, target, n_imp, rng)

        # sampled_vals shape: [n_imp, 4] = [r_norm, theta, z_rel, p_nuclear]
        sampled_vals = _sanitize_sampled_vals(sampled_vals, n_imp)

        r_norms = sampled_vals[:, 0]
        thetas = sampled_vals[:, 1]
        z_rels = sampled_vals[:, 2]
        p_nuc_sampled = sampled_vals[:, 3]

        abs_x, abs_y, abs_z, p_nuc_geom = convert_to_absolute_fast(
            r_norms,
            thetas,
            z_rels,
            cid,
        )

        if "ct_label" in target:
            ct_label = str(target["ct_label"])
        else:
            ct_label = str(cell_type_labels[row])

        for k in range(n_imp):
            global_idx = generated + k

            records.append({
                "transcript_id": f"{method_key}_imputed_{global_idx}",
                "imputed_molecule_id": f"{method_key}_imputed_{global_idx}",

                # CosMx cell IDs remain strings.
                "cell_id": cid,

                "overlaps_nucleus": int(p_nuc_geom[k] > 0.5),
                "gene_id": gid,

                "x": float(abs_x[k]),
                "y": float(abs_y[k]),
                "z": float(abs_z[k]),

                "quality": np.nan,

                # Compatibility and CosMx-native cell-type columns.
                "Assigned_Xenium_Cell_Type": ct_label,
                "cell_type": ct_label,
                "Final_CosMx_Cell_Type": ct_label,

                "r_norm": float(r_norms[k]),
                "theta": float(thetas[k]),
                "z_rel": float(z_rels[k]),

                # Geometry-derived nuclear assignment after absolute conversion.
                "p_nuclear": float(p_nuc_geom[k]),

                # Nuclear value sampled from empirical pool before geometry conversion.
                "p_nuclear_sampled": float(p_nuc_sampled[k]),

                "status": "imputed",
                "weight": 1.0,
                "is_imputed": True,
                "imputation_confidence": 1.0,

                "imputation_model": method_key,
                "source_run": RUN_NAME,
                "baseline_method": method_key,
                "baseline_label": method_label,
                "baseline_seed": int(BASELINE_SEED + seed_offset),
                "n_impute_for_pair": n_imp,
            })

        generated += n_imp

        if generated > 0 and generated % 500_000 < n_imp:
            elapsed_min = (time.time() - t0) / 60
            print(f"  {method_key}: generated {generated:,}/{int(total_to_generate):,} ({elapsed_min:.1f} min)")

    elapsed = time.time() - t0

    print("\nConverting records to DataFrame...")
    baseline_df = pd.DataFrame(records)

    baseline_df["cell_id"] = baseline_df["cell_id"].astype("string")
    baseline_df["gene_id"] = baseline_df["gene_id"].astype("string")
    baseline_df["transcript_id"] = baseline_df["transcript_id"].astype("string")
    baseline_df["imputed_molecule_id"] = baseline_df["imputed_molecule_id"].astype("string")
    baseline_df["status"] = baseline_df["status"].astype("string")
    baseline_df["imputation_model"] = baseline_df["imputation_model"].astype("string")
    baseline_df["source_run"] = baseline_df["source_run"].astype("string")
    baseline_df["baseline_method"] = baseline_df["baseline_method"].astype("string")
    baseline_df["baseline_label"] = baseline_df["baseline_label"].astype("string")
    baseline_df["Assigned_Xenium_Cell_Type"] = baseline_df["Assigned_Xenium_Cell_Type"].astype("string")
    baseline_df["cell_type"] = baseline_df["cell_type"].astype("string")
    baseline_df["Final_CosMx_Cell_Type"] = baseline_df["Final_CosMx_Cell_Type"].astype("string")

    print(f"{method_key} generated molecules: {len(baseline_df):,}")
    print(f"Expected molecules          : {int(total_to_generate):,}")
    print(f"Difference                  : {len(baseline_df) - int(total_to_generate):,}")
    print(f"Elapsed                     : {elapsed / 60:.1f} min")

    if len(baseline_df) != int(total_to_generate):
        raise RuntimeError(
            f"{method_key} generated count mismatch: "
            f"{len(baseline_df):,} vs {int(total_to_generate):,}"
        )

    out_path = baseline_paths[method_key]
    tmp_path = out_path + ".tmp"

    print(f"Saving {method_key} imputed records:")
    print(f"  {out_path}")

    if os.path.exists(tmp_path):
        os.remove(tmp_path)

    baseline_df.to_parquet(tmp_path, index=False, compression="snappy")
    os.replace(tmp_path, out_path)

    print(f"Saved {method_key} baseline records.")

    # Small quality summary
    summary = {
        "platform": "CosMx",
        "run_name": RUN_NAME,
        "baseline_method": method_key,
        "baseline_label": method_label,
        "n_imputed_molecules": int(len(baseline_df)),
        "expected_imputed_molecules": int(total_to_generate),
        "seed": int(BASELINE_SEED + seed_offset),
        "output_path": out_path,
        "elapsed_min": float(elapsed / 60),
        "r_norm_mean": float(baseline_df["r_norm"].mean()),
        "r_norm_median": float(baseline_df["r_norm"].median()),
        "frac_r_lt_001": float((baseline_df["r_norm"] < 0.001).mean()),
        "frac_r_lt_01": float((baseline_df["r_norm"] < 0.01).mean()),
        "p_nuclear_mean": float(baseline_df["p_nuclear"].mean()),
        "unique_cells": int(baseline_df["cell_id"].nunique()),
        "unique_genes": int(baseline_df["gene_id"].nunique()),
    }

    # Free memory before next baseline.
    del records
    del baseline_df
    gc.collect()

    return summary


baseline_summaries = []

baseline_summaries.append(
    generate_baseline_imputed_records(
        method_key="gene_emp",
        method_label="Gene-level empirical distribution",
        seed_offset=0,
    )
)

baseline_summaries.append(
    generate_baseline_imputed_records(
        method_key="ct_gene_emp",
        method_label="Cell-type gene empirical distribution",
        seed_offset=1000,
    )
)

baseline_summaries.append(
    generate_baseline_imputed_records(
        method_key="spatial_knn_emp",
        method_label="Spatial-kNN empirical distribution",
        seed_offset=2000,
    )
)

baseline_summary_df = pd.DataFrame(baseline_summaries)
baseline_summary_df.to_csv(baseline_summary_path, index=False)

with open(baseline_summary_json_path, "w") as f:
    json.dump(baseline_summaries, f, indent=2)

print("\n" + "=" * 90)
print("COSMX 9E EMPIRICAL BASELINE IMPUTATION COMPLETE")
print("=" * 90)

display(baseline_summary_df)

print(f"Saved baseline summary:")
print(f"  {baseline_summary_path}")
print(f"  {baseline_summary_json_path}")

print(f"Total elapsed: {(time.time() - t0_all) / 60:.1f} min")

In [ ]:
# ==============================================================================
# COSMX CELL S5-9E-DOWNSTREAM-1 — Corrected downstream validation
# Raw / Step4 / Learned 9E / 3 empirical baselines
#
# Compares:
#   1. Raw observed counts
#   2. Step4 denoised counts
#   3. Learned 9E completed counts
#   4. Gene empirical completed counts
#   5. Cell-type gene empirical completed counts
#   6. Spatial-kNN empirical completed counts
#
# FIXES IN THIS VERSION:
#   - Raw and Step4 clustering are recomputed together using cell-level-style logic.
#   - Raw matrix is taken directly from denoised_adata.layers["raw"].
#   - Step4 matrix is taken directly from denoised_adata.X.
#   - Cell order is taken directly from denoised_adata.obs_names.
#   - Gene order is taken directly from denoised_adata.var_names.
#   - Learned/baseline completed matrices are built in the same authoritative order.
#   - Same fixed 50k-cell sample is reused for all matrices.
#   - Existing output filenames are NOT changed, so rerun overwrites old Drive files.
#
# CosMx-safe:
#   - Keeps CosMx cell IDs as strings like "1_1"; never converts them to int.
#   - Accepts CosMx cell-type columns.
#
# GPU not needed.
# ==============================================================================

import os
import gc
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from scipy import sparse

# ------------------------------------------------------------------------------
# 0. Imports
# ------------------------------------------------------------------------------

try:
    import scanpy as sc
    import anndata as ad
except Exception as e:
    print(f"Scanpy import failed: {e}")
    print("Installing scanpy/leiden dependencies...")
    !pip install -q scanpy leidenalg igraph anndata
    import scanpy as sc
    import anndata as ad

from sklearn.metrics import (
    adjusted_rand_score,
    normalized_mutual_info_score,
    silhouette_score,
)

print("=" * 90)
print("COSMX SUBSTEP 5J-9E: Downstream validation using cell-level-exact Step4 logic")
print("=" * 90)

# ------------------------------------------------------------------------------
# 1. Paths / configuration
# ------------------------------------------------------------------------------

CHECKPOINT_DIR = "/content/drive/MyDrive/diffusion/step4_cosmx/imputation"
RUN_NAME = "cosmx_9E_distribution_empirical_baselines"

LEARNED_IMPUTED_PATH = os.path.join(
    CHECKPOINT_DIR,
    f"{RUN_NAME}_imputed_records.parquet"
)

BASELINE_PATHS = {
    "Gene empirical completed counts": os.path.join(
        CHECKPOINT_DIR,
        f"{RUN_NAME}_gene_emp_imputed_records.parquet"
    ),
    "Cell-type gene empirical completed counts": os.path.join(
        CHECKPOINT_DIR,
        f"{RUN_NAME}_ct_gene_emp_imputed_records.parquet"
    ),
    "Spatial-kNN empirical completed counts": os.path.join(
        CHECKPOINT_DIR,
        f"{RUN_NAME}_spatial_knn_emp_imputed_records.parquet"
    ),
}

# IMPORTANT:
# These filenames are intentionally unchanged from your previous cell.
# Rerunning this cell will overwrite the existing saved files with latest results.
COUNT_METRICS_PATH = os.path.join(
    CHECKPOINT_DIR,
    f"{RUN_NAME}_downstream_count_matrix_metrics_step4matched.csv"
)

CLUSTERING_METRICS_PATH = os.path.join(
    CHECKPOINT_DIR,
    f"{RUN_NAME}_downstream_clustering_metrics_step4matched.csv"
)

REPORT_PATH = os.path.join(
    CHECKPOINT_DIR,
    f"{RUN_NAME}_downstream_validation_report_step4matched.txt"
)

SAMPLE_INDEX_PATH = os.path.join(
    CHECKPOINT_DIR,
    f"{RUN_NAME}_downstream_step4matched_sample_idx.npy"
)

# Match Step 4 validation setup.
SAMPLE_SIZE = 50_000
RANDOM_STATE = 42
N_PCS = 30
LEIDEN_RESOLUTION = 0.5

print(f"CHECKPOINT_DIR    : {CHECKPOINT_DIR}")
print(f"RUN_NAME          : {RUN_NAME}")
print(f"SAMPLE_SIZE       : {SAMPLE_SIZE:,}")
print(f"RANDOM_STATE      : {RANDOM_STATE}")
print(f"N_PCS             : {N_PCS}")
print(f"LEIDEN_RESOLUTION : {LEIDEN_RESOLUTION}")

print("\nInput files:")
print(f"  Learned 9E: {LEARNED_IMPUTED_PATH}")
for label, path in BASELINE_PATHS.items():
    print(f"  {label}: {path}")

# ------------------------------------------------------------------------------
# 2. Required variable checks
# ------------------------------------------------------------------------------

required_vars = [
    "denoised_adata",
]

missing = [v for v in required_vars if v not in globals()]
if missing:
    raise NameError(
        "Missing required variable(s):\n"
        + "\n".join(missing)
        + "\n\nRun the CosMx recovery/load cell first."
    )

if "was_corrected" not in globals():
    print("WARNING: was_corrected not found. Corrected-pair metrics will be skipped.")
    was_corrected = None

# ------------------------------------------------------------------------------
# 3. Helper functions
# ------------------------------------------------------------------------------

def ensure_dense(X):
    if sparse.issparse(X):
        return X.toarray()
    return np.asarray(X)


def get_cell_type_column(adata):
    """
    CosMx-safe cell-type column selection.
    """
    candidate_cols = [
        "cell_type",
        "Final_CosMx_Cell_Type",
        "Assigned_Xenium_Cell_Type",
        "Reference_Cell_Type",
    ]

    for col in candidate_cols:
        if col in adata.obs.columns:
            return col

    raise KeyError(
        "No cell-type column found. Expected one of: "
        "cell_type, Final_CosMx_Cell_Type, Assigned_Xenium_Cell_Type, Reference_Cell_Type."
    )


ct_col = get_cell_type_column(denoised_adata)
print(f"\nCell-type column: {ct_col}")

# ------------------------------------------------------------------------------
# 4. Use denoised_adata as the authoritative Step4 object
# ------------------------------------------------------------------------------

print("\n" + "=" * 90)
print("Checking authoritative Step4 object")
print("=" * 90)

if "raw" not in denoised_adata.layers:
    raise KeyError(
        "denoised_adata.layers['raw'] is missing. "
        "This cell needs the raw molecule-count layer from the corrected Step4 object."
    )

# Authoritative order for all downstream validation.
matrix_cell_ids = np.array([str(c) for c in denoised_adata.obs_names], dtype=str)
matrix_genes = np.array([str(g) for g in denoised_adata.var_names], dtype=str)

cell_idx_map = {cid: i for i, cid in enumerate(matrix_cell_ids)}
gene_idx_map = {g: j for j, g in enumerate(matrix_genes)}

print(f"Authoritative cells from denoised_adata.obs_names: {len(matrix_cell_ids):,}")
print(f"Authoritative genes from denoised_adata.var_names: {len(matrix_genes):,}")

print("\nFirst 10 authoritative genes:")
print(list(matrix_genes[:10]))

# Optional consistency checks against older variables, if present.
if "cell_ids_step4" in globals():
    old_cell_ids = np.array([str(c) for c in cell_ids_step4], dtype=str)
    print("\ncell_ids_step4 exists.")
    print(f"  same length as denoised_adata.obs_names? {len(old_cell_ids) == len(matrix_cell_ids)}")
    print(f"  exact same cell order? {np.array_equal(old_cell_ids, matrix_cell_ids)}")

if "shared_genes" in globals():
    old_genes = np.array([str(g) for g in shared_genes], dtype=str)
    print("\nshared_genes exists.")
    print(f"  same length as denoised_adata.var_names? {len(old_genes) == len(matrix_genes)}")
    print(f"  exact same gene order? {np.array_equal(old_genes, matrix_genes)}")

# ------------------------------------------------------------------------------
# 5. Convert base matrices from denoised_adata directly
# ------------------------------------------------------------------------------

print("\nLoading/confirming raw and Step4 matrices from denoised_adata...")

X_raw_full = ensure_dense(denoised_adata.layers["raw"]).astype(np.float32)
X_step4_full = ensure_dense(denoised_adata.X).astype(np.float32)

if X_raw_full.shape != X_step4_full.shape:
    raise ValueError(
        f"Shape mismatch: raw {X_raw_full.shape}, Step4 {X_step4_full.shape}"
    )

if X_raw_full.shape != denoised_adata.shape:
    raise ValueError(
        f"Raw matrix shape {X_raw_full.shape} does not match denoised_adata shape {denoised_adata.shape}"
    )

print(f"Raw matrix shape     : {X_raw_full.shape}")
print(f"Step4 matrix shape   : {X_step4_full.shape}")
print(f"Raw matrix sum       : {X_raw_full.sum(dtype=np.float64):,.0f}")
print(f"Step4 matrix sum     : {X_step4_full.sum(dtype=np.float64):,.2f}")
print(f"Step4 added vs raw   : {(X_step4_full - X_raw_full).sum(dtype=np.float64):,.2f}")
print(f"Negative delta entries: {int(((X_step4_full - X_raw_full) < -1e-6).sum()):,}")

# Optional consistency check against X_raw_counts / X_denoised if present.
if "X_raw_counts" in globals():
    X_raw_counts_check = ensure_dense(X_raw_counts).astype(np.float32)
    if X_raw_counts_check.shape == X_raw_full.shape:
        max_diff_raw = np.max(np.abs(X_raw_counts_check - X_raw_full))
        print(f"\nMax abs diff: X_raw_counts vs denoised_adata.layers['raw'] = {max_diff_raw:.6f}")
    else:
        print("\nWARNING: X_raw_counts exists but shape differs from denoised_adata raw layer.")

if "X_denoised" in globals():
    X_denoised_check = ensure_dense(X_denoised).astype(np.float32)
    if X_denoised_check.shape == X_step4_full.shape:
        max_diff_den = np.max(np.abs(X_denoised_check - X_step4_full))
        print(f"Max abs diff: X_denoised vs denoised_adata.X = {max_diff_den:.6f}")
    else:
        print("WARNING: X_denoised exists but shape differs from denoised_adata.X.")

# Validate correction mask shape, if available.
if was_corrected is not None:
    if was_corrected.shape != X_raw_full.shape:
        print(
            f"\nWARNING: was_corrected shape {was_corrected.shape} "
            f"does not match matrix shape {X_raw_full.shape}. It will be ignored."
        )
        was_corrected = None
    else:
        was_corrected = was_corrected.astype(bool)
        print(f"\nwas_corrected entries: {was_corrected.sum():,}")

# ------------------------------------------------------------------------------
# 6. Fixed Step4-style sampling
# ------------------------------------------------------------------------------

print("\nCreating fixed 50k-cell sample using Step4 logic...")

n_cells = X_raw_full.shape[0]
n_sample = min(SAMPLE_SIZE, n_cells)

rng = np.random.default_rng(RANDOM_STATE)
sample_idx = rng.choice(n_cells, size=n_sample, replace=False)

obs_sub = denoised_adata.obs.iloc[sample_idx].copy()
var_sub = denoised_adata.var.copy()
true_labels = obs_sub[ct_col].astype(str).values

print(f"Sampled cells: {n_sample:,} / {n_cells:,}")
print(f"Sampling seed: {RANDOM_STATE}")
print(f"Unique true labels in sample: {pd.Series(true_labels).nunique()}")

# Same filename as before; this overwrites previous sample index.
np.save(SAMPLE_INDEX_PATH, sample_idx)
print(f"Saved fixed sample index:")
print(f"  {SAMPLE_INDEX_PATH}")

# ------------------------------------------------------------------------------
# 7. Count matrix helper functions
# ------------------------------------------------------------------------------

def add_imputed_counts_from_file(base_counts, imputed_path, label):
    """
    Builds completed count matrix:
        completed = X_raw_from_denoised_adata + counts(imputed_records)

    CosMx-safe:
      - reads only cell_id/gene_id
      - keeps cell_id as string
      - uses denoised_adata.obs_names / var_names as authoritative order
    """
    if not os.path.exists(imputed_path):
        raise FileNotFoundError(f"Missing imputed records for {label}: {imputed_path}")

    print("\n" + "-" * 80)
    print(f"Loading imputed counts for {label}")
    print("-" * 80)
    print(f"  path: {imputed_path}")
    print(f"  size: {os.path.getsize(imputed_path) / 1e9:.3f} GB")

    imp = pd.read_parquet(imputed_path, columns=["cell_id", "gene_id"])

    print(f"  Loaded imputed rows: {len(imp):,}")

    imp["cell_id"] = imp["cell_id"].astype(str)
    imp["gene_id"] = imp["gene_id"].astype(str)

    counts = (
        imp
        .groupby(["cell_id", "gene_id"])
        .size()
        .reset_index(name="count")
    )

    print(f"  Unique imputed cell-gene pairs: {len(counts):,}")

    rr = counts["cell_id"].map(cell_idx_map)
    cc = counts["gene_id"].map(gene_idx_map)

    ok = rr.notna() & cc.notna()

    n_unmapped = len(counts) - int(ok.sum())
    if n_unmapped > 0:
        print(
            f"  WARNING: dropped {n_unmapped:,} "
            "unmapped cell-gene count rows."
        )

    X = np.asarray(base_counts, dtype=np.float32).copy()

    X[
        rr[ok].astype(int).to_numpy(),
        cc[ok].astype(int).to_numpy(),
    ] += counts.loc[ok, "count"].to_numpy(dtype=np.float32)

    print(f"  Completed matrix sum: {X.sum(dtype=np.float64):,.0f}")

    del imp, counts, rr, cc, ok
    gc.collect()

    return X


def count_matrix_metrics(name, X, X_target, X_raw, corrected_mask=None):
    """
    Count-level comparison against Step4 denoised target and raw counts.
    """
    X = np.asarray(X, dtype=np.float32)
    X_target = np.asarray(X_target, dtype=np.float32)
    X_raw = np.asarray(X_raw, dtype=np.float32)

    diff_target = X - X_target
    diff_raw = X - X_raw

    row = {
        "dataset": name,
        "total_counts": float(X.sum(dtype=np.float64)),
        "mean_counts_per_cell": float(X.sum(axis=1, dtype=np.float64).mean()),
        "nonzero_entries": int((X > 0).sum()),
        "mae_vs_step4_all": float(np.mean(np.abs(diff_target))),
        "rmse_vs_step4_all": float(np.sqrt(np.mean(diff_target ** 2))),
        "total_abs_diff_vs_step4": float(np.abs(diff_target).sum(dtype=np.float64)),
        "total_added_vs_raw": float(np.maximum(diff_raw, 0).sum(dtype=np.float64)),
    }

    if corrected_mask is not None and corrected_mask.shape == X.shape:
        m = corrected_mask.astype(bool)

        if int(m.sum()) > 0:
            row["mae_vs_step4_corrected_pairs"] = float(np.mean(np.abs(diff_target[m])))
            row["rmse_vs_step4_corrected_pairs"] = float(np.sqrt(np.mean(diff_target[m] ** 2)))
            row["n_corrected_pairs"] = int(m.sum())
        else:
            row["mae_vs_step4_corrected_pairs"] = np.nan
            row["rmse_vs_step4_corrected_pairs"] = np.nan
            row["n_corrected_pairs"] = 0
    else:
        row["mae_vs_step4_corrected_pairs"] = np.nan
        row["rmse_vs_step4_corrected_pairs"] = np.nan
        row["n_corrected_pairs"] = np.nan

    return row

# ------------------------------------------------------------------------------
# 8. Cell-level-exact clustering functions
# ------------------------------------------------------------------------------

def run_scanpy_clustering_celllevel_style(
    adata_tmp,
    random_state=42,
    n_pcs=30,
    leiden_resolution=0.5,
):
    """
    Matches the current cell-level validation style:
      - normalize_total(target_sum=1e4)
      - log1p
      - sc.pp.pca(..., random_state=42)
      - sc.pp.neighbors(..., n_pcs=n_pcs_use)
      - sc.tl.leiden(..., resolution=0.5, random_state=42)

    IMPORTANT:
      Do not add extra Leiden/neighbor parameters here if the goal is to stay
      closest to the already-current cell-level file's output.
    """
    n_pcs_use = min(n_pcs, adata_tmp.n_vars - 1)

    sc.pp.normalize_total(adata_tmp, target_sum=1e4)
    sc.pp.log1p(adata_tmp)

    sc.pp.pca(
        adata_tmp,
        n_comps=n_pcs_use,
        random_state=random_state,
    )

    sc.pp.neighbors(
        adata_tmp,
        n_pcs=n_pcs_use,
    )

    sc.tl.leiden(
        adata_tmp,
        resolution=leiden_resolution,
        random_state=random_state,
        key_added="leiden",
    )

    return adata_tmp, n_pcs_use


def evaluate_raw_step4_clustering_celllevel_exact(
    X_raw,
    X_step4,
    sample_idx,
    obs_sub,
    var_sub,
    true_labels,
    var_names,
    random_state=42,
    n_pcs=30,
    leiden_resolution=0.5,
):
    """
    Recomputes Raw and Step4 together using the same structure as the cell-level file.

    This is the key fix:
      - Raw and Step4 are evaluated together.
      - Same sample_idx.
      - Same obs_sub.
      - Same var_sub.
      - Same var_names from denoised_adata.var_names.
      - Same Scanpy processing calls.
    """
    print("\n" + "-" * 80)
    print("Evaluating Raw and Step4 clustering together with cell-level-exact logic")
    print("-" * 80)

    X_raw = np.asarray(X_raw, dtype=np.float32)
    X_step4 = np.asarray(X_step4, dtype=np.float32)

    adata_raw = ad.AnnData(
        X=X_raw[sample_idx, :].copy(),
        obs=obs_sub.copy(),
        var=var_sub.copy(),
    )

    adata_den = ad.AnnData(
        X=X_step4[sample_idx, :].copy(),
        obs=obs_sub.copy(),
        var=var_sub.copy(),
    )

    # Authoritative gene names from denoised_adata.
    adata_raw.var_names = pd.Index(var_names).astype(str)
    adata_den.var_names = pd.Index(var_names).astype(str)

    adata_raw.var_names_make_unique()
    adata_den.var_names_make_unique()

    print(f"  Raw AnnData shape      : {adata_raw.shape}")
    print(f"  Step4 AnnData shape    : {adata_den.shape}")
    print(f"  Raw sampled counts     : {X_raw[sample_idx, :].sum(dtype=np.float64):,.2f}")
    print(f"  Step4 sampled counts   : {X_step4[sample_idx, :].sum(dtype=np.float64):,.2f}")

    print("  Processing Raw...")
    adata_raw, n_pcs_use_raw = run_scanpy_clustering_celllevel_style(
        adata_raw,
        random_state=random_state,
        n_pcs=n_pcs,
        leiden_resolution=leiden_resolution,
    )

    print("  Processing Step4 denoised...")
    adata_den, n_pcs_use_den = run_scanpy_clustering_celllevel_style(
        adata_den,
        random_state=random_state,
        n_pcs=n_pcs,
        leiden_resolution=leiden_resolution,
    )

    clusters_raw = adata_raw.obs["leiden"].astype(str).values
    clusters_den = adata_den.obs["leiden"].astype(str).values

    unique_labels = np.unique(true_labels)

    ari_raw = adjusted_rand_score(true_labels, clusters_raw)
    ari_den = adjusted_rand_score(true_labels, clusters_den)

    nmi_raw = normalized_mutual_info_score(true_labels, clusters_raw)
    nmi_den = normalized_mutual_info_score(true_labels, clusters_den)

    if len(unique_labels) >= 2 and len(unique_labels) < len(true_labels):
        sil_raw = silhouette_score(
            adata_raw.obsm["X_pca"],
            true_labels,
            metric="euclidean",
        )
        sil_den = silhouette_score(
            adata_den.obsm["X_pca"],
            true_labels,
            metric="euclidean",
        )
    else:
        sil_raw = np.nan
        sil_den = np.nan

    raw_row = {
        "dataset": "Raw observed counts",
        "n_cells_sampled": int(adata_raw.n_obs),
        "n_genes": int(adata_raw.n_vars),
        "total_counts_sampled": float(X_raw[sample_idx, :].sum(dtype=np.float64)),
        "ARI": float(ari_raw),
        "NMI": float(nmi_raw),
        "silhouette": float(sil_raw) if np.isfinite(sil_raw) else np.nan,
        "n_clusters": int(pd.Series(clusters_raw).nunique()),
        "n_true_labels": int(len(unique_labels)),
        "leiden_resolution": float(leiden_resolution),
        "n_pcs": int(n_pcs_use_raw),
        "random_state": int(random_state),
        "clustering_config": "cell_level_exact_raw_step4_joint_recomputed",
    }

    step4_row = {
        "dataset": "Step4 denoised counts",
        "n_cells_sampled": int(adata_den.n_obs),
        "n_genes": int(adata_den.n_vars),
        "total_counts_sampled": float(X_step4[sample_idx, :].sum(dtype=np.float64)),
        "ARI": float(ari_den),
        "NMI": float(nmi_den),
        "silhouette": float(sil_den) if np.isfinite(sil_den) else np.nan,
        "n_clusters": int(pd.Series(clusters_den).nunique()),
        "n_true_labels": int(len(unique_labels)),
        "leiden_resolution": float(leiden_resolution),
        "n_pcs": int(n_pcs_use_den),
        "random_state": int(random_state),
        "clustering_config": "cell_level_exact_raw_step4_joint_recomputed",
    }

    print("\n  Raw observed counts:")
    print(f"    ARI        : {raw_row['ARI']:.6f}")
    print(f"    NMI        : {raw_row['NMI']:.6f}")
    print(f"    silhouette : {raw_row['silhouette']:.6f}")
    print(f"    clusters   : {raw_row['n_clusters']}")

    print("\n  Step4 denoised counts:")
    print(f"    ARI        : {step4_row['ARI']:.6f}")
    print(f"    NMI        : {step4_row['NMI']:.6f}")
    print(f"    silhouette : {step4_row['silhouette']:.6f}")
    print(f"    clusters   : {step4_row['n_clusters']}")

    del adata_raw, adata_den
    gc.collect()

    return [raw_row, step4_row]


def evaluate_clustering_quality_one_matrix(
    dataset_name,
    X_full,
    sample_idx,
    obs_sub,
    var_sub,
    true_labels,
    var_names,
    random_state=42,
    n_pcs=30,
    leiden_resolution=0.5,
):
    """
    Clustering evaluation for learned/baseline completed matrices.

    Uses:
      - same fixed sample_idx
      - same obs_sub
      - same true_labels
      - authoritative denoised_adata.var_names
      - same Scanpy calls as raw/Step4 evaluator
    """
    print("\n" + "-" * 80)
    print(f"Evaluating clustering quality: {dataset_name}")
    print("-" * 80)

    X_full = np.asarray(X_full, dtype=np.float32)

    adata_tmp = ad.AnnData(
        X=X_full[sample_idx, :].copy(),
        obs=obs_sub.copy(),
        var=var_sub.copy(),
    )

    # Critical fix:
    # Use denoised_adata.var_names order, not separately loaded shared_genes.
    adata_tmp.var_names = pd.Index(var_names).astype(str)
    adata_tmp.var_names_make_unique()

    print(f"  AnnData shape: {adata_tmp.shape}")
    print(f"  Total counts in sampled matrix: {adata_tmp.X.sum(dtype=np.float64):,.2f}")

    adata_tmp, n_pcs_use = run_scanpy_clustering_celllevel_style(
        adata_tmp,
        random_state=random_state,
        n_pcs=n_pcs,
        leiden_resolution=leiden_resolution,
    )

    pred_clusters = adata_tmp.obs["leiden"].astype(str).values

    ari = adjusted_rand_score(true_labels, pred_clusters)
    nmi = normalized_mutual_info_score(true_labels, pred_clusters)

    unique_labels = np.unique(true_labels)

    if len(unique_labels) >= 2 and len(unique_labels) < len(true_labels):
        sil = silhouette_score(
            adata_tmp.obsm["X_pca"],
            true_labels,
            metric="euclidean",
        )
    else:
        sil = np.nan

    n_clusters = int(pd.Series(pred_clusters).nunique())

    row = {
        "dataset": dataset_name,
        "n_cells_sampled": int(adata_tmp.n_obs),
        "n_genes": int(adata_tmp.n_vars),
        "total_counts_sampled": float(X_full[sample_idx, :].sum(dtype=np.float64)),
        "ARI": float(ari),
        "NMI": float(nmi),
        "silhouette": float(sil) if np.isfinite(sil) else np.nan,
        "n_clusters": int(n_clusters),
        "n_true_labels": int(len(unique_labels)),
        "leiden_resolution": float(leiden_resolution),
        "n_pcs": int(n_pcs_use),
        "random_state": int(random_state),
        "clustering_config": "cell_level_style_completed_matrix_recomputed",
    }

    print(f"  ARI        : {ari:.6f}")
    print(f"  NMI        : {nmi:.6f}")
    print(f"  silhouette : {sil:.6f}" if np.isfinite(sil) else "  silhouette : nan")
    print(f"  clusters   : {n_clusters}")

    del adata_tmp
    gc.collect()

    return row

# ------------------------------------------------------------------------------
# 9. Evaluate raw and Step4 first
# ------------------------------------------------------------------------------

count_metric_rows = []
clustering_metric_rows = []

print("\n" + "=" * 90)
print("1. Evaluating raw observed counts and Step4 denoised counts")
print("=" * 90)

count_metric_rows.append(
    count_matrix_metrics(
        name="Raw observed counts",
        X=X_raw_full,
        X_target=X_step4_full,
        X_raw=X_raw_full,
        corrected_mask=was_corrected,
    )
)

count_metric_rows.append(
    count_matrix_metrics(
        name="Step4 denoised counts",
        X=X_step4_full,
        X_target=X_step4_full,
        X_raw=X_raw_full,
        corrected_mask=was_corrected,
    )
)

# Key fix: raw and Step4 are clustered together, cell-level-style.
clustering_metric_rows.extend(
    evaluate_raw_step4_clustering_celllevel_exact(
        X_raw=X_raw_full,
        X_step4=X_step4_full,
        sample_idx=sample_idx,
        obs_sub=obs_sub,
        var_sub=var_sub,
        true_labels=true_labels,
        var_names=matrix_genes,
        random_state=RANDOM_STATE,
        n_pcs=N_PCS,
        leiden_resolution=LEIDEN_RESOLUTION,
    )
)

# ------------------------------------------------------------------------------
# 10. Evaluate learned 9E completed counts
# ------------------------------------------------------------------------------

print("\n" + "=" * 90)
print("2. Evaluating learned 9E completed counts")
print("=" * 90)

X_learned_full = add_imputed_counts_from_file(
    base_counts=X_raw_full,
    imputed_path=LEARNED_IMPUTED_PATH,
    label="Learned 9E completed counts",
)

count_metric_rows.append(
    count_matrix_metrics(
        name="Learned 9E completed counts",
        X=X_learned_full,
        X_target=X_step4_full,
        X_raw=X_raw_full,
        corrected_mask=was_corrected,
    )
)

clustering_metric_rows.append(
    evaluate_clustering_quality_one_matrix(
        dataset_name="Learned 9E completed counts",
        X_full=X_learned_full,
        sample_idx=sample_idx,
        obs_sub=obs_sub,
        var_sub=var_sub,
        true_labels=true_labels,
        var_names=matrix_genes,
        random_state=RANDOM_STATE,
        n_pcs=N_PCS,
        leiden_resolution=LEIDEN_RESOLUTION,
    )
)

del X_learned_full
gc.collect()

# ------------------------------------------------------------------------------
# 11. Evaluate empirical baseline completed counts
# ------------------------------------------------------------------------------

print("\n" + "=" * 90)
print("3. Evaluating empirical baseline completed counts")
print("=" * 90)

for dataset_name, path in BASELINE_PATHS.items():
    X_base_full = add_imputed_counts_from_file(
        base_counts=X_raw_full,
        imputed_path=path,
        label=dataset_name,
    )

    count_metric_rows.append(
        count_matrix_metrics(
            name=dataset_name,
            X=X_base_full,
            X_target=X_step4_full,
            X_raw=X_raw_full,
            corrected_mask=was_corrected,
        )
    )

    clustering_metric_rows.append(
        evaluate_clustering_quality_one_matrix(
            dataset_name=dataset_name,
            X_full=X_base_full,
            sample_idx=sample_idx,
            obs_sub=obs_sub,
            var_sub=var_sub,
            true_labels=true_labels,
            var_names=matrix_genes,
            random_state=RANDOM_STATE,
            n_pcs=N_PCS,
            leiden_resolution=LEIDEN_RESOLUTION,
        )
    )

    del X_base_full
    gc.collect()

# ------------------------------------------------------------------------------
# 12. Save metrics
# ------------------------------------------------------------------------------

print("\n" + "=" * 90)
print("4. Saving downstream validation metrics")
print("=" * 90)

count_metrics_df = pd.DataFrame(count_metric_rows)
clustering_metrics_df = pd.DataFrame(clustering_metric_rows)

# Same filenames as before. These overwrite existing files.
count_metrics_df.to_csv(COUNT_METRICS_PATH, index=False)
clustering_metrics_df.to_csv(CLUSTERING_METRICS_PATH, index=False)

print("\nCount matrix metrics:")
display(count_metrics_df)

print("\nClustering metrics:")
display(clustering_metrics_df)

print(f"\nSaved count matrix metrics:")
print(f"  {COUNT_METRICS_PATH}")

print(f"Saved clustering metrics:")
print(f"  {CLUSTERING_METRICS_PATH}")

# ------------------------------------------------------------------------------
# 13. Print learned-vs-baseline summary
# ------------------------------------------------------------------------------

print("\n" + "=" * 90)
print("5. Summary: learned 9E vs raw / Step4 / empirical baselines")
print("=" * 90)

raw_row = clustering_metrics_df[
    clustering_metrics_df["dataset"] == "Raw observed counts"
].iloc[0]

step4_row = clustering_metrics_df[
    clustering_metrics_df["dataset"] == "Step4 denoised counts"
].iloc[0]

learned_row = clustering_metrics_df[
    clustering_metrics_df["dataset"] == "Learned 9E completed counts"
].iloc[0]

print("\nClustering change relative to raw observed counts:")
for _, row in clustering_metrics_df.iterrows():
    if row["dataset"] == "Raw observed counts":
        continue

    print(
        f"{row['dataset']:45s} "
        f"ΔARI={row['ARI'] - raw_row['ARI']:+.4f}, "
        f"ΔNMI={row['NMI'] - raw_row['NMI']:+.4f}, "
        f"Δsilhouette={row['silhouette'] - raw_row['silhouette']:+.4f}, "
        f"Δclusters={int(row['n_clusters']) - int(raw_row['n_clusters']):+d}"
    )

print("\nClustering change relative to Step4 denoised counts:")
for _, row in clustering_metrics_df.iterrows():
    if row["dataset"] == "Step4 denoised counts":
        continue

    print(
        f"{row['dataset']:45s} "
        f"ΔARI={row['ARI'] - step4_row['ARI']:+.4f}, "
        f"ΔNMI={row['NMI'] - step4_row['NMI']:+.4f}, "
        f"Δsilhouette={row['silhouette'] - step4_row['silhouette']:+.4f}, "
        f"Δclusters={int(row['n_clusters']) - int(step4_row['n_clusters']):+d}"
    )

# ------------------------------------------------------------------------------
# 14. Save readable report
# ------------------------------------------------------------------------------

# Same filename as before. This overwrites existing report.
with open(REPORT_PATH, "w") as f:
    f.write("COSMX 9E DOWNSTREAM VALIDATION REPORT — STEP4-MATCHED\n")
    f.write("=" * 90 + "\n\n")

    f.write(f"CHECKPOINT_DIR    : {CHECKPOINT_DIR}\n")
    f.write(f"RUN_NAME          : {RUN_NAME}\n")
    f.write(f"SAMPLE_SIZE       : {SAMPLE_SIZE}\n")
    f.write(f"RANDOM_STATE      : {RANDOM_STATE}\n")
    f.write(f"N_PCS             : {N_PCS}\n")
    f.write(f"LEIDEN_RESOLUTION : {LEIDEN_RESOLUTION}\n")
    f.write(f"Cell-type column  : {ct_col}\n\n")

    f.write("AUTHORITATIVE MATRIX ORDER\n")
    f.write("-" * 90 + "\n")
    f.write("Cell order source : denoised_adata.obs_names\n")
    f.write("Gene order source : denoised_adata.var_names\n")
    f.write("Raw source        : denoised_adata.layers['raw']\n")
    f.write("Step4 source      : denoised_adata.X\n\n")

    f.write("INPUT FILES\n")
    f.write("-" * 90 + "\n")
    f.write(f"Learned 9E: {LEARNED_IMPUTED_PATH}\n")

    for label, path in BASELINE_PATHS.items():
        f.write(f"{label}: {path}\n")

    f.write("\nCOUNT MATRIX METRICS\n")
    f.write("-" * 90 + "\n")
    f.write(count_metrics_df.to_string(index=False))
    f.write("\n\n")

    f.write("CLUSTERING METRICS\n")
    f.write("-" * 90 + "\n")
    f.write(clustering_metrics_df.to_string(index=False))
    f.write("\n\n")

    f.write("CHANGE RELATIVE TO RAW OBSERVED COUNTS\n")
    f.write("-" * 90 + "\n")

    for _, row in clustering_metrics_df.iterrows():
        if row["dataset"] == "Raw observed counts":
            continue

        f.write(
            f"{row['dataset']}: "
            f"ΔARI={row['ARI'] - raw_row['ARI']:.4f}, "
            f"ΔNMI={row['NMI'] - raw_row['NMI']:.4f}, "
            f"Δsilhouette={row['silhouette'] - raw_row['silhouette']:.4f}, "
            f"Δclusters={int(row['n_clusters']) - int(raw_row['n_clusters'])}\n"
        )

    f.write("\nCHANGE RELATIVE TO STEP4 DENOISED COUNTS\n")
    f.write("-" * 90 + "\n")

    for _, row in clustering_metrics_df.iterrows():
        if row["dataset"] == "Step4 denoised counts":
            continue

        f.write(
            f"{row['dataset']}: "
            f"ΔARI={row['ARI'] - step4_row['ARI']:.4f}, "
            f"ΔNMI={row['NMI'] - step4_row['NMI']:.4f}, "
            f"Δsilhouette={row['silhouette'] - step4_row['silhouette']:.4f}, "
            f"Δclusters={int(row['n_clusters']) - int(step4_row['n_clusters'])}\n"
        )

print(f"\nSaved downstream validation report:")
print(f"  {REPORT_PATH}")

print("\n" + "=" * 90)
print("COSMX 9E DOWNSTREAM VALIDATION COMPLETE — STEP4-MATCHED")
print("=" * 90)

print("Saved outputs:")
print(f"  Count matrix metrics : {COUNT_METRICS_PATH}")
print(f"  Clustering metrics   : {CLUSTERING_METRICS_PATH}")
print(f"  Fixed sample index   : {SAMPLE_INDEX_PATH}")
print(f"  Report               : {REPORT_PATH}")

In [ ]:
# ==============================================================================
# COSMX FINAL SAFETY CHECKPOINT CELL — 9E Step 5 / imputation / downstream
# Purpose:
#   1. Verify all important CosMx Step 5 files are saved in Google Drive.
#   2. Check key row counts / count reconciliation.
#   3. Save a complete manifest of files, sizes, row counts, and runtime variables.
#   4. Create a final "safe to disconnect" report.
#
# GPU not needed.
# ==============================================================================

import os
import gc
import json
import time
import glob
import platform
from datetime import datetime

import numpy as np
import pandas as pd

print("=" * 100)
print("COSMX FINAL SAFETY CHECKPOINT — STEP 5 9E")
print("=" * 100)

# ------------------------------------------------------------------------------
# 0. Paths / run name
# ------------------------------------------------------------------------------

CHECKPOINT_DIR = "/content/drive/MyDrive/diffusion/step4_cosmx/imputation"
STEP4_EXPORT_DIR = "/content/drive/MyDrive/diffusion/step4_cosmx/step4_exports"

RUN_NAME = "cosmx_9E_distribution_empirical_baselines"

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

MANIFEST_JSON_PATH = os.path.join(
    CHECKPOINT_DIR,
    f"{RUN_NAME}_FINAL_MANIFEST_{timestamp}.json"
)

MANIFEST_CSV_PATH = os.path.join(
    CHECKPOINT_DIR,
    f"{RUN_NAME}_FINAL_FILE_MANIFEST_{timestamp}.csv"
)

SAFE_REPORT_PATH = os.path.join(
    CHECKPOINT_DIR,
    f"{RUN_NAME}_SAFE_TO_DISCONNECT_REPORT_{timestamp}.txt"
)

LATEST_SAFE_REPORT_PATH = os.path.join(
    CHECKPOINT_DIR,
    f"{RUN_NAME}_SAFE_TO_DISCONNECT_REPORT_LATEST.txt"
)

print(f"CHECKPOINT_DIR  : {CHECKPOINT_DIR}")
print(f"STEP4_EXPORT_DIR: {STEP4_EXPORT_DIR}")
print(f"RUN_NAME        : {RUN_NAME}")
print(f"Timestamp       : {timestamp}")

# ------------------------------------------------------------------------------
# 1. File helpers
# ------------------------------------------------------------------------------

def file_size_gb(path):
    return os.path.getsize(path) / 1e9 if os.path.exists(path) else np.nan


def get_parquet_metadata(path):
    """
    Fast Parquet metadata reader. Does not load full table into RAM.
    """
    meta = {
        "parquet_rows": np.nan,
        "parquet_columns": np.nan,
        "parquet_error": "",
    }

    if not os.path.exists(path):
        meta["parquet_error"] = "file_missing"
        return meta

    try:
        import pyarrow.parquet as pq
        pf = pq.ParquetFile(path)
        meta["parquet_rows"] = int(pf.metadata.num_rows)
        meta["parquet_columns"] = int(pf.metadata.num_columns)
    except Exception as e:
        meta["parquet_error"] = str(e)

    return meta


def check_file(path, category, required=True):
    exists = os.path.exists(path)

    row = {
        "category": category,
        "required": bool(required),
        "exists": bool(exists),
        "path": path,
        "size_gb": file_size_gb(path) if exists else np.nan,
        "parquet_rows": np.nan,
        "parquet_columns": np.nan,
        "notes": "",
    }

    if exists and path.endswith(".parquet"):
        meta = get_parquet_metadata(path)
        row["parquet_rows"] = meta["parquet_rows"]
        row["parquet_columns"] = meta["parquet_columns"]
        row["notes"] = meta["parquet_error"]

    return row


# ------------------------------------------------------------------------------
# 2. Important expected files
# ------------------------------------------------------------------------------

critical_files = []

# Step 4 exports needed to reproduce / reload Step 5.
for fname in [
    "molecules.parquet",
    "X_raw_counts.npy",
    "denoised_adata.h5ad",
    "was_corrected.npy",
    "step4_config.json",
]:
    critical_files.append((
        os.path.join(STEP4_EXPORT_DIR, fname),
        "Step4 CosMx exports",
        True,
    ))

# Optional Step4 geometry/cell files. Names may differ depending on your export cell.
for fname in [
    "cell_data.npz",
    "clean_preprocessing_summary.json",
]:
    critical_files.append((
        os.path.join(STEP4_EXPORT_DIR, fname),
        "Optional Step4 CosMx exports",
        False,
    ))

# Step 5 processed molecule / geometry checkpoints.
for fname in [
    "step5_mol_processed.parquet",
    "step5_mol_processed_summary.json",
    "checkpoint_cosmx_geometry_after_5A.pkl",
    "nuc_centroids.pkl",
    f"{RUN_NAME}_cosmx_summary_geometry_cache.pkl",
]:
    critical_files.append((
        os.path.join(CHECKPOINT_DIR, fname),
        "Step5 preprocessing/geometry checkpoints",
        True,
    ))

# 9E model / training / final evaluation artifacts.
for fname in [
    f"{RUN_NAME}_best_coordNN_model.pt",
    f"{RUN_NAME}_final_large_recovery_metrics.csv",
    f"{RUN_NAME}_final_large_recovery_metrics.json",
    f"{RUN_NAME}_final_large_recovery_report.txt",
]:
    critical_files.append((
        os.path.join(CHECKPOINT_DIR, fname),
        "9E trained model/final evaluation",
        True,
    ))

# 9E learned imputation outputs.
for fname in [
    f"{RUN_NAME}_imputation_targets.csv",
    f"{RUN_NAME}_imputed_records.parquet",
    f"{RUN_NAME}_completed_molecule_table.parquet",
    f"{RUN_NAME}_count_reconciliation.csv",
    f"{RUN_NAME}_imputation_summary.csv",
    f"{RUN_NAME}_imputation_summary.json",
    f"{RUN_NAME}_infer2_generation_summary.json",
]:
    critical_files.append((
        os.path.join(CHECKPOINT_DIR, fname),
        "9E learned imputation outputs",
        True,
    ))

# 3 empirical baseline imputation outputs.
for fname in [
    f"{RUN_NAME}_gene_emp_imputed_records.parquet",
    f"{RUN_NAME}_ct_gene_emp_imputed_records.parquet",
    f"{RUN_NAME}_spatial_knn_emp_imputed_records.parquet",
    f"{RUN_NAME}_baseline_imputation_summary.csv",
    f"{RUN_NAME}_baseline_imputation_summary.json",
]:
    critical_files.append((
        os.path.join(CHECKPOINT_DIR, fname),
        "Empirical baseline imputation outputs",
        True,
    ))

# Downstream outputs.
for fname in [
    f"{RUN_NAME}_downstream_count_matrix_metrics_step4matched.csv",
    f"{RUN_NAME}_downstream_clustering_metrics_step4matched.csv",
    f"{RUN_NAME}_downstream_step4matched_sample_idx.npy",
    f"{RUN_NAME}_downstream_validation_report_step4matched.txt",
]:
    critical_files.append((
        os.path.join(CHECKPOINT_DIR, fname),
        "Downstream validation outputs",
        True,
    ))

# Optional artifacts: include if present, do not fail if absent.
optional_globs = [
    os.path.join(CHECKPOINT_DIR, f"{RUN_NAME}*training*curve*.png"),
    os.path.join(CHECKPOINT_DIR, f"{RUN_NAME}*curve*.png"),
    os.path.join(CHECKPOINT_DIR, f"{RUN_NAME}*checkpoint*.pt"),
    os.path.join(CHECKPOINT_DIR, f"{RUN_NAME}*report*.json"),
    os.path.join(CHECKPOINT_DIR, f"{RUN_NAME}*report*.txt"),
    os.path.join(CHECKPOINT_DIR, f"{RUN_NAME}*metrics*.csv"),
    os.path.join(CHECKPOINT_DIR, f"{RUN_NAME}*summary*.json"),
]

optional_files = []
for pattern in optional_globs:
    optional_files.extend(glob.glob(pattern))

optional_files = sorted(set(optional_files))

required_paths_set = set(p for p, _, _ in critical_files)

for path in optional_files:
    if path not in required_paths_set:
        critical_files.append((path, "Optional discovered CosMx 9E artifacts", False))


# ------------------------------------------------------------------------------
# 3. Check file existence / sizes / row counts
# ------------------------------------------------------------------------------

print("\nChecking important files in Drive...")

file_rows = []

for path, category, required in critical_files:
    file_rows.append(check_file(path, category, required=required))

file_manifest_df = pd.DataFrame(file_rows)

required_missing = file_manifest_df[
    (file_manifest_df["required"] == True)
    & (file_manifest_df["exists"] == False)
].copy()

print("\nFile manifest:")
display(file_manifest_df)

file_manifest_df.to_csv(MANIFEST_CSV_PATH, index=False)

print(f"\nSaved file manifest CSV:")
print(f"  {MANIFEST_CSV_PATH}")


# ------------------------------------------------------------------------------
# 4. Check important count reconciliation files
# ------------------------------------------------------------------------------

print("\nChecking count reconciliation and summaries...")

check_results = {
    "count_reconciliation_ok": None,
    "count_reconciliation_mismatched_pairs": None,
    "count_reconciliation_max_abs_diff": None,
    "imputation_summary_ok": None,
    "baseline_summary_ok": None,
    "downstream_metrics_ok": None,
    "final_large_recovery_ok": None,
}

COUNT_CHECK_PATH = os.path.join(
    CHECKPOINT_DIR,
    f"{RUN_NAME}_count_reconciliation.csv"
)

IMPUTATION_SUMMARY_PATH = os.path.join(
    CHECKPOINT_DIR,
    f"{RUN_NAME}_imputation_summary.csv"
)

BASELINE_SUMMARY_PATH = os.path.join(
    CHECKPOINT_DIR,
    f"{RUN_NAME}_baseline_imputation_summary.csv"
)

DOWNSTREAM_CLUSTERING_PATH = os.path.join(
    CHECKPOINT_DIR,
    f"{RUN_NAME}_downstream_clustering_metrics_step4matched.csv"
)

DOWNSTREAM_COUNT_PATH = os.path.join(
    CHECKPOINT_DIR,
    f"{RUN_NAME}_downstream_count_matrix_metrics_step4matched.csv"
)

FINAL_RECOVERY_METRICS_PATH = os.path.join(
    CHECKPOINT_DIR,
    f"{RUN_NAME}_final_large_recovery_metrics.csv"
)

# Count reconciliation
if os.path.exists(COUNT_CHECK_PATH):
    count_check = pd.read_csv(COUNT_CHECK_PATH)

    if "diff" in count_check.columns:
        n_bad = int((count_check["diff"] != 0).sum())
        max_abs_diff = int(count_check["diff"].abs().max()) if len(count_check) else 0
        expected_total = int(count_check["n_imputed_expected"].sum()) if "n_imputed_expected" in count_check.columns else None
        actual_total = int(count_check["n_imputed_actual"].sum()) if "n_imputed_actual" in count_check.columns else None

        check_results["count_reconciliation_ok"] = bool(
            n_bad == 0
            and max_abs_diff == 0
            and expected_total == actual_total
        )
        check_results["count_reconciliation_mismatched_pairs"] = n_bad
        check_results["count_reconciliation_max_abs_diff"] = max_abs_diff
        check_results["count_reconciliation_expected_total"] = expected_total
        check_results["count_reconciliation_actual_total"] = actual_total

        print(f"Count reconciliation target pairs      : {len(count_check):,}")
        print(f"Count reconciliation expected molecules: {expected_total:,}")
        print(f"Count reconciliation actual molecules  : {actual_total:,}")
        print(f"Count reconciliation mismatched pairs   : {n_bad:,}")
        print(f"Count reconciliation max abs diff       : {max_abs_diff}")
    else:
        check_results["count_reconciliation_ok"] = False
        print("WARNING: count_reconciliation.csv exists but has no 'diff' column.")
else:
    check_results["count_reconciliation_ok"] = False
    print("WARNING: count_reconciliation.csv missing.")

# Imputation summary
if os.path.exists(IMPUTATION_SUMMARY_PATH):
    imp_summary = pd.read_csv(IMPUTATION_SUMMARY_PATH)
    check_results["imputation_summary_ok"] = True

    print("\nImputation summary:")
    display(imp_summary)
else:
    check_results["imputation_summary_ok"] = False
    print("WARNING: imputation_summary.csv missing.")

# Baseline summary
if os.path.exists(BASELINE_SUMMARY_PATH):
    baseline_summary = pd.read_csv(BASELINE_SUMMARY_PATH)
    check_results["baseline_summary_ok"] = True

    print("\nBaseline imputation summary:")
    display(baseline_summary)
else:
    check_results["baseline_summary_ok"] = False
    print("WARNING: baseline_imputation_summary.csv missing.")

# Downstream outputs
downstream_ok = os.path.exists(DOWNSTREAM_CLUSTERING_PATH) and os.path.exists(DOWNSTREAM_COUNT_PATH)
check_results["downstream_metrics_ok"] = bool(downstream_ok)

if os.path.exists(DOWNSTREAM_CLUSTERING_PATH):
    clustering_df = pd.read_csv(DOWNSTREAM_CLUSTERING_PATH)
    print("\nDownstream clustering metrics:")
    display(clustering_df)
else:
    print("WARNING: downstream clustering metrics missing.")

if os.path.exists(DOWNSTREAM_COUNT_PATH):
    count_metrics_df = pd.read_csv(DOWNSTREAM_COUNT_PATH)
    print("\nDownstream count matrix metrics:")
    display(count_metrics_df)
else:
    print("WARNING: downstream count matrix metrics missing.")

# Final large recovery metrics
if os.path.exists(FINAL_RECOVERY_METRICS_PATH):
    final_recovery_df = pd.read_csv(FINAL_RECOVERY_METRICS_PATH)
    check_results["final_large_recovery_ok"] = True
    print("\nFinal large recovery metrics:")
    display(final_recovery_df)
else:
    final_recovery_df = None
    check_results["final_large_recovery_ok"] = False
    print("WARNING: final large recovery metrics missing.")


# ------------------------------------------------------------------------------
# 5. Fast row-count sanity checks for important parquet files
# ------------------------------------------------------------------------------

print("\nRunning fast Parquet row-count sanity checks...")

expected_rows = {
    f"{RUN_NAME}_imputed_records.parquet": 4_990_120,
    f"{RUN_NAME}_gene_emp_imputed_records.parquet": 4_990_120,
    f"{RUN_NAME}_ct_gene_emp_imputed_records.parquet": 4_990_120,
    f"{RUN_NAME}_spatial_knn_emp_imputed_records.parquet": 4_990_120,
    f"{RUN_NAME}_completed_molecule_table.parquet": 35_119_986,
    "step5_mol_processed.parquet": 30_129_866,
}

row_count_checks = []

for fname, expected in expected_rows.items():
    path = os.path.join(CHECKPOINT_DIR, fname)

    meta = get_parquet_metadata(path)
    actual = meta["parquet_rows"]

    ok = bool(os.path.exists(path) and pd.notna(actual) and int(actual) == int(expected))

    row_count_checks.append({
        "file": fname,
        "path": path,
        "expected_rows": int(expected),
        "actual_rows": int(actual) if pd.notna(actual) else np.nan,
        "ok": ok,
        "notes": meta["parquet_error"],
    })

row_count_df = pd.DataFrame(row_count_checks)

print("\nParquet row-count checks:")
display(row_count_df)


# ------------------------------------------------------------------------------
# 6. Runtime variable snapshot
# ------------------------------------------------------------------------------

print("\nCapturing runtime variable snapshot...")

def describe_var(name):
    if name not in globals():
        return {
            "name": name,
            "exists_in_runtime": False,
            "type": None,
            "shape_or_len": None,
            "notes": "",
        }

    obj = globals()[name]

    info = {
        "name": name,
        "exists_in_runtime": True,
        "type": type(obj).__name__,
        "shape_or_len": None,
        "notes": "",
    }

    try:
        if hasattr(obj, "shape"):
            info["shape_or_len"] = str(obj.shape)
        elif hasattr(obj, "__len__"):
            info["shape_or_len"] = str(len(obj))
    except Exception as e:
        info["notes"] = str(e)

    return info


important_runtime_vars = [
    "mol",
    "completed_mol",
    "imputed_df",
    "imputed_records",
    "imputation_targets_df",
    "imputation_targets",
    "X_raw_counts",
    "X_denoised",
    "was_corrected",
    "denoised_adata",
    "shared_genes",
    "cell_ids_step4",
    "cell_edge_lookup",
    "nuc_centroids",
    "baseline_gene_pools",
    "baseline_ct_gene_pools",
    "baseline_spatial_index",
    "model",
    "device",
    "count_metrics_df",
    "clustering_metrics_df",
]

runtime_snapshot = [describe_var(v) for v in important_runtime_vars]
runtime_snapshot_df = pd.DataFrame(runtime_snapshot)

print("\nRuntime variable snapshot:")
display(runtime_snapshot_df)


# ------------------------------------------------------------------------------
# 7. Build final manifest JSON
# ------------------------------------------------------------------------------

print("\nWriting final manifest JSON/report...")

safe_required_files = bool(len(required_missing) == 0)
safe_count_reconciliation = bool(check_results["count_reconciliation_ok"] is True)
safe_row_counts = bool(row_count_df["ok"].fillna(False).all())
safe_downstream = bool(check_results["downstream_metrics_ok"] is True)
safe_final_recovery = bool(check_results["final_large_recovery_ok"] is True)

safe_to_disconnect = bool(
    safe_required_files
    and safe_count_reconciliation
    and safe_row_counts
    and safe_downstream
    and safe_final_recovery
)

manifest = {
    "platform": "CosMx",
    "run_name": RUN_NAME,
    "timestamp": timestamp,
    "checkpoint_dir": CHECKPOINT_DIR,
    "step4_export_dir": STEP4_EXPORT_DIR,
    "python_version": platform.python_version(),
    "platform_info": platform.platform(),
    "safe_to_disconnect": safe_to_disconnect,
    "safety_checks": {
        "required_files_present": safe_required_files,
        "count_reconciliation_ok": safe_count_reconciliation,
        "parquet_row_counts_ok": safe_row_counts,
        "downstream_metrics_present": safe_downstream,
        "final_large_recovery_present": safe_final_recovery,
    },
    "check_results": check_results,
    "required_missing_files": required_missing.to_dict(orient="records"),
    "file_manifest_csv": MANIFEST_CSV_PATH,
    "safe_report_path": SAFE_REPORT_PATH,
    "latest_safe_report_path": LATEST_SAFE_REPORT_PATH,
    "row_count_checks": row_count_df.to_dict(orient="records"),
    "runtime_snapshot": runtime_snapshot,
}

with open(MANIFEST_JSON_PATH, "w") as f:
    json.dump(manifest, f, indent=2)


# ------------------------------------------------------------------------------
# 8. Write human-readable safety report
# ------------------------------------------------------------------------------

report_lines = []

report_lines.append("=" * 100)
report_lines.append("COSMX FINAL SAFETY CHECKPOINT REPORT — STEP 5 9E")
report_lines.append("=" * 100)
report_lines.append(f"Run name: {RUN_NAME}")
report_lines.append(f"Timestamp: {timestamp}")
report_lines.append(f"Checkpoint dir: {CHECKPOINT_DIR}")
report_lines.append(f"Step4 export dir: {STEP4_EXPORT_DIR}")
report_lines.append("")
report_lines.append("SAFETY CHECKS")
report_lines.append("-" * 100)
report_lines.append(f"Required files present          : {safe_required_files}")
report_lines.append(f"Count reconciliation OK         : {safe_count_reconciliation}")
report_lines.append(f"Parquet row counts OK           : {safe_row_counts}")
report_lines.append(f"Downstream metrics present      : {safe_downstream}")
report_lines.append(f"Final large recovery present    : {safe_final_recovery}")
report_lines.append(f"SAFE TO DISCONNECT              : {'YES' if safe_to_disconnect else 'NO'}")
report_lines.append("")

report_lines.append("KEY EXPECTED OUTPUTS")
report_lines.append("-" * 100)
for fname in [
    f"{RUN_NAME}_best_coordNN_model.pt",
    f"{RUN_NAME}_final_large_recovery_metrics.csv",
    f"{RUN_NAME}_imputed_records.parquet",
    f"{RUN_NAME}_completed_molecule_table.parquet",
    f"{RUN_NAME}_gene_emp_imputed_records.parquet",
    f"{RUN_NAME}_ct_gene_emp_imputed_records.parquet",
    f"{RUN_NAME}_spatial_knn_emp_imputed_records.parquet",
    f"{RUN_NAME}_downstream_clustering_metrics_step4matched.csv",
    f"{RUN_NAME}_downstream_count_matrix_metrics_step4matched.csv",
]:
    p = os.path.join(CHECKPOINT_DIR, fname)
    if os.path.exists(p):
        report_lines.append(f"✓ {fname} ({file_size_gb(p):.3f} GB)")
    else:
        report_lines.append(f"✗ {fname}")

report_lines.append("")
report_lines.append("ROW COUNT CHECKS")
report_lines.append("-" * 100)
report_lines.append(row_count_df.to_string(index=False))

if len(required_missing) > 0:
    report_lines.append("")
    report_lines.append("MISSING REQUIRED FILES")
    report_lines.append("-" * 100)
    report_lines.append(required_missing.to_string(index=False))

report_lines.append("")
report_lines.append("MANIFEST FILES")
report_lines.append("-" * 100)
report_lines.append(f"Manifest JSON: {MANIFEST_JSON_PATH}")
report_lines.append(f"File manifest CSV: {MANIFEST_CSV_PATH}")
report_lines.append(f"Safety report: {SAFE_REPORT_PATH}")
report_lines.append("")

report_text = "\n".join(report_lines)

with open(SAFE_REPORT_PATH, "w") as f:
    f.write(report_text)

with open(LATEST_SAFE_REPORT_PATH, "w") as f:
    f.write(report_text)

print(report_text)

print("\nSaved final manifest files:")
print(f"  JSON manifest      : {MANIFEST_JSON_PATH}")
print(f"  File manifest CSV  : {MANIFEST_CSV_PATH}")
print(f"  Safety report      : {SAFE_REPORT_PATH}")
print(f"  Latest report copy : {LATEST_SAFE_REPORT_PATH}")


# ------------------------------------------------------------------------------
# 9. Final verdict
# ------------------------------------------------------------------------------

print("\n" + "=" * 100)

if safe_to_disconnect:
    print("SAFE TO DISCONNECT: YES")
    print("All critical CosMx Step 5 files, row counts, count reconciliation, final recovery metrics, and downstream outputs are saved.")
    print("You can disconnect/delete the runtime now.")
else:
    print("SAFE TO DISCONNECT: NO")
    print("One or more required checks failed. Review the warnings above before disconnecting.")

print("=" * 100)

gc.collect()

============VALIDATION DONE=============

In [ ]:
# ==============================================================================
# LIST ALL FILES IN COSMX IMPUTATION FOLDER WITH FULL NAMES, SIZE, AND TIMESTAMP
# ==============================================================================

import os
from pathlib import Path
from datetime import datetime
import pandas as pd

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

IMPUTATION_DIR = Path("/content/drive/MyDrive/diffusion/step4_cosmx/imputation")

if not IMPUTATION_DIR.exists():
    raise FileNotFoundError(f"Folder does not exist: {IMPUTATION_DIR}")

records = []

for path in sorted(IMPUTATION_DIR.iterdir(), key=lambda p: p.name.lower()):
    if path.is_file():
        stat = path.stat()

        records.append({
            "filename": path.name,
            "full_path": str(path),
            "extension": path.suffix,
            "size_bytes": stat.st_size,
            "size_MB": stat.st_size / (1024 ** 2),
            "size_GB": stat.st_size / (1024 ** 3),
            "modified_time_local": datetime.fromtimestamp(stat.st_mtime).strftime("%Y-%m-%d %H:%M:%S"),
        })

files_df = pd.DataFrame(records)

if len(files_df) == 0:
    print(f"No files found in: {IMPUTATION_DIR}")
else:
    files_df = files_df.sort_values(
        by=["modified_time_local", "filename"],
        ascending=[False, True]
    ).reset_index(drop=True)

    pd.set_option("display.max_rows", None)
    pd.set_option("display.max_colwidth", None)
    pd.set_option("display.width", 240)

    print("=" * 120)
    print(f"FILES IN: {IMPUTATION_DIR}")
    print("=" * 120)
    print(f"Total files: {len(files_df)}")
    print()

    display(files_df)

    # Save manifest into the same folder.
    manifest_path = IMPUTATION_DIR / "cosmx_imputation_folder_file_manifest.csv"
    files_df.to_csv(manifest_path, index=False)

    print("\nSaved file manifest CSV:")
    print(manifest_path)